# LB 0.477 · Two 0.472 Fixes, Merged the Right Way

Two public changes to the same 0.468 leader-safe recipe each reach a displayed 0.472: Roman Tamrazov's fragment check, and alex chilton's heavier neural weight on ranks 2-25 (shown by its fork; his own page shows 0.467). Stacked the plain way they still score 0.472; our explanation is that the fragment check's leader rule picks its contenders from the order it is handed. Run on the order it was built on, with ranks 2-25 from the 0.75 order, the two add up.

| | |
|---|---|
| result | **public LB 0.477 (submission 57722897, read 2026-10-29 06:05 UTC)** |
| beats | every public notebook on the Code tab at 2026-10-28 07:48 UTC: [Heliosli](https://www.kaggle.com/code/heliosli/happy-to-make-the-dataset-public-if-it-helps) at 0.475 (0.002 below ours, about half of one rank-2-to-1 move; it needs a private dataset), then ten notebooks at 0.472, among them [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) and the w_mid 0.70-0.80 forks |
| runtime | commit run 21-30 min in our runs (12 molecules; m1's took 21); a scoring run takes hours on Kaggle T4, internet off |


**The task in one paragraph.** A tandem mass spectrum (MS/MS) records the pieces a molecule breaks into in a collision
cell: a list of fragment masses with their intensities. For each hidden molecule a notebook hands in up to 25
structures (SMILES), best guess first. The score is MRR@25: 1 for a right first guess, 1/2 for a right second guess,
down to 1/25, and 0 when the right structure is not in the list, averaged over molecules. An answer counts as right
when its atom connectivity matches (the first 14 characters of the InChIKey after RDKit's tautomer canonicalisation),
so a wrong stereo centre or tautomer is not penalised.

**What this notebook is.** Roman Tamrazov's [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) (version 22) with his provenance reranker switched off and one
change in how its fragment check is used. The check's rank-1 rule runs on the 0.468 order it was built on (V2,
ranks 2-25 at neural weight 0.50), and only its leader moves are kept; ranks 2-25 come from alex chilton's order at
0.75. This is run m1, the best public score of our 13 (section 5), tied with m2, m4, m6. The code that builds the answer lists is his and the code he pins, plus one merge cell of
ours; four of his cells carry our string patches, listed in section 6. What is ours: the explanation that the two
public 0.472 changes collide in the fragment check's contender window (m1 tests it together with dropping the check's
swaps and last-place change, not alone), the merge that keeps them apart, the
13 hidden-test submissions that measured it, the field record and the per-layer audit of this run.

**Reading map.** 1 The field · 2 One lineage, two branches · 3 The pipeline · 4 What this run did · 5 Every
submission of this pipeline · 6 Run it yourself · 7 What we concluded · 8 Credits

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
VARIANT = 'CUSTOM'
#   V0  baseline 0.465 answers, no neural term
#   V1  public 0.466 recipe: neural reciprocal-rank fusion, weight 0.25 everywhere
#   V2  leader-safe consensus: rank 1 as V1, ranks 2-25 at neural weight 0.50   (upstream default)
#   V3  V2 + tail rescue: outside candidates may take the last 3 places by neural score
#   V4  bold consensus: rank 1 as V1, ranks 2-25 at neural weight 1.00
#   'CUSTOM' uses the weights below (w_top=0 keeps the pipeline's own rank 1)
CUSTOM = dict(w_top=0.25, w_mid=0.75, rescue=0)

RRF_OFFSET = 3.0              # k in 1/(k + rank), as in the public recipe
COMMIT_MODE = 'smoke'         # 'smoke': the commit run ranks 12 visible molecules and draws every figure
                              # 'fast' : the commit run skips the pipeline (saves ~30 GPU minutes);
                              #          the scoring rerun on the hidden test always runs in full
OUTSIDERS_PER_MOLECULE = 20   # candidate pool for the tail rescue (V3, or CUSTOM with rescue > 0)
TIME_GUARD_HOURS = 7.5        # optional stages are skipped once the session has run this long
QUIET_LOGS = True             # one line per pipeline stage here; full logs stay in /kaggle/working/*.log
ENABLE_TAIL_SLOT = True
PHYSICAL_POST_SECONDS = 1500

# This notebook: CUSTOM, rank 1 at neural weight 0.25, ranks 2-25 at 0.75. PHYSICAL switches the fragment check,
# PROVENANCE the provenance reranker; neural model: Saner Turhaner v1.
PHYSICAL = True
PROVENANCE = False
# variant M: the fragment check reads variants/submission_V2.csv (cell 5 as upstream, the input of Spectral Parallax V17);
# the merge cell after the provenance cell keeps only its rank-1 moves, on the CUSTOM order
FRAG_ON_V2 = True   # False leaves submission.csv as the fragment check wrote it (V2 order)

# Fragment scorer (frag_rescore.py, casmi26 fork): credit for 2 cleavages inside one ring (a ring-opening event such
# as retro-Diels-Alder). None = upstream scorer exactly. 0.8 on our same-formula isomer bench: +0.016 MRR on
# enveda-np-examples, +0.006 on natural-product panels, ~0 on held-out libraries, -0.008 on enveda-180.
# Reaches both users (the 0.465 pipeline's fragment gap and the fragment check) through the environment.
FRAG_W_RING = None   # 0.8 scored 0.473 public (v1) vs 0.477 without it; off since v3

# Final layer (cell 15, from baidalinadilzhan's public LB 0.479 notebook): documentation-prior weights on ranks 2-25
# (PubChem popularity / spectral-library record / natural product), the library bonus only for molecules whose best
# library match reaches LIB_HIT_GATE, the pipeline's leader pinned at rank 2 after a fragment-check leader move,
# metric-exact deduplication and empty slots filled with scored outside candidates.
POP_W, SPEC_W, NP_W = 0.10, 0.05, 0.02
LIB_HIT_GATE = 0.87
PIN_OLD_LEADER = True
OUTSIDER_FILL = True

# Class-3 analog insertion (cell after the final layer; class3/ in our repo): for molecules whose best library match is
# below CLASS3_CFG['gate'], generate analogs of the spectral neighbours (41 mass-delta biotransformation rules), rank
# them with a LambdaRank model trained on held-out library molecules and insert the best n_insert at rank `position`.
CLASS3 = True
# rerank: reorder the same gated lists with the isomer reranker (rerank/ in our repo) before the insertion;
# 'freeze1' keeps the pipeline's leader, 'all' may change it. Placement probes (v5): rank 2 or rank 1 inserts cost 0.017.
CLASS3_CFG = dict(gate=0.87, position=3, n_insert=3, seconds=2400, rerank='freeze1')
import os
os.environ['CASMI_FRAG_W_RING'] = '' if FRAG_W_RING is None else str(FRAG_W_RING)


In [ ]:
# ── Session: paths, run mode, design tokens and small display helpers ─────────────────────────
import os, sys, re, io, json, time, math, base64, shutil, hashlib, subprocess, collections, warnings
from pathlib import Path

NB_T0 = time.time()
warnings.filterwarnings('ignore')
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
CODE = ROOT / 'p2s_code'; CODE.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

# Design tokens: chart ink and a fixed categorical order (identity never follows rank).
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE, CARD = '#e1e0d9', '#c3c2b7', '#fcfcfb', '#f7f8fb'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
QUIET_GRAY = '#c9c8c1'
ROLE = {'main': BLUE, 'engine2': ORANGE, 'pubchem': AQUA, 'new': VIOLET}
FONT = "system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10.5, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': INK, 'axes.titlepad': 10, 'axes.labelcolor': INK2, 'axes.edgecolor': AXIS,
    'axes.linewidth': 0.8, 'xtick.color': AXIS, 'ytick.color': AXIS, 'xtick.labelcolor': INK2,
    'ytick.labelcolor': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'grid.linestyle': '-', 'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160,
    'savefig.bbox': 'tight'})
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass


def tiles(items):
    """A row of stat tiles: items = [(label, value, note), ...]."""
    cells = ''.join(
        f'<div style="display:inline-block;vertical-align:top;min-width:150px;margin:0 10px 10px 0;'
        f'padding:12px 16px;border-radius:12px;background:#ffffff;border:1px solid rgba(11,11,11,0.10);">'
        f'<div style="font-size:12px;color:{INK2};">{label}</div>'
        f'<div style="font-size:25px;font-weight:650;color:{INK};line-height:1.3;">{value}</div>'
        f'<div style="font-size:11.5px;color:{MUTED};">{note}</div></div>' for label, value, note in items)
    display(HTML(f'<div style="font-family:{FONT};margin:4px 0 2px;">{cells}</div>'))


def note(title, text, color=VIOLET):
    display(HTML(f'<div style="font-family:{FONT};border-left:4px solid {color};background:{CARD};'
                 f'padding:10px 14px;border-radius:8px;margin:8px 0;font-size:13.5px;line-height:1.5;">'
                 f'<b style="color:{INK};">{title}</b> <span style="color:{INK2};">{text}</span></div>'))


def table_html(frame, highlight=None, numeric=()):
    """Plain, readable HTML table; numbers right-aligned; `highlight` = index label of a row to emphasise."""
    align = {c: 'right' if c in set(numeric) else 'left' for c in frame.columns}
    head = ''.join(f'<th style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {AXIS};'
                   f'color:{INK2};font-weight:600;background:transparent;">{c}</th>' for c in frame.columns)
    body = []
    for idx, row in frame.iterrows():
        on = highlight is not None and idx == highlight
        style = 'background:#efedfb;font-weight:600;' if on else 'background:transparent;'
        body.append(f'<tr style="{style}">' + ''.join(
            f'<td style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {GRID};color:{INK};'
            f'font-variant-numeric:tabular-nums;">{v}</td>' for c, v in zip(frame.columns, row.values)) + '</tr>')
    display(HTML(f'<table style="font-family:{FONT};font-size:13px;border-collapse:collapse;margin:6px 0 10px;">'
                 f'<thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'))


def minutes(seconds):
    return f'{seconds / 60:,.1f} min' if seconds >= 60 else f'{seconds:,.0f} s'


def gpu_name():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return ' + '.join(x.strip() for x in out.splitlines() if x.strip()) or 'none'
    except Exception:
        return 'none'


# Run mode: the commit run sees the visible format example; the scoring rerun sees the hidden test.
COMP = next((p for p in (INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-casmi26-molecule-id-mass-spectra')
             if all((p / f).is_file() for f in ('train.parquet', 'test.parquet', 'sample_submission.csv'))), None)
assert COMP is not None, 'Attach the competition data (enveda-CASMI26-molecule-id-mass-spectra)'
_ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
del _ids
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
SHOW = not IS_RERUN                      # figures are drawn for readers; the scoring rerun skips them
SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
assert VARIANT in ('V0', 'V1', 'V2', 'V3', 'V4', 'CUSTOM'), VARIANT

GPU = gpu_name()
MODE = 'Scoring rerun' if IS_RERUN else ('Commit · smoke run' if RUN_PIPELINE else 'Commit · fast path')
tiles([('Run mode', MODE, 'hidden test' if IS_RERUN else 'visible format example'),
       ('Variant', VARIANT, 'set in the configuration cell'),
       ('Accelerator', GPU.split(' + ')[0] if GPU != 'none' else 'none', GPU if ' + ' in GPU else 'GPU required'),
       ('Python', '.'.join(map(str, sys.version_info[:3])), 'pipeline expects 3.12'),
       ('Answer rows', f'{len(SAMPLE_IDS):,}', 'molecules in sample_submission')])

## 1. The field

On 2026-10-29 06:00 UTC the Code tab listed 1009 public notebooks for this competition, 697 of them
with a public score. Of those, 133 score 0.460 or more, and 122 of these 133 attach the
same dataset, Dmitrii Gluzdov's v4g reproduction assets (Figure 1). Most of the top of the Code tab is one pipeline
with different constants.

The leaderboard on 2026-10-29 06:04 UTC had 4,295 teams. Of these, 231 sat at exactly 0.477 (the
score of this notebook, which 11 public notebooks now display), 175 at 0.472, 107 at
0.468 and 158 at 0.464. The board does not record what each team submitted, so this matches scores and does not prove
copying. The gold line was rank 18 at 0.481, and first place was 0.516. No member of the top
ten teams is the author of any of the 1009 public notebooks.

In [ ]:
# House style for our own figures (engine/33_style.py at repo commit 30b76796, without its ink and grid
# colours, so the upstream tiles and tables keep theirs). A later cell imports another `apply`; ours stays as style_apply.
ACCENT = "#d9480f"      # the one colour that means "this is the point"
SECOND = "#1c7ed6"      # used only when two things must be told apart
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 10,
    })


def label_point(ax, x, y, text: str, color: str = ACCENT) -> None:
    """Print the number that matters on the chart, next to the point it describes."""
    ax.annotate(text, (x, y), xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=11, fontweight="semibold")


def first_screen(claim: str, number: str, beats: str | None = None, runtime: str | None = None,
                 accel: str | None = None) -> str:
    """The markdown every busyaprime notebook opens with. Returns it so a builder can write it into cell 0.

    claim   one sentence: what this notebook shows that the reader did not know
    number  the result, with its metric and where it was measured (public LB, CV on stated folds)
    beats   the public work it beats, with that work's number and a link
    runtime wall clock on the stated accelerator
    """
    rows = [f"| result | **{number}** |"]
    if beats:
        rows.append(f"| beats | {beats} |")
    if runtime:
        rows.append(f"| runtime | {runtime}{' on ' + accel if accel else ''} |")
    return f"{claim}\n\n| | |\n|---|---|\n" + "\n".join(rows) + "\n"
style_apply = apply

In [ ]:
# The field as the Code tab showed it on 2026-10-28 07:48 UTC: every public notebook with a displayed score.
FIELD = {"taken_utc":"2026-10-28 07:48","notebooks_listed":974,"scored":[[0.475,44,"E",1,1,"2026-10-22"],[0.472,92,"M",1,1,"2026-10-23"],[0.472,66,"E",1,1,"2026-10-25"],[0.472,8,"C",1,1,"2026-10-27"],[0.472,1,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.472,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-27"],[0.471,0,"C",1,1,"2026-10-28"],[0.471,0,"C",1,1,"2026-10-28"],[0.47,0,"C",1,1,"2026-10-26"],[0.47,4,"C",1,1,"2026-10-26"],[0.468,61,"E",1,0,"2026-09-28"],[0.468,98,"M",1,1,"2026-10-13"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-26"],[0.468,0,"E",1,1,"2026-10-26"],[0.468,0,"C",1,1,"2026-10-27"],[0.467,59,"E",1,1,"2026-10-25"],[0.467,1,"C",1,1,"2026-10-25"],[0.467,0,"E",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-25"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,1,"C",0,0,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,0,"C",1,1,"2026-10-26"],[0.467,3,"C",1,1,"2026-10-26"],[0.466,2,"E",1,1,"2026-09-17"],[0.466,80,"E",1,1,"2026-10-21"],[0.466,21,"C",1,1,"2026-10-24"],[0.466,6,"E",1,0,"2026-10-24"],[0.466,1,"E",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"M",1,1,"2026-10-25"],[0.466,1,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-25"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,0,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-26"],[0.466,0,"C",1,1,"2026-10-27"],[0.465,182,"M",1,0,"2026-10-16"],[0.465,22,"C",1,0,"2026-10-24"],[0.465,0,"C",1,0,"2026-10-24"],[0.465,3,"C",1,0,"2026-10-24"],[0.465,1,"C",1,0,"2026-10-24"],[0.465,0,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,1,"C",1,1,"2026-10-25"],[0.465,0,"M",1,1,"2026-10-26"],[0.465,0,"C",1,0,"2026-10-26"],[0.465,0,"C",1,1,"2026-10-26"],[0.465,0,"E",0,1,"2026-10-28"],[0.464,201,"M",1,0,"2026-09-15"],[0.464,3,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,1,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,0,"C",1,0,"2026-10-22"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,3,"C",1,0,"2026-10-23"],[0.464,5,"C",1,0,"2026-10-23"],[0.464,10,"M",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,2,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-23"],[0.464,0,"C",1,0,"2026-10-24"],[0.464,1,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-25"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,0,"C",1,0,"2026-10-26"],[0.464,1,"C",1,1,"2026-10-26"],[0.464,1,"E",0,1,"2026-10-28"],[0.463,5,"M",0,0,"2026-10-21"],[0.463,7,"M",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-22"],[0.463,0,"C",0,0,"2026-10-23"],[0.463,0,"C",1,0,"2026-10-24"],[0.463,0,"C",1,0,"2026-10-26"],[0.463,0,"C",1,1,"2026-10-26"],[0.463,0,"E",0,0,"2026-10-27"],[0.462,1,"C",1,0,"2026-10-26"],[0.462,0,"C",1,0,"2026-10-27"],[0.461,0,"C",1,0,"2026-10-26"],[0.461,0,"C",1,0,"2026-10-27"],[0.46,1,"C",0,0,"2026-10-19"],[0.46,0,"C",1,0,"2026-10-24"],[0.46,0,"C",1,0,"2026-10-27"],[0.459,1,"C",0,0,"2026-10-21"],[0.459,0,"C",1,0,"2026-10-22"],[0.459,0,"C",1,0,"2026-10-26"],[0.459,0,"C",1,1,"2026-10-28"],[0.458,0,"C",1,0,"2026-10-23"],[0.457,0,"C",0,0,"2026-10-23"],[0.457,0,"C",1,0,"2026-10-24"],[0.457,0,"C",1,0,"2026-10-25"],[0.457,0,"C",1,0,"2026-10-26"],[0.456,155,"M",0,0,"2026-09-15"],[0.456,9,"E",0,0,"2026-10-17"],[0.456,77,"C",0,0,"2026-10-18"],[0.456,1,"C",0,0,"2026-10-19"],[0.456,2,"C",0,0,"2026-10-19"],[0.456,1,"E",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-20"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"E",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-21"],[0.456,0,"C",1,0,"2026-10-21"],[0.456,0,"C",0,0,"2026-10-22"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,1,"C",1,0,"2026-10-23"],[0.456,0,"C",1,0,"2026-10-23"],[0.456,0,"C",1,1,"2026-10-27"],[0.456,0,"E",0,0,"2026-10-27"],[0.455,166,"M",1,0,"2026-09-15"],[0.455,2,"E",0,0,"2026-10-18"],[0.455,40,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,8,"M",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,0,"C",0,0,"2026-10-18"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,1,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",0,0,"2026-10-19"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,0,"2026-10-23"],[0.455,0,"C",1,1,"2026-10-27"],[0.455,0,"E",0,0,"2026-10-27"],[0.454,37,"C",0,0,"2026-10-17"],[0.454,15,"C",0,0,"2026-10-17"],[0.454,2,"E",0,0,"2026-10-18"],[0.454,1,"C",0,0,"2026-10-18"],[0.454,0,"M",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-19"],[0.454,0,"C",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"E",0,0,"2026-10-20"],[0.454,0,"M",0,0,"2026-10-22"],[0.454,1,"C",1,0,"2026-10-24"],[0.452,36,"E",0,0,"2026-10-14"],[0.452,0,"E",0,0,"2026-10-27"],[0.451,8,"E",0,0,"2026-10-02"],[0.451,1,"C",0,0,"2026-10-12"],[0.451,73,"E",0,0,"2026-10-15"],[0.451,48,"E",0,0,"2026-10-15"],[0.451,14,"E",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,2,"C",0,0,"2026-10-16"],[0.451,3,"C",0,0,"2026-10-17"],[0.451,0,"E",0,0,"2026-10-17"],[0.451,4,"M",0,0,"2026-10-17"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-18"],[0.451,0,"C",0,0,"2026-10-19"],[0.45,0,"C",0,0,"2026-10-22"],[0.448,2,"E",1,0,"2026-10-17"],[0.448,2,"E",0,0,"2026-10-18"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,2,"M",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-19"],[0.448,0,"C",0,0,"2026-10-20"],[0.448,0,"E",0,0,"2026-10-25"],[0.448,1,"E",0,0,"2026-10-25"],[0.448,0,"C",1,1,"2026-10-27"],[0.447,0,"C",0,0,"2026-10-17"],[0.447,0,"C",0,0,"2026-10-18"],[0.447,1,"M",0,0,"2026-10-20"],[0.447,0,"E",0,0,"2026-10-24"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.447,0,"C",1,1,"2026-10-25"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,0,"C",0,0,"2026-10-14"],[0.446,63,"E",0,0,"2026-10-14"],[0.446,7,"M",0,0,"2026-10-15"],[0.446,2,"E",0,0,"2026-10-15"],[0.446,1,"C",0,0,"2026-10-15"],[0.446,1,"M",0,0,"2026-10-17"],[0.446,0,"E",0,0,"2026-10-20"],[0.446,0,"C",0,0,"2026-10-22"],[0.445,0,"C",0,0,"2026-10-15"],[0.445,0,"C",0,0,"2026-10-17"],[0.445,0,"C",0,0,"2026-10-20"],[0.445,0,"C",0,0,"2026-10-20"],[0.444,2,"C",0,0,"2026-10-13"],[0.444,0,"C",0,0,"2026-10-15"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-19"],[0.444,0,"C",0,0,"2026-10-20"],[0.444,0,"C",0,0,"2026-10-20"],[0.443,18,"C",0,0,"2026-10-13"],[0.443,0,"C",0,0,"2026-10-15"],[0.443,3,"C",0,0,"2026-10-15"],[0.443,0,"C",0,0,"2026-10-20"],[0.442,25,"C",0,0,"2026-10-13"],[0.442,1,"C",0,0,"2026-10-17"],[0.442,0,"C",0,0,"2026-10-22"],[0.441,62,"E",0,0,"2026-10-10"],[0.441,69,"M",0,0,"2026-10-12"],[0.441,12,"E",0,0,"2026-10-13"],[0.441,2,"C",0,0,"2026-10-13"],[0.441,3,"C",0,0,"2026-10-13"],[0.441,0,"E",0,0,"2026-10-14"],[0.441,0,"C",0,0,"2026-10-14"],[0.441,1,"C",0,0,"2026-10-15"],[0.44,139,"M",0,0,"2026-09-17"],[0.44,3,"E",0,0,"2026-09-28"],[0.44,125,"M",0,0,"2026-10-10"],[0.44,3,"C",0,0,"2026-10-10"],[0.44,19,"E",0,0,"2026-10-11"],[0.44,26,"E",0,0,"2026-10-11"],[0.44,0,"E",0,0,"2026-10-11"],[0.44,2,"C",0,0,"2026-10-12"],[0.44,1,"C",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-12"],[0.44,1,"M",0,0,"2026-10-12"],[0.44,1,"E",0,0,"2026-10-12"],[0.44,0,"C",0,0,"2026-10-13"],[0.44,1,"E",0,0,"2026-10-21"],[0.439,1,"M",0,0,"2026-10-17"],[0.438,2,"C",0,0,"2026-10-09"],[0.438,59,"C",0,0,"2026-10-09"],[0.438,0,"C",0,0,"2026-10-11"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,0,"C",0,0,"2026-10-15"],[0.438,2,"C",0,0,"2026-10-16"],[0.438,0,"C",0,0,"2026-10-18"],[0.438,0,"E",0,0,"2026-10-21"],[0.438,0,"C",0,0,"2026-10-21"],[0.436,1,"M",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-14"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-16"],[0.436,0,"C",0,0,"2026-10-17"],[0.436,1,"C",0,0,"2026-10-17"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"C",0,0,"2026-10-18"],[0.436,0,"E",0,0,"2026-10-25"],[0.435,25,"E",0,0,"2026-10-09"],[0.435,0,"C",0,0,"2026-10-15"],[0.435,0,"C",0,0,"2026-10-16"],[0.435,0,"C",0,0,"2026-10-20"],[0.435,0,"C",1,0,"2026-10-26"],[0.434,0,"C",0,0,"2026-10-12"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-16"],[0.434,0,"C",0,0,"2026-10-22"],[0.433,42,"E",0,0,"2026-09-17"],[0.433,29,"E",0,0,"2026-10-07"],[0.433,5,"C",0,0,"2026-10-07"],[0.433,48,"C",0,0,"2026-10-08"],[0.433,8,"C",0,0,"2026-10-09"],[0.433,0,"C",0,0,"2026-10-12"],[0.433,0,"M",1,0,"2026-10-13"],[0.433,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-14"],[0.431,0,"C",0,0,"2026-10-14"],[0.431,2,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"M",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-20"],[0.431,0,"C",0,0,"2026-10-21"],[0.43,0,"C",0,0,"2026-10-12"],[0.43,1,"C",0,0,"2026-10-12"],[0.429,0,"C",0,0,"2026-10-14"],[0.429,0,"E",0,0,"2026-10-25"],[0.428,86,"C",0,0,"2026-10-07"],[0.428,0,"C",0,0,"2026-10-11"],[0.426,2,"C",0,0,"2026-10-13"],[0.425,107,"C",0,0,"2026-10-05"],[0.425,7,"C",0,0,"2026-10-06"],[0.425,23,"E",0,0,"2026-10-06"],[0.425,9,"C",0,0,"2026-10-07"],[0.425,0,"C",0,0,"2026-10-20"],[0.425,0,"C",0,0,"2026-10-21"],[0.424,3,"C",0,0,"2026-10-07"],[0.424,2,"C",0,0,"2026-10-07"],[0.423,1,"E",0,0,"2026-10-07"],[0.423,3,"M",0,0,"2026-10-08"],[0.423,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"C",0,0,"2026-10-09"],[0.422,0,"E",0,0,"2026-10-24"],[0.421,56,"C",0,0,"2026-10-02"],[0.421,2,"C",0,0,"2026-10-06"],[0.421,0,"C",0,0,"2026-10-08"],[0.421,0,"C",0,0,"2026-10-08"],[0.42,16,"E",0,0,"2026-09-23"],[0.42,0,"C",0,0,"2026-10-07"],[0.42,0,"C",0,0,"2026-10-18"],[0.419,0,"C",0,0,"2026-10-06"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"E",0,0,"2026-10-08"],[0.419,0,"M",0,0,"2026-10-20"],[0.419,0,"C",0,0,"2026-10-21"],[0.419,0,"E",0,0,"2026-10-27"],[0.418,0,"C",0,0,"2026-10-07"],[0.418,0,"C",0,0,"2026-10-08"],[0.418,0,"C",0,0,"2026-10-09"],[0.418,3,"C",0,0,"2026-10-09"],[0.418,0,"E",0,0,"2026-10-09"],[0.418,0,"C",0,0,"2026-10-10"],[0.417,79,"C",0,0,"2026-10-03"],[0.417,6,"E",0,0,"2026-10-03"],[0.417,1,"E",0,0,"2026-10-05"],[0.417,0,"C",0,0,"2026-10-06"],[0.417,0,"E",0,0,"2026-10-06"],[0.417,0,"C",0,0,"2026-10-08"],[0.416,0,"E",0,0,"2026-10-06"],[0.415,93,"C",0,0,"2026-10-01"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-07"],[0.415,0,"C",0,0,"2026-10-08"],[0.415,0,"C",0,0,"2026-10-09"],[0.415,0,"C",1,0,"2026-10-26"],[0.414,0,"C",0,0,"2026-10-07"],[0.413,5,"C",0,0,"2026-10-01"],[0.413,33,"C",0,0,"2026-10-03"],[0.413,2,"C",0,0,"2026-10-04"],[0.413,9,"C",0,0,"2026-10-04"],[0.413,0,"C",0,0,"2026-10-04"],[0.413,8,"C",0,0,"2026-10-05"],[0.413,0,"C",0,0,"2026-10-08"],[0.411,2,"C",0,0,"2026-10-01"],[0.411,2,"C",0,0,"2026-10-03"],[0.411,0,"C",0,0,"2026-10-05"],[0.411,0,"C",0,0,"2026-10-08"],[0.409,1,"C",0,0,"2026-10-01"],[0.409,27,"C",0,0,"2026-10-01"],[0.409,4,"C",0,0,"2026-10-02"],[0.409,2,"C",0,0,"2026-10-02"],[0.409,1,"C",0,0,"2026-10-03"],[0.409,0,"C",0,0,"2026-10-05"],[0.409,0,"E",0,0,"2026-10-06"],[0.409,0,"C",0,0,"2026-10-08"],[0.407,1,"C",0,0,"2026-10-02"],[0.407,0,"C",0,0,"2026-10-04"],[0.407,0,"C",0,0,"2026-10-05"],[0.406,1,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-04"],[0.406,0,"C",0,0,"2026-10-06"],[0.405,30,"E",0,0,"2026-09-28"],[0.405,0,"C",0,0,"2026-10-04"],[0.404,2,"E",0,0,"2026-10-02"],[0.404,0,"E",0,0,"2026-10-06"],[0.404,0,"C",1,1,"2026-10-24"],[0.403,1,"C",0,0,"2026-10-05"],[0.402,0,"C",0,0,"2026-10-02"],[0.402,0,"E",0,0,"2026-10-05"],[0.401,32,"C",0,0,"2026-09-30"],[0.4,1,"C",0,0,"2026-10-01"],[0.4,2,"E",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-02"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"C",0,0,"2026-10-16"],[0.4,0,"E",0,0,"2026-10-24"],[0.399,56,"E",0,0,"2026-09-30"],[0.399,5,"M",0,0,"2026-09-30"],[0.399,1,"C",0,0,"2026-10-01"],[0.399,0,"C",0,0,"2026-10-01"],[0.399,1,"C",0,0,"2026-10-02"],[0.399,0,"C",0,0,"2026-10-06"],[0.398,1,"C",0,0,"2026-10-12"],[0.397,0,"E",0,0,"2026-10-01"],[0.397,0,"C",0,0,"2026-10-01"],[0.397,1,"C",0,0,"2026-10-02"],[0.396,0,"E",0,0,"2026-10-01"],[0.396,0,"E",0,0,"2026-10-02"],[0.396,0,"C",0,0,"2026-10-02"],[0.395,0,"C",1,0,"2026-10-01"],[0.394,0,"C",0,0,"2026-10-06"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,1,"C",0,0,"2026-10-02"],[0.393,0,"E",0,0,"2026-10-03"],[0.393,0,"M",0,0,"2026-10-03"],[0.393,0,"E",0,0,"2026-10-04"],[0.393,0,"E",0,0,"2026-10-04"],[0.39,0,"C",0,0,"2026-09-30"],[0.389,0,"C",0,0,"2026-10-04"],[0.386,17,"E",0,0,"2026-09-29"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"M",0,0,"2026-10-02"],[0.386,0,"C",0,0,"2026-10-02"],[0.386,0,"C",1,0,"2026-10-09"],[0.384,23,"M",0,0,"2026-09-29"],[0.384,5,"C",0,0,"2026-09-29"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-09-30"],[0.384,1,"C",0,0,"2026-09-30"],[0.384,0,"C",0,0,"2026-10-01"],[0.384,0,"M",0,0,"2026-10-18"],[0.384,0,"E",0,0,"2026-10-25"],[0.383,0,"C",0,0,"2026-10-02"],[0.383,0,"M",0,0,"2026-10-17"],[0.382,0,"C",0,0,"2026-09-29"],[0.382,0,"C",1,0,"2026-10-02"],[0.382,0,"C",0,0,"2026-10-06"],[0.381,29,"M",0,0,"2026-09-29"],[0.381,0,"C",0,0,"2026-10-02"],[0.381,0,"M",0,0,"2026-10-19"],[0.38,0,"C",0,0,"2026-09-29"],[0.38,1,"C",0,0,"2026-10-11"],[0.379,0,"C",0,0,"2026-09-29"],[0.378,24,"M",0,0,"2026-09-29"],[0.378,1,"C",0,0,"2026-10-01"],[0.378,0,"E",0,0,"2026-10-03"],[0.378,0,"E",0,0,"2026-10-04"],[0.377,0,"C",0,0,"2026-09-30"],[0.377,0,"C",0,0,"2026-10-02"],[0.377,0,"E",0,0,"2026-10-04"],[0.375,0,"C",0,0,"2026-10-02"],[0.375,0,"C",0,0,"2026-10-03"],[0.375,1,"C",0,0,"2026-10-04"],[0.375,0,"E",0,0,"2026-10-05"],[0.374,2,"E",1,0,"2026-09-30"],[0.374,0,"C",0,0,"2026-10-03"],[0.374,0,"E",0,0,"2026-10-05"],[0.373,19,"M",0,0,"2026-09-28"],[0.372,0,"C",0,0,"2026-10-04"],[0.371,0,"C",0,0,"2026-10-06"],[0.37,23,"M",0,0,"2026-09-26"],[0.37,0,"C",0,0,"2026-09-29"],[0.37,0,"C",0,0,"2026-10-05"],[0.368,0,"C",0,0,"2026-09-29"],[0.367,0,"C",0,0,"2026-09-29"],[0.366,0,"C",0,0,"2026-09-28"],[0.365,0,"C",0,0,"2026-09-30"],[0.364,0,"C",1,0,"2026-10-03"],[0.362,20,"M",0,0,"2026-09-26"],[0.359,0,"C",1,1,"2026-10-26"],[0.359,0,"C",1,1,"2026-10-27"],[0.358,30,"M",0,0,"2026-09-21"],[0.356,0,"C",0,0,"2026-09-29"],[0.356,0,"C",0,0,"2026-09-30"],[0.354,45,"M",0,0,"2026-09-21"],[0.348,0,"C",0,0,"2026-09-28"],[0.347,42,"C",0,0,"2026-09-17"],[0.345,0,"C",0,0,"2026-09-28"],[0.344,0,"C",0,0,"2026-09-28"],[0.344,1,"C",0,0,"2026-10-24"],[0.342,16,"M",0,0,"2026-09-18"],[0.341,40,"E",0,0,"2026-09-20"],[0.341,2,"E",0,0,"2026-09-21"],[0.341,2,"C",0,0,"2026-09-22"],[0.341,3,"C",0,0,"2026-09-25"],[0.341,5,"E",0,0,"2026-09-25"],[0.341,1,"C",0,0,"2026-09-25"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,0,"C",0,0,"2026-09-27"],[0.341,7,"E",0,0,"2026-09-29"],[0.339,270,"E",0,0,"2026-09-15"],[0.339,0,"C",0,0,"2026-09-25"],[0.339,1,"C",0,0,"2026-09-26"],[0.339,0,"C",0,0,"2026-10-24"],[0.338,0,"C",0,0,"2026-09-21"],[0.338,0,"C",0,0,"2026-09-26"],[0.337,81,"E",0,0,"2026-09-16"],[0.337,33,"E",0,0,"2026-09-17"],[0.337,0,"E",0,0,"2026-09-20"],[0.337,0,"C",0,0,"2026-09-25"],[0.337,0,"C",0,0,"2026-09-25"],[0.335,0,"E",0,0,"2026-09-20"],[0.334,13,"E",0,0,"2026-09-19"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"C",0,0,"2026-09-21"],[0.334,0,"E",0,0,"2026-09-22"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-09-25"],[0.334,0,"C",0,0,"2026-10-24"],[0.333,3,"E",0,0,"2026-09-19"],[0.333,0,"E",0,0,"2026-09-20"],[0.333,0,"C",0,0,"2026-09-24"],[0.333,0,"E",0,0,"2026-09-25"],[0.333,3,"E",0,0,"2026-09-25"],[0.333,0,"C",0,0,"2026-09-27"],[0.333,0,"C",0,0,"2026-10-24"],[0.332,2,"E",0,0,"2026-09-20"],[0.332,0,"C",0,0,"2026-09-26"],[0.332,0,"C",0,0,"2026-09-26"],[0.331,3,"C",0,0,"2026-09-18"],[0.331,0,"E",0,0,"2026-09-20"],[0.331,0,"E",0,0,"2026-09-20"],[0.33,0,"C",0,0,"2026-09-20"],[0.33,1,"C",0,0,"2026-09-21"],[0.33,1,"C",0,0,"2026-09-24"],[0.329,0,"C",0,0,"2026-10-24"],[0.328,2,"E",0,0,"2026-09-17"],[0.328,1,"E",0,0,"2026-09-17"],[0.328,0,"C",0,0,"2026-09-18"],[0.328,0,"C",0,0,"2026-09-20"],[0.328,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-21"],[0.327,0,"C",0,0,"2026-09-23"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.326,0,"E",0,0,"2026-09-19"],[0.323,1,"E",0,0,"2026-09-18"],[0.323,2,"E",0,0,"2026-09-19"],[0.323,0,"C",0,0,"2026-09-21"],[0.322,1,"E",0,0,"2026-09-17"],[0.322,0,"C",0,0,"2026-09-25"],[0.32,1,"E",0,0,"2026-09-22"],[0.319,1,"C",0,0,"2026-09-18"],[0.319,1,"E",0,0,"2026-09-19"],[0.317,1,"C",0,0,"2026-09-20"],[0.315,0,"C",0,0,"2026-09-26"],[0.314,0,"C",0,0,"2026-09-25"],[0.311,0,"C",0,0,"2026-09-28"],[0.309,0,"C",0,0,"2026-09-26"],[0.308,1,"E",0,0,"2026-09-21"],[0.308,1,"E",0,0,"2026-09-21"],[0.301,2,"C",0,0,"2026-09-18"],[0.298,0,"C",0,0,"2026-09-25"],[0.297,1,"E",0,0,"2026-09-19"],[0.293,0,"C",0,0,"2026-09-17"],[0.293,1,"E",0,0,"2026-09-19"],[0.291,0,"C",0,0,"2026-10-11"],[0.29,1,"E",0,0,"2026-09-19"],[0.29,1,"E",0,0,"2026-09-19"],[0.288,0,"C",0,0,"2026-10-11"],[0.285,32,"M",0,0,"2026-09-15"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,1,"E",0,0,"2026-09-20"],[0.285,0,"C",0,0,"2026-09-25"],[0.274,0,"C",0,0,"2026-10-26"],[0.274,0,"C",0,0,"2026-10-26"],[0.273,0,"C",0,0,"2026-10-26"],[0.272,2,"C",0,0,"2026-10-14"],[0.269,2,"E",0,0,"2026-09-15"],[0.264,0,"C",0,0,"2026-09-26"],[0.261,0,"C",0,0,"2026-10-03"],[0.255,0,"C",0,0,"2026-09-21"],[0.249,12,"E",0,0,"2026-09-17"],[0.241,8,"E",0,0,"2026-09-15"],[0.241,3,"C",0,0,"2026-09-16"],[0.239,0,"C",0,0,"2026-10-26"],[0.237,0,"C",0,0,"2026-10-22"],[0.236,0,"C",0,0,"2026-09-19"],[0.236,2,"C",0,0,"2026-09-19"],[0.236,0,"C",0,0,"2026-09-21"],[0.233,3,"M",0,0,"2026-09-15"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-12"],[0.231,0,"C",0,0,"2026-10-26"],[0.228,0,"E",0,0,"2026-10-14"],[0.227,4,"C",0,0,"2026-09-16"],[0.226,1,"C",0,0,"2026-10-12"],[0.224,1,"C",0,0,"2026-09-19"],[0.221,2,"C",0,0,"2026-10-11"],[0.217,0,"C",0,0,"2026-10-25"],[0.21,0,"C",0,0,"2026-10-25"],[0.207,0,"C",0,0,"2026-10-11"],[0.204,0,"C",0,0,"2026-10-26"],[0.191,1,"E",0,0,"2026-09-15"],[0.185,0,"C",0,0,"2026-10-17"],[0.182,2,"C",0,0,"2026-09-18"],[0.18,3,"E",0,0,"2026-09-15"],[0.18,2,"C",0,0,"2026-09-21"],[0.179,0,"C",0,0,"2026-10-25"],[0.176,2,"C",0,0,"2026-10-11"],[0.174,3,"C",0,0,"2026-09-16"],[0.174,0,"E",0,0,"2026-09-17"],[0.171,0,"C",0,0,"2026-10-24"],[0.169,1,"E",0,0,"2026-09-18"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.166,1,"E",0,0,"2026-09-15"],[0.165,2,"E",0,0,"2026-09-27"],[0.163,1,"E",0,0,"2026-09-15"],[0.161,0,"C",0,0,"2026-10-25"],[0.16,8,"M",0,0,"2026-09-15"],[0.158,24,"E",0,0,"2026-09-15"],[0.157,8,"E",0,0,"2026-09-15"],[0.156,32,"E",0,0,"2026-09-15"],[0.156,0,"C",0,0,"2026-09-18"],[0.156,0,"E",0,0,"2026-09-21"],[0.156,0,"C",0,0,"2026-10-26"],[0.155,4,"M",0,0,"2026-09-15"],[0.153,2,"E",0,0,"2026-09-15"],[0.153,0,"C",0,0,"2026-09-24"],[0.153,8,"C",0,0,"2026-10-13"],[0.152,0,"C",0,0,"2026-09-17"],[0.151,2,"M",0,0,"2026-09-15"],[0.151,10,"E",0,0,"2026-09-20"],[0.151,9,"E",0,0,"2026-09-20"],[0.15,2,"C",0,0,"2026-10-11"],[0.149,11,"E",0,0,"2026-09-15"],[0.149,2,"C",0,0,"2026-09-16"],[0.149,0,"C",0,0,"2026-10-13"],[0.146,0,"E",0,0,"2026-09-15"],[0.146,1,"E",0,0,"2026-09-15"],[0.145,0,"C",0,0,"2026-09-25"],[0.145,0,"C",0,0,"2026-10-09"],[0.143,3,"E",0,0,"2026-09-15"],[0.142,6,"C",0,0,"2026-09-16"],[0.142,1,"C",0,0,"2026-09-16"],[0.14,24,"E",0,0,"2026-09-15"],[0.14,0,"C",0,0,"2026-10-15"],[0.136,0,"C",0,0,"2026-10-04"],[0.134,2,"C",0,0,"2026-09-21"],[0.134,0,"C",0,0,"2026-10-01"],[0.13,0,"C",0,0,"2026-09-22"],[0.128,0,"E",0,0,"2026-09-15"],[0.115,0,"C",0,0,"2026-10-01"],[0.11,2,"E",0,0,"2026-09-17"],[0.106,0,"C",0,0,"2026-09-20"],[0.103,0,"C",0,0,"2026-09-18"],[0.099,0,"M",0,0,"2026-10-26"],[0.098,0,"C",0,0,"2026-10-04"],[0.096,3,"E",0,0,"2026-09-15"],[0.094,0,"C",0,0,"2026-09-24"],[0.093,0,"C",0,0,"2026-10-13"],[0.09,0,"C",0,0,"2026-09-19"],[0.09,0,"C",0,0,"2026-10-01"],[0.089,26,"E",0,0,"2026-09-21"],[0.088,0,"C",0,0,"2026-09-22"],[0.086,0,"C",0,0,"2026-10-23"],[0.083,0,"E",0,0,"2026-09-17"],[0.082,0,"E",0,0,"2026-09-29"],[0.064,0,"E",0,0,"2026-09-24"],[0.05,0,"C",0,0,"2026-10-18"],[0.046,0,"C",0,0,"2026-10-24"],[0.038,9,"E",0,0,"2026-09-20"],[0.0,419,"S",0,0,"2026-09-14"],[0.0,0,"C",0,0,"2026-09-17"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"E",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-18"],[0.0,0,"C",0,0,"2026-09-23"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,0,"C",0,0,"2026-09-25"],[0.0,3,"E",0,0,"2026-09-27"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",0,0,"2026-10-22"],[0.0,0,"C",1,1,"2026-10-26"]]}
if SHOW:
    import matplotlib.pyplot as plt
    style_apply()
    sc = np.array([r[0] for r in FIELD['scored']])
    v4g = np.array([r[3] for r in FIELD['scored']], bool)
    lo = 0.40
    fig, ax = plt.subplots(figsize=(10, 4.4))
    bins = np.arange(lo - 0.0005, 0.4806, 0.001)   # one bar per printed score (three decimals)
    ax.hist([sc[(sc >= lo) & ~v4g], sc[(sc >= lo) & v4g]], bins=bins, stacked=True, color=['#ced4da', '#495057'],
            label=['other notebooks', "attach Dmitrii Gluzdov's v4g assets"])
    top = ax.get_ylim()[1]
    for x, name in [(0.465, '0.465 frozen pipeline'), (0.468, '0.468 leader-safe V2'), (0.472, '0.472 two ways'), (0.477, 'this notebook 0.477')]:
        ours = name.startswith('this')
        ax.axvline(x, color=ACCENT if ours else MUTED, lw=1.4 if ours else 1.0, ls='-' if ours else ':')
        ax.text(x - 0.0006, top * 0.98, name, rotation=90, va='top', ha='right', fontsize=8.5,
                color=ACCENT if ours else INK, bbox=dict(facecolor='white', edgecolor='none', pad=1.0, alpha=0.85))
    ax.set_xlim(lo, 0.48)
    ax.set_xlabel('public LB shown on the notebook page'); ax.set_ylabel('notebooks')
    ax.set_title(f"{int((sc >= 0.46).sum())} notebooks at 0.460 or more, {int(v4g[sc >= 0.46].sum())} of them on one set of assets")
    ax.legend(loc='upper left')
    print(f"{FIELD['notebooks_listed']} public notebooks, {len(sc)} with a score, {int((sc < lo).sum())} of those below {lo} not drawn")
    plt.tight_layout(); plt.show()

*Figure 1. Public scores as the Code tab shows them (the best over each notebook's versions), 0.40 and above. Dark:
notebooks that attach Dmitrii Gluzdov's v4g assets.*

## 2. One lineage, two branches

| step | notebook | what it changed | public LB shown |
|---|---|---|---:|
| 1 | Dmitrii Gluzdov, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) | assembles and documents the shared pipeline: library search, a PubChem channel, GLACIER and ICEBERG forward models, two-engine fusion (its own credits name canqiang's E006 and Ahmed Berat Özer's engines as the base) | 0.465 |
| 2 | Saner Turhaner, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466) | a neural spectrum-to-structure scorer, fused by reciprocal rank at weight 0.25 | 0.466 |
| 3 | Roman Rozen, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures) | leader-safe consensus V2: rank 1 as in step 2, ranks 2-25 at neural weight 0.50 (Kozykorpesh's [Balanced Acquisition Evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) ran this V2 in its versions 6-7, the pipeline Spectral Parallax thanks) | 0.468 |
| 4a | alex chilton, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75) | ranks 2-25 at neural weight 0.75 | 0.472 on [a fork that changes only the commit mode](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472); his own page shows 0.467 |
| 4b | Roman Tamrazov, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax) | an untrained fragment check on the V2 lists | 0.472 (best of its versions; version 22, whose code runs here, shows no score of its own) |
| 5 | this notebook | 4b's leader moves from the V2 order, ranks 2-25 from 4a's 0.75 order | **0.477** |

Scores are what the Code tab displayed on 2026-10-27, Kaggle's best public score over each notebook's versions. The
current versions of some linked notebooks run something else: Roman Rozen's now stacks a private isomer reranker
(skipped when its dataset is absent), the fragment check and a tail recovery on ranks 2-25 at 1.00, and since
2026-10-28 his page shows 0.472.

The fusion rule in one line: a candidate at pipeline rank b and neural rank n scores 1/(3 + b) + w/(3 + n). Rank 1
uses w = 0.25, so the neural model can take the lead from the pipeline's leader only when that leader sits at neural
rank 18 or worse and the runner-up is the neural favourite. Ranks 2-25 use w_mid; at 0.75 the neural model sorts them
much harder than at 0.50.

**What the fragment check does.** 4b has three parts:

- **Rank 1.** When the leader's match to the reference library is below 0.87 and the molecule has at least three
  usable spectra, a candidate from ranks 2-6 with the leader's formula (an isomer the precursor mass cannot tell
  apart) can take rank 1. It must explain clearly more fragment intensity than the leader and score higher with the
  neural model, and it needs support from the engines' own top-5 lists or from the GLACIER and ICEBERG forward
  models.
- **Swaps.** Up to two adjacent swaps of same-formula candidates in ranks 2-11, with no library gate.
- **Last place.** One same-formula outside candidate may take rank 25.

**Why stacking 4a and 4b scored 0.472, and what scored 0.477.** 4a never touches rank 1, so the two changes looked
independent. Run together (our e1) they scored 0.472, the same as each alone. The rank-1 rule takes its contenders
from ranks 2-6 of the order it is handed. On the 0.75 order that window holds different candidates: on Saner
Turhaner's labelled panel, the same-formula candidates in ranks 2-6 that pass the rule's neural gates change in 17 of
86 eligible lists, each change adds a wrong structure and no right one drops out (the fragment gates need the
spectra and were not replayed). Our reading, which no single submission isolates: on the 0.75 order the rule keeps
its good leader moves and adds wrong ones. The whole fragment check (rank-1 rule, swaps and last place) on the
long-trained neural model's 0.75 order cost 0.008 (e3 0.453 against e4 0.461); that pair does not separate the rank-1
rule from the other two parts. This
notebook runs the rule on the V2 order, keeps only the leader it picks there, and places the old leader second and
the 0.75 order below them: 0.477.

The field already swept the knobs around 4a (displayed scores, knobs read from each notebook's configuration cell):
w_mid 0.70, 0.75 and 0.80 all show 0.472 ([v1220](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1220-golden-ratio-070), [fork-lb472](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472),
[v1210](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1210-sweet-spot-080)); rank-1 weight 0 shows 0.472 ([v1230](https://www.kaggle.com/code/rajrajak99/casmi26-apex-v1230-pure-shield-075)), 0.50 shows 0.448 and 1.00 shows
0.359 ([fork-wt05](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt05), [fork-wt10](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt10)). On the step-3 line (w_mid 0.50), offset 1 instead
of 3 shows 0.466 ([v520](https://www.kaggle.com/code/guhongbin/casmi26-v520-p2s-wmid050off1)). Moving rank 1 with the neural model is where scores are lost.

## 3. The pipeline

The configuration and session cells at the top of this notebook and the cells below are Spectral Parallax's, in his
order. Hidden: the pinned sources (Saner Turhaner's six helper modules from the 0.466 notebooks, Roman Rozen's
`consensus.py` and `outsiders.py`, and the 0.465 pipeline embedded as base64, each checked by SHA-256), the stage
runner, and the fragment check's source. Shown: the run cell (pipeline in one process, the neural scorer on the final
lists and on up to 20 outside candidates per molecule, the leader-safe consensus), the fragment-check cell and the
provenance cell. A failing overlay restores the previous answer file; the scoring rerun refuses to submit without a
real pipeline run and a neural score.

In [ ]:
# ── Pinned sources (generated; hidden in the viewer) ─────────────────────────────────────────
# MODULES: the six helper modules of the public 0.466 notebooks, unchanged, plus this notebook's
# consensus.py and outsiders.py. FROZEN_B64: the public 0.465 pipeline notebook, byte for byte.
MODULES = {}
MODULES['runner.py'] = '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n'
MODULES['export_evidence.py'] = '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n'
MODULES['neural_scores.py'] = '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n'
MODULES['submission.py'] = 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n'
MODULES['ranking.py'] = '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n'
MODULES['bundle.py'] = "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"
MODULES['consensus.py'] = '"""Leader-safe consensus re-ranking for CASMI26 answer lists.\n\nRuns after the frozen 0.465 pipeline and after the neural scorer. It never reruns a\nmodel: it reorders the candidates of each molecule and, only when asked, lets a few\ncandidates that the pipeline left outside its list compete for the last places.\n\nOrder rule\n    rank 1   = leader of   1/(k + b) + w_top / (k + n)\n    ranks 2+ = the others, ordered by   1/(k + b) + w_mid / (k + n)\nwith b the baseline rank, n the neural rank among the same candidates and k the\noffset. With w_top == w_mid this is exactly the fixed-weight reciprocal-rank fusion\nof the public 0.466 notebook (``ranking.rrf_order``), which build_variants re-checks.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ranking import rrf_order, stable_rank\n\nPRESETS = {\n    \'V0\': dict(w_top=0.00, w_mid=0.00, rescue=0, label=\'Baseline 0.465 - no neural term\'),\n    \'V1\': dict(w_top=0.25, w_mid=0.25, rescue=0, label=\'Neural RRF 0.25 - public 0.466 recipe\'),\n    \'V2\': dict(w_top=0.25, w_mid=0.50, rescue=0, label=\'Leader-safe consensus 0.25 / 0.50\'),\n    \'V3\': dict(w_top=0.25, w_mid=0.50, rescue=3, label=\'Leader-safe consensus + 3-slot tail rescue\'),\n    \'V4\': dict(w_top=0.25, w_mid=1.00, rescue=0, label=\'Bold consensus 0.25 / 1.00\'),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef leader_safe_order(neural_scores, w_top, w_mid, offset=3.0):\n    """Indices of the new order (baseline order = 0..n-1)."""\n    s = np.asarray(neural_scores, dtype=float)\n    if len(s) == 0:\n        return np.zeros(0, dtype=int)\n    first = np.asarray(rrf_order(s, w_top, offset), dtype=int)\n    if w_mid == w_top:\n        return first\n    lead = int(first[0])\n    rest = [int(i) for i in rrf_order(s, w_mid, offset) if int(i) != lead]\n    return np.asarray([lead] + rest, dtype=int)\n\n\ndef rescue_tail(smiles, scores, outsiders, slots, limit=25):\n    """Let outside candidates compete for the last `slots` places by neural score.\n\n    smiles, scores: the reordered list and its neural scores.\n    outsiders: [(smiles, neural_score)] whose metric identities differ from every\n    listed candidate and from each other (guaranteed by outsiders.py).\n    Free places of a list shorter than `limit` are offered as well.\n    Returns (new_list, number_of_outsiders_admitted).\n    """\n    smiles = list(smiles)\n    scores = [float(v) for v in scores]\n    if slots <= 0 or not outsiders or not smiles:\n        return smiles, 0\n    k = min(int(slots), len(smiles))\n    free = max(0, int(limit) - len(smiles))\n    head = smiles[:len(smiles) - k]\n    pool = [(s, v, 0) for s, v in zip(smiles[len(smiles) - k:], scores[len(scores) - k:])]\n    pool += [(str(s), float(v), 1) for s, v in outsiders if np.isfinite(float(v)) and str(s) not in smiles]\n    pool.sort(key=lambda t: (-t[1], t[2]))  # stable; on ties a listed candidate keeps its place\n    chosen = pool[:k + free]\n    out = (head + [s for s, _, _ in chosen])[:limit]\n    return out, int(sum(t[2] for t in chosen))\n\n\ndef read_outsiders(path):\n    out = {}\n    if not path or not Path(path).is_file():\n        return out\n    frame = pd.read_parquet(path)\n    if frame.empty or \'neural_ok\' not in frame.columns:\n        return out\n    score = pd.to_numeric(frame.neural_score, errors=\'coerce\')\n    frame = frame[frame.neural_ok.astype(bool) & np.isfinite(score)]\n    for mid, g in frame.groupby(\'molecule_id\', sort=False):\n        g = g.sort_values(\'baseline_rank\', kind=\'stable\')\n        out[str(mid)] = list(zip(g.smiles.astype(str), g.neural_score.astype(float)))\n    return out\n\n\ndef build_variants(evidence, baseline, outdir, presets=None, offset=3.0, outsiders=None):\n    """Write submission_<name>.csv for every preset; return (report, moves)."""\n    presets = dict(PRESETS if presets is None else presets)\n    frame = pd.read_parquet(evidence)\n    base = pd.read_csv(baseline, dtype=str, keep_default_na=False)\n    assert list(base.columns) == [\'molecule_id\', \'smiles\'] and base.molecule_id.is_unique\n    groups = {str(m): g.sort_values(\'baseline_rank\', kind=\'stable\') for m, g in frame.groupby(\'molecule_id\', sort=False)}\n    assert set(groups) == set(base.molecule_id), \'evidence and baseline cover different molecules\'\n    outs = read_outsiders(outsiders)\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    report, moves = {}, []\n    for name, cfg in presets.items():\n        rows = []\n        stats = dict(active_rows=0, changed_rows=0, changed_top1=0, rescued=0)\n        for mid, text in base.itertuples(index=False):\n            original = text.split(\';\')\n            g = groups[mid]\n            assert g.smiles.tolist() == original, f\'evidence order differs from the baseline: {mid}\'\n            active = bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n            if active:\n                order = leader_safe_order(g.neural_score.values, cfg[\'w_top\'], cfg[\'w_mid\'], offset)\n            else:\n                order = np.arange(len(g))\n            new = g.smiles.iloc[order].tolist()\n            new_scores = g.neural_score.astype(float).values[order].tolist()\n            rescued = 0\n            if active and int(cfg.get(\'rescue\', 0)) > 0 and outs.get(mid):\n                new, rescued = rescue_tail(new, new_scores, outs[mid], int(cfg[\'rescue\']))\n            assert 1 <= len(new) <= 25 and len(set(new)) == len(new), mid\n            if not rescued:\n                assert sorted(new) == sorted(original), f\'candidate set changed: {mid}\'\n            stats[\'active_rows\'] += int(active)\n            stats[\'changed_rows\'] += int(new != original)\n            stats[\'changed_top1\'] += int(new[0] != original[0])\n            stats[\'rescued\'] += int(rescued)\n            rows.append((mid, \';\'.join(new)))\n            if active:\n                b_rank = {s: i + 1 for i, s in enumerate(original)}\n                n_rank = dict(zip(g.smiles, stable_rank(g.neural_score.values)))\n                for pos, s in enumerate(new, 1):\n                    moves.append((name, mid, s, pos, b_rank.get(s, 0), int(n_rank.get(s, 0))))\n        out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n        assert out.molecule_id.tolist() == base.molecule_id.tolist()\n        path = outdir / f\'submission_{name}.csv\'\n        out.to_csv(path, index=False)\n        report[name] = dict(cfg, offset=offset, file=path.name, sha256=sha256(path), **stats)\n    moves = pd.DataFrame(moves, columns=[\'variant\', \'molecule_id\', \'smiles\', \'final_rank\', \'baseline_rank\', \'neural_rank\'])\n    (outdir / \'consensus_report.json\').write_text(json.dumps(report, indent=2))\n    return report, moves\n\n\ndef validate(path, sample_ids, baseline_path=None, allow_new=False, identity_of=None):\n    """Raise unless `path` is a well-formed answer file for the expected molecules."""\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    assert list(frame.columns) == [\'molecule_id\', \'smiles\'], \'columns\'\n    assert frame.molecule_id.tolist() == list(sample_ids), \'molecule order\'\n    lists = frame.smiles.str.split(\';\')\n    assert lists.map(len).between(1, 25).all(), \'list length\'\n    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), \'empty or repeated SMILES\'\n    if baseline_path is not None:\n        base = pd.read_csv(baseline_path, dtype=str, keep_default_na=False)\n        assert base.molecule_id.tolist() == frame.molecule_id.tolist(), \'baseline order\'\n        for before, after in zip(base.smiles, frame.smiles):\n            b, a = before.split(\';\'), after.split(\';\')\n            if allow_new:\n                assert len(a) >= min(len(b), 25), \'list got shorter\'\n            else:\n                assert sorted(b) == sorted(a), \'candidate set changed\'\n    if identity_of is not None:\n        for answer in frame.smiles:\n            keys = [identity_of.get(s) for s in answer.split(\';\')]\n            keys = [k for k in keys if k]\n            assert len(keys) == len(set(keys)), \'duplicate metric identity\'\n    return frame\n'
MODULES['outsiders.py'] = '"""Collect candidates that the frozen pipeline considered but left outside its final list.\n\nReads the pipeline\'s own record of its candidate lists (e004_candidate_inputs.json),\nwalks the fusion overflow, Engine 2, the main-engine tail and the PubChem channel in\nround-robin rank order, and keeps structures whose official metric identity (the frozen\nnotebook\'s own metric_identity, executed from its source) is new for the molecule.\nOutput rows follow the evidence schema the neural scorer reads; baseline_rank 26, 27, ...\nonly fixes their order.\n\nusage: python outsiders.py <frozen.ipynb> <root> <evidence.parquet> <out.parquet> [per_molecule] [max_checks]\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOLUMNS = [\'molecule_id\', \'smiles\', \'metric_key\', \'source\', \'source_rank\', \'baseline_rank\', \'processed\']\n\n\ndef load_identity(frozen):\n    cells = json.loads(Path(frozen).read_text())[\'cells\']\n    hits = [\'\'.join(c[\'source\']) for c in cells\n            if c[\'cell_type\'] == \'code\' and \'def metric_identity\' in \'\'.join(c[\'source\'])]\n    assert len(hits) == 1, \'metric identity cell not found in the frozen source\'\n    scope = {\'__name__\': \'frozen_metric_identity\'}\n    exec(compile(hits[0], \'frozen_metric_identity\', \'exec\'), scope)\n    return scope[\'metric_identity\']\n\n\ndef source_lists(payload, mid):\n    eng = payload.get(\'engine2\', {}).get(mid) or {}\n    pc = payload.get(\'pubchem\', {}).get(mid) or {}\n    return [(\'fusion_overflow\', list(payload.get(\'fusion_overflow\', {}).get(mid) or [])),\n            (\'engine2\', list(eng.get(\'smiles\') or [])),\n            (\'main_tail\', list(payload.get(\'base_tail\', {}).get(mid) or [])),\n            (\'pubchem\', list(pc.get(\'pc\') or []))]\n\n\ndef collect(payload, evidence, identity, per_molecule=12, max_checks=30):\n    rows = []\n    active = evidence[evidence.processed.astype(bool)]\n    for mid, g in active.groupby(\'molecule_id\', sort=False):\n        mid = str(mid)\n        if mid not in payload.get(\'prefusion\', {}):\n            continue\n        listed = g.sort_values(\'baseline_rank\', kind=\'stable\').smiles.astype(str).tolist()\n        taken = {k for k in g.metric_key.tolist() if isinstance(k, str) and k}\n        seen = set(listed)\n        lists = source_lists(payload, mid)\n        depth = max((len(v) for _, v in lists), default=0)\n        picked = checks = 0\n        for r in range(depth):\n            if picked >= per_molecule or checks >= max_checks:\n                break\n            for name, values in lists:\n                if r >= len(values) or picked >= per_molecule or checks >= max_checks:\n                    continue\n                s = str(values[r]).strip()\n                if not s or s == \'CCO\' or s in seen:\n                    continue\n                seen.add(s)\n                checks += 1\n                key = identity(s)\n                if key is None or key[0] in taken:\n                    continue\n                taken.add(key[0])\n                picked += 1\n                rows.append((mid, s, key[0], name, r + 1, 25 + picked, True))\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\nif __name__ == \'__main__\':\n    frozen, root, evidence_path, out = sys.argv[1:5]\n    per = int(sys.argv[5]) if len(sys.argv) > 5 else 12\n    checks = int(sys.argv[6]) if len(sys.argv) > 6 else 30\n    t0 = time.time()\n    identity = load_identity(frozen)\n    payload = json.loads((Path(root) / \'e004_candidate_inputs.json\').read_text())\n    evidence = pd.read_parquet(evidence_path)\n    frame = collect(payload, evidence, identity, per, checks)\n    frame.to_parquet(out, index=False)\n    by_source = frame.source.value_counts().to_dict() if len(frame) else {}\n    print(\'OUTSIDERS\', len(frame), \'candidates for\', frame.molecule_id.nunique(), \'molecules\',\n          json.dumps(by_source), f\'{time.time() - t0:.0f}s\', flush=True)\n'
MODULE_SHA256 = {
 "runner.py": "86b74f3b4cd5767a2cbe1d291735c292b66b4fbf286a705c6cb777a1865883db",
 "export_evidence.py": "d4ec36a9f6216d2607fff4f481bc465dbc8fbc616ec4b24f12932e4c5e546e19",
 "neural_scores.py": "cb9e4d6732992e3310e951533fd0c644d3b515134ad03768c35e5a86551267d2",
 "submission.py": "d0027a893b1bfefe8066ede4f625c584fe0d4fa16bfe4bee32265ef34a7f007c",
 "ranking.py": "6f909cd7e41e88387c23109b93dd590885d2283509804279a58c624a758987d7",
 "bundle.py": "8595ef16ead9c9e6ac34a3398bff4b748f5668cdc517c396be18100a5b745cea",
 "consensus.py": "9e67ab49a852404587e62ca84e0fc551d70ce68ec8f7a47739aada6ca08d3989",
 "outsiders.py": "4dcf6c2b9c9b7be3ac8f7a616683495430fcb4e0c25b449684d132daaa8e7b5c"
}
MODEL_PINS = {
 "features.py": "a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455",
 "model.py": "2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39",
 "config.json": "563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03",
 "final_model.pt": "44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642"
}
FROZEN_SHA256 = '44db201061e909ba67d88bd820bac73d9d8d585121f203f2906125624b613877'
FROZEN_B64 = '...'
print(f'{len(MODULES)} modules and the pinned pipeline source ready ({len(FROZEN_B64) * 3 // 4 / 1e6:.2f} MB)')

In [ ]:
# ── Stage machinery: process runner, compact trace, and the four stage functions ───────────────
import contextlib, importlib

STAGES = []           # (stage, seconds, status) for the run summary
STAGE_MARKS = {}      # stage -> [(frozen cell, START/DONE, seconds since stage start)]
FROZEN_STAGE = {
    2: 'Official RDKit 2026.03.3', 6: 'Competition files', 9: 'Asset inventory', 12: 'Documentation prior table',
    14: 'Prior benchmark table', 16: 'Natural-product panel', 18: 'Natural-product knowledge table',
    26: 'Datasets, priors and wheels', 27: 'Determinism patch', 28: 'Preflight', 30: 'Commit or rerun switch',
    32: 'Engine 2 · two rankers', 34: 'PubChem channel', 36: 'Engine 1 · library, pool, spectrum cache',
    37: 'Engine 1 · models and rankers', 39: 'Engine 1 · candidate lists', 41: 'GLACIER + ICEBERG forward models',
    43: 'Gated PubChem merge', 45: 'Two-engine fusion', 48: 'Metric audit and safe tail', 49: 'Metric-identity fusion',
    50: 'Postflight record', 51: 'Fragment score gap', 52: 'Window-40 overlay', 53: 'Engine 2 rank-2 placement',
    54: 'Execution receipt', 56: 'Example answer table'}
CELL_MARK = re.compile(r'^PIPELINE CELL (\d+) (START|DONE)')
KEY_LINE = re.compile(r'FAILED|ERROR|Traceback|merge stats|RANK2_OVERLAY|EXPORTED FROZEN|NEURAL FEATURES|'
                      r'NEURAL input failures|OUTSIDERS|engine lists \d+ |pubchem lists \d+|errors \d+ \|')
BASELINE = ROOT / 'submission_baseline0465.csv'
VARIANTS_DIR = ROOT / 'variants'
RESCUE_SLOTS = {'V3': 3, 'CUSTOM': int(CUSTOM.get('rescue', 0))}.get(VARIANT, 0)
PIPELINE_DONE = NEURAL_OK = OUTSIDERS_OK = CONSENSUS_OK = False
REPORT, MOVES, SUBMITTED, consensus = {}, pd.DataFrame(), None, None


def run_stage(args, log, name, quiet=None):
    """Run one stage in its own process; keep the full log on disk and print a compact trace."""
    quiet = QUIET_LOGS if quiet is None else quiet
    env = {**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    t0, marks, tail = time.time(), [], collections.deque(maxlen=150)
    print(f'▶ {name}', flush=True)
    with open(log, 'w') as handle:
        proc = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line); handle.flush()
            t = time.time() - t0
            tail.append(line)
            m = CELL_MARK.match(line)
            if m:
                cell, kind = int(m.group(1)), m.group(2)
                marks.append((cell, kind, t))
                if quiet and kind == 'DONE' and cell in FROZEN_STAGE:
                    start = next((s for c, k, s in reversed(marks) if c == cell and k == 'START'), t)
                    if t - start >= 1:
                        print(f'  ✓ {t / 60:6.1f} min   {FROZEN_STAGE[cell]:<40s} {t - start:8,.0f} s', flush=True)
                continue
            if not quiet:
                if len(line) < 5000:
                    print(line, end='', flush=True)
            elif KEY_LINE.search(line) and len(line) < 800:
                print('      ' + line.strip()[:180], flush=True)
        status = proc.wait()
    seconds = time.time() - t0
    STAGE_MARKS[name] = marks
    STAGES.append((name, seconds, 'ok' if status == 0 else f'exit {status}'))
    if status:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed with exit code {status}; full log: {log}')
    print(f'  done in {minutes(seconds)}', flush=True)
    return seconds


def materialise_sources():
    """Stage 0: write and verify the pinned sources, find the neural bundle, install the official RDKit."""
    global FROZEN
    t0 = time.time()
    for name, text in MODULES.items():
        (CODE / name).write_text(text)
        assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == MODULE_SHA256[name], name
    FROZEN = CODE / 'frozen_0465.ipynb'
    FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
    assert hashlib.sha256(FROZEN.read_bytes()).hexdigest() == FROZEN_SHA256, 'Pinned pipeline source differs'
    assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 or 3.13 is required'
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))
    from bundle import find_bundle
    bundle = find_bundle(str(INPUT), ROOT, MODEL_PINS)
    (CODE / 'bootstrap.py').write_text(''.join(json.loads(FROZEN.read_text())['cells'][2]['source']))
    run_stage([sys.executable, CODE / 'bootstrap.py'], ROOT / 'bootstrap.log', 'Official RDKit wheel', quiet=False)
    STAGES[-1] = ('Sources and RDKit wheel', time.time() - t0, 'ok')
    tiles([('Pipeline source', FROZEN_SHA256[:12] + '…', 'SHA-256 verified'),
           ('Helper modules', f'{len(MODULES)}', '6 public + 2 new, SHA-256 verified'),
           ('Neural bundle', Path(bundle).name, '4 files, SHA-256 verified'),
           ('Stage 0', minutes(time.time() - t0), 'including the wheel install')])
    return bundle


def run_public_pipeline():
    """Stage 1: every cell of the pinned pipeline in one process, then the export of its final lists."""
    if not RUN_PIPELINE:
        note('Fast commit.', 'The pipeline is skipped in this commit run; the scoring rerun executes every stage.', BLUE)
        return False
    assert not (ROOT / 'work/spec_meta.parquet').exists() and not (ROOT / 'submission.csv').exists(), \
        'Start from a fresh session (Run All after a restart)'
    seconds = run_stage([sys.executable, CODE / 'runner.py', FROZEN, FROZEN_SHA256, ROOT],
                        ROOT / 'submission_pipeline.log', 'Public 0.465 pipeline')
    receipt = json.loads((ROOT / 'execution_receipt.json').read_text())
    full = receipt['is_full_run']
    tiles([('Molecules ranked', f"{receipt['processed_molecules']:,}", 'hidden test' if full else 'visible smoke subset'),
           ('Answer rows', f'{len(SAMPLE_IDS):,}', 'all ranked' if full else 'placeholders elsewhere'),
           ('Pipeline time', minutes(seconds), 'one process, every cell'),
           ('Baseline file', BASELINE.name, 'kept for reference')])
    return True


def score_final_lists():
    """Stage 2a: the public scorer on exactly the exported candidates."""
    try:
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'frozen_candidate_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'neural_candidate_features.parquet', 'cuda'],
                  ROOT / 'submission_neural.log', 'Neural scorer · final lists')
        return True
    except Exception as exc:
        note('Neural scorer failed.', f'The 0.465 answers will be submitted. {type(exc).__name__}: {str(exc)[:300]}', RED)
        return False


def score_outside_candidates(needed):
    """Stage 2b (optional): collect and score candidates ranked just outside the final lists."""
    elapsed_h = (time.time() - NB_T0) / 3600
    if not needed:
        return False
    if elapsed_h >= TIME_GUARD_HOURS:
        note('Tail rescue skipped.', f'The session has run {elapsed_h:.1f} h (guard {TIME_GUARD_HOURS} h).', YELLOW)
        return False
    try:
        run_stage([sys.executable, CODE / 'outsiders.py', FROZEN, ROOT, ROOT / 'frozen_candidate_evidence.parquet',
                   ROOT / 'outsider_evidence.parquet', OUTSIDERS_PER_MOLECULE],
                  ROOT / 'outsiders.log', 'Outside candidates · collect')
        if not len(pd.read_parquet(ROOT / 'outsider_evidence.parquet')):
            return False
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'outsider_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'outsider_neural_features.parquet', 'cuda'],
                  ROOT / 'outsiders_neural.log', 'Neural scorer · outside candidates')
        return True
    except Exception as exc:
        note('Tail rescue disabled.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
        return False


def show_neural_summary():
    if not NEURAL_OK:
        return
    nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    act = nf[nf.processed.astype(bool)]
    n_out = len(pd.read_parquet(ROOT / 'outsider_neural_features.parquet')) if OUTSIDERS_OK else 0
    tiles([('Candidates scored', f'{len(act):,}', f'{act.molecule_id.nunique():,} molecules'),
           ('Scored molecules', f'{int(act.groupby("molecule_id").neural_ok.all().sum()):,}', 'all spectra usable'),
           ('Spectra per molecule', f'{act.groupby("molecule_id").n_spectra.first().median():.0f}', 'median used by the scorer'),
           ('Outside candidates', f'{n_out:,}', 'scored for the tail rescue' if n_out else 'not needed for this variant')])


def build_consensus():
    """Stage 3a: the public 0.466 file, its exact rebuild, then every variant."""
    global consensus
    try:
        consensus = importlib.import_module('consensus')
        from submission import apply as public_apply
        evidence = ROOT / 'neural_candidate_features.parquet'
        with contextlib.redirect_stdout(io.StringIO()):      # the public helper prints its own report
            public_apply(evidence, BASELINE, ROOT / 'submission_public0466.csv', weight=0.25, offset=3.0)
        consensus.build_variants(evidence, BASELINE, ROOT / 'variants_check', {'V1': consensus.PRESETS['V1']}, 3.0)
        mine = pd.read_csv(ROOT / 'variants_check/submission_V1.csv', dtype=str, keep_default_na=False)
        public = pd.read_csv(ROOT / 'submission_public0466.csv', dtype=str, keep_default_na=False)
        assert mine.equals(public), 'V1 differs from the public 0.466 file'
        presets = dict(consensus.PRESETS)
        if VARIANT == 'CUSTOM':
            presets['CUSTOM'] = dict(w_top=float(CUSTOM['w_top']), w_mid=float(CUSTOM['w_mid']),
                                     rescue=int(CUSTOM.get('rescue', 0)), label='Custom weights')
        report, moves = consensus.build_variants(
            evidence, BASELINE, VARIANTS_DIR, presets, RRF_OFFSET,
            ROOT / 'outsider_neural_features.parquet' if OUTSIDERS_OK else None)
        print('V1 rebuilt exactly: the new code reproduces the public 0.466 file row for row.')
        return True, report, moves
    except Exception as exc:
        note('Consensus layer disabled.', f'{type(exc).__name__}: {str(exc)[:400]}', RED)
        return False, {}, pd.DataFrame()


def identity_map():
    keys = {}
    for path in (ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet'):
        if path.is_file():
            f = pd.read_parquet(path, columns=['smiles', 'metric_key'])
            keys.update({s: k for s, k in zip(f.smiles, f.metric_key) if isinstance(k, str)})
    return keys


def well_formed(path, same_candidates=True):
    """Minimal checks that do not depend on the new modules."""
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, 'rows'
    lists = f.smiles.str.split(';')
    assert lists.map(len).between(1, 25).all(), 'list length'
    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    if same_candidates and BASELINE.is_file():
        b = pd.read_csv(BASELINE, dtype=str, keep_default_na=False)
        assert all(sorted(x.split(';')) == sorted(y.split(';')) for x, y in zip(b.smiles, f.smiles)), 'candidates'


def choose_and_submit():
    """Stage 3b: chosen variant -> V1 -> V0 -> public 0.466 file -> baseline file (-> sample in a fast commit)."""
    t0 = time.time()
    chain = []
    if CONSENSUS_OK:
        chain += [(n, VARIANTS_DIR / f'submission_{n}.csv') for n in dict.fromkeys([VARIANT, 'V1', 'V0'])]
    chain += [('public 0.466 file', ROOT / 'submission_public0466.csv'), ('baseline file', BASELINE)]
    chosen = None
    for name, path in chain:
        if not path.is_file():
            continue
        try:
            rescue = name in REPORT and REPORT[name].get('rescued', 0) > 0
            if rescue:
                consensus.validate(path, SAMPLE_IDS, BASELINE, allow_new=True, identity_of=identity_map())
            well_formed(path, same_candidates=not rescue)
            chosen = (name, path)
            break
        except Exception as exc:
            note(f'{name} rejected.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
    if chosen is None:
        assert not IS_RERUN, 'No valid answer file in the scoring rerun'
        chosen = ('sample (fast commit)', COMP / 'sample_submission.csv')
    shutil.copyfile(chosen[1], ROOT / 'submission.csv')
    STAGES.append(('Consensus and checks', time.time() - t0, 'ok'))
    return chosen[0]


def show_variant_board():
    if CONSENSUS_OK:
        board = pd.DataFrame([{
            'Variant': k, 'Rule': v['label'], 'Leader weight': f"{v['w_top']:.2f}",
            'Ranks 2–25 weight': f"{v['w_mid']:.2f}", 'Tail rescue': v['rescue'] or '–',
            'Rows re-ordered': f"{v['changed_rows']:,}", 'Rank 1 changed': f"{v['changed_top1']:,}",
            'Rescued': f"{v['rescued']:,}"} for k, v in REPORT.items()])
        board.index = board.Variant
        table_html(board, highlight=SUBMITTED, numeric=board.columns[2:])
    sub = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    tiles([('Submitted', SUBMITTED, 'copied to submission.csv'),
           ('Rows', f'{len(sub):,}', 'one per molecule, sample order'),
           ('Candidates per row', f'{sub.smiles.str.count(";").add(1).median():.0f}', 'median, at most 25'),
           ('SHA-256', hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest()[:12] + '…', 'of submission.csv')])

In [ ]:
from pathlib import Path
_physical = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\ncasmi26 fork: optional ppm tolerance, ring-opening credit (`w_ring`, default from env CASMI_FRAG_W_RING),\nspectrum aggregation and precursor removal; with all of them off it scores exactly like the upstream file.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport os\n\nimport numpy as np\n\n\ndef _env_weight(name):\n    """Optional float from the environment (\'\' / unset / \'none\' -> None), so a notebook switch reaches subprocesses."""\n    v = os.environ.get(name, \'\').strip().lower()\n    return None if v in (\'\', \'none\') else float(v)\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n    # extensions (casmi26 fork); the defaults below reproduce the upstream scorer exactly\n    ppm=0.0,                     # if > 0, tolerance per peak = max(tol, ppm * 1e-6 * mz)\n    # credit for 2 cleavages inside one ring (one ring-opening event, e.g. RDA); None = w2.\n    # Read from CASMI_FRAG_W_RING (bench: 0.8 -> +0.016 MRR on enveda-np-examples, -0.008 on enveda-180)\n    w_ring=_env_weight(\'CASMI_FRAG_W_RING\'),\n    agg=\'mean\',                  # \'mean\' | \'npeaks\' (spectra weighted by sqrt(peak count)) | \'max\'\n    drop_prec=0.0,               # if > 0, ignore peaks within this many Da below the precursor and above it\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)), w_ring=None)\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    if out[\'agg\'] not in (\'mean\', \'npeaks\', \'max\'):\n        raise ValueError(\'frag_rescore: agg must be mean, npeaks or max\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components, same_ring):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces.\n\n    A 2-bond cleavage whose bonds share a ring (`same_ring[i, j]`) is emitted with k = 4 (ring-opening event).\n    """\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt,\n                                       4 if same_ring[i, j] else 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef same_ring_matrix(smi, nb):\n    """nb x nb bool: bonds i and j lie in one SSSR ring (all False if the SMILES does not parse)."""\n    from rdkit import Chem\n    out = np.zeros((nb, nb), np.bool_)\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return out\n    for ring in m.GetRingInfo().BondRings():\n        ix = np.array(ring, np.int64)\n        out[np.ix_(ix, ix)] = True\n    np.fill_diagonal(out, False)\n    return out\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40, ring=False):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse. With `ring`, 2-bond cleavages inside one ring form an extra last level, ranked\n    between k=1 and k=2 (a mass reachable both ways counts as the ring event).\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    codes = list(range(int(max_breaks) + 1)) + ([4] if ring else [])\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in codes[1:]]\n    same_ring = same_ring_matrix(smi, nb) if ring and mb >= 2 else np.zeros((nb, nb), np.bool_)\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces, same_ring)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    cost = np.array([0, 1, 3, 4, 2], np.int8)[fk]  # ring event (k=4) sorts between 1 and 2 cleavages\n    key = np.round(fm, 5)\n    order = np.lexsort((cost, key))                # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in codes]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if c[\'drop_prec\'] and s.get(\'prec\'):\n        keep = mz < float(s[\'prec\']) - float(c[\'drop_prec\'])\n        mz, it = mz[keep], it[keep]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    tol = float(c[\'tol\'])\n    if c[\'ppm\']:\n        tol = np.tile(np.maximum(tol, float(c[\'ppm\']) * 1e-6 * mz), len(shifts))\n    return iw / tot, q, len(shifts), len(mz), tol\n\n\ndef _score_spectrum(levels, weights, prep):\n    iw, q, ns, npk, tol = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    ring = c[\'w_ring\'] is not None\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\'])) + ((\'ring\',) if ring else ())\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'], ring=ring)\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    if c[\'w_ring\'] is not None:\n        weights.append(float(c[\'w_ring\']))\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    sw = np.sqrt([p[3] for p in preps]) if c[\'agg\'] == \'npeaks\' else None\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        per = [_score_spectrum(lv, weights, p) for p in preps]\n        out.append(float(np.max(per) if c[\'agg\'] == \'max\' else np.average(per, weights=sw)))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
_post = "from __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\n\n@lru_cache(maxsize=30000)\ndef mol_formula(s):\n    try:\n        m = Chem.MolFromSmiles(str(s))\n        return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n    except Exception:\n        return None\n\ndef finite(x,default=np.nan):\n    try:\n        t=float(x)\n        return t if np.isfinite(t) else default\n    except (ValueError,TypeError):\n        return default\n\ndef select_spectra(frame,max_n=12):\n    out=[]\n    by_bucket={}\n    for r in frame.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs,dtype=float).ravel()\n        it=np.asarray(r.ms2_normalized_intensities,dtype=float).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        good=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if good.sum()<5:continue\n        order=np.argsort(it[good])[-120:]\n        mz,it=mz[good][order],it[good][order]\n        e=finite(getattr(r,'collision_energy_ev',30),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        item={'mz':mz,'it':it,'adduct':adduct,'prec':finite(getattr(r,'precursor_mz',None),float(mz.max())),'mode':mode}\n        bucket=(adduct,round(e/10))\n        by_bucket.setdefault(bucket,[]).append((int(good.sum()),item))\n    for key in sorted(by_bucket):\n        out += [v for _,v in sorted(by_bucket[key],key=lambda x:-x[0])[:2]]\n    return out[:max_n]\n\ndef support(payload,mid,s):\n    eng=(payload.get('engine2',{}).get(mid) or {}).get('smiles') or []\n    pc=(payload.get('pubchem',{}).get(mid) or {}).get('pc') or []\n    main=payload.get('base_tail',{}).get(mid) or []\n    fusion=payload.get('fusion_overflow',{}).get(mid) or []\n    def rank(v):return v.index(s)+1 if s in v else 0\n    return dict(engine2=rank(eng),pubchem=rank(pc),main=rank(main),fusion=rank(fusion))\n\ndef score_candidates(frag,spec,candidates,cache):\n    if not spec:return None\n    opt=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cache=cache)],float)\n    old=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cfg=frag.BASELINE_CFG,cache=cache)],float)\n    if not np.isfinite(opt).all() or not np.isfinite(old).all():return None\n    chunks=[spec[i:i+2] for i in range(0,len(spec),2)]\n    group=np.array([[np.nan if v is None else v for v in frag.score_molecule(ch,candidates,cache=cache)] for ch in chunks],float)\n    return opt,old,group\n\ndef decision(mid,control,listed,outs,query,payload,frag,cache,allow_tail=True,allow_top=True):\n    diag={'molecule_id':mid,'top1_changed':False,'tail_changed':False,'local_swaps':0,'reason':'unchanged','n_spectra':0,\n          'candidate_count':len(control),'top1_frag_margin':0.,'top1_neural_margin':0.,'new_candidate_source':''}\n    bysmiles=listed.set_index('smiles',drop=False)\n    if len(bysmiles)!=len(listed) or any(s not in bysmiles.index for s in control):\n        diag['reason']='evidence_mismatch';return control,diag\n    if not bool(listed.neural_ok.all()) or not bool(listed.processed.all()):\n        diag['reason']='unscored';return control,diag\n    n=len(control)\n    if n<3:return control,diag\n    anchor=mol_formula(control[0])\n    if anchor is None:\n        diag['reason']='formula_missing';return control,diag\n    cand=list(control[:min(12,n)])\n    outrows=[]\n    if allow_tail and outs is not None and len(outs) and n>=20:\n        for row in outs.sort_values('baseline_rank').itertuples(index=False):\n            if not bool(row.neural_ok) or not bool(row.processed):continue\n            s=str(row.smiles)\n            if s in control or mol_formula(s)!=anchor:continue\n            ev=support(payload,mid,s)\n            sources=int(bool(ev['engine2']))+int(bool(ev['pubchem']))+int(bool(ev['main']))\n            if sources<2 and not (0<ev['fusion']<=3):continue\n            outrows.append(row)\n            cand.append(s)\n            if len(outrows)>=8:break\n    spec=select_spectra(query)\n    diag['n_spectra']=len(spec)\n    if len(spec)<2:\n        diag['reason']='few_spectra';return control,diag\n    data=score_candidates(frag,spec,cand,cache)\n    if data is None:\n        diag['reason']='frag_unavailable';return control,diag\n    frag_new,frag_old,energy=data\n    pos={s:i for i,s in enumerate(cand)}\n    leader=control[0];lp=pos[leader]\n    neu_vals=np.asarray([finite(x) for x in listed.neural_score],float)\n    neural_sd=max(float(np.nanstd(neu_vals)),1e-6)\n    result=list(control)\n    lib=finite(bysmiles.loc[leader,'library_max'],0.)\n    if allow_top and lib<.87 and len(spec)>=3:\n        contenders=[]\n        for ix,s in enumerate(control[1:min(6,n)],1):\n            if mol_formula(s)!=anchor:continue\n            j=pos[s]; ev=bysmiles.loc[s]; lead_ev=bysmiles.loc[leader]\n            if not np.isfinite(frag_new[j]) or frag_new[j]<.27:continue\n            margin=float(frag_new[j]-frag_new[lp]); oldmargin=float(frag_old[j]-frag_old[lp])\n            if margin<.095 or oldmargin<.028:continue\n            votes=float((energy[:,j]>energy[:,lp]+.025).mean())\n            if votes<.70:continue\n            nmargin=(finite(ev.neural_score)-finite(lead_ev.neural_score))/neural_sd\n            if not np.isfinite(nmargin) or nmargin<.23:continue\n            if finite(ev.neural_cos)<finite(lead_ev.neural_cos)+.003 or finite(ev.neural_fp)<finite(lead_ev.neural_fp)-.012:continue\n            supports=0\n            for col in ('engine2_rank','pubchem_rank','main_rank'):\n                rank=finite(ev.get(col),0)\n                if 0<rank<=5:supports+=1\n            simulator_vote=0\n            for col in ('ice','gl'):\n                a,b=finite(ev.get(col)),finite(lead_ev.get(col))\n                if np.isfinite(a) and np.isfinite(b):\n                    if a>=b:simulator_vote+=1\n                    elif a<b-.35:simulator_vote-=1\n            if supports<1 and simulator_vote<1:continue\n            if simulator_vote<=-1:continue\n            contenders.append((margin+.05*nmargin+.025*simulator_vote+.01*supports,s,margin,nmargin))\n        if contenders:\n            _,winner,margin,nmargin=max(contenders,key=lambda v:v[0])\n            result.remove(winner);result.insert(0,winner)\n            diag.update(top1_changed=True,top1_frag_margin=round(margin,5),top1_neural_margin=round(nmargin,5),reason='fragment_neural_top1')\n    swaps=0\n    for i in range(2,min(11,n)):\n        prev,s=result[i-1],result[i]\n        if mol_formula(prev)!=mol_formula(s):continue\n        j,k=pos.get(s),pos.get(prev)\n        if j is None or k is None:continue\n        if frag_new[j]-frag_new[k]<.085 or frag_old[j]-frag_old[k]<.015:continue\n        if (energy[:,j]>energy[:,k]+.010).mean()<.65:continue\n        neui,neup=finite(bysmiles.loc[s,'neural_score']),finite(bysmiles.loc[prev,'neural_score'])\n        if not np.isfinite(neui) or neui<neup-.08*neural_sd:continue\n        result[i-1],result[i]=result[i],result[i-1]\n        swaps+=1\n        if swaps>=2:break\n    diag['local_swaps']=swaps\n    if not diag['top1_changed'] and swaps:diag['reason']='adjacent_fragment_consensus'\n    if allow_tail and outrows and len(result)==25:\n        last=result[-1]\n        if last in pos:\n            lastp=pos[last]\n            eligible=[]\n            listed_keys=set(listed.metric_key.dropna().astype(str))\n            for row in outrows:\n                s=str(row.smiles);j=pos[s]\n                if not isinstance(row.metric_key,str) or row.metric_key in listed_keys:continue\n                if not np.isfinite(frag_new[j]) or frag_new[j]<.28:continue\n                fm=float(frag_new[j]-frag_new[lastp])\n                if fm<.065 or frag_old[j]-frag_old[lastp]<.02:continue\n                if (energy[:,j]>energy[:,lastp]+.01).mean()<.65:continue\n                if finite(row.neural_score)<finite(bysmiles.loc[last,'neural_score'])+.22*neural_sd:continue\n                if finite(row.neural_cos)<=finite(bysmiles.loc[last,'neural_cos']) or finite(row.neural_fp)<=finite(bysmiles.loc[last,'neural_fp']):continue\n                ev=support(payload,mid,s)\n                supports=int(bool(ev['main']))+int(bool(ev['engine2']))+int(bool(ev['pubchem']))\n                eligible.append((fm,s,ev,supports))\n            if eligible:\n                margin,s,ev,supports=max(eligible,key=lambda q:q[0]+.015*q[3])\n                result[-1]=s\n                diag.update(tail_changed=True,new_candidate_source='+' .join(k for k in ('main','engine2','pubchem','fusion') if ev[k]),reason='tail_supported' if not diag['top1_changed'] and not swaps else diag['reason'])\n    assert len(result)==len(control) and len(set(result))==len(result)\n    assert set(result[:-1]).issubset(set(control)) and not (diag['top1_changed'] and result[0]==control[0])\n    return result,diag\n\ndef apply(root,comp,max_seconds=1500,allow_tail=True,allow_top=True):\n    import frag_rescore as frag\n    root=Path(root);comp=Path(comp)\n    control=root/'variants/submission_V2.csv'\n    if not control.is_file():raise FileNotFoundError(control)\n    src=pd.read_csv(control,dtype=str,keep_default_na=False)\n    need=['molecule_id','smiles','metric_key','processed','neural_ok','neural_score','neural_cos','neural_fp','baseline_rank']\n    listed=pd.read_parquet(root/'neural_candidate_features.parquet')\n    if not set(need).issubset(listed.columns):raise ValueError('Neural evidence columns missing')\n    outsiders_path=root/'outsider_neural_features.parquet'\n    outsider=pd.read_parquet(outsiders_path) if outsiders_path.is_file() else pd.DataFrame()\n    q=pd.read_parquet(root/'effective_query_spectra.parquet')\n    payload=json.loads((root/'e004_candidate_inputs.json').read_text())\n    lmap={str(mid):g.sort_values('baseline_rank') for mid,g in listed.groupby('molecule_id',sort=False)}\n    omap={str(mid):g for mid,g in outsider.groupby('molecule_id',sort=False)} if len(outsider) else {}\n    qmap={str(mid):g for mid,g in q.groupby('molecule_id',sort=False)}\n    start=time.monotonic()\n    cache={}; rows=[];audits=[];expired=False\n    frag.warmup()\n    for i,(mid,smiles) in enumerate(src.itertuples(index=False)):\n        mid=str(mid);old=smiles.split(';')\n        if time.monotonic()-start>max_seconds:\n            expired=True\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='time_budget',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        if mid not in lmap or mid not in qmap:\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='not_processed',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        try:\n            new,meta=decision(mid,old,lmap[mid],omap.get(mid),qmap[mid],payload,frag,cache,allow_tail=allow_tail,allow_top=allow_top)\n        except Exception as exc:\n            new,meta=old,dict(molecule_id=mid,reason='error:'+type(exc).__name__,top1_changed=False,tail_changed=False,local_swaps=0)\n        rows.append((mid,';'.join(new)));audits.append(meta)\n        if (i+1)%40==0:print('PARALLAX',i+1,'/',len(src),'elapsed',round(time.monotonic()-start,1),'s',flush=True)\n        if len(cache)>35000:cache.clear()\n    dest=pd.DataFrame(rows,columns=['molecule_id','smiles'])\n    ids=pd.read_parquet(comp/'test.parquet',columns=['molecule_id']).molecule_id.astype(str).unique().tolist()\n    if dest.molecule_id.tolist()!=src.molecule_id.tolist() or set(ids)!=set(dest.molecule_id):raise AssertionError('Molecule coverage changed')\n    for initial,new in zip(src.smiles,dest.smiles):\n        before=initial.split(';');after=new.split(';')\n        if not 1<=len(after)<=25 or len(after)!=len(set(after)) or any(not k for k in after):raise AssertionError('Submission format')\n        if after[0] not in before or not set(after[:-1]).issubset(set(before)):raise AssertionError('Unsafe candidate replacement')\n        if len(set(after)-set(before))>1:raise AssertionError('Too many outsiders')\n    out=root/'submission.csv';temp=root/'submission.parallax.tmp'\n    dest.to_csv(temp,index=False)\n    reread=pd.read_csv(temp,dtype=str,keep_default_na=False)\n    if not reread.equals(dest):raise AssertionError('CSV roundtrip failed')\n    temp.replace(out)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'spectral_parallax_audit.csv',index=False)\n    diagnostics={'control_sha256':hashlib.sha256(control.read_bytes()).hexdigest(),\n        'submission_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),\n        'rows':len(dest),'changed_rows':int((src.smiles!=dest.smiles).sum()),\n        'top1_changed':int(sum(bool(r.get('top1_changed')) for r in audits)),\n        'tail_changed':int(sum(bool(r.get('tail_changed')) for r in audits)),\n        'adjacent_swaps':int(sum(int(r.get('local_swaps',0)) for r in audits)),\n        'time_budget_exceeded':expired,'elapsed_seconds':round(time.monotonic()-start,2),\n        'outcome':'experimental_unvalidated','baseline':'V2_0.468_user_reported'}\n    (root/'spectral_parallax_report.json').write_text(json.dumps(diagnostics,indent=2))\n    print('PARALLAX REPORT',json.dumps(diagnostics),flush=True)\n    return diagnostics\n"
assert __import__('hashlib').sha256(_physical.encode()).hexdigest() == '5235ffc2926b7a04f0967300439212c8590ca869acff4aa51bb1219b49db0a22'
assert __import__('hashlib').sha256(_post.encode()).hexdigest() == 'cd1594b3bbfa75ce30997559c5403438b7cec011879dcb9c0d48e111715dd85d'
(ROOT/'frag_rescore.py').write_text(_physical)
(CODE/'parallax_physical.py').write_text(_post)
print('Fragment source:', __import__('hashlib').sha256(_physical.encode()).hexdigest()[:12])


In [ ]:
BUNDLE = materialise_sources()
PIPELINE_DONE = run_public_pipeline()
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    shutil.copyfile(COMP / 'sample_submission.csv', ROOT / 'submission.csv')
    raise SystemExit('fast commit: sample file written, the scoring rerun runs every stage')
if not PIPELINE_DONE:
    raise RuntimeError('A real baseline run is required; no placeholder-only submission')
NEURAL_OK = score_final_lists()
if not NEURAL_OK:
    raise RuntimeError('The 0.468 V2 neural checkpoint is required; refusing silent 0.465 fallback')
OUTSIDERS_OK = score_outside_candidates(needed=True)
CONSENSUS_OK, REPORT, MOVES = build_consensus()
if not CONSENSUS_OK:
    raise RuntimeError('Unable to reproduce the V2 control')
SUBMITTED = choose_and_submit()
if SUBMITTED != VARIANT:
    raise RuntimeError(f'Expected {VARIANT}, got {SUBMITTED}')
shutil.copy2(ROOT/'submission.csv', ROOT/'submission_control_V2.csv')
print('Exact V2 control:', hashlib.sha256((ROOT/'submission_control_V2.csv').read_bytes()).hexdigest()[:16])


In [ ]:
import sys, traceback
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(CODE))
from parallax_physical import apply
try:
    _remaining = max(0, int(8.55 * 3600 - (time.time() - NB_T0)))
    if not PHYSICAL:
        PARALLAX_DIAGNOSTICS = {'status': 'disabled in this experiment'}
    elif _remaining < 180:
        PARALLAX_DIAGNOSTICS = {'status':'skipped_due_to_runtime_budget','remaining_seconds':_remaining,'control_retained':True}
    else:
        PARALLAX_DIAGNOSTICS = apply(ROOT, COMP, max_seconds=min(PHYSICAL_POST_SECONDS, _remaining - 120), allow_tail=True, allow_top=True)
except Exception:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PARALLAX_DIAGNOSTICS = {'error':'physical_overlay_failed', 'control_restored':True}
    (ROOT/'spectral_parallax_report.json').write_text(json.dumps(PARALLAX_DIAGNOSTICS,indent=2))
assert (ROOT/'submission.csv').is_file()
final = pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.notna().all() and final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(x.strip() for x in a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','rows',len(final),'control',SUBMITTED)
print('EXPERIMENT:', json.dumps(PARALLAX_DIAGNOSTICS))


In [ ]:
from pathlib import Path
import hashlib
_PUBLIC_RERANKER = "from __future__ import annotations\nimport json\nimport math\nimport time\nimport hashlib\nimport shutil\nfrom pathlib import Path\nfrom functools import lru_cache\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\nimport frag_rescore as frag\n\n@lru_cache(maxsize=40000)\ndef formula(s):\n    m = Chem.MolFromSmiles(str(s))\n    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n\ndef number(x, default=float('nan')):\n    try:\n        v=float(x)\n        return v if math.isfinite(v) else default\n    except Exception:\n        return default\n\ndef member_keys(comp):\n    frame=pd.read_parquet(Path(comp)/'train.parquet',columns=['inchikey14'])\n    return set(frame.inchikey14.dropna().astype(str))\n\ndef balanced_spectra(query, max_groups=12):\n    groups={}\n    for r in query.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs, dtype=np.float64).ravel()\n        it=np.asarray(r.ms2_normalized_intensities, dtype=np.float64).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        ok=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if ok.sum()<6:continue\n        order=np.argsort(it[ok])[-120:]\n        mz,it=mz[ok][order],it[ok][order]\n        energy=number(getattr(r,'collision_energy_ev',30.),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        spec={'mz':mz,'it':it,'prec':number(getattr(r,'precursor_mz',None),float(mz.max())),\n              'adduct':adduct,'mode':mode}\n        bucket=(adduct,int(round(energy/10.)))\n        groups.setdefault(bucket,[]).append((int(ok.sum()), spec))\n    chosen=[]\n    for group in sorted(groups):\n        chosen.extend(s for _,s in sorted(groups[group],key=lambda x:-x[0])[:2])\n    return chosen[:max_groups]\n\ndef evidence_rows(df, order):\n    data=df.drop_duplicates('smiles').set_index('smiles',drop=False)\n    if any(s not in data.index for s in order):\n        return None\n    if ('neural_ok' in data and not data.loc[order,'neural_ok'].astype(bool).all()):\n        return None\n    if ('processed' in data and not data.loc[order,'processed'].astype(bool).all()):\n        return None\n    return data\n\ndef source_votes(candidate, leader):\n    good=0;bad=0\n    for label in ('engine2_rank','pubchem_rank','main_rank'):\n        c=number(candidate.get(label),0.);l=number(leader.get(label),0.)\n        if 0<c<=6:good+=1\n        if 0<l<=4 and (c<=0 or c>12):bad+=1\n    sim=0\n    for label in ('ice','gl'):\n        c=number(candidate.get(label));l=number(leader.get(label))\n        if math.isfinite(c) and math.isfinite(l):\n            if c>=l-.025:sim+=1\n            elif c<l-.30:sim-=1\n    return good,bad,sim\n\ndef paired_scores(spectra, smiles, cache):\n    matrix=[];matrix_old=[]\n    for spectrum in spectra:\n        vals=frag.score_molecule([spectrum],smiles,cache=cache)\n        old=frag.score_molecule([spectrum],smiles,cfg=frag.BASELINE_CFG,cache=cache)\n        if any(x is None or not math.isfinite(float(x)) for x in vals+old):return None\n        matrix.append(vals)\n        matrix_old.append(old)\n    return np.asarray(matrix,dtype=float),np.asarray(matrix_old,dtype=float)\n\ndef propose(order, original, data, query, train, cache):\n    record={'action':'none','top1_changed':False,'swaps':0,'n_spectra':0,\n            'protected_train':False,'physical_gain':None,'neural_gain_sd':None}\n    if len(order)<4 or not order or original[0]!=order[0]:\n        record['action']='protected_previous_top1';return order,record\n    evid=evidence_rows(data,order)\n    if evid is None:\n        record['action']='incomplete_evidence';return order,record\n    keys=[str(evid.loc[s,'metric_key']) if 'metric_key' in evid.columns else '' for s in order]\n    trained=np.asarray([k in train for k in keys],dtype=bool)\n    record['protected_train']=bool(trained.any())\n    leader=order[0]\n    if formula(leader) is None:\n        return order,record\n    lib=number(evid.loc[leader].get('library_max'),0.)\n    if lib>=.90:\n        record['action']='library_protected';return order,record\n    selected=balanced_spectra(query)\n    record['n_spectra']=len(selected)\n    if len(selected)<3:\n        record['action']='insufficient_spectra';return order,record\n    head=order[:min(10,len(order))]\n    p=paired_scores(selected,head,cache)\n    if p is None:\n        record['action']='fragment_unavailable';return order,record\n    a,b=p\n    neu=np.asarray([number(evid.loc[s].get('neural_score')) for s in order],float)\n    if not np.isfinite(neu).all():\n        record['action']='neural_unavailable';return order,record\n    ns=max(float(neu.std()),.00001)\n    updated=list(order)\n    candidates=[]\n    if not trained[0]:\n        for i in range(1,min(6,len(head))):\n            if trained[i] or formula(head[i])!=formula(leader):continue\n            new_margin=a[:,i]-a[:,0]\n            old_margin=b[:,i]-b[:,0]\n            mean=float(np.mean(new_margin));med=float(np.median(new_margin))\n            low=float(np.quantile(new_margin,.25));votes=float((new_margin>.020).mean())\n            old=float(np.mean(old_margin))\n            nz=float((neu[i]-neu[0])/ns)\n            if mean<.095 or med<.070 or low<-.010 or votes<.75 or old<.022 or nz<.18:continue\n            ev=evid.loc[head[i]];lv=evid.loc[leader]\n            if number(ev.get('neural_cos'),0.)<number(lv.get('neural_cos'),0.)+.002:continue\n            if number(ev.get('neural_fp'),0.)<number(lv.get('neural_fp'),0.)-.010:continue\n            votes_source,bad,sim=source_votes(ev,lv)\n            if votes_source<1 and sim<1:continue\n            if bad>=2 or sim<0:continue\n            confidence=mean+.045*nz+.020*sim+.010*votes_source+.018*max(0,low)\n            candidates.append((confidence,i,mean,nz))\n        if candidates:\n            _,idx,gain,nz=max(candidates)\n            updated.insert(0,updated.pop(idx))\n            record.update(action='provenance_spectral_top1',top1_changed=True,\n                          physical_gain=round(gain,5),neural_gain_sd=round(nz,4))\n    swaps=0\n    for pos in range(1,min(8,len(updated)-1)):\n        before=updated[pos];after=updated[pos+1]\n        if formula(before)!=formula(after):continue\n        i,j=head.index(before) if before in head else -1, head.index(after) if after in head else -1\n        if i<0 or j<0:continue\n        if trained[i] or trained[j]:continue\n        d=a[:,j]-a[:,i];do=b[:,j]-b[:,i]\n        if float(np.mean(d))<.068 or float(np.median(d))<.045 or float((d>.010).mean())<.67:continue\n        if float(np.mean(do))<.015:continue\n        if neu[j]-neu[i]<-.04*ns:continue\n        c=source_votes(evid.loc[after],evid.loc[before])[0]\n        if c<1 and neu[j]<=neu[i]:continue\n        updated[pos],updated[pos+1]=updated[pos+1],updated[pos]\n        swaps+=1\n        if swaps>=2:break\n    record['swaps']=swaps\n    if swaps and not record['top1_changed']:record['action']='paired_energy_swap'\n    # The private 0.475 model freezes known training candidates. This public analogue\n    # only allows permutations among untrained structures, preserving every training slot.\n    for i,s in enumerate(order):\n        if trained[i] and updated[i]!=s:\n            return order,dict(record,action='training_slot_guard',top1_changed=False,swaps=0)\n    return updated,record\n\ndef run(root,comp,time_budget=900):\n    root=Path(root);comp=Path(comp)\n    path=root/'submission.csv'\n    control=root/'submission_control_V2.csv'\n    for req in [path,control,root/'neural_candidate_features.parquet',root/'effective_query_spectra.parquet',comp/'train.parquet']:\n        if not req.exists():raise FileNotFoundError(req)\n    existing=pd.read_csv(path,dtype=str,keep_default_na=False)\n    first=pd.read_csv(control,dtype=str,keep_default_na=False)\n    assert existing.molecule_id.tolist()==first.molecule_id.tolist()\n    assert existing.molecule_id.is_unique\n    shutil.copy2(path,root/'submission_physical0472_control.csv')\n    nfeatures=pd.read_parquet(root/'neural_candidate_features.parquet')\n    spectra=pd.read_parquet(root/'effective_query_spectra.parquet')\n    groups={str(m):df for m,df in nfeatures.groupby('molecule_id',sort=False)}\n    specgroups={str(m):df for m,df in spectra.groupby('molecule_id',sort=False)}\n    known=member_keys(comp)\n    frag.warmup()\n    t0=time.monotonic();cache={};newrows=[];audits=[]\n    for idx,(record,ctrl) in enumerate(zip(existing.itertuples(index=False),first.itertuples(index=False))):\n        mid=str(record.molecule_id)\n        arr=str(record.smiles).split(';');original=str(ctrl.smiles).split(';')\n        if time.monotonic()-t0>time_budget:\n            new=arr;meta={'action':'runtime_budget','top1_changed':False,'swaps':0}\n        elif mid not in groups or mid not in specgroups:\n            new=arr;meta={'action':'no_source_spectra','top1_changed':False,'swaps':0}\n        else:\n            try:\n                new,meta=propose(arr,original,groups[mid],specgroups[mid],known,cache)\n            except Exception as e:\n                new=arr;meta={'action':'failed_'+type(e).__name__,'top1_changed':False,'swaps':0}\n        if set(new)!=set(arr) or len(new)!=len(arr):\n            raise AssertionError('Candidate identities may not change')\n        newrows.append((mid,';'.join(new)))\n        audits.append({'molecule_id':mid,**meta})\n        if len(cache)>25000:cache.clear()\n        if (idx+1)%50==0:\n            print('PROVENANCE',idx+1,'/',len(existing),'elapsed',round(time.monotonic()-t0,1),'s',flush=True)\n    result=pd.DataFrame(newrows,columns=['molecule_id','smiles'])\n    assert result.molecule_id.tolist()==existing.molecule_id.tolist()\n    assert result.molecule_id.is_unique\n    assert result.smiles.str.split(';').map(len).between(1,25).all()\n    assert result.smiles.str.split(';').map(lambda a:len(set(a))==len(a) and all(a)).all()\n    temp=root/'submission.provenance.tmp.csv'\n    result.to_csv(temp,index=False)\n    assert pd.read_csv(temp,dtype=str,keep_default_na=False).equals(result)\n    temp.replace(path)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'provenance_tournament_audit.csv',index=False)\n    summary={'source':'0.472 physical consensus public pipeline',\n             'private_rr3_weights_used':False,'train_structure_keys':len(known),\n             'rows':len(result),'changed':int((existing.smiles!=result.smiles).sum()),\n             'top1_changed':int(report.top1_changed.fillna(False).astype(bool).sum()),\n             'adjacent_swaps':int(report.swaps.fillna(0).sum()),\n             'elapsed_seconds':round(time.monotonic()-t0,2),\n             'control_hash':hashlib.sha256((root/'submission_physical0472_control.csv').read_bytes()).hexdigest(),\n             'final_hash':hashlib.sha256(path.read_bytes()).hexdigest(),\n             'note':'Leaderboard improvement not validated'}\n    (root/'provenance_tournament_report.json').write_text(json.dumps(summary,indent=2))\n    print('PROVENANCE_REPORT',json.dumps(summary),flush=True)\n    return summary\n"
PROVENANCE_CODE = ROOT/'parallax_public_provenance.py'
PROVENANCE_CODE.write_text(_PUBLIC_RERANKER)
assert hashlib.sha256(PROVENANCE_CODE.read_bytes()).hexdigest() == hashlib.sha256(_PUBLIC_RERANKER.encode()).hexdigest()
print('Public provenance-aware reranker:',PROVENANCE_CODE)

In [ ]:
import traceback
try:
    import importlib.util
    spec=importlib.util.spec_from_file_location('parallax_public_provenance',PROVENANCE_CODE)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    _left=max(0,int(8.65*3600-(time.time()-NB_T0)))
    if not PROVENANCE:
        PROVENANCE_REPORT={'status':'disabled in this experiment'}
    elif _left<240:
        PROVENANCE_REPORT={'status':'skipped_because_of_kaggle_runtime','seconds_remaining':_left}
    else:
        PROVENANCE_REPORT=module.run(ROOT,COMP,time_budget=min(960,_left-180))
except Exception as exc:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_physical0472_control.csv' if (ROOT/'submission_physical0472_control.csv').is_file() else ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PROVENANCE_REPORT={'status':'restored_control_after_error','exception':repr(exc)}

final=pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert list(final.columns)==['molecule_id','smiles']
assert final.molecule_id.tolist()==SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a))==len(a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','molecules',len(final),'provenance',PROVENANCE_REPORT)


In [ ]:
LIB_MAX = {}
# ── Variant M + final layer: rank 1 from the fragment check on V2, ranks 2-25 from CUSTOM + documentation prior,
# metric-key deduplication and outsider fill. Ported verbatim from baidalinadilzhan, casmi26 midx exact dedup fill
# (public LB 0.479): https://www.kaggle.com/code/baidalinadilzhan/casmi26-midx-exact-dedup-fill ─────────────
import os, json, shutil, hashlib, traceback, glob
from pathlib import Path
from functools import lru_cache
import pandas as pd
import numpy as np


def _load_pop_prior(root, input_root):
    """Load PubChem documentation prior and spectral annotations for InChIKey14 lookups."""
    pop_map, spec_map, np_map = {}, {}, {}
    try:
        hits = sorted(glob.glob(str(input_root / '**/pool_popularity.csv'), recursive=True))
        if hits:
            cols = ['inchikey14', 'pop', 'src_spectral_library', 'np_coconut', 'np_lotus', 'src_natural_product']
            p_df = pd.read_csv(hits[0], usecols=cols)
            pop_map = dict(zip(p_df['inchikey14'], p_df['pop']))
            spec_map = dict(zip(p_df['inchikey14'], p_df['src_spectral_library']))
            np_map = dict(zip(p_df['inchikey14'], (p_df['np_coconut'] | p_df['np_lotus'] | p_df['src_natural_product']).astype(int)))
            print(f'Documentation Prior: loaded {len(p_df)} pool records from {Path(hits[0]).name}')
    except Exception as e:
        print(f'Note: pool_popularity lookup exception ({e}); proceeding with neutral prior.')
    return pop_map, spec_map, np_map


def pxm_merge(physical, v2, custom, pop_map=None, spec_map=None, np_map=None, nf_evidence=None):
    """physical, v2, custom: molecule_id, smiles frames in the same row order. Returns (frame, counts)."""
    assert list(physical.columns) == list(v2.columns) == list(custom.columns) == ['molecule_id', 'smiles']
    assert physical.molecule_id.tolist() == v2.molecule_id.tolist() == custom.molecule_id.tolist(), 'row order differs'
    rows = []
    counts = dict(rows=0, merged=0, leader_moves=0, ranks_reordered=0, skipped={}, skipped_leader_moves=0, moved_ids=[])
    
    pop_map = pop_map or {}
    spec_map = spec_map or {}
    np_map = np_map or {}
    evidence_by_mid = nf_evidence or {}

    for mid, ps, vs, cs in zip(custom.molecule_id, physical.smiles, v2.smiles, custom.smiles):
        p, v, c = ps.split(';'), vs.split(';'), cs.split(';')
        counts['rows'] += 1
        
        # Variant M Leader Rule (Strictly Locked Rank 1):
        # If the fragment check on V2 identified a superior physical bond cleavage leader (p[0] != v[0]):
        # Place p[0] at Rank 1, followed by the high-resolution CUSTOM order.
        if p[0] != v[0]:
            leader = p[0]
            rest = [s for s in c if s != p[0]]
            counts['merged'] += 1
            counts['leader_moves'] += 1
            counts['moved_ids'].append(mid)
        else:
            leader = c[0]
            rest = c[1:]
            counts['merged'] += 1

        # Re-rank Ranks 2-25 by Documentation Prior + Spectral Library + Natural Product Bonus
        cinfo = evidence_by_mid.get(str(mid), {})
        lib_hit = float(LIB_MAX.get(str(mid), 0.0)) >= LIB_HIT_GATE
        if rest and cinfo:
            def _score_cand(s):
                if s in cinfo:
                    br, nr, k14 = cinfo[s]
                    p_val = float(pop_map.get(k14, 0.0))
                    s_val = float(spec_map.get(k14, 0.0))
                    np_val = float(np_map.get(k14, 0.0))
                    return 1.0 / (3.0 + br) + 0.75 / (3.0 + nr) + POP_W * (p_val / 20.0) + (SPEC_W if lib_hit else 0.0) * s_val + NP_W * np_val
                idx = c.index(s) if s in c else 25
                return 1.0 / (3.0 + idx + 1)
            
            old_rest = list(rest)
            rest.sort(key=_score_cand, reverse=True)
            if rest != old_rest:
                counts['ranks_reordered'] += 1
            
        if PIN_OLD_LEADER and p[0] != v[0] and v[0] in rest and rest[0] != v[0]:
            rest = [v[0]] + [x for x in rest if x != v[0]]
            counts['old_leader_pinned'] = counts.get('old_leader_pinned', 0) + 1
        rows.append((mid, [leader] + rest))
    return rows, counts


# ---- run (everything above is definitions) ----
def _pxm_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()


def _pxm_main():
    rep = dict(status='disabled')
    if not globals().get('FRAG_ON_V2', False):
        return rep
    assert not globals().get('PROVENANCE', False), 'variant M is defined without the provenance reranker'
    sub_path = ROOT / 'submission.csv'
    v_path, c_path = ROOT / 'variants/submission_V2.csv', ROOT / 'variants/submission_CUSTOM.csv'
    rep['input_sha256'] = dict(physical_v2=_pxm_sha(sub_path), v2=_pxm_sha(v_path), custom=_pxm_sha(c_path))
    diag = globals().get('PARALLAX_DIAGNOSTICS') or {}
    rep['overlay'] = {k: diag[k] for k in ('status', 'error', 'control_sha256', 'submission_sha256', 'changed_rows',
                                           'top1_changed', 'tail_changed', 'adjacent_swaps') if k in diag}
    if diag.get('control_sha256'):
        assert diag['control_sha256'] == rep['input_sha256']['v2'], 'the fragment check did not read the V2 file'
    if diag.get('submission_sha256'):
        assert diag['submission_sha256'] == rep['input_sha256']['physical_v2'], \
            'submission.csv is not the file the fragment check wrote'
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in (sub_path, v_path, c_path)]
    for f in frames:
        assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, \
            'an input file differs from sample_submission'

    from rdkit import Chem

    from rdkit.Chem.MolStandardize import rdMolStandardize
    _TAUT = rdMolStandardize.TautomerEnumerator()
    _IDMAP = identity_map()

    @lru_cache(maxsize=100000)
    def _to_k14(s):
        k = _IDMAP.get(s)
        if isinstance(k, str) and k:
            return k
        try:
            m = Chem.MolFromSmiles(str(s))
            if m is None:
                return ''
            Chem.RemoveStereochemistry(m)
            return Chem.MolToInchiKey(_TAUT.Canonicalize(m))[:14]
        except Exception:
            return ''

    global LIB_MAX
    LIB_MAX = {}
    out_by_mid = {}
    try:
        _nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet', columns=['molecule_id', 'library_max'])
        LIB_MAX = {str(m): float(v) for m, v in _nf.groupby('molecule_id', sort=False).library_max.max().items()}
        _op = ROOT / 'outsider_neural_features.parquet'
        if OUTSIDER_FILL and _op.is_file():
            _of = pd.read_parquet(_op)
            _sc = pd.to_numeric(_of.neural_score, errors='coerce')
            _of = _of[_of.neural_ok.astype(bool) & np.isfinite(_sc)].assign(_s=_sc)
            for _m, _g in _of.groupby('molecule_id', sort=False):
                out_by_mid[str(_m)] = _g.sort_values('_s', ascending=False, kind='stable').smiles.astype(str).tolist()
    except Exception as e:
        print(f'Note: final-layer inputs exception ({e}); gate off, no outsider fill')
    rep['lib_hit_molecules'] = int(sum(v >= LIB_HIT_GATE for v in LIB_MAX.values()))

    # Load documentation prior and candidate evidence
    pop_map, spec_map, np_map = _load_pop_prior(ROOT, INPUT)
    evidence_by_mid = {}
    nf_path = ROOT / 'neural_candidate_features.parquet'
    if nf_path.is_file():
        try:
            nf = pd.read_parquet(nf_path)
            for mid, g in nf.groupby('molecule_id', sort=False):
                order = np.argsort(-g['neural_score'].to_numpy(), kind='stable')
                nranks = np.empty(len(g), dtype=int)
                nranks[order] = np.arange(1, len(g) + 1)
                cand_info = {}
                for s, br, nr, mk in zip(g['smiles'], g['baseline_rank'], nranks, g['metric_key']):
                    k14 = mk if (isinstance(mk, str) and len(mk) == 14) else _to_k14(s)
                    cand_info[s] = (int(br), int(nr), k14)
                evidence_by_mid[str(mid)] = cand_info
        except Exception as e:
            print(f'Note: neural evidence indexing exception ({e})')

    merged_tuples, counts = pxm_merge(*frames, pop_map=pop_map, spec_map=spec_map, np_map=np_map, nf_evidence=evidence_by_mid)

    # In-Pool InChIKey14 Skeletal Deduplication
    reclaimed_smiles = []
    dedup_swaps = 0
    outsider_fills = 0
    for (mid, prim_list), c_s, v_s in zip(merged_tuples, frames[2].smiles, frames[1].smiles):
        seen_k = set()
        res = []
        # 1. Primary order (Variant M merged + DocPrior reordered ranks 2-25)
        for s in prim_list:
            if not s: continue
            k = _to_k14(s)
            if not k or k not in seen_k:
                if k: seen_k.add(k)
                res.append(s)
                if len(res) >= 25: break

        # 2. In-pool backfill strictly from c and v (no outside distractors)
        if len(res) < 25:
            for extra in (c_s, v_s):
                for s in extra.split(';'):
                    if not s or s in res: continue
                    k = _to_k14(s)
                    if not k or k not in seen_k:
                        if k: seen_k.add(k)
                        res.append(s)
                        dedup_swaps += 1
                        if len(res) >= 25: break
                if len(res) >= 25: break

        # 2b. Empty slots: scored outside candidates with a new metric identity, best neural score first
        if len(res) < 25:
            for s in out_by_mid.get(str(mid), []):
                if s in res:
                    continue
                k = _to_k14(s)
                if not k or k in seen_k:
                    continue
                seen_k.add(k)
                res.append(s)
                outsider_fills += 1
                if len(res) >= 25:
                    break

        # 3. Fallback fill if still < 25
        if len(res) < 25:
            for extra in (prim_list, c_s.split(';'), v_s.split(';')):
                for s in extra:
                    if s and s not in res:
                        res.append(s)
                        if len(res) >= 25: break
                if len(res) >= 25: break

        reclaimed_smiles.append(';'.join(res[:25]))

    new = pd.DataFrame({'molecule_id': [m for m, _ in merged_tuples], 'smiles': reclaimed_smiles})
    rep['dedup_swaps'] = dedup_swaps
    rep['outsider_fills'] = outsider_fills
    rep['old_leader_pinned'] = counts.get('old_leader_pinned', 0)
    print(f'Final layer: outsider fills {outsider_fills}, old leader pinned {rep["old_leader_pinned"]}, '
          f'library-hit molecules {rep.get("lib_hit_molecules")}, POP_W {POP_W}, NP_W {NP_W}')
    rep['ranks_reordered'] = counts['ranks_reordered']
    print(f'InChIKey14 Deduplication: reclaimed {dedup_swaps} redundant isomer slots!')
    print(f'Documentation Prior: reordered secondary ranks in {counts["ranks_reordered"]} / {counts["rows"]} rows.')

    assert new.molecule_id.tolist() == SAMPLE_IDS
    assert new.smiles.str.split(';').map(len).between(1, 25).all()
    assert new.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
    tmp = ROOT / 'submission.merge.tmp'
    new.to_csv(tmp, index=False)
    out_sha = _pxm_sha(tmp)
    os.replace(tmp, sub_path)
    rep.update(status='applied', **counts, output_sha256=out_sha,
               output_equals_custom=out_sha == rep['input_sha256']['custom'])
    return rep


try:
    if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
        MERGE_REPORT = {'status': 'skipped_fast_commit'}
    else:
        MERGE_REPORT = _pxm_main()
except Exception as exc:
    traceback.print_exc()
    MERGE_REPORT = dict(status='restored_custom_after_error', exception=repr(exc)[:600])
    try:
        _tmp = ROOT / 'submission.merge.tmp'
        shutil.copyfile(ROOT / 'variants/submission_CUSTOM.csv', _tmp)
        os.replace(_tmp, ROOT / 'submission.csv')
        MERGE_REPORT['output_sha256'] = _pxm_sha(ROOT / 'submission.csv')
    except Exception as exc2:
        traceback.print_exc()
        MERGE_REPORT['restore_error'] = repr(exc2)[:300]
try:
    _tmp = ROOT / 'merge_report.json.tmp'
    _tmp.write_text(json.dumps(MERGE_REPORT, indent=2, default=str))
    os.replace(_tmp, ROOT / 'merge_report.json')
except Exception:
    traceback.print_exc()

final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), 'MERGE', json.dumps(
    {k: MERGE_REPORT.get(k) for k in ('status', 'rows', 'merged', 'leader_moves', 'ranks_reordered', 'dedup_swaps',
                                      'output_equals_custom', 'outsider_fills', 'old_leader_pinned', 'lib_hit_molecules', 'output_sha256', 'exception') if k in MERGE_REPORT},
    default=str))


In [ ]:
# ── Class-3 analog insertion (ours): new structures for molecules the pipeline cannot confirm ───────────────────
# Sources: class3/ in our repository (library, transforms, neural, candidates, stage) and the LambdaRank ranker trained
# by class3/ranker.py. The neural scorer is our re-implementation of Saner Turhaner's casmi_spectrum_model_v1
# (Apache-2.0); only its config and weights are read from the attached dataset. Any error restores the input file.
import sys, json, time, shutil, traceback, glob
from pathlib import Path
C3_FILES = {'library.py': '"""Compact structure-level spectral library for analog (neighbour) search.\n\nUp to MAX_PER spectra per (inchikey14, ionisation mode) with a test adduct, from every source library except the\nsynthetic enveda-180. Each spectrum becomes two sparse unit vectors over 0.05 Da bins: fragment m/z and neutral loss\n(neutral precursor mass - fragment m/z). Cached to data/cache/class3_library.npz + .parquet.\n\n    .venv/bin/python -I class3/library.py --data data/train.parquet\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\nimport scipy.sparse as sp\n\nADDUCTS = {  # adduct -> (mass added to M per charge, multiplier of M); charge is always 1\n    \'[M+H]+\': (1.007276, 1), \'[M+NH4]+\': (18.033823, 1), \'[M-H2O+H]+\': (-17.003289, 1),\n    \'[M-2H2O+H]+\': (-35.013853, 1), \'[M+Na]+\': (22.989218, 1), \'[M+K]+\': (38.963158, 1),\n    \'[M-H]-\': (-1.007276, 1), \'[M-H2O-H]-\': (-19.017841, 1), \'[M+CH2O2-H]-\': (44.998201, 1),\n    \'[M+Cl]-\': (34.969402, 1),\n}\nBIN = 0.05\nMAX_MZ = 2000.0\nNBINS = int(MAX_MZ / BIN) + 1\nTOP_PEAKS = 60\nMAX_PER = 3\nEXCLUDE_LIBS = (\'enveda-180\',)\n\n\ndef neutral_mass(prec, adduct):\n    add, mult = ADDUCTS[adduct]\n    return (prec - add) / mult\n\n\ndef spectrum_vectors(mz, it, prec, adduct):\n    """(frag_bins, frag_w, nl_bins, nl_w) for one spectrum: top peaks, sqrt intensity, unit norm, summed per bin."""\n    mz = np.asarray(mz, float)\n    it = np.asarray(it, float)\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz < prec - 0.5)  # drop the precursor region\n    mz, it = mz[ok], it[ok]\n    if len(mz) > TOP_PEAKS:\n        keep = np.argsort(it)[-TOP_PEAKS:]\n        mz, it = mz[keep], it[keep]\n    w = np.sqrt(it)\n    m = neutral_mass(prec, adduct)\n    out = []\n    for vals in (mz, m - mz):\n        b = np.round(vals / BIN).astype(np.int64)\n        good = (b > 0) & (b < NBINS)\n        bb, ww = np.unique(b[good], return_inverse=True)\n        acc = np.zeros(len(bb))\n        np.add.at(acc, ww, w[good])\n        n = np.linalg.norm(acc)\n        out += [bb, acc / n if n > 0 else acc]\n    return out\n\n\ndef to_csr(rows):\n    indptr = np.cumsum([0] + [len(b) for b, _ in rows])\n    idx = np.concatenate([b for b, _ in rows]) if rows else np.zeros(0, np.int64)\n    val = np.concatenate([w for _, w in rows]) if rows else np.zeros(0)\n    return sp.csr_matrix((val, idx, indptr), shape=(len(rows), NBINS))\n\n\ndef build(data, out_dir):\n    t0 = time.time()\n    pf = pq.ParquetFile(data)\n    light_cols = [\'ingest_lib\', \'inchikey14\', \'normalized_smiles\', \'ionization_mode\', \'adduct\', \'num_peaks\']\n    light = pf.read(columns=light_cols).to_pandas()\n    light[\'row\'] = np.arange(len(light))\n    sel = light[~light.ingest_lib.isin(EXCLUDE_LIBS) & light.adduct.isin(list(ADDUCTS)) & (light.num_peaks >= 5)]\n    # prefer the commonest adducts and the richest spectra, MAX_PER per structure and mode\n    pref = {\'[M+H]+\': 0, \'[M-H]-\': 0}\n    sel = sel.assign(p=sel.adduct.map(lambda a: pref.get(a, 1)))\n    sel = sel.sort_values([\'inchikey14\', \'ionization_mode\', \'p\', \'num_peaks\'], ascending=[True, True, True, False])\n    sel = sel.groupby([\'inchikey14\', \'ionization_mode\'], sort=False).head(MAX_PER)\n    wanted = set(sel.row.tolist())\n    print(f\'selected {len(sel):,} spectra of {sel.inchikey14.nunique():,} structures ({time.time() - t0:.0f}s)\', flush=True)\n\n    meta, frag, nl = [], [], []\n    start = 0\n    for g in range(pf.num_row_groups):\n        n = pf.metadata.row_group(g).num_rows\n        idx = [r - start for r in range(start, start + n) if r in wanted]\n        if idx:\n            tab = pf.read_row_group(g, columns=[\'precursor_mz\', \'adduct\', \'ms2_mzs\', \'ms2_normalized_intensities\'])\n            tab = tab.take(idx).to_pandas()\n            for r, row in zip(idx, tab.itertuples(index=False)):\n                fb, fw, nb, nw = spectrum_vectors(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz, row.adduct)\n                if len(fb) < 3:\n                    continue\n                frag.append((fb, fw))\n                nl.append((nb, nw))\n                meta.append(start + r)\n        start += n\n        print(f\'row group {g + 1}/{pf.num_row_groups}: {len(meta):,} spectra ({time.time() - t0:.0f}s)\', flush=True)\n\n    m = light.set_index(\'row\').loc[meta, [\'inchikey14\', \'normalized_smiles\', \'ionization_mode\', \'adduct\', \'ingest_lib\']]\n    m = m.reset_index()\n    out_dir.mkdir(parents=True, exist_ok=True)\n    m.to_parquet(out_dir / \'class3_library.parquet\')\n    sp.save_npz(out_dir / \'class3_frag.npz\', to_csr(frag))\n    sp.save_npz(out_dir / \'class3_nl.npz\', to_csr(nl))\n    print(f\'library: {len(m):,} spectra, {m.inchikey14.nunique():,} structures ({time.time() - t0:.0f}s)\')\n\n\ndef load(out_dir):\n    m = pd.read_parquet(out_dir / \'class3_library.parquet\')\n    return m, sp.load_npz(out_dir / \'class3_frag.npz\'), sp.load_npz(out_dir / \'class3_nl.npz\')\n\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--data\', default=\'data/train.parquet\')\n    ap.add_argument(\'--out\', default=\'data/cache\')\n    a = ap.parse_args()\n    build(a.data, Path(a.out))\n', 'transforms.py': '"""Mass-delta-driven biotransformations: turn a known neighbour structure into candidates of the query\'s mass.\n\nEach rule is a reaction SMARTS with its monoisotopic mass change. For a neighbour of neutral mass m and a query of\nneutral mass M, the rules (and pairs of rules) whose summed delta matches M - m within tolerance are applied at every\nsite; products are sanitised and de-duplicated.\n\n    generate(neighbour_smiles, query_mass, ppm=10, abs_tol=0.003, max_steps=2) -> list[str]\n"""\nfrom __future__ import annotations\n\nfrom functools import lru_cache\nfrom itertools import combinations_with_replacement\n\nfrom rdkit import Chem, RDLogger\nfrom rdkit.Chem import AllChem, Descriptors\n\nRDLogger.DisableLog(\'rdApp.*\')\n\nHEX = \'C1OC(CO)C(O)C(O)C1O\'          # hexose (as an O-substituent)\nPEN = \'C1OCC(O)C(O)C1O\'              # pentose\nDHX = \'C1OC(C)C(O)C(O)C1O\'           # 6-deoxyhexose (rhamnose-like)\nGLA = \'C1OC(C(=O)O)C(O)C(O)C1O\'      # glucuronic acid\n\n# (name, reaction SMARTS); the delta is computed from the SMARTS products below\nRULES = [\n    (\'hydroxylation_ar\', \'[cH1:1]>>[c:1]O\'),\n    (\'hydroxylation_al\', \'[CX4;H1,H2;!$(C[OX2]);!$(C[NX3]):1]>>[C:1]O\'),\n    (\'dehydroxylation\', \'[CX4:1][OH1]>>[C:1]\'),\n    (\'o_methylation\', \'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]C\'),\n    (\'n_methylation\', \'[NX3;H1,H2;!$(NC=O);!$(N-[a]):1]>>[N:1]C\'),\n    (\'o_demethylation\', \'[OX2:1][CH3]>>[O:1]\'),\n    (\'n_demethylation\', \'[NX3:1][CH3]>>[N:1]\'),\n    (\'methoxylation_ar\', \'[cH1:1]>>[c:1]OC\'),\n    (\'hydrogenation\', \'[C:1]=[C:2]>>[C:1][C:2]\'),\n    (\'dehydrogenation\', \'[CX4;H1,H2;!R:1]-[CX4;H1,H2;!R:2]>>[C:1]=[C:2]\'),\n    (\'dehydrogenation_ring\', \'[CX4;H1,H2;R:1]-[CX4;H1,H2;R:2]>>[C:1]=[C:2]\'),\n    (\'ketone_from_alcohol\', \'[CX4;H1:1][OH1:2]>>[C:1]=[O:2]\'),\n    (\'alcohol_from_ketone\', \'[CX3;!$(C[O,N;!R]);$(C([#6])[#6]):1]=[O:2]>>[C:1][O:2]\'),\n    (\'hexosylation\', f\'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]{HEX}\'),\n    (\'pentosylation\', f\'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]{PEN}\'),\n    (\'deoxyhexosylation\', f\'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]{DHX}\'),\n    (\'glucuronidation\', f\'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]{GLA}\'),\n    (\'dehexosylation\', \'[#6:2][OX2:1][CH1]1O[CH1]([CH2][OH1])[CH1]([OH1])[CH1]([OH1])[CH1]1[OH1]>>[#6:2][O:1]\'),\n    (\'depentosylation\', \'[#6:2][OX2:1][CH1]1O[CH2][CH1]([OH1])[CH1]([OH1])[CH1]1[OH1]>>[#6:2][O:1]\'),\n    (\'dedeoxyhexosylation\', \'[#6:2][OX2:1][CH1]1O[CH1]([CH3])[CH1]([OH1])[CH1]([OH1])[CH1]1[OH1]>>[#6:2][O:1]\'),\n    (\'acetylation\', \'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]C(C)=O\'),\n    (\'deacetylation\', \'[OX2:1]C(=O)[CH3]>>[O:1]\'),\n    (\'malonylation\', \'[OX2H1;$(O[CX4]):1]>>[O:1]C(=O)CC(=O)O\'),\n    (\'sulfation\', \'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]S(=O)(=O)O\'),\n    (\'prenylation_ar\', \'[cH1:1]>>[c:1]CC=C(C)C\'),\n    (\'decarboxylation\', \'[#6:1]C(=O)[OH1]>>[#6:1]\'),\n    (\'carboxylic_from_methyl\', \'[CH3;$(C[#6]):1]>>[C:1](=O)O\'),\n    (\'hydration\', \'[C:1]=[C:2]>>[C:1][C:2]O\'),\n    (\'dehydration\', \'[CX4;H1,H2:1][CX4:2][OH1]>>[C:1]=[C:2]\'),\n    # carbon skeleton: homologues (the commonest unmatched neighbour deltas on held-out GNPS: +-14, 28, 42, 56)\n    (\'c_methylation_al\', \'[CX4;H2,H3;!$(C[O,N,S]):1]>>[C:1]C\'),\n    (\'c_methylation_ar\', \'[cH1:1]>>[c:1]C\'),\n    (\'c_demethylation\', \'[#6:1][CH3]>>[#6:1]\'),\n    (\'ethylation\', \'[CH3;$(C[CX4]):1]>>[C:1]CC\'),\n    (\'propylation\', \'[CH3;$(C[CX4]):1]>>[C:1]CCC\'),\n    # heteroatom swaps and small acyl groups\n    (\'amine_to_hydroxyl\', \'[CX4,c:2][NH2]>>[*:2]O\'),\n    (\'hydroxyl_to_amine\', \'[CX4,c:2][OH1]>>[*:2]N\'),\n    (\'amide_to_acid\', \'[C:2](=O)[NH2]>>[C:2](=O)O\'),\n    (\'acid_to_amide\', \'[C:2](=O)[OH1]>>[C:2](=O)N\'),\n    (\'formylation\', \'[NX3;H1,H2;!$(NC=O):1]>>[N:1]C=O\'),\n    (\'carbamoylation\', \'[OX2H1;$(O[#6]);!$(OC=O):1]>>[O:1]C(N)=O\'),\n    (\'aza_ar\', \'[cH1;r6:1]>>[n:1]\'),\n]\n\n\ndef _delta(rule):\n    """Mass change of a rule, measured on a probe molecule it applies to."""\n    probes = [\'OCC(O)Cc1ccccc1C\', \'COc1ccc(CC=CCC(=O)C)cc1O\', \'CN(C)CCc1ccc(OC(C)=O)cc1\',\n              \'OC1C(O)C(O)C(CO)OC1Oc1ccccc1\', \'OC1C(O)C(O)COC1Oc1ccccc1\', \'OC1C(O)C(O)C(C)OC1Oc1ccccc1\',\n              \'OC(=O)CCc1ccccc1\', \'CNCCc1ccccc1\', \'OCCC1CCCCC1C(C)=O\', \'NCCc1ccccc1\', \'NC(=O)CCc1ccccc1\']\n    rxn = AllChem.ReactionFromSmarts(rule)\n    for p in probes:\n        m = Chem.MolFromSmiles(p)\n        for prods in rxn.RunReactants((m,)):\n            q = prods[0]\n            try:\n                Chem.SanitizeMol(q)\n            except Exception:\n                continue\n            return Descriptors.ExactMolWt(q) - Descriptors.ExactMolWt(m)\n    raise ValueError(f\'no probe for {rule}\')\n\n\n@lru_cache(maxsize=1)\ndef compiled():\n    return [(name, AllChem.ReactionFromSmarts(s), _delta(s)) for name, s in RULES]\n\n\ndef plans(delta, tol, max_steps=2):\n    """Rule-index tuples whose summed mass change matches `delta` within `tol`."""\n    rules = compiled()\n    out = []\n    for k in range(1, max_steps + 1):\n        for combo in combinations_with_replacement(range(len(rules)), k):\n            if abs(sum(rules[i][2] for i in combo) - delta) <= tol:\n                out.append(combo)\n    return out\n\n\ndef _apply(rxn, smiles_list, cap):\n    """Products of `rxn` on each input, in a deterministic order (the cap must not depend on hash seeds)."""\n    out = {}\n    for smi in smiles_list:\n        m = Chem.MolFromSmiles(smi)\n        if m is None:\n            continue\n        for prods in rxn.RunReactants((m,)):\n            q = prods[0]\n            try:\n                Chem.SanitizeMol(q)\n                s = Chem.MolToSmiles(q)\n            except Exception:\n                continue\n            if \'.\' not in s:\n                out[s] = None\n            if len(out) >= cap:\n                return list(out)\n    return list(out)\n\n\ndef generate_plans(neighbour, query_mass, ppm=10.0, abs_tol=0.003, max_steps=2, cap=400):\n    """{candidate smiles: rule-name tuple of the first plan that produced it} for 1..max_steps rules."""\n    m = Chem.MolFromSmiles(neighbour)\n    if m is None:\n        return {}\n    base = Descriptors.ExactMolWt(m)\n    tol = max(abs_tol, ppm * 1e-6 * query_mass)\n    rules = compiled()\n    root = Chem.MolToSmiles(m)\n    out = {}\n    for combo in plans(query_mass - base, tol, max_steps):\n        cur = [root]\n        for i in combo:\n            cur = _apply(rules[i][1], cur, cap)\n            if not cur:\n                break\n        for s in cur:\n            out.setdefault(s, tuple(rules[i][0] for i in combo))\n        if len(out) >= cap:\n            break\n    out.pop(root, None)\n    return out\n\n\ndef generate(neighbour, query_mass, ppm=10.0, abs_tol=0.003, max_steps=2, cap=400):\n    """Candidates of mass `query_mass` reachable from `neighbour` by 1..max_steps rules (neighbour itself excluded)."""\n    return sorted(generate_plans(neighbour, query_mass, ppm, abs_tol, max_steps, cap))\n', 'neural.py': '"""Spectrum-candidate scorer compatible with Saner Turhaner\'s casmi_spectrum_model_v1 checkpoints (Apache-2.0).\n\nOur own implementation of the architecture and features described by that bundle, so that only data is read from\nit: config.json and a weights-only checkpoint (torch.load(..., weights_only=True)). No bundle code is executed.\n\n    Scorer(bundle_dir, checkpoint).score_candidates(spectra, smiles) -> np.ndarray (larger = better, -inf invalid)\n\nspectra: dicts with mzs, intensities, precursor_mz, adduct, ionization_mode, collision_energy.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nADDUCTS = [\'[M+H]+\', \'[M-H]-\', \'[M+Na]+\', \'[M+K]+\', \'[M+NH4]+\', \'[M+Cl]-\', \'[M+FA-H]-\', \'[M+H-H2O]+\', \'[M+2H]2+\',\n           \'[M-2H]2-\']\n\n\nclass SpectrumCandidateModel(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg[\'width\']\n        self.cfg = cfg\n        self.register_buffer(\'freq\', 2. ** torch.arange(10).float())\n        self.peak = nn.Linear(44, d)\n        self.meta = nn.Sequential(nn.Linear(4, d), nn.GELU(), nn.Linear(d, d))\n        self.adduct = nn.Embedding(11, d)\n        self.cls = nn.Parameter(torch.zeros(1, 1, d))\n        layer = nn.TransformerEncoderLayer(d, cfg[\'heads\'], d * 4, cfg[\'dropout\'], batch_first=True, norm_first=True)\n        self.transformer = nn.TransformerEncoder(layer, cfg[\'layers\'], enable_nested_tensor=False)\n        self.norm = nn.LayerNorm(d)\n        self.spec_projection = nn.Linear(d, cfg[\'embedding\'])\n        self.fp_head = nn.Linear(d, cfg[\'fp_bits\'])\n        self.molecule = nn.Sequential(nn.Linear(cfg[\'fp_bits\'], 512), nn.GELU(), nn.Dropout(cfg[\'dropout\']),\n                                      nn.Linear(512, cfg[\'embedding\']))\n\n    def encode_spectra(self, x, lengths, meta, adduct):\n        a = x[:, :, [0, 3]].unsqueeze(-1) * self.freq * torch.pi\n        p = torch.cat([x, a.sin().flatten(2), a.cos().flatten(2)], -1)\n        tokens = self.peak(p)\n        cls = self.cls.expand(len(x), -1, -1) + self.meta(meta)[:, None] + self.adduct(adduct)[:, None]\n        tokens = torch.cat([cls, tokens], 1)\n        mask = torch.arange(x.shape[1] + 1, device=x.device)[None] > lengths[:, None]\n        h = self.norm(self.transformer(tokens, src_key_padding_mask=mask)[:, 0])\n        return F.normalize(self.spec_projection(h), dim=-1), self.fp_head(h)\n\n    def encode_molecules(self, fp):\n        return F.normalize(self.molecule(fp), dim=-1)\n\n\ndef peaks(mz, it, precursor, max_peaks=96):\n    mz, it = np.asarray(mz, np.float32).ravel(), np.asarray(it, np.float32).ravel()\n    keep = np.isfinite(mz) & np.isfinite(it) & (mz > 0) & (it > 0)\n    mz, it = mz[keep], it[keep]\n    if not len(mz):\n        raise ValueError(\'No valid peaks\')\n    ix = np.argsort(-it, kind=\'stable\')[:max_peaks]\n    ix = ix[np.argsort(mz[ix], kind=\'stable\')]\n    mz, it = mz[ix], it[ix]\n    it = it / it.max()\n    out = np.zeros((max_peaks, 4), np.float32)\n    out[:len(mz)] = np.stack([mz / 1000, np.sqrt(it), np.log1p(100 * it) / np.log(101), (float(precursor) - mz) / 1000], 1)\n    return out, len(mz)\n\n\ndef metadata(precursor, adduct, mode, ce):\n    try:\n        a = np.asarray(ce if ce is not None else [], np.float32).ravel()\n        a = a[np.isfinite(a)]\n        ev, known = (float(a.mean()), 1.) if len(a) else (0., 0.)\n    except (ValueError, TypeError):\n        ev, known = 0., 0.\n    pol = 1. if str(mode).lower() == \'positive\' else -1.\n    return (np.array([float(precursor) / 1000, np.clip(ev, 0, 500) / 100, known, pol], np.float32),\n            ADDUCTS.index(adduct) + 1 if adduct in ADDUCTS else 0)\n\n\ndef fingerprint(smiles, nbits):\n    from rdkit import Chem\n    from rdkit.Chem import rdFingerprintGenerator\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    mol = Chem.MolFromSmiles(str(smiles))\n    if mol is None:\n        return None\n    Chem.RemoveStereochemistry(mol)\n    mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)\n    return rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=nbits).GetFingerprintAsNumPy(mol)\n\n\nclass Scorer:\n    def __init__(self, bundle, checkpoint=\'final_model.pt\'):\n        root = Path(bundle)\n        self.cfg = json.loads((root / \'config.json\').read_text())\n        self.model = SpectrumCandidateModel(self.cfg)\n        obj = torch.load(root / checkpoint, map_location=\'cpu\', weights_only=True)\n        self.model.load_state_dict(obj[\'state_dict\'])\n        self.model.eval()\n        self.cache = {}\n\n    @torch.inference_mode()\n    def score_candidates(self, spectra, candidates):\n        rec = []\n        for s in spectra:\n            p, n = peaks(s[\'mzs\'], s[\'intensities\'], s[\'precursor_mz\'], self.cfg[\'max_peaks\'])\n            m, a = metadata(s[\'precursor_mz\'], s.get(\'adduct\', \'\'), s.get(\'ionization_mode\', \'\'), s.get(\'collision_energy\'))\n            rec.append((p, n, m, a))\n        t = [torch.as_tensor(np.array([r[j] for r in rec])) for j in range(4)]\n        z, p = self.model.encode_spectra(t[0].float(), t[1].long(), t[2].float(), t[3].long())\n        z, p = z.mean(0), p.mean(0)  # equal to the averaged per-spectrum score\n        out = np.full(len(candidates), -np.inf, np.float32)\n        valid, fps = [], []\n        for i, s in enumerate(candidates):\n            if s not in self.cache:\n                self.cache[s] = fingerprint(s, self.cfg[\'fp_bits\'])\n            if self.cache[s] is not None:\n                valid.append(i)\n                fps.append(self.cache[s])\n        if valid:\n            fp = torch.as_tensor(np.array(fps), dtype=torch.float32)\n            score = (self.model.encode_molecules(fp) @ z) / self.cfg[\'temperature\'] \\\n                + self.cfg[\'fp_score_weight\'] * (fp @ p) / self.cfg[\'fp_bits\']\n            out[valid] = score.numpy()\n        return out\n', 'candidates.py': '"""Shared class-3 candidate machinery for the bench and the Kaggle notebook stage.\n\nneighbour_scores: spectral similarity of a query to every library structure.\nbuild: generate candidates from the query\'s neighbours, score them (fragment scorer, neural scorer) and compute the\nranker features. The bench and the notebook call exactly this code, so their features match.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem, DataStructs, RDLogger\nfrom rdkit.Chem import rdFingerprintGenerator\n\nimport library as L\nimport transforms as T\n\nRDLogger.DisableLog(\'rdApp.*\')\nCAP_PER_NEIGHBOUR = 150\nFPG = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\n# our adduct names -> the neural model\'s vocabulary (sanpier casmi_spectrum_model_v1 features.ADDUCTS)\nNEURAL_ADDUCT = {\'[M+CH2O2-H]-\': \'[M+FA-H]-\', \'[M-H2O+H]+\': \'[M+H-H2O]+\'}\n\n\ndef k14(smi):\n    m = Chem.MolFromSmiles(smi) if smi else None\n    return Chem.MolToInchiKey(m)[:14] if m is not None else None\n\n\ndef zscore(x):\n    x = np.asarray(x, float)\n    sd = np.nanstd(x)\n    return np.nan_to_num((x - np.nanmean(x)) / sd if sd > 0 else np.zeros_like(x), nan=-3.0)\n\n\ndef neighbour_scores(q, lib_meta, frag, nl):\n    """Best combined cosine of each library structure to any of the query\'s spectra (same polarity only)."""\n    best = np.zeros(len(lib_meta))\n    pos = (lib_meta.ionization_mode == \'positive\').to_numpy()\n    for s in q[\'spectra\']:\n        fb, fw, nb, nw = L.spectrum_vectors(s[\'mz\'], s[\'it\'], s[\'prec\'], s[\'adduct\'])\n        if len(fb) < 3:\n            continue\n        qf = np.zeros(L.NBINS); qf[fb] = fw\n        qn = np.zeros(L.NBINS); qn[nb] = nw\n        sim = 0.5 * (frag @ qf + nl @ qn)\n        sim[pos != (s[\'mode\'] > 0)] = 0.0\n        best = np.maximum(best, sim)\n    return pd.Series(best).groupby(lib_meta.inchikey14.to_numpy()).max()\n\n\ndef neural_scores(scorer, q, smiles):\n    if scorer is None:\n        return np.full(len(smiles), np.nan)\n    spectra = [dict(mzs=s[\'mz\'], intensities=s[\'it\'], precursor_mz=s[\'prec\'],\n                    adduct=NEURAL_ADDUCT.get(s[\'adduct\'], s[\'adduct\']),\n                    ionization_mode=\'positive\' if s[\'mode\'] > 0 else \'negative\', collision_energy=s.get(\'ce\'))\n               for s in q[\'spectra\']]\n    out = scorer.score_candidates(spectra, smiles).astype(float)\n    out[~np.isfinite(out)] = np.nan\n    return out\n\n\ndef build(q, neighbours, steps, scorer, qid=\'\', truth=None):\n    """Candidates generated from `neighbours` [(smiles, similarity), ...] best first.\n\n    Returns (keys, raw, feats): candidate InChIKey14 blocks; raw per-candidate arrays (frag, neural, parent_sim,\n    parent_rank, z_frag, z_neural, z_parent); one feature dict per candidate (label = key == truth).\n    """\n    import frag_rescore as frag\n    cands = {}    # key -> (smiles, parent rank, parent similarity, rule names, parent smiles)\n    support = {}  # key -> number of neighbours that generate it\n    for rank, (smi, sim) in enumerate(neighbours, 1):\n        for c, plan in T.generate_plans(smi, q[\'mass\'], max_steps=steps, cap=CAP_PER_NEIGHBOUR).items():\n            key = k14(c)\n            if not key:\n                continue\n            support[key] = support.get(key, 0) + 1\n            if key not in cands:\n                cands[key] = (c, rank, sim, plan, smi)\n    if not cands:\n        return [], {}, []\n    keys = list(cands)\n    smiles = [cands[k][0] for k in keys]\n    fs = np.array([np.nan if v is None else v for v in frag.score_molecule(q[\'spectra\'], smiles, cache={})])\n    fs = np.nan_to_num(fs, nan=-1.0)\n    psim = np.array([cands[k][2] for k in keys])\n    prank = np.array([cands[k][1] for k in keys])\n    ns = neural_scores(scorer, q, smiles)\n    zf, zn, zp = zscore(fs), zscore(ns), zscore(psim)\n    raw = dict(frag=fs, neural=ns, parent_sim=psim, parent_rank=prank, z_frag=zf, z_neural=zn, z_parent=zp)\n\n    rank_of = lambda v: (-np.nan_to_num(v, nan=-1e9)).argsort().argsort() + 1\n    rf, rn = rank_of(fs), rank_of(ns)\n    feats = []\n    for j, k in enumerate(keys):\n        smi, _, _, plan, parent = cands[k]\n        m, pm = Chem.MolFromSmiles(smi), Chem.MolFromSmiles(parent)\n        tan = DataStructs.TanimotoSimilarity(FPG.GetFingerprint(m), FPG.GetFingerprint(pm)) if m and pm else np.nan\n        feats.append(dict(\n            qid=qid, cand=k, label=int(k == truth), smiles=smi,\n            frag=fs[j], neural=ns[j], parent_sim=psim[j], parent_rank=int(prank[j]), z_frag=zf[j], z_neural=zn[j],\n            z_parent=zp[j], rank_frag=int(rf[j]), rank_neural=int(rn[j]), support=support[k], n_steps=len(plan),\n            rule1=plan[0], rule2=plan[1] if len(plan) > 1 else \'\', tan_parent=tan,\n            heavy=m.GetNumHeavyAtoms() if m else 0, heavy_parent=pm.GetNumHeavyAtoms() if pm else 0,\n            n_cands=len(keys)))\n    return keys, raw, feats\n', 'stage.py': '"""Class-3 insertion stage for the Kaggle notebook (runs after the final merge layer).\n\nFor each molecule whose best library match is below `gate` (the pipeline\'s leader is not library-confirmed), find\nits spectral neighbours in a library built from train.parquet, generate analogs of the query mass, rank them with\nthe learned LambdaRank model and insert the best `n_insert` new structures at rank `position` (1-based); ranks\nabove it are kept, the rest shift down and the list is cut at 25. When the ranker\'s top-1 margin over the runner-up\nis at least `confident_margin`, the top pick goes to `confident_position` instead (eval: P(top1) ~0.40 at margin 0.8).\n\n`rerank` (\'all\' | \'freeze1\' | None): before the insertion, the same gated lists are reordered by the isomer reranker\n(rerank/; trained on held-out molecules whose spectra were removed from the library); \'freeze1\' keeps the leader.\n\n`variants` {name: overrides of gate/position/n_insert/confident_*} also writes submission_<name>.csv from the same\ncandidates, so one notebook run yields several LB probes.\n\n    run(root, comp, bundle, lib_max, cfg) -> report dict   (submission.csv in `root` is rewritten in place)\n"""\nfrom __future__ import annotations\n\nimport json\nimport multiprocessing as mp\nimport os\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport candidates as C\nimport features as RF\nimport library as L\n\nDEFAULTS = dict(gate=0.87, position=3, n_insert=3, confident_margin=None, confident_position=1, k=30, steps=2,\n                max_spectra=6, seconds=2400, workers=0, checkpoint=\'final_model.pt\', limit=0, variants={},\n                rerank=None)\nPLACEMENT = (\'gate\', \'position\', \'n_insert\', \'confident_margin\', \'confident_position\')\n_SCORER = None\n\n\ndef _init(bundle, checkpoint):\n    global _SCORER\n    import torch\n    torch.set_num_threads(1)\n    from neural import Scorer\n    _SCORER = Scorer(bundle, checkpoint=checkpoint)\n\n\ndef _one(args):\n    mid, q, neighbours, steps, rr = args\n    keys, _, feats = C.build(q, neighbours, steps, _SCORER, qid=mid)\n    rows = []\n    if rr is not None:\n        smiles, rr_neighbours = rr\n        rows = RF.features(q, [C.k14(s) for s in smiles], smiles, {1.0: rr_neighbours}, _SCORER,\n                           short=len(smiles), order=[C.k14(s) for s in smiles])[0][1.0]\n    return mid, feats, rows\n\n\ndef rerank_list(old, rows, model, meta, mode):\n    """`old` reordered by the reranker score; candidates without a valid structure stay at the end."""\n    if not rows:\n        return old\n    s = model.predict(pd.DataFrame(rows)[meta[\'numeric\']])\n    ranked = [rows[i][\'smiles\'] for i in np.argsort(-s, kind=\'stable\')]\n    if mode == \'freeze1\' and old:\n        ranked = [old[0]] + [x for x in ranked if x != old[0]]\n    return ranked + [x for x in old if x not in set(ranked)]\n\n\ndef queries_from_test(test_path, max_spectra):\n    t = pd.read_parquet(test_path)\n    t = t[t.adduct.isin(list(L.ADDUCTS)) & (t.ms2_mzs.map(len) >= 5)]\n    out = {}\n    for mid, g in t.groupby(\'molecule_id\', sort=False):\n        g = g.head(max_spectra)\n        spectra = [dict(mz=np.asarray(r.ms2_mzs, float), it=np.asarray(r.ms2_normalized_intensities, float),\n                        adduct=r.adduct, prec=float(r.precursor_mz), ce=r.collision_energy_ev,\n                        mode=1 if str(r.ionization_mode).lower() == \'positive\' else -1)\n                   for r in g.itertuples(index=False)]\n        out[str(mid)] = dict(spectra=spectra,\n                             mass=float(np.median([L.neutral_mass(s[\'prec\'], s[\'adduct\']) for s in spectra])))\n    return out\n\n\ndef rank_features(feats, model, meta):\n    x = pd.DataFrame(feats)[meta[\'numeric\'] + meta[\'categorical\']]\n    for col in meta[\'categorical\']:\n        x[col] = pd.Categorical(x[col], categories=meta[\'categories\'][col])\n    return model.predict(x)\n\n\ndef insert(old, new, position, limit=25):\n    """old[:position-1] + new + the rest of old, without duplicates, at most `limit` entries."""\n    head, tail = old[:position - 1], old[position - 1:]\n    out = []\n    for s in head + new + tail:\n        if s and s not in out:\n            out.append(s)\n    return out[:limit]\n\n\ndef run(root, comp, bundle, lib_max, cfg=None, ranker_dir=None, cache=None):\n    import lightgbm as lgb\n    c = dict(DEFAULTS, **(cfg or {}))\n    t0 = time.time()\n    root, comp = Path(root), Path(comp)\n    ranker_dir = Path(ranker_dir or Path(__file__).resolve().parent / \'ranker\')\n    sub = pd.read_csv(root / \'submission.csv\', dtype=str, keep_default_na=False)\n    max_gate = max([c[\'gate\']] + [v.get(\'gate\', c[\'gate\']) for v in c[\'variants\'].values()])\n    gated = [m for m in sub.molecule_id if float(lib_max.get(str(m), 0.0)) < max_gate]\n    if c[\'limit\']:\n        gated = gated[:c[\'limit\']]\n    rep = dict(status=\'started\', cfg=c, molecules=len(sub), gated=len(gated))\n    if not gated:\n        return dict(rep, status=\'nothing_gated\')\n\n    cache = Path(cache or root / \'class3_cache\')\n    if not (cache / \'class3_frag.npz\').is_file():\n        L.build(comp / \'train.parquet\', cache)\n    meta_lib, frag_m, nl_m = L.load(cache)\n    structs = meta_lib.drop_duplicates(\'inchikey14\').set_index(\'inchikey14\').normalized_smiles\n    queries = queries_from_test(comp / \'test.parquet\', c[\'max_spectra\'])\n    sub_lists = {str(m): x.split(\';\') for m, x in zip(sub.molecule_id, sub.smiles)}\n    rep[\'library_seconds\'] = round(time.time() - t0, 1)\n\n    jobs = []\n    for mid in gated:\n        q = queries.get(str(mid))\n        if q is None:\n            continue\n        top = C.neighbour_scores(q, meta_lib, frag_m, nl_m).reindex(structs.index).fillna(0.0).nlargest(c[\'k\'] + 5)\n        rr = None\n        old = sub_lists[str(mid)]\n        if c[\'rerank\'] and len(old) > 1 and len({C.k14(x) for x in old} - {None}) == len(old):\n            rr = (old, [(i, structs[i], float(v)) for i, v in top.items()])\n        top = top.head(c[\'k\'])\n        jobs.append((str(mid), q, [(structs[i], float(v)) for i, v in top.items()], c[\'steps\'], rr))\n    rep[\'neighbour_seconds\'] = round(time.time() - t0, 1)\n\n    model = lgb.Booster(model_file=str(ranker_dir / \'ranker.txt\'))\n    meta = json.loads((ranker_dir / \'meta.json\').read_text())\n    rr_dir = ranker_dir.parent / \'rerank\'\n    if c[\'rerank\']:\n        rr_model = lgb.Booster(model_file=str(rr_dir / \'ranker.txt\'))\n        rr_meta = json.loads((rr_dir / \'meta.json\').read_text())\n    reranked = {}\n    workers = c[\'workers\'] or max(1, (os.cpu_count() or 2))\n    ctx = mp.get_context(\'fork\') if \'fork\' in mp.get_all_start_methods() else None\n    picks, timed_out = {}, 0\n    with ProcessPoolExecutor(workers, mp_context=ctx, initializer=_init, initargs=(str(bundle), c[\'checkpoint\'])) as pool:\n        futures = [pool.submit(_one, j) for j in jobs]\n        for f in futures:\n            left = c[\'seconds\'] - (time.time() - t0)\n            try:\n                mid, feats, rr_rows = f.result(timeout=max(left, 1))\n            except Exception:  # timeout or a failed molecule: that molecule keeps its list\n                timed_out += 1\n                continue\n            if rr_rows and float(lib_max.get(mid, 0.0)) < c[\'gate\']:\n                new = rerank_list(sub_lists[mid], rr_rows, rr_model, rr_meta, c[\'rerank\'])\n                if new != sub_lists[mid]:\n                    reranked[mid] = new\n            if feats:\n                s = rank_features(feats, model, meta)\n                order = np.argsort(-s, kind=\'stable\')\n                picks[mid] = ([feats[i][\'smiles\'] for i in order], s[order])\n        for f in futures:\n            f.cancel()\n\n    rep.update(jobs=len(jobs), with_candidates=len(picks), timed_out=timed_out, reranked=len(reranked),\n               new_leader=sum(v[0] != sub_lists[m][0] for m, v in reranked.items()))\n    main = {k: c[k] for k in PLACEMENT}\n    out, stats = place(sub, picks, lib_max, main, reranked)\n    _write(out, root / \'submission.csv\')\n    rep[\'variants\'] = {}\n    for name, v in c[\'variants\'].items():\n        vout, vstats = place(sub, picks, lib_max, dict(main, **v), reranked)\n        _write(vout, root / f\'submission_{name}.csv\')\n        rep[\'variants\'][name] = vstats\n    return dict(rep, status=\'applied\', **stats, seconds=round(time.time() - t0, 1))\n\n\ndef place(sub, picks, lib_max, p, reranked=None):\n    """Submission with the (reranked) lists and the ranked picks inserted under placement `p`; unchanged rows pass\n    through byte for byte."""\n    reranked = reranked or {}\n    changed, inserted, confident = 0, 0, 0\n    rows = []\n    for mid, smi in zip(sub.molecule_id, sub.smiles):\n        text = \';\'.join(reranked[mid]) if mid in reranked else smi\n        if mid in picks and float(lib_max.get(str(mid), 0.0)) < p[\'gate\']:\n            old = text.split(\';\')\n            have = {C.k14(x) for x in old}\n            ranked, scores = picks[mid]\n            keep = [i for i, x in enumerate(ranked) if C.k14(x) not in have][:p[\'n_insert\']]\n            add = [ranked[i] for i in keep]\n            if add:\n                margin = scores[0] - scores[1] if len(scores) > 1 else np.inf\n                if p[\'confident_margin\'] is not None and keep[0] == 0 and margin >= p[\'confident_margin\']:\n                    new = insert(insert(old, add[1:], p[\'position\']), add[:1], p[\'confident_position\'])\n                    confident += 1\n                else:\n                    new = insert(old, add, p[\'position\'])\n                assert 1 <= len(new) <= 25 and len(set(new)) == len(new) and all(new), mid\n                inserted += len(set(new) - set(old))\n                changed += 1\n                text = \';\'.join(new)\n        rows.append((mid, text))\n    out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n    assert out.molecule_id.tolist() == sub.molecule_id.tolist()\n    return out, dict(changed_rows=int(changed), inserted=int(inserted), confident=int(confident), placement=p)\n\n\ndef _write(df, path):\n    tmp = path.with_suffix(\'.tmp\')\n    df.to_csv(tmp, index=False)\n    os.replace(tmp, path)\n', 'ranker/ranker.txt': 'tree\nversion=v4\nnum_class=1\nnum_tree_per_iteration=1\nlabel_index=0\nmax_feature_idx=16\nobjective=lambdarank\nfeature_names=frag neural parent_sim parent_rank z_frag z_neural z_parent rank_frag rank_neural support n_steps tan_parent heavy heavy_parent n_cands rule1 rule2\nfeature_infos=[0:0.98438761776581418] [-2.8350811004638672:9.2908010482788086] [0.15185204666802016:0.99484965783878931] [1:30] [-8.1226434510091572:8.4261497731763555] [-6.891894778425808:3.8956916713143368] [-9.682743885563049:14.207740014744942] [1:2893] [1:2893] [1:14] [1:2] [0.1111111111111111:1] [8:83] [6:80] [1:2893] -1:7:28:16:29:26:36:18:17:32:8:35:34:6:24:0:33:25:37:19:30:10:15:12:21:38:27:2:39:1:23:5:31:4:22:9:11:40:3:13:14 -1:8:7:17:9:0:22:19:16:38:26:18:40:33:20:25:37:27:36:1:11:13:31:6:23:28:30:3:10:15:14:32:35:39:2:29:34:4:41:12:21:5\ntree_sizes=1795 1786 1778 1786 1779 1787 1762 1784 1772 1781 1785 1785 1784 1783 1788 1764 1791 1786 1773 1782 1789 1787 1788 1785 1787 1784 1783 1771 1796 1787 1784 1770 1782 1778 1796\n\nTree=0\nnum_leaves=15\nnum_cat=3\nsplit_feature=10 5 4 4 15 4 15 15 8 2 11 2 0 5\nsplit_gain=955.168 568.851 230.569 135.323 124.579 83.6591 76.8407 60.9706 48.0018 41.5475 36.3895 33.0557 51.2999 45.4813\nthreshold=1.5000000000000002 0.67979967121712248 0.22389574069364052 0.21887964173907506 0 1.3594231215139634 1 2 5.5000000000000009 0.49051945057336427 0.61311827956989251 0.61684767800942397 0.36408115561319404 -1.1659230274635746\ndecision_type=2 2 2 2 9 2 9 9 2 2 2 2 2 2\nleft_child=3 4 -3 7 -2 11 8 -1 -4 -10 -7 -6 13 -13\nright_child=1 2 6 -5 5 10 -8 -9 9 -11 -12 12 -14 -15\nleaf_value=0.03528202897674098 0.033544244926131526 -0.053102542797593223 0.078684856924205376 0.07651672357663214 -0.086490048900957198 -0.055277219431226793 -0.03076364272758891 -0.041113117717206077 -0.03133371629206267 0.044105401298485149 0.041806843758340623 0.065050620058517652 -0.077470898149066547 -0.044861249483699198\nleaf_weight=55.976903585731634 28.869989545302815 84.609274610505963 97.102139805421757 325.05157589451846 314.53314942908037 15.916643255244706 29.505328025639756 32.154094842910126 20.915934742195532 115.01926372052367 21.476196415518643 13.125864800691486 81.226010327058248 27.256505367920909\nleaf_count=2244 3293 17736 1154 5927 80986 2342 4697 3332 4707 7650 2259 1051 13261 6168\ninternal_value=-0.000611459 -0.0309327 0.0182603 0.0616571 -0.0648929 -0.0701803 0.0413725 0.00660692 0.0497855 0.0323538 0.000118611 -0.0777993 -0.0547993 -0.00842692\ninternal_weight=1262.74 849.556 347.152 413.183 502.404 473.534 262.543 88.131 233.037 135.935 37.3928 436.142 121.608 40.3824\ninternal_count=156807 145304 35944 11503 109360 106067 18208 5576 13511 12357 4601 101466 20480 7219\ncat_boundaries=0 1 3 5\ncat_threshold=17301572 1462047172 16 344260677 159\nis_linear=0\nshrinkage=0.05\n\n\nTree=1\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 15 9 6 5 15 5 11 7 11 7 6 16 12\nsplit_gain=1193.19 608.107 388.96 378.992 185.675 167.749 131.734 119.726 107.672 102.19 94.1933 90.7623 88.8327 77.8086\nthreshold=0 1 1.5000000000000002 1.3596833190455928 1.1196042404780329 2 0.22834880344223121 0.78487818157070921 6.5000000000000009 0.57459036439227207 3.5000000000000004 -1.0000000180025095e-35 3 18.500000000000004\ndecision_type=9 9 2 2 2 9 2 2 2 2 2 2 9 2\nleft_child=1 3 5 4 7 -2 11 10 13 12 -1 -3 -8 -7\nright_child=2 6 -4 -5 -6 8 9 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0054012429672839609 0.01513388121339556 -0.029198829655119163 0.024332675753459418 0.036926612957173928 0.0098328505618958811 -0.095195029166372416 0.04008780177819242 -0.00037336996073599507 -0.084958385982748602 0.06429699554715336 -0.067736484232507985 0.035604325999902278 -0.050331686095117373 0.00093097450356251438\nleaf_weight=74.526256932935212 59.516344309435226 77.657468055491336 136.90483963681618 340.68061905584182 242.88402016594773 29.043126047123224 65.036606014415156 125.91346579944366 526.69184551632497 665.83502075949218 327.68937656108756 172.16584762965795 31.696143089211546 70.275738133699633\nleaf_count=438 4133 7673 5289 7838 4079 355 694 5927 81700 7201 23813 4702 1133 1391\ninternal_value=-0.00110726 0.0187061 -0.0518371 -0.00686242 -0.0263228 -0.0678171 0.0465544 -0.0429377 -0.074764 0.0573291 -0.0562285 0.0154242 0.00920031 -0.0278472\ninternal_weight=2946.52 2124.08 822.432 1111.69 771.013 685.527 1012.39 528.129 626.011 762.568 402.216 249.823 96.7327 99.3189\ninternal_count=156366 63498 92868 42095 34257 87579 21403 30178 83446 9028 24251 12375 1827 1746\ncat_boundaries=0 2 4 5 7\ncat_threshold=2454509755 596 954435505 68 19140608 262161 4\nis_linear=0\nshrinkage=0.05\n\n\nTree=2\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 4 15 5 5 15 15 4 3 15 2 11 3 6\nsplit_gain=1055.36 476.306 239.702 232.222 177.5 164.795 132.867 113.349 100.376 95.1323 79.6719 75.8707 73.5555 68.75\nthreshold=0 0.70748822644173848 1 0.81817707692661434 1.3260108918225957 2 3 1.0884655409089907 6.5000000000000009 4 0.36152556798409846 0.54677083333333343 2.5000000000000004 1.6213361902259147\ndecision_type=9 2 9 2 2 9 9 2 2 9 2 2 2 2\nleft_child=1 2 -1 9 12 10 -5 -8 -7 13 -3 -6 -4 -2\nright_child=3 5 4 6 11 8 7 -9 -10 -11 -12 -13 -14 -15\nleaf_value=0.023410119218390979 -0.038559853838642001 -0.089343769600574691 -0.021962377896539435 0.017354237022240943 -0.090614361101671759 0.029675677530416625 -0.085779464436178579 0.016637310515450275 -0.025991520902732192 -0.074682671287383304 0.060779735358587605 0.040948485570572804 -0.066356241012944614 0.056295202175613547\nleaf_weight=335.47237307665637 90.245894112857059 8.1207618570188043 165.7580928200623 177.52464362350293 11.441464362200348 433.33814705640543 81.184573928243481 39.819653855753131 98.717987399897538 453.00990932318382 629.37900042941328 93.708195878658444 216.7400946902344 23.223173146718182\nleaf_count=7468 15183 480 7203 6410 475 6657 11040 1460 3089 69021 5305 1696 19545 1415\ninternal_value=0.000224078 0.0202367 -0.00879614 -0.0455558 -0.0308034 0.0408326 -0.010346 -0.0482607 0.018962 -0.0648628 0.0578248 0.0259142 -0.0472253 -0.0173118\ninternal_weight=2857.68 1992.68 823.12 865.008 487.648 1169.56 298.529 121.004 532.056 566.479 637.5 105.15 382.498 113.469\ninternal_count=156447 51918 36387 104529 28919 15531 18910 12500 9746 85619 5785 2171 26748 16598\ncat_boundaries=0 2 4 6 7 9\ncat_threshold=2285432987 276 1166831694 26 386158666 26 67440704 3442224652 480\nis_linear=0\nshrinkage=0.05\n\n\nTree=3\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 15 5 5 16 3 7 11 11 6 4 6 15 3\nsplit_gain=806.006 351.677 295.385 248.377 138.879 138.923 101.533 99.5935 96.763 86.7838 98.0538 81.4032 79.5504 72.6815\nthreshold=0 1 1.2834354001852917 1.2232175415258248 2 1.5000000000000002 17.500000000000004 0.99342105263157909 0.5100040016006403 2.2726853643837845 0.88682283986442145 0.81316807225276388 3 18.500000000000004\ndecision_type=9 9 2 2 9 2 2 2 2 2 2 2 9 2\nleft_child=1 7 11 4 13 6 -6 9 -4 10 -1 -2 -13 -3\nright_child=2 3 8 -5 5 -7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=0.014153010156017123 -0.067800293659216013 0.0073337340655835297 -0.088552083363836423 0.034623319369421396 0.021282668215534289 -0.077010315480778552 -0.056493626631930687 -0.093744123244831432 0.030022385881091268 0.076342347347785319 0.051598677092428803 0.0099607246411629914 -0.063519934425997848 -0.058620343626873711\nleaf_weight=313.33809590199962 498.11189736571396 418.53186042251764 18.482169294264168 324.99394916696474 98.55093571357429 154.83208384353202 71.601974369084928 12.686355372541582 147.63214794185478 156.29979735001689 397.38573970703874 81.347826771670952 55.886728271783795 45.390693854307756\nleaf_count=9275 79065 9288 1118 4472 924 22735 5725 240 5323 344 2312 5672 7067 2651\ninternal_value=-0.000372028 0.0166573 -0.0423794 -0.0020795 -0.0172504 -0.0416195 -0.0115852 0.0401148 0.0163786 0.0426413 0.035133 -0.0584096 -0.0240807 0.000752326\ninternal_weight=2795.07 1993.61 801.461 1113.9 788.908 324.985 170.153 879.71 166.114 867.024 710.724 635.346 137.235 463.923\ninternal_count=156211 57966 98245 45795 41323 29384 6649 12171 6441 11931 11587 91804 12739 11939\ncat_boundaries=0 2 4 6 8\ncat_threshold=440203449 788 120352846 139 269496353 532 69546241 64\nis_linear=0\nshrinkage=0.05\n\n\nTree=4\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 9 4 3 5 16 6 11 12 9 8 16 11 6\nsplit_gain=784.307 474.563 283.319 138.83 135.494 111.042 84.3774 82.3106 83.6228 79.101 72.5135 71.2879 66.807 65.3678\nthreshold=0 1.5000000000000002 0.48634079056164314 3.5000000000000004 0.19880277570399549 1 1.2615302718886507 0.59197099892588623 25.500000000000004 4.5000000000000009 1.5000000000000002 2 0.57459036439227207 1.9543342892696052\ndecision_type=9 2 2 2 2 9 2 2 2 2 2 9 2 2\nleft_child=2 6 3 5 12 -1 -2 -5 -9 10 -6 -12 -4 -7\nright_child=1 -3 4 7 9 13 -8 8 -10 -11 11 -13 -14 -15\nleaf_value=0.037675809427447859 -0.066671549385673393 0.024751222761640958 -0.063533501682557836 -0.076817618437633559 0.066900449752024194 -0.047458013089571233 -0.0084294991439451804 0.005907709664092149 -0.059897610608473165 0.082293373376402815 0.035840273040007212 -0.00093144391702265281 0.0085113025204893705 0.024660943182875254\nleaf_weight=173.90639686957002 594.62736230774317 215.17679747444345 36.896383434650488 84.38821585860569 144.02889048540965 88.897188287228346 69.784720659488812 130.89143351488747 75.343459182884544 93.92300027818419 638.23595909535652 174.0646091485396 212.28592046603444 47.527303979557473\nleaf_count=1806 97933 6416 1882 6074 234 5862 5607 2959 9768 176 5123 4376 6694 1638\ninternal_value=-0.000501762 -0.039252 0.017559 -0.0107252 0.0307729 0.0125169 -0.0605761 -0.0355011 -0.0183059 0.0386305 0.0342608 0.0283984 -0.00236767 -0.0208216\ninternal_weight=2779.98 879.589 1900.39 600.954 1299.43 310.331 664.412 290.623 206.235 1050.25 956.329 812.301 249.182 136.424\ninternal_count=156548 109956 46592 28107 18485 9306 103540 18801 12727 9909 9733 9499 8576 7500\ncat_boundaries=0 2 4 6\ncat_threshold=2550722715 20 4107 16 2416484353 4\nis_linear=0\nshrinkage=0.05\n\n\nTree=5\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 9 4 15 8 15 6 5 6 11 1 8 9 4\nsplit_gain=739.305 358.261 247.012 173.663 157.244 132.045 131.887 109.833 87.0816 86.7073 62.4875 59.3447 69.2097 57.4224\nthreshold=0 2.5000000000000004 0.47215838872972526 1 5.5000000000000009 2 0.51476368267003603 0.61651355393786489 2.2726853643837845 0.5707610146862484 7.0392909049987802 7.5000000000000009 1.5000000000000002 0.91534489266762964\ndecision_type=9 2 2 9 2 9 2 2 2 2 2 2 2 2\nleft_child=2 4 3 9 13 7 -5 -4 -6 -1 -8 -7 -13 -2\nright_child=1 -3 5 6 8 11 10 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.062007585109704645 -0.031705009119490528 0.044514902500662701 -0.031167075707806913 -0.068237839173368786 -0.063253297944983175 0.056588696325708635 -0.030800783153049308 0.024277149079503239 0.023373326223302075 0.036906897207908775 0.022810782180698194 0.02019539639855493 0.064242849442047584 0.043374852735817114\nleaf_weight=23.896982971695252 47.120249099680223 115.19625828851713 128.15699837810826 138.15645328426035 607.47314946557162 398.70640856376849 115.52501418424072 288.78667783364654 29.977214446524158 206.19079719227739 100.90840827929787 381.69128443545196 116.09452895497088 53.14880369475577\nleaf_count=1272 1423 1798 6195 13334 104375 786 6641 3208 1270 3466 3132 8114 1092 315\ninternal_value=0.00161848 -0.0367726 0.0189735 -0.00795185 -0.0500066 0.0311006 -0.0294324 0.00703305 -0.0591493 0.0253143 -0.00581535 0.0417303 0.0305764 0.00812729\ninternal_weight=2751.03 852.916 1898.11 584.678 737.719 1313.44 354.59 416.944 637.45 230.088 216.433 896.492 497.786 100.269\ninternal_count=156421 109181 47240 27845 107383 19395 23107 9403 105645 4738 9773 9992 9206 1738\ncat_boundaries=0 2 4 6\ncat_threshold=2284386459 20 1166876743 10 1210287521 132\nis_linear=0\nshrinkage=0.05\n\n\nTree=6\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 5 16 9 9 15 3 15 9 11 5 7 8 7\nsplit_gain=674.733 263.582 258.68 258.671 118.02 100.535 82.654 79.096 100.045 86.6229 72.031 67.185 46.425 44.7765\nthreshold=0 0.80787295344800214 1 1.5000000000000002 2.5000000000000004 2 2.5000000000000004 3 3.5000000000000004 0.61572802197802212 1.2232175415258248 2.5000000000000004 87.500000000000014 22.500000000000004\ndecision_type=9 2 9 2 2 9 2 9 2 2 2 2 2 2\nleft_child=1 2 5 13 11 6 -1 -3 9 -9 -11 -4 -7 -2\nright_child=3 7 4 -5 -6 12 -8 8 -10 10 -12 -13 -14 -15\nleaf_value=0.018190356092785475 -0.036844631943787676 0.046032787296947389 0.021996284770384414 0.013871389948336249 0.028045594698408161 0.011846484728910249 -0.059598798746030003 -0.035224403666934384 0.06420560037326814 -0.017736240923004864 0.03287001608853151 -0.058501788764838164 0.044800110026521747 -0.068320309757114839\nleaf_weight=65.440465927589685 165.90203724993626 498.96288931183517 28.109441927052103 156.30195207084762 55.35956161073409 212.65420008345973 69.596703888033517 92.422973834327422 93.821610260172747 96.329924831807148 254.16362979367841 273.63687146926532 215.86627837485867 384.64568164967932\nleaf_count=730 6973 3005 315 4930 1305 1664 3274 3717 322 4108 3097 36785 4032 82121\ninternal_value=-0.000743041 0.0144026 -0.00498884 -0.0422301 -0.0376137 0.0160955 -0.0205352 0.0317674 0.0172679 0.00759196 0.0189505 -0.050955 0.0278637 -0.0589194\ninternal_weight=2663.21 1956.36 920.664 706.85 357.106 563.558 135.037 1035.7 536.738 442.917 350.494 301.746 428.52 550.548\ninternal_count=156378 62354 48105 94024 38405 9700 4004 14249 11244 10922 7205 37100 5696 89094\ncat_boundaries=0 2 4 6 8\ncat_threshold=443340985 789 49153 16 39977376 7 100929609 15\nis_linear=0\nshrinkage=0.05\n\n\nTree=7\nnum_leaves=15\nnum_cat=6\nsplit_feature=16 4 9 5 16 15 15 15 9 11 6 8 5 16\nsplit_gain=659.827 363.614 215.722 158.418 158.186 113.854 91.3082 90.6682 81.1012 105.553 66.4668 62.7487 61.8082 60.3743\nthreshold=0 0.22389574069364052 2.5000000000000004 0.17072489627481388 1 2 3 4 3.5000000000000004 0.65069803021610262 1.0837699989682761 5.5000000000000009 -1.0752009023380464 5\ndecision_type=9 2 2 2 9 9 9 9 2 2 2 2 2 9\nleft_child=2 4 11 5 6 13 -2 -5 9 10 -9 -1 -7 -3\nright_child=1 3 -4 7 -6 12 -8 8 -10 -11 -12 -13 -14 -15\nleaf_value=-0.010532237317832538 0.025579291466088363 0.03955623658824893 0.02836256376796438 0.043724020002408964 -0.052431376921355181 0.0082835936757732712 -0.030521008655529581 -0.071544713060317178 0.061316065662608048 0.022307183056873698 0.003487086959314753 -0.066268801618632384 -0.057296820982829982 -0.049042931849003983\nleaf_weight=57.709139785263687 169.95468263048679 114.56557300500572 80.568061964004301 792.57378290116321 227.84930927486857 52.70632051571738 110.23788684851024 56.456882683211006 82.199546211631969 319.08146084414329 60.512941938708536 427.9401522433036 110.3460730361403 14.386475980572866\nleaf_count=1241 4629 1765 1142 7408 27016 3019 6583 3063 313 6544 1704 80758 9775 1602\ninternal_value=0.00109884 0.0139844 -0.0463087 0.0257308 -0.0227777 -0.00755286 0.00318823 0.0331453 0.0157682 0.0074222 -0.0330274 -0.0596809 -0.0340983 0.0265939\ninternal_weight=2677.09 2110.87 566.217 1602.83 508.042 292.004 280.193 1310.82 518.251 436.051 116.97 485.649 163.052 128.952\ninternal_count=156562 73421 83141 35193 38228 16161 11212 19032 11624 11311 4767 81999 12794 3367\ncat_boundaries=0 2 4 6 8 10 12\ncat_threshold=1652754244 232 1331211 533 1294254151 1 1157963781 27 386666570 153 71335937 258\nis_linear=0\nshrinkage=0.05\n\n\nTree=8\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 4 9 6 15 15 15 4 5 11 16 5 8 0\nsplit_gain=617.181 277.595 225.192 127.26 124.002 82.8214 78.1933 67.7666 66.0231 55.8284 51.3819 45.4141 43.3576 39.1074\nthreshold=0 0.71447990227228164 2.5000000000000004 1.4229107095596134 1 2 3 0.78859579615429165 0.61651355393786489 0.5157072368421054 4 0.3942940573926389 15.500000000000002 0.68216519083278071\ndecision_type=9 2 2 2 9 9 9 2 2 2 9 2 2 2\nleft_child=1 3 5 4 12 -2 -3 -7 -8 -5 -6 -9 13 -1\nright_child=2 6 -4 9 10 7 8 11 -10 -11 -12 -13 -14 -15\nleaf_value=-0.051877795416530183 0.011587480147466729 0.043166465896159731 0.0232283052204596 -0.065058041025364957 0.023580450084392121 -0.066073140599586452 -0.012492354536073306 -0.059918084726023814 0.028133650450287304 0.029487778624543489 -0.02394064015984709 -0.00767680369721276 -0.064217405321595492 0.0036347274661072768\nleaf_weight=62.078791088657454 54.234970583172981 620.87494534743018 125.91100941016339 15.923265213146804 140.2211614632979 426.88930170371896 146.36158149148105 68.767614636628423 311.55153259180952 210.72676881746156 81.004219223337714 103.90556604001904 146.98802485043416 63.584211242618039\nleaf_count=1424 6899 4954 1779 724 2275 79316 6012 11451 3170 4375 6192 7579 19453 648\ninternal_value=-0.000527582 0.0155734 -0.0373764 -0.00836486 -0.0225871 -0.0496364 0.0316989 -0.0544705 0.0148265 0.0225889 0.00587026 -0.0287073 -0.044213 -0.0239821\ninternal_weight=2579.02 1799.31 779.708 720.526 493.876 653.797 1078.79 599.562 457.913 226.65 221.225 172.673 272.651 125.663\ninternal_count=156251 49227 107024 35091 29992 105245 14136 98346 9182 5099 8467 19030 21525 2072\ncat_boundaries=0 2 4 5 7 8\ncat_threshold=2282281115 276 342070757 128 17567748 354701402 139 9\nis_linear=0\nshrinkage=0.05\n\n\nTree=9\nnum_leaves=15\nnum_cat=6\nsplit_feature=16 9 4 5 16 15 16 9 15 15 5 6 3 6\nsplit_gain=506.414 247.854 230.743 119.266 101.997 73.3936 72.4249 70.8358 84.8825 63.2735 58.1941 58.1534 47.4 47.3356\nthreshold=0 1.5000000000000002 0.22389574069364052 0.17072489627481388 1 2 3 3.5000000000000004 4 5 0.7470770372477219 0.993734750602976 8.5000000000000018 1.7488359477048623\ndecision_type=9 2 2 2 9 9 9 2 9 9 2 2 2 2\nleft_child=2 9 4 6 5 -1 12 8 -5 -2 -3 -7 -4 -10\nright_child=1 10 3 7 -6 11 -8 -9 13 -11 -12 -13 -14 -15\nleaf_value=0.029321573713301786 0.0066680654763846912 -0.034801737973641111 0.025348023454550197 0.038335371757114864 -0.049168613410045185 -0.048322831527039131 -0.043681298959455397 0.06087931361738546 6.3879040512890639e-05 -0.059199688461549264 0.030060379889509617 0.014017388363516981 -0.04364751001336023 0.03486045551139752\nleaf_weight=117.03702010447159 45.158386616851203 45.376892993284855 129.14344968926162 552.99558980995789 154.23927139129955 89.475880193698686 81.631103523424827 159.16781690833159 361.83625975606265 462.10790891654324 135.64619893417694 63.161311912583187 29.828056955360807 132.90717969601974\nleaf_count=2515 6275 3743 1576 3157 17730 7127 9723 409 8856 89221 1991 1115 1255 1658\ninternal_value=0.000385614 -0.0359504 0.0138795 0.0234518 -0.0184579 0.00100319 -0.00866272 0.0298559 0.0250894 -0.0543679 0.0136996 -0.0212218 0.011492 0.00927593\ninternal_weight=2559.71 688.289 1871.42 1447.51 423.913 269.674 240.603 1206.91 1047.74 507.266 181.023 152.637 158.972 494.743\ninternal_count=156351 101230 55121 26634 28487 10757 12554 14080 13671 95496 5734 8242 2831 10514\ncat_boundaries=0 2 4 6 8 10 11\ncat_threshold=2553084091 788 268718091 528 1166352389 138 2147516417 16 117706826 157 17567744\nis_linear=0\nshrinkage=0.05\n\n\nTree=10\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 9 4 16 7 15 9 11 2 15 4 11 3 2\nsplit_gain=510.749 201.651 197.943 111.468 106.724 83.6957 57.0768 78.1588 55.6695 54.9138 50.3296 46.3747 45.0054 43.4536\nthreshold=0 2.5000000000000004 0.70748822644173848 1 17.500000000000004 2 3.5000000000000004 0.56030303030303041 0.36152556798409846 3 1.242459270799247 0.68048396334478822 3.5000000000000004 0.46000357143213427\ndecision_type=9 2 2 9 2 9 2 2 2 9 2 2 2 2\nleft_child=2 10 3 9 -5 6 7 -4 -7 -1 -2 -12 -11 -9\nright_child=1 -3 5 4 -6 8 -8 13 -10 12 11 -13 -14 -15\nleaf_value=0.018914522401028689 -0.056333524921376478 0.021732503376671718 -0.070611204664286276 0.010905376408407434 -0.052804379816275705 -0.082958862478368256 0.051977881307877172 -0.044640513660437572 0.038470376551414248 0.011975563304466998 -0.043784934170188299 0.04096137812687007 -0.050791910739105266 0.014754219644273196\nleaf_weight=349.6209058976965 473.22215728176525 116.39983959647361 34.973104765522294 99.950657453970052 189.36145105736796 8.6549868946894986 59.024597739684395 34.072441861382686 742.41301591484807 52.86921551648993 38.420266419707332 26.371085446793586 60.316982880642172 252.54097475355957\nleaf_count=7621 92875 1584 1682 1095 24648 636 215 1158 6631 961 4061 2626 5367 5417\ninternal_value=0.000467075 -0.0372221 0.0136975 -0.00610705 -0.0298057 0.0269745 0.00703503 -0.00113135 0.0364776 0.0091845 -0.0506842 -0.00919355 -0.0198838 0.00756275\ninternal_weight=2538.21 654.413 1883.8 752.119 289.312 1131.68 380.611 321.587 751.068 462.807 538.014 64.7914 113.186 286.613\ninternal_count=156577 101146 55431 39692 25743 15739 8472 8257 7267 13949 99562 6687 6328 6575\ncat_boundaries=0 2 4 6 8\ncat_threshold=2553084091 788 2412555 16 1784874420 4 1694847053 29\nis_linear=0\nshrinkage=0.05\n\n\nTree=11\nnum_leaves=15\nnum_cat=2\nsplit_feature=16 9 5 11 5 4 4 3 12 5 3 11 9 16\nsplit_gain=463.628 205.865 167.148 122.014 89.8415 87.644 60.8505 57.6513 56.7656 47.0139 44.021 38.8439 35.9133 60.3215\nthreshold=0 2.5000000000000004 0.69873624032267434 0.56030303030303041 1.1043457755849995 1.3108848532147344 0.73329725171479743 17.500000000000004 20.500000000000004 1.6789404729405224 12.500000000000002 0.86700680272108854 2.5000000000000004 1\ndecision_type=9 2 2 2 2 2 2 2 2 2 2 2 2 9\nleft_child=2 4 3 -1 -2 10 -6 9 -9 12 -5 -3 13 -4\nright_child=1 11 7 5 6 -7 -8 8 -10 -11 -12 -13 -14 -15\nleaf_value=-0.04998684815914875 -0.05465850099200438 0.027577452013106869 0.028693523015853362 0.0010925009492205541 -0.048438345554631501 0.038434809906384107 0.021218921180149158 0.021880794158608212 -0.073875695244714865 0.049893033043602972 -0.041640161253639861 -0.089681673702208572 0.047225276515391795 -0.0065607570151655558\nleaf_weight=118.29646860109642 489.62715286400635 135.80455012782477 411.21474928030511 387.78986687166616 52.275777859380469 149.40561683813576 75.257154321239796 35.583346260711551 25.943280509731267 238.3559126277687 70.346411448263098 6.4847858267021357 136.620930626872 170.75862047052942\nleaf_count=7473 96703 1591 2935 18423 7844 1673 2809 323 2176 1179 6465 258 680 5956\ninternal_value=0.000306304 -0.0320306 0.0144961 -0.00375402 -0.0449389 0.00531547 -0.00748832 0.0276355 -0.0190316 0.0306527 -0.00554546 0.0216448 0.0242102 0.0187356\ninternal_weight=2503.76 759.449 1744.32 725.838 617.16 607.542 127.533 1018.48 61.5266 956.95 458.136 142.289 718.594 581.973\ninternal_count=156488 109205 47283 34034 107356 26561 10653 13249 2499 10750 24888 1849 9571 8891\ncat_boundaries=0 2 4\ncat_threshold=269015195 276 268439571 4\nis_linear=0\nshrinkage=0.05\n\n\nTree=12\nnum_leaves=15\nnum_cat=2\nsplit_feature=16 9 15 6 11 8 11 5 9 6 11 5 8 5\nsplit_gain=403.16 430.452 86.5909 119.867 77.8348 75.8602 62.5465 60.8576 58.6873 47.543 38.3273 37.7694 37.1409 29.9819\nthreshold=0 1.5000000000000002 1 0.92467986616559794 0.61572802197802212 15.500000000000002 0.78487818157070921 1.2625707354522766 2.5000000000000004 1.9543342892696052 0.94160272804774092 1.167275749598238 3.5000000000000004 0.98556948785963472\ndecision_type=9 2 9 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 10 6 9 -6 -4 8 -3 12 13 -10 -2 -1\nright_child=1 7 3 -5 5 -7 -8 -9 11 -11 -12 -13 -14 -15\nleaf_value=0.023729241041809856 -0.016733187719763794 -0.029845423581492048 -0.043045741343817334 0.026042938707206378 0.010062973556427201 -0.039323010045344164 0.010969894938769381 0.040076494646385109 0.026050483976463448 0.0024369139171856197 -0.046261051079634158 -0.086147266580671081 -0.065524132086343972 0.045449348267864016\nleaf_weight=328.44053302996326 44.455726779997349 87.982267685292754 140.44320515898289 284.92530143714976 109.34884723240975 265.21625341696199 85.739203146513319 194.45637285953853 153.76597382582258 32.370538458228111 14.233467176323755 6.920357860508374 373.22699332580669 315.38383757101838\nleaf_count=3748 772 6244 7474 6194 2772 65746 2791 1274 2357 1612 257 119 53889 943\ninternal_value=-0.000914659 -0.0203569 0.0201679 0.00442309 -0.0418573 -0.0249171 -0.0226118 0.0191993 0.00277847 -0.0558697 0.0321229 0.0207155 -0.0603835 0.034423\ninternal_weight=2436.91 1267.74 1169.17 511.108 824.618 374.565 226.182 443.125 248.669 450.053 658.058 160.686 417.683 643.824\ninternal_count=156192 134785 21407 16459 124791 68518 10265 9994 8720 56273 4948 2476 54661 4691\ncat_boundaries=0 2 4\ncat_threshold=2674713 20 84168783 159\nis_linear=0\nshrinkage=0.05\n\n\nTree=13\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 3 15 9 15 16 1 11 8 4 8 8 7 4\nsplit_gain=404.153 187.177 130.722 125.207 116.296 106.886 67.4104 82.1313 66.3507 57.5604 53.4401 46.5946 44.3085 40.3982\nthreshold=0 6.5000000000000009 1 2.5000000000000004 2 3 6.9058468341827401 0.78659176029962563 33.500000000000007 1.3321543814615286 5.5000000000000009 2.5000000000000004 169.50000000000003 1.7630476406113533\ndecision_type=9 2 9 2 9 9 2 2 2 2 2 2 2 2\nleft_child=3 4 5 10 6 12 7 8 13 -4 -1 -7 -3 -2\nright_child=1 2 9 -5 -6 11 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0027102885138493549 -0.014968815428447666 0.032737932635119399 -0.049842254050612156 0.012109978481488308 0.036298050002631969 0.045018252825043549 0.021403014116374998 0.029095070126985974 -0.052638521511340609 0.011193904589480799 -0.058879022584758856 -0.066962518121511258 -0.061497414850584887 0.048076747083679969\nleaf_weight=48.284854968253057 118.32754211663269 214.53611298440956 219.59714399001678 93.456012669485062 646.63741457639844 10.424386287690139 405.24032680585515 104.16835329309106 109.5975515791215 46.431087286910042 349.16780762159033 54.521616959187668 12.334720592712982 31.284754976630211\nleaf_count=1224 2143 2008 26219 1133 7246 88 8623 2233 13921 950 77942 10596 1733 76\ninternal_value=0.00102681 0.0111579 -0.013298 -0.0390654 0.0208789 0.0100146 0.00730987 -0.00822183 -0.0232725 -0.0377728 -0.0520792 -0.0421821 0.0262166 -0.00155396\ninternal_weight=2464.01 1973.1 557.845 490.909 1415.26 291.817 768.619 363.378 259.21 266.028 397.453 64.946 226.871 149.612\ninternal_count=156135 75836 41594 80299 34242 14425 26996 18373 16140 27169 79166 10684 3741 2219\ncat_boundaries=0 2 4 6 8\ncat_threshold=1619199812 232 621023306 24 948142480 20 69230593 276\nis_linear=0\nshrinkage=0.05\n\n\nTree=14\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 9 15 6 15 0 11 11 9 2 9 8 7 11\nsplit_gain=399.355 169.804 155.13 89.8506 60.7176 56.5222 46.8575 45.4686 52.9378 41.5545 39.7002 39.6825 37.2578 35.2854\nthreshold=0 1.5000000000000002 1 1.4229107095596134 2 0.48910583036620175 0.53589837997054501 0.65582584170632829 2.5000000000000004 0.38064325420363493 1.5000000000000002 7.5000000000000009 2.5000000000000004 0.5157072368421054\ndecision_type=9 2 9 2 9 2 2 2 2 2 2 2 2 2\nleft_child=2 12 6 5 -3 10 -1 8 -7 -8 -4 -5 -2 -13\nright_child=1 4 3 11 -6 7 9 -9 -10 -11 -12 13 -14 -15\nleaf_value=-0.024452038770147132 0.0093964159490676404 0.019004819330562635 -0.040248124780505913 0.043343695345139405 -0.054670367669147957 -0.033000716717595989 -0.032417782384447653 0.012949879093811132 0.028349158152711208 0.03675587653049852 0.023849185796523391 -0.05481392842426213 -0.052171710910822545 0.013915653766917394\nleaf_weight=35.037362063769251 25.670339758566115 160.06050795986084 185.2909084384446 119.97559982049279 22.721353027329315 151.36099922360154 21.758058318227995 258.49795638327487 44.871699306531809 623.13352017040597 27.085033867158927 19.269107112661004 456.53760172473267 238.90380816539982\nleaf_count=1547 607 3471 19648 327 2241 7350 1350 6584 293 6417 1287 947 97068 7101\ninternal_value=-0.000257256 -0.0328592 0.0124396 0.000261048 0.00765808 -0.0107695 0.0308594 -0.000805588 -0.0188995 0.0343754 -0.0320001 0.0197045 -0.0488815 0.00859422\ninternal_weight=2390.17 664.99 1725.18 1045.26 182.782 667.107 679.929 454.731 196.233 644.892 212.376 378.149 482.208 258.173\ninternal_count=156238 103387 52851 43537 5712 35162 9314 14227 7643 7767 20935 8375 97675 8048\ncat_boundaries=0 2 4 6\ncat_threshold=2419650745 790 1158434894 11 67441856 16\nis_linear=0\nshrinkage=0.05\n\n\nTree=15\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 8 6 11 3 5 5 1 4 10 5 1 11 4\nsplit_gain=352.941 155.112 112.9 99.1765 62.2856 54.271 50.9186 53.3192 50.4926 48.2911 38.2457 33.319 33.4568 32.361\nthreshold=0.45422973912831383 7.5000000000000009 0.70692866265342036 0.77831196581196593 8.5000000000000018 0.66228488081928416 0.19880277570399549 4.7508065700531015 0.81460006249793526 1.5000000000000002 0.87512723910509405 3.1318092346191411 0.55028089887640463 0.53491892770497662\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 10 9 6 -5 -4 7 11 13 -1 -2 -3 -13 -8\nright_child=1 3 5 4 -6 -7 8 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.015567807219678712 -0.012861577564700187 -0.074226569901404402 -0.026261138347891871 0.032787336031640003 -0.03633140425836686 0.013701817550815379 0.0086607485912828189 -0.050495683900156435 0.010371567632010095 -0.051672563344155945 0.036925826962745358 -0.063027846407056651 0.024704713194551097 -0.047717396064170525\nleaf_weight=126.24331648135558 40.952325003221631 12.270662529917898 156.68591378105339 268.74670913856244 36.235416988143697 183.25764995708596 38.155800583015662 126.88685723085655 312.79576444829581 358.58699525293196 558.31494274939178 11.889019071531946 68.920683091215324 73.855910158948973\nleaf_count=5066 254 808 12493 6995 5939 5636 2460 19852 10960 72823 2289 1524 2655 6611\ninternal_value=-0.000454353 0.0136121 -0.0268573 0.00103648 0.0244833 -0.0047408 -0.0100748 -0.0295418 5.1064e-05 -0.0423228 0.0335077 -0.00074439 0.0111845 -0.028606\ninternal_weight=2373.8 1549.02 824.774 949.757 304.982 339.944 644.775 219.967 424.807 484.83 599.267 93.0804 80.8097 112.012\ninternal_count=156365 60347 96018 57804 12934 18129 44870 24839 20031 77889 2543 4987 4179 9071\nis_linear=0\nshrinkage=0.05\n\n\nTree=16\nnum_leaves=15\nnum_cat=5\nsplit_feature=4 16 16 5 15 15 5 5 0 9 16 11 2 9\nsplit_gain=362.986 150.54 137.201 78.0852 75.4422 70.3768 74.7485 42.8506 48.9646 38.4246 41.3312 45.1423 36.5969 35.2827\nthreshold=0.47215838872972526 0 1 1.0421956977107312 2 3 0.61651355393786489 0.67979967121712248 0.63666382790553666 2.5000000000000004 4 0.5157072368421054 0.60622425308797945 5.5000000000000009\ndecision_type=2 9 9 2 9 9 2 2 2 2 9 2 2 2\nleft_child=2 5 4 -3 -1 6 -2 -4 -9 10 11 -7 -6 -8\nright_child=1 3 7 -5 12 9 13 8 -10 -11 -12 -13 -14 -15\nleaf_value=0.026303467851508111 -0.030422244671063348 -0.040410718524322847 -0.054721496243235961 0.013135347587235648 -0.03716834666578675 -0.079130819585855106 0.010958271838270651 -0.046132665013376824 0.0049588188468202071 0.052606859237149334 0.031183084817305924 0.010176475282012022 0.013612938272652099 0.065760533919623068\nleaf_weight=137.97921167418826 113.38062040612567 169.89981528150383 316.65867027948843 112.52333863795502 124.00687071826542 14.632156836218199 244.6763478803914 109.79906377050793 80.999547444633208 152.38016725319903 549.27948220493272 148.25556191179203 48.931617404276039 32.490514877717942\nleaf_count=3024 7450 31745 64880 4298 8462 462 3806 16113 4013 519 4473 5844 1359 42\ninternal_value=-0.000133096 0.0141837 -0.0270151 -0.0184546 -0.000148835 0.0215125 0.00352344 -0.0426025 -0.0245301 0.0294924 0.0248934 0.00162122 -0.0215647 0.0175957\ninternal_weight=2355.89 1537.52 818.375 282.423 310.918 1255.09 390.547 507.457 190.799 864.547 712.167 162.888 172.938 277.167\ninternal_count=156490 58639 97851 36043 12845 22596 11298 85006 20126 11298 10779 6306 9821 3848\ncat_boundaries=0 2 4 6 8 10\ncat_threshold=2422794385 278 1141952555 532 1158422533 15 1214481825 4 275286144 16\nis_linear=0\nshrinkage=0.05\n\n\nTree=17\nnum_leaves=15\nnum_cat=2\nsplit_feature=16 9 7 11 6 0 16 7 6 14 1 2 11 11\nsplit_gain=301.537 261.661 94.8708 77.5555 77.0441 51.7118 58.8431 51.5107 41.7686 40.9555 38.5535 30.8888 27.4583 45.9812\nthreshold=0 2.5000000000000004 18.500000000000004 0.5707610146862484 1.1342138978903009 0.68216519083278071 1 31.500000000000004 0.89440751322627077 66.500000000000014 6.0276613235473642 0.49208951090305225 0.80094339622641531 0.77158385093167714\ndecision_type=9 2 2 2 2 2 9 2 2 2 2 2 2 2\nleft_child=4 2 3 -2 5 6 -1 9 -4 -7 -3 12 13 -8\nright_child=1 10 8 -5 -6 7 11 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.045918587400663027 -0.04393851666431358 -0.059681682305515529 -0.056767013437214048 0.0096107877223798252 0.030118662152940442 0.0062989791363524759 -0.026646005166721606 -0.028331404544722646 -0.024031742134691562 0.04271569521636099 0.028753723740241724 0.016018192442334524 -0.057577878008715756 0.038880858237576921\nleaf_weight=67.43881833148771 110.85832402750384 12.089739317307247 364.0381108194706 170.24828611325938 538.91970431798836 139.7860071407631 75.555081547587179 48.796617704094388 135.94329287437722 172.5654531379696 233.76733876112849 182.57046905223979 28.857003240729682 40.17710043140687\nleaf_count=10669 4040 452 95821 4492 6690 939 3291 2746 17902 1149 2330 3757 1610 532\ninternal_value=-0.000422216 -0.0205338 -0.0348838 -0.011588 0.0155724 0.00537914 -0.00712673 0.0190253 -0.0479321 0.0264903 0.0241809 0.00231393 -0.0148225 -0.00375977\ninternal_weight=2321.61 1026.95 781.088 281.107 1294.67 755.747 394.598 361.148 499.981 312.351 245.857 327.16 144.589 115.732\ninternal_count=156420 125037 122255 8532 31383 24693 19859 4834 113723 2088 2782 9190 5433 3823\ncat_boundaries=0 2 4\ncat_threshold=2945051 276 794624 272\nis_linear=0\nshrinkage=0.05\n\n\nTree=18\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 5 9 16 4 15 3 5 15 15 2 5 9 2\nsplit_gain=345.343 134.95 98.3505 97.2662 81.1456 68.6838 57.8416 48.8023 46.7389 46.298 38.2295 37.7942 34.8049 33.3328\nthreshold=0 0.29850567396771482 1.5000000000000002 1 0.22389574069364052 2 14.500000000000002 1.2040308936940882 3 4 0.42529670702354377 -2.0776512493288348 6.5000000000000009 0.41427175468698008\ndecision_type=9 2 2 9 2 9 2 2 9 9 2 2 2 2\nleft_child=1 3 -2 6 5 7 8 -3 11 -6 -11 -1 13 -4\nright_child=2 4 12 -5 9 -7 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=0.055253629220517159 -0.04517765184679768 -0.021800125629582518 0.040542728073034578 -0.041656724930420309 0.035674966011455589 -0.033433187797761665 -0.056387051173006381 0.050334363439663771 0.02385319569630576 -0.060301146936116191 0.016549346227539216 -0.045795774783979136 0.054929753584805431 -0.021609900003237502\nleaf_weight=10.460767752607351 428.74523012479767 36.644074308860581 26.081310256849974 156.72307184385136 567.22078256175155 109.96589549200144 32.259032243862748 61.922016557538882 212.71192691160832 15.792398043442516 470.83283272595145 49.060349052539095 24.283126971684396 105.55538618657738\nleaf_count=87 95987 1921 453 22268 4521 9055 2328 654 2909 942 7565 1886 45 5365\ninternal_value=0.000567339 0.0118419 -0.0322846 -0.0112182 0.020363 -0.00810273 0.00571032 0.0214162 0.0134572 0.025998 0.0136567 -0.0235292 0.00117318 -0.00908271\ninternal_weight=2308.26 1723.59 584.665 461.215 1262.38 208.532 304.492 98.5661 272.233 1053.85 486.625 59.5211 155.92 131.637\ninternal_count=155986 54136 101850 29478 24658 11630 7210 2575 4882 13028 8507 1973 5863 5818\ncat_boundaries=0 2 4 6 8 10\ncat_threshold=472447163 788 268484609 16 1143283715 13 40534224 6 386273352 11\nis_linear=0\nshrinkage=0.05\n\n\nTree=19\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 5 4 6 9 2 16 7 16 8 5 9 6 9\nsplit_gain=273.757 139.702 112.594 108.873 44.9236 36.5353 34.2429 62.4809 32.2978 40.6488 31.3316 29.7644 26.6372 26.6071\nthreshold=0 1.2625707354522766 0.71447990227228164 0.63386352193863271 1.5000000000000002 0.33379148537332293 1 15.500000000000002 2 19.500000000000004 1.3983563208293723 2.5000000000000004 1.7488359477048623 2.5000000000000004\ndecision_type=9 2 2 2 2 2 9 2 9 2 2 2 2 2\nleft_child=1 13 3 10 -3 -4 7 -5 -7 11 -2 -10 -9 -1\nright_child=2 4 5 6 -6 8 -8 12 9 -11 -12 -13 -14 -15\nleaf_value=-0.050262368119104242 -0.035877775223614107 -0.024892834101692247 -0.077440720999933654 0.03107417558171682 0.032985881457565085 0.027108525556161306 0.021358741350651404 -0.059355580176193294 0.0042637804444536648 -0.017094989720428666 0.0085958421883624014 0.045028744353791963 -0.0044135554914518874 -0.0097017938520387575\nleaf_weight=371.88385229470441 250.35016923758667 52.451136190735269 8.5202143045607937 58.07632989884587 89.21836529369466 676.80023907299619 258.26365767640527 49.175411730073392 108.97063845046796 110.95458807697287 46.602489860495552 75.292939525097609 39.678484593518078 45.958566057495773\nleaf_count=88281 27663 5950 609 355 769 6986 3809 7056 1063 8593 1641 152 2488 1129\ninternal_value=-0.000817303 -0.0307657 0.00928086 -0.00594406 0.0115328 0.0202727 0.010904 -0.00837065 0.0212079 0.00649995 -0.0288928 0.0210741 -0.0351429 -0.0458349\ninternal_weight=2242.2 559.512 1682.69 702.147 141.67 980.539 405.194 146.93 972.018 295.218 296.953 184.264 88.8539 417.842\ninternal_count=156544 96129 60415 43012 6719 17403 13708 9899 16794 9808 29304 1215 9544 89410\ncat_boundaries=0 2 3 5\ncat_threshold=1690502980 489 2283290752 2148057091 4\nis_linear=0\nshrinkage=0.05\n\n\nTree=20\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 15 9 5 7 12 15 6 9 4 7 0 0 0\nsplit_gain=229.625 125.867 120.629 99.5986 50.0049 49.614 45.2485 43.978 40.717 37.6005 31.955 31.5071 30.8038 28.4407\nthreshold=0 1 1.5000000000000002 1.1196042404780329 27.500000000000004 45.500000000000007 2 -0.10458719046694327 1.5000000000000002 0.82392250492806862 2.5000000000000004 0.5235184930606408 0.19802510594200221 0.099086256596991759\ndecision_type=9 9 2 2 2 2 9 2 2 2 2 2 2 2\nleft_child=1 9 10 4 7 -6 -4 -3 11 13 -2 12 -5 -1\nright_child=2 3 6 8 5 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.052190869167402676 0.010768661344427456 -0.037139504444513083 0.021862994779961342 0.013453496488921239 -0.042655916267046436 0.012468499486674244 -0.044095406198451856 0.0067622779124974237 0.039164590321675047 0.034905997151036462 -0.047218639117964173 0.016286332986545814 -0.053516317376087266 0.015367474467725233\nleaf_weight=15.389599380781872 25.038331409101374 72.374797136406414 121.56852543004788 30.254254122963175 214.16217384807533 49.894861573120579 21.998817950836383 255.39923562121112 132.06231353105977 391.72638254379854 367.41118315196945 164.18157837406034 37.731515134626534 327.54929484336753\nleaf_count=1466 591 2385 2647 1308 27124 8367 2354 3777 881 2518 89463 2302 1923 9237\ninternal_value=-0.000473803 0.00858935 -0.0286504 -0.00339728 -0.0160832 -0.0322321 0.00998323 -0.00301502 0.0171105 0.0240082 -0.0435011 0.00445137 -0.0239512 0.0121928\ninternal_weight=2226.74 1690.73 536.017 956.061 591.831 264.057 143.567 327.774 364.23 734.665 392.45 232.167 67.9858 342.939\ninternal_count=156343 61288 95055 48067 41653 35491 5001 6162 6414 13221 90054 5533 3231 10703\ncat_boundaries=0 2 4 6\ncat_threshold=2424107193 788 1160015950 154 67441792 16\nis_linear=0\nshrinkage=0.05\n\n\nTree=21\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 9 4 11 8 16 8 2 8 16 9 11 4 11\nsplit_gain=244.139 131.316 79.4572 67.5582 50.2303 54.1161 45.2304 41.1935 36.5777 36.7186 62.313 43.3998 35.1827 35.7851\nthreshold=0 2.5000000000000004 0.78151929183826352 0.53589837997054501 25.500000000000004 1 73.500000000000014 0.36152556798409846 1.5000000000000002 2 1.5000000000000002 0.61311827956989251 1.242459270799247 0.6751623376623378\ndecision_type=9 2 2 2 2 9 2 2 2 9 2 2 2 2\nleft_child=2 12 3 -1 -5 6 -6 -4 -9 -10 11 -11 -2 -14\nright_child=1 -3 7 4 5 -7 -8 8 9 10 -12 -13 13 -15\nleaf_value=-0.036904315742243426 -0.045049849546072246 0.019142922869664653 -0.063866561634071548 0.018778681430351479 -0.033067463327518795 -0.038256518076674004 0.018461310032618305 0.047993531557088034 0.025459601680046448 -0.068943302325498487 0.025110023389057615 -0.0038652494400434481 -0.038667985615719179 0.038578439705894897\nleaf_weight=111.09823762282031 390.95148109612637 119.6786195813329 13.192036338499745 329.55141938201268 56.23997233598493 95.177347627119161 165.51245484256651 118.31340901739895 526.18857983441558 36.658222997677512 133.73225646710489 80.53943410620559 31.785950320423581 26.632680968090426\nleaf_count=7723 96636 1692 741 3977 1894 18834 6310 200 3707 2559 1374 3939 4104 2770\ninternal_value=0.00116664 -0.0268063 0.0108263 -0.00106819 0.00513776 -0.00906086 0.00507517 0.0208527 0.0221707 0.0181827 0.00180078 -0.0246321 -0.0396543 -0.00339525\ninternal_weight=2235.25 569.049 1666.2 757.579 646.481 316.93 221.752 908.624 895.432 777.118 250.93 117.198 449.37 58.4186\ninternal_count=156460 105202 51258 38738 31015 27038 8204 12520 11779 11579 7872 6498 103510 6874\ncat_boundaries=0 2 4 6\ncat_threshold=2552821915 788 2154505 528 2415976451 4\nis_linear=0\nshrinkage=0.05\n\n\nTree=22\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 15 8 9 6 3 4 7 11 9 16 11 15 4\nsplit_gain=229.735 123.497 85.2506 75.3713 63.8472 55.2209 50.2386 49.2304 55.6674 43.1503 58.7351 32.8648 37.8944 28.6382\nthreshold=0 1 39.500000000000007 4.5000000000000009 1.397635517509048 16.500000000000004 0.78151929183826352 22.500000000000004 0.73183303997988436 2.5000000000000004 2 0.78487818157070921 3 0.85800049017530078\ndecision_type=9 9 2 2 2 2 2 2 2 2 9 2 9 2\nleft_child=3 5 4 7 6 9 -3 8 -1 10 13 -4 -13 -2\nright_child=1 2 11 -5 -6 -7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.029012954974217627 0.014460703929691327 -0.029995490886728978 -0.044462317429973121 0.027232514778927637 0.02740663289261018 -0.018172575439168417 0.0084148578801366836 -0.050061447115983794 0.051317794806356121 0.0470007512111357 -0.0097765776449913119 0.027573396335611591 -0.033474799530888888 0.037245347041030351\nleaf_weight=106.65356657089433 263.6824420405319 148.96610576746752 156.14865423404262 48.481265184935182 196.85838722297922 82.057471317646559 196.14790841622744 280.71822765065008 26.009779807529412 171.42853243893478 141.60854451492196 38.329452206555288 41.93622368760407 292.89216508166282\nleaf_count=6002 5632 5830 26074 293 1560 7798 2005 78836 753 880 12033 748 6171 1636\ninternal_value=0.000258919 0.00865037 -0.00610062 -0.0306495 0.00475769 0.0206473 -0.00820347 -0.0382268 -0.0129973 0.0245642 0.0189959 -0.0311651 -0.00506793 0.0260294\ninternal_weight=2191.92 1730.06 778.387 461.863 541.972 951.669 345.114 413.382 132.663 869.612 698.183 236.414 80.2657 556.575\ninternal_count=156251 70367 42388 85884 9395 27979 7835 85591 6755 20181 19301 32993 6919 7268\ncat_boundaries=0 2 4 6 8\ncat_threshold=2827159364 234 1462530124 154 335560859 260 536871169 1\nis_linear=0\nshrinkage=0.05\n\n\nTree=23\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 9 15 4 3 15 8 15 2 13 16 2 9 6\nsplit_gain=211.49 121.513 116.395 77.219 55.0815 32.2408 31.8434 38.7799 36.3462 43.9031 28.5762 25.9044 25.7746 29.1734\nthreshold=0 1.5000000000000002 1 0.72470403997357347 10.500000000000002 2 6.5000000000000009 3 0.50776508281990418 18.500000000000004 4 0.3776613151524853 2.5000000000000004 1.8794842944366945\ndecision_type=9 2 9 2 2 9 2 9 2 2 9 2 2 2\nleft_child=2 -2 11 4 10 -3 -5 -8 -9 -10 -4 -1 13 -12\nright_child=1 5 3 6 -6 -7 7 8 9 -11 12 -13 -14 -15\nleaf_value=-0.02737672789109144 -0.044159896116348109 0.017030633622419463 0.0070521155301328537 0.026890827243927037 -0.049377298368020225 -0.042144954291754771 0.019536191096389235 -0.045472412970587309 -0.072418413417136673 0.008627191570742718 -0.045635951917481814 0.029852428262034322 0.0085555792677513062 0.0065079032604011726\nleaf_weight=19.776731814432424 354.99122754301061 138.48084382130764 183.32861721341033 207.56018251890782 96.246401729484205 16.780739354435354 183.42523004919349 56.267950677894987 17.626699360553175 161.38848247728311 103.16218754899455 540.20151114987675 53.001406127586961 35.819487284636125\nleaf_count=1756 93103 3548 4556 788 14036 1993 5755 2352 371 4549 14582 6264 1077 1955\ninternal_value=0.000282679 -0.0275214 0.00896375 -0.000566936 -0.0158356 0.00879893 0.0109277 0.00296934 -0.0103623 0.00028909 -0.00712569 0.0272994 -0.0199618 -0.0322465\ninternal_weight=2168.06 510.253 1657.8 1097.83 471.558 155.262 626.269 418.708 235.283 179.015 375.312 559.978 191.983 138.982\ninternal_count=156685 98644 58041 50021 36206 5541 13815 13027 7272 4920 22170 8020 17614 16537\ncat_boundaries=0 2 4 6 8 9\ncat_threshold=2554132635 788 84168782 10 604308672 16 805863441 4 268701705\nis_linear=0\nshrinkage=0.05\n\n\nTree=24\nnum_leaves=15\nnum_cat=4\nsplit_feature=4 6 5 16 9 16 16 6 5 0 5 9 16 13\nsplit_gain=182.213 129.502 64.0993 83.9371 68.8985 64.0577 45.1368 32.5379 31.8636 36.5602 34.2207 25.5944 30.1009 23.4941\nthreshold=0.70025987113965937 0.65445637638847654 1.4526685446458756 0 5.5000000000000009 1 2 -1.7225419984650159 0.67979967121712248 0.63666382790553666 0.92323772426952411 1.5000000000000002 3 28.500000000000004\ndecision_type=2 2 2 9 2 9 9 2 2 2 2 2 9 2\nleft_child=1 6 3 11 -5 -3 7 -1 -7 10 -10 12 -2 -8\nright_child=2 5 -4 4 -6 8 13 -9 9 -11 -12 -13 -14 -15\nleaf_value=0.029316489554715459 -0.026103788212421764 0.018538314249097382 0.033334876631866781 -0.024019374158050083 0.054040458387678229 -0.041426387800237705 -0.0045981451064922123 -0.051995530404587403 0.0095777859154025889 0.016982681185043782 -0.078610489843147682 0.040119543911216117 0.016326171579349694 -0.041854670185549503\nleaf_weight=12.275514441716949 37.402870974910911 265.19853653165046 295.62449589202879 261.45282385605969 30.730480080470443 86.670350606204011 156.83042405568995 270.15445770998485 19.630431694618892 83.905788486241363 23.46784025686793 103.3100723438547 426.95007356211136 57.105055180931231\nleaf_count=1073 3634 4881 2347 27628 58 14546 8088 67727 1686 1601 2528 999 5743 14009\ninternal_value=-0.000137308 -0.0160441 0.0132906 0.00636838 -0.0151368 0.00251 -0.0339133 -0.046722 -0.0174287 -0.00221795 -0.0391361 0.0174046 0.0126496 -0.0140382\ninternal_weight=2130.71 975.238 1155.47 859.846 292.183 478.873 496.365 282.43 213.674 127.004 43.0983 567.663 464.353 213.935\ninternal_count=156548 116139 40409 38062 27686 25242 90897 68800 20361 5815 4214 10376 9377 22097\ncat_boundaries=0 2 4 6 7\ncat_threshold=2430132243 6 373602315 274 2630560642 256 4759552\nis_linear=0\nshrinkage=0.05\n\n\nTree=25\nnum_leaves=15\nnum_cat=3\nsplit_feature=16 15 9 6 6 4 5 5 4 5 4 2 0 15\nsplit_gain=192.901 110.121 108.371 90.2205 33.7654 39.5458 31.8141 31.5923 30.9745 29.3368 24.7404 23.4648 23.1532 23.1254\nthreshold=0 1 2.5000000000000004 1.3596833190455928 2.6294216724632671 0.82392250492806862 1.3260108918225957 0.60757964625900185 0.73329725171479743 1.2834354001852917 1.1360565039478134 0.92425931159627261 0.3818364376436737 2\ndecision_type=9 9 2 2 2 2 2 2 2 2 2 2 2 9\nleft_child=1 4 6 7 5 -1 13 -3 -8 10 -5 12 -9 -2\nright_child=2 3 -4 9 -6 -7 8 11 -10 -11 -12 -13 -14 -15\nleaf_value=0.0056428208129056801 0.0029403239608166098 -0.031009861230868709 0.016269391709303107 -0.0088304318796491379 0.050510954739621722 0.03003851070669086 -0.049508134863150755 -0.053834925777866485 0.022591007596321846 0.035954430480625486 0.026879539999391019 0.03221409648838524 -0.0052987591291835424 -0.046979020677605954\nleaf_weight=301.01845706428867 32.398007132171188 221.21054765433655 119.24732595865498 123.82900516170776 98.008135966956615 371.21589902148116 22.852158458175836 27.033616929285927 38.702496298996266 123.53614172455855 78.378726112889126 34.931061684037559 201.47638874693075 315.99197473906679\nleaf_count=9897 6412 22254 1626 5412 394 2764 4266 3526 1573 872 1309 382 5203 90594\ninternal_value=0.00071914 0.00947436 -0.0251055 -0.0034165 0.0228944 0.019139 -0.0377621 -0.0171219 -0.00453933 0.0168603 0.00507544 -0.00538712 -0.0112503 -0.0436022\ninternal_weight=2109.83 1580.64 529.192 810.395 770.242 672.234 409.945 484.652 61.5547 325.744 202.208 263.441 228.51 348.39\ninternal_count=156484 52013 104471 38958 13055 12661 102845 31365 5839 7593 6721 9111 8729 97006\ncat_boundaries=0 2 4 5\ncat_threshold=271382675 788 86798414 27 1099174400\nis_linear=0\nshrinkage=0.05\n\n\nTree=26\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 15 5 15 6 7 15 5 16 2 7 4 6 8\nsplit_gain=167.104 111.175 66.4555 59.1885 47.1794 62.3833 54.6081 34.2496 33.4082 61.0101 37.5024 29.194 27.4496 26.3305\nthreshold=0 1 1.2834354001852917 2 1.0837699989682761 35.500000000000007 3 -0.93249115854835274 4 0.73145901676379144 17.500000000000004 1.049463203728759 1.4456576627456437 7.5000000000000009\ndecision_type=9 9 2 9 2 2 9 2 9 2 2 2 2 2\nleft_child=3 4 8 -1 5 7 -7 -2 9 -3 -11 -5 13 -4\nright_child=1 2 12 11 -6 6 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=0.00078337087920630892 -0.058613762243248616 -0.059286465810262604 0.011148370356318528 -0.05300638120800806 0.032615794595122226 0.018553317225865878 -0.038979501040212736 0.023916764448857274 -0.0055222304855453977 0.016602870681192362 -0.03919515699626179 -0.0109483945084561 0.034178062916824424 -0.046112209781655103\nleaf_weight=116.90255208365852 12.12901994702406 104.21801755361957 101.976337208529 207.90866507566534 371.61519252840662 62.195768885198049 92.651721464921138 334.53489688632544 372.78726816159906 69.8440749211004 51.42722678440623 52.170254500379087 108.93983269867022 23.938541524723405\nleaf_count=13135 532 18528 1097 60115 4506 6111 13326 3168 18388 850 6012 7416 807 2495\ninternal_value=-0.000148569 0.00654868 -0.00649243 -0.0297834 0.0189184 0.00910049 -0.0172109 0.0208689 -0.015412 -0.030093 -0.0071863 -0.0429972 0.0159233 -1.53876e-05\ninternal_weight=2083.24 1706.26 833.131 376.981 873.127 501.511 154.847 346.664 598.277 225.489 121.271 260.079 234.855 125.915\ninternal_count=156486 75820 48177 80666 27643 23137 19437 3700 43778 25390 6862 67531 4399 3592\ncat_boundaries=0 2 4 6 8 10\ncat_threshold=545457996 232 386166862 11 3314294280 256 17063940 8 272640130 256\nis_linear=0\nshrinkage=0.05\n\n\nTree=27\nnum_leaves=15\nnum_cat=5\nsplit_feature=16 9 15 5 15 9 11 15 1 3 4 7 15 0\nsplit_gain=179.65 95.4267 71.9688 72.8513 48.7575 39.5642 46.439 39.1357 43.2911 30.7624 34.0197 29.4959 24.8475 24.6923\nthreshold=0 1.5000000000000002 1 0.61651355393786489 2 3.5000000000000004 0.60510071474983762 3 5.8141963481903085 2.5000000000000004 -0.35739206006570806 2.5000000000000004 4 0.536844156764411\ndecision_type=9 2 9 2 9 2 2 9 2 2 2 2 9 2\nleft_child=2 4 9 7 -2 6 13 8 -4 -1 -11 -6 -3 -5\nright_child=1 12 3 5 11 -7 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=0.039798111782455521 0.013722921927479374 0.01428377179551988 -0.032560294497346876 -0.062291510940712239 0.0067695035475953836 0.045182424617775593 0.015160463373246414 -0.02940665522865182 0.01747602010608058 -0.038769363877524526 0.020676773105141506 -0.04734642607666064 -0.030547919539194113 -0.0070527983257482087\nleaf_weight=225.34869655757211 43.862242560542654 157.91705564004951 61.750466553203296 27.097773752815556 26.994723572046496 82.063854142324999 433.46773409345769 196.35506610234734 137.96473944457102 26.309360446350183 205.58211717300583 330.86938354239101 27.450822602666449 73.382030279957689\nleaf_count=808 5993 3611 4595 2527 569 290 8503 17677 6690 1603 2548 96268 2557 1859\ninternal_value=0.000506135 -0.0226256 0.00984488 0.00229958 -0.0366761 0.0130709 0.00806038 -0.014386 0.0018289 0.0260997 0.0137948 -0.0420841 0.00664374 -0.0224168\ninternal_weight=2056.42 587.094 1469.32 1012.08 401.726 616.011 533.948 396.07 199.715 457.24 231.891 357.864 185.368 100.48\ninternal_count=156098 108998 47100 42141 102830 13179 12889 28962 11285 4959 4151 96837 6168 4386\ncat_boundaries=0 2 4 5 7 8\ncat_threshold=271112347 534 117456972 11 19664896 948436994 144 69534912\nis_linear=0\nshrinkage=0.05\n\n\nTree=28\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 5 15 6 6 15 5 4 11 3 16 14 11 1\nsplit_gain=193.423 87.9445 75.6325 67.2073 32.6538 33.5111 30.8574 30.196 30.0685 29.7215 27.4324 26.602 25.5208 23.2589\nthreshold=0 1.2232175415258248 1 0.92467986616559794 0.72287946255114133 2 0.19880277570399549 0.45998124015447445 0.68123905080426839 17.500000000000004 3 69.500000000000014 0.57459036439227207 7.6776890754699716\ndecision_type=9 2 9 2 2 9 2 2 2 2 9 2 2 2\nleft_child=2 4 7 9 -2 -6 11 8 -1 10 -4 -5 -3 -9\nright_child=1 12 3 6 5 -7 -8 13 -10 -11 -12 -13 -14 -15\nleaf_value=-0.024190178449950299 -0.046982856701778347 -0.033512535861462946 -0.00021726201100761989 0.012847724298284784 -0.049860068563015746 0.003704969930797098 0.016837076017799976 0.021229752354223749 0.018564702263082985 -0.058186252425311627 -0.032526519545439107 -0.042822378561331087 0.016421015274508625 0.045079740057871645\nleaf_weight=60.643248760083225 280.14713615528308 31.052302000753116 177.55963058353518 40.257703177630901 34.023376753844786 85.710142462019576 384.56196410622215 363.56953316455474 122.62426863395376 40.988634736277163 97.373929321649484 44.092491716495715 126.29772042215336 143.73814765550196\nleaf_count=3097 84882 2560 6198 472 6769 8595 7912 3601 3186 5544 11405 5505 5944 418\ninternal_value=3.47133e-05 -0.0247777 0.00951961 -0.00115391 -0.0376562 -0.0156258 0.0108336 0.0214779 0.00436261 -0.018961 -0.0129234 -0.0164136 0.00641745 0.0280627\ninternal_weight=2032.64 557.231 1475.41 784.834 399.881 119.734 468.912 690.575 183.268 315.922 274.934 84.3502 157.35 507.308\ninternal_count=156088 108750 47338 37036 100246 15364 13889 10302 6283 23147 17603 5977 8504 4019\ncat_boundaries=0 2 4 6 8\ncat_threshold=405336217 534 117706830 155 1375862976 31 402653185 20\nis_linear=0\nshrinkage=0.05\n\n\nTree=29\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 15 5 5 7 6 3 8 6 11 15 2 16 11\nsplit_gain=168.93 94.3637 80.8818 57.5621 41.7292 42.6031 34.478 33.8043 33.6566 32.1545 29.2904 27.2622 25.9987 25.8581\nthreshold=0 1 0.95923833406345205 1.2625707354522766 16.500000000000004 0.61869964386369225 17.500000000000004 596.50000000000011 2.2726853643837845 0.77158385093167714 2 0.54420962937350625 3 0.99342105263157909\ndecision_type=9 9 2 2 2 2 2 2 2 2 9 2 9 2\nleft_child=1 7 4 -2 5 -3 -4 8 13 -6 11 -11 -5 -1\nright_child=3 2 6 12 9 -7 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=0.018927453078859565 -0.036106387181067345 -0.032360600798700956 0.017032578324819716 0.020157593998369163 -0.045077663626739199 0.015722581269546624 -0.063838516108885843 -0.075481756720721147 0.045891268418104064 0.041262901039202687 -0.031270460523467256 -0.077484094125476979 -0.02365266786221859 -0.076665380736299249\nleaf_weight=574.72576250118436 352.92271269325283 69.388553769909777 352.7547196696396 76.923301788920071 156.22579776219209 132.32904933142709 12.721905540034639 7.8744471727404735 130.29459677194245 36.681748652044917 43.346127750060987 4.6087053079681928 44.429602991353022 6.1977540308143935\nleaf_count=14917 87972 2052 6042 2454 24697 1381 1551 3211 578 812 6274 499 3711 178\ninternal_value=0.000341744 0.00845297 -0.00329489 -0.0253827 -0.0176926 -0.000895527 0.014051 0.021505 0.0229852 -0.0316979 -0.00676295 0.021424 0.00376995 0.0177776\ninternal_weight=2001.42 1527.15 808.057 474.276 442.58 201.718 365.477 719.093 711.218 240.862 84.6366 41.2905 121.353 580.924\ninternal_count=156329 62192 43308 94137 35715 3433 7593 18884 15673 32282 7585 1311 6165 15095\ncat_boundaries=0 2 4 6 8\ncat_threshold=343734427 516 386682958 138 536903937 16 2281701632 272\nis_linear=0\nshrinkage=0.05\n\n\nTree=30\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 9 7 11 15 3 0 5 16 7 5 15 5 9\nsplit_gain=159.883 227.827 53.3342 67.3425 44.1446 46.3452 37.4904 35.0738 31.4716 26.9103 25.5657 24.5789 24.5226 22.8797\nthreshold=0 2.5000000000000004 39.500000000000007 0.5100040016006403 1 3.5000000000000004 0.68216519083278071 -0.090262261503627914 2 15.500000000000002 1.4526685446458756 3 0.64429665732646468 4.5000000000000009\ndecision_type=9 2 2 2 9 2 2 2 9 2 2 9 2 2\nleft_child=4 2 3 -2 -1 7 -7 -6 11 -8 -10 -5 -13 -3\nright_child=1 13 -4 8 5 6 9 -9 10 -11 -12 12 -14 -15\nleaf_value=0.023649581080941676 -0.049971207539972834 0.012255349291182606 -0.041606553239529914 0.024520556019267611 -0.044051613198808272 -0.046882599557026165 0.024925314370909512 0.022989007351986255 -0.038796807136620683 -0.03594254461361978 0.021522171732612873 -0.053631351187526544 0.0065320541138026988 0.040736887276800901\nleaf_weight=592.32689623744227 99.416975708561949 165.7037298103096 357.99526913938462 107.35211622389033 20.798338180640712 68.819699406682048 50.646682310383767 190.21768052177504 79.000837600324303 27.013054761453532 21.804566910024732 23.629195187764708 53.657944214355666 123.43425851815846\nleaf_count=6436 6172 2786 116171 1940 723 6257 334 1577 8706 1160 467 2122 984 488\ninternal_value=0.000602165 -0.01296 -0.0277329 -0.0147533 0.0153111 0.00120157 -0.0202935 0.0162047 -0.00235571 0.00356535 -0.0233707 0.00990697 -0.0109174 0.0245127\ninternal_weight=1981.82 1031.99 742.857 384.862 949.822 357.495 146.479 211.016 285.445 77.6597 100.805 184.639 77.2871 289.138\ninternal_count=156323 139836 136562 20391 16487 10051 7751 2300 14219 1494 9173 5046 3106 3274\ncat_boundaries=0 2 4 6 8\ncat_threshold=402706441 20 621572175 30 2214731904 258 335807552 17\nis_linear=0\nshrinkage=0.05\n\n\nTree=31\nnum_leaves=15\nnum_cat=5\nsplit_feature=11 7 6 16 9 15 7 5 15 6 15 5 15 4\nsplit_gain=136.863 77.2715 77.5479 62.4623 76.5584 44.7403 42.4281 38.7084 38.142 37.4657 37.0197 43.485 33.277 23.2359\nthreshold=0.61311827956989251 41.500000000000007 0.54675896212523611 0 1.5000000000000002 1 25.500000000000004 1.4526685446458756 2 3.0394815027411126 3 1.3983563208293723 4 -0.41337908038667354\ndecision_type=2 2 2 9 2 9 2 2 9 2 9 2 9 2\nleft_child=3 5 12 8 9 -2 -6 -7 -1 -5 11 -4 -3 -10\nright_child=1 2 10 4 6 7 -8 -9 13 -11 -12 -13 -14 -15\nleaf_value=0.018773401240798984 0.032791481412466537 0.0044534688318278596 -0.059645573078626613 -0.047439514810601636 0.019387976011664337 0.00047994983662192031 -0.047847490967910744 0.033548899510920972 0.01971479678344162 0.045575574184347156 0.016127182060769713 0.029368055255255029 -0.040020417181229062 -0.067830097361445504\nleaf_weight=133.96146932500415 338.24541475594742 61.67668761589448 38.146085016487632 284.25419388432056 93.91117397043854 355.74336524057435 30.172309234680142 116.80673160078004 10.137548761034848 10.494041046360506 303.53751827363158 20.367458115564659 149.81070954576717 25.168875706614926\nleaf_count=3600 4134 10478 6256 54068 958 8039 2961 471 816 157 15875 272 46957 1323\ninternal_value=0.00067652 0.00926056 -0.00478091 -0.019511 -0.029453 0.0191868 0.00278709 0.00852757 0.00646791 -0.043985 0.00926931 -0.0245617 -0.0287643 -0.0335906\ninternal_weight=1972.43 1384.33 573.538 588.1 418.832 810.796 124.083 472.55 169.268 294.748 362.051 58.5135 211.487 35.3064\ninternal_count=156365 92482 79838 63883 58144 12644 3919 8510 5739 54225 22403 6528 57435 2139\ncat_boundaries=0 2 4 6 8 10\ncat_threshold=270353 4 69685260 15 922751168 28 33689984 6 2404147759 96\nis_linear=0\nshrinkage=0.05\n\n\nTree=32\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 9 8 15 6 6 2 4 15 9 11 16 8 4\nsplit_gain=138.402 122.429 59.8369 56.176 47.1257 39.0351 30.1647 37.9719 36.4643 27.3678 26.1128 25.401 20.5093 17.7612\nthreshold=0 2.5000000000000004 15.500000000000002 1 1.7488359477048623 1.6053745853402284 0.4809574483944104 0.22389574069364052 2 5.5000000000000009 0.57459036439227207 3 159.50000000000003 0.22389574069364052\ndecision_type=9 2 2 9 2 2 2 2 9 2 2 9 2 2\nleft_child=3 2 10 -1 6 -4 8 -8 -5 12 11 -2 -3 -7\nright_child=1 9 5 4 -6 13 7 -9 -10 -11 -12 -13 -14 -15\nleaf_value=0.030910966921386587 0.013233082529204306 0.0083353018105669693 -0.043478792351317207 0.0047492046466518474 0.027099521475787671 -0.049697871833897414 -0.028225843028921716 0.012220314628406531 -0.041533923726982641 0.040624182965842213 0.0080504701997788867 -0.041463383444040637 -0.07574502080558726 0.010104916642923945\nleaf_weight=287.6259655226022 28.809144638944417 156.82956734544132 353.2456144199241 83.328096049139276 182.86747526546242 14.846692949126007 69.105588102043839 336.44573798240162 79.113911962078419 76.38465311890468 143.65904480981408 57.790724300488364 6.605220742058008 56.392208796343766\nleaf_count=3554 504 1986 112960 3740 2073 2572 4972 6416 5397 229 3797 3048 832 4047\ninternal_value=9.10672e-05 -0.0142315 -0.0255751 0.0124476 0.00503087 -0.0366871 -0.00202093 0.00527585 -0.0201588 0.0161249 -0.00502002 -0.0265468 0.00449722 -0.00287385\ninternal_weight=1933.05 894.563 654.743 1038.49 750.861 424.485 567.993 405.551 162.442 239.819 230.259 86.5999 163.435 71.2389\ninternal_count=156127 129975 126928 26152 22598 119579 20525 11388 9137 3047 7349 3552 2818 6619\ncat_boundaries=0 2 4 6 7\ncat_threshold=336121881 20 84693005 14 545325072 17 2281711680\nis_linear=0\nshrinkage=0.05\n\n\nTree=33\nnum_leaves=15\nnum_cat=4\nsplit_feature=16 9 7 15 6 11 9 16 6 9 7 16 2 7\nsplit_gain=132.922 128.048 63.3919 44.9182 52.7621 39.507 36.5563 32.6957 25.8065 20.8901 31.7166 20.4114 19.7167 18.1007\nthreshold=0 2.5000000000000004 18.500000000000004 1 2.1859327562415869 0.5100040016006403 1.5000000000000002 2 1.6053745853402284 4.5000000000000009 33.500000000000007 3 0.4809574483944104 25.500000000000004\ndecision_type=9 2 2 9 2 2 2 9 2 2 2 9 2 2\nleft_child=3 2 5 -1 6 -2 7 12 11 10 -3 -4 -5 -14\nright_child=1 9 8 4 -6 -7 -8 -9 -10 -11 -12 -13 13 -15\nleaf_value=0.024418576227559396 -0.044933708176673144 0.022090406701197173 -0.00022700577770279909 -0.018391828509303088 0.033580700635906478 0.0055587423734761391 0.028901891685178434 -0.036555336887269078 -0.01039766005821678 0.03429127526210482 -0.030744329648553684 -0.049874331121728094 0.013710775242730282 -0.012020340104623217\nleaf_weight=417.73841404909035 49.277219328214414 79.111414523562416 32.562115596141666 126.47790878434898 121.22464355675038 168.09602986677783 78.66537199675804 70.501852868474089 61.228451979433885 98.314371631713584 43.01802170148585 269.17437708987563 201.29438397136983 102.30720485682832\nleaf_count=5648 2694 620 13384 4706 1275 5805 1344 10598 6500 461 1830 96467 1310 3801\ninternal_value=0.000498123 -0.0149395 -0.027436 0.0115869 0.00367522 -0.00604094 -0.00256714 -0.0075602 -0.0401974 0.0172595 0.00338172 -0.04619 -0.00184259 0.00502895\ninternal_weight=1918.99 800.782 580.338 1118.21 700.471 217.373 579.247 500.581 362.965 220.444 122.129 301.736 430.079 303.602\ninternal_count=156443 127761 124850 28682 23034 8499 21759 20415 116351 2911 2450 109851 9817 5111\ncat_boundaries=0 2 4 5 7\ncat_threshold=2684955 276 84168715 30 4123 101711904 547\nis_linear=0\nshrinkage=0.05\n\n\nTree=34\nnum_leaves=15\nnum_cat=5\nsplit_feature=11 9 15 16 11 5 15 2 16 0 8 11 15 1\nsplit_gain=107.576 62.4778 60.5163 56.1082 46.6124 44.582 39.3854 34.3899 38.754 27.6236 25.6899 32.765 28.6854 26.4391\nthreshold=0.61311827956989251 3.5000000000000004 0 1 0.99342105263157909 1.6789404729405224 2 0.93311231644491166 3 0.5756699792131007 25.500000000000004 0.65235814296241723 4 5.2658069133758554\ndecision_type=2 2 9 9 2 2 9 2 9 2 2 2 9 2\nleft_child=1 3 5 -1 10 7 -5 8 -2 -7 -4 -12 13 -13\nright_child=2 -3 4 6 -6 9 -8 -9 -10 -11 11 12 -14 -15\nleaf_value=-0.038223632148885439 -0.041891073879370799 0.025108932134689434 0.024764949132187047 0.011637019389418449 -0.078527359998974644 -0.039882629853146834 -0.036247970425694928 0.026026293761556962 -0.0063460848045524439 0.048350010145847014 -0.04418725002482124 -0.058560851031308564 0.021871358636976399 0.0019807647997726781\nleaf_weight=301.19794526524493 111.4364341077744 68.875080395373516 442.88110484380741 135.49324390871334 12.120463504572397 10.018410241580566 49.868070064883796 47.201207008212805 240.74481336648751 48.326473558379803 27.847735464252764 20.212061519312556 219.74739811250765 119.95490372888162\nleaf_count=52522 28162 455 3739 5997 1003 725 4744 1718 19223 323 6463 2254 15293 13761\ninternal_value=-0.000794445 -0.0191944 0.00707283 -0.0255045 0.0149126 -0.00752377 -0.00255447 -0.0136672 -0.0190457 0.0327828 0.0165481 0.00712375 0.0112053 -0.00663249\ninternal_weight=1855.93 555.434 1300.49 486.559 842.764 457.727 185.361 399.382 352.181 58.3449 830.643 387.762 359.914 140.167\ninternal_count=156382 63718 92664 63263 42513 50151 10741 49103 47385 1048 41510 37771 31308 16015\ncat_boundaries=0 2 4 6 8 10\ncat_threshold=973778864 96 80415694 18 922751040 26 138617280 258 1075908609 30\nis_linear=0\nshrinkage=0.05\n\n\nend of trees\n\nfeature_importances:\nrule1=68\nrule2=67\nz_neural=55\nsupport=53\ntan_parent=46\nz_frag=41\nz_parent=40\nrank_frag=25\nrank_neural=25\nparent_sim=20\nparent_rank=19\nfrag=13\nneural=8\nheavy=4\nn_steps=2\nheavy_parent=2\nn_cands=2\n\nparameters:\n[boosting: gbdt]\n[objective: lambdarank]\n[metric: ndcg]\n[tree_learner: serial]\n[device_type: cpu]\n[data_sample_strategy: bagging]\n[data: ]\n[valid: ]\n[num_iterations: 2000]\n[learning_rate: 0.05]\n[num_leaves: 15]\n[num_threads: 0]\n[seed: 0]\n[deterministic: 0]\n[force_col_wise: 0]\n[force_row_wise: 0]\n[histogram_pool_size: -1]\n[max_depth: -1]\n[min_data_in_leaf: 50]\n[min_sum_hessian_in_leaf: 0.001]\n[bagging_fraction: 0.8]\n[pos_bagging_fraction: 1]\n[neg_bagging_fraction: 1]\n[bagging_freq: 1]\n[bagging_seed: 7719]\n[bagging_by_query: 0]\n[feature_fraction: 0.8]\n[feature_fraction_bynode: 1]\n[feature_fraction_seed: 2437]\n[extra_trees: 0]\n[extra_seed: 11797]\n[early_stopping_round: 0]\n[early_stopping_min_delta: 0]\n[first_metric_only: 0]\n[max_delta_step: 0]\n[lambda_l1: 0]\n[lambda_l2: 1]\n[linear_lambda: 0]\n[min_gain_to_split: 0]\n[drop_rate: 0.1]\n[max_drop: 50]\n[skip_drop: 0.5]\n[xgboost_dart_mode: 0]\n[uniform_drop: 0]\n[drop_seed: 21238]\n[top_rate: 0.2]\n[other_rate: 0.1]\n[min_data_per_group: 100]\n[max_cat_threshold: 32]\n[cat_l2: 10]\n[cat_smooth: 10]\n[max_cat_to_onehot: 4]\n[top_k: 20]\n[monotone_constraints: ]\n[monotone_constraints_method: basic]\n[monotone_penalty: 0]\n[feature_contri: ]\n[forcedsplits_filename: ]\n[refit_decay_rate: 0.9]\n[cegb_tradeoff: 1]\n[cegb_penalty_split: 0]\n[cegb_penalty_feature_lazy: ]\n[cegb_penalty_feature_coupled: ]\n[path_smooth: 0]\n[interaction_constraints: ]\n[verbosity: -1]\n[saved_feature_importance_type: 0]\n[use_quantized_grad: 0]\n[num_grad_quant_bins: 4]\n[quant_train_renew_leaf: 0]\n[stochastic_rounding: 1]\n[linear_tree: 0]\n[max_bin: 255]\n[max_bin_by_feature: ]\n[min_data_in_bin: 3]\n[bin_construct_sample_cnt: 200000]\n[data_random_seed: 38]\n[is_enable_sparse: 1]\n[enable_bundle: 1]\n[use_missing: 1]\n[zero_as_missing: 0]\n[feature_pre_filter: 1]\n[pre_partition: 0]\n[two_round: 0]\n[header: 0]\n[label_column: ]\n[weight_column: ]\n[group_column: ]\n[ignore_column: ]\n[categorical_feature: 15,16]\n[forcedbins_filename: ]\n[precise_float_parser: 0]\n[parser_config_file: ]\n[objective_seed: 8855]\n[num_class: 1]\n[is_unbalance: 0]\n[scale_pos_weight: 1]\n[sigmoid: 1]\n[boost_from_average: 1]\n[reg_sqrt: 0]\n[alpha: 0.9]\n[fair_c: 1]\n[poisson_max_delta_step: 0.7]\n[tweedie_variance_power: 1.5]\n[lambdarank_truncation_level: 30]\n[lambdarank_norm: 1]\n[label_gain: ]\n[lambdarank_position_bias_regularization: 0]\n[eval_at: 1,5,25]\n[multi_error_top_k: 1]\n[auc_mu_weights: ]\n[num_machines: 1]\n[local_listen_port: 12400]\n[time_out: 120]\n[machine_list_filename: ]\n[machines: ]\n[gpu_platform_id: -1]\n[gpu_device_id: -1]\n[gpu_device_id_list: ]\n[gpu_use_dp: 0]\n[num_gpu: 1]\n\nend of parameters\n\npandas_categorical:[["acetylation", "acid_to_amide", "alcohol_from_ketone", "amide_to_acid", "amine_to_hydroxyl", "aza_ar", "c_demethylation", "c_methylation_al", "c_methylation_ar", "carbamoylation", "carboxylic_from_methyl", "deacetylation", "decarboxylation", "dedeoxyhexosylation", "dehexosylation", "dehydration", "dehydrogenation", "dehydrogenation_ring", "dehydroxylation", "deoxyhexosylation", "depentosylation", "ethylation", "formylation", "glucuronidation", "hexosylation", "hydration", "hydrogenation", "hydroxyl_to_amine", "hydroxylation_al", "hydroxylation_ar", "ketone_from_alcohol", "malonylation", "methoxylation_ar", "n_demethylation", "n_methylation", "o_demethylation", "o_methylation", "pentosylation", "prenylation_ar", "propylation", "sulfation"], ["", "acetylation", "acid_to_amide", "alcohol_from_ketone", "amide_to_acid", "amine_to_hydroxyl", "aza_ar", "c_demethylation", "c_methylation_al", "c_methylation_ar", "carbamoylation", "carboxylic_from_methyl", "deacetylation", "decarboxylation", "dedeoxyhexosylation", "dehexosylation", "dehydration", "dehydrogenation", "dehydrogenation_ring", "dehydroxylation", "deoxyhexosylation", "depentosylation", "ethylation", "formylation", "glucuronidation", "hexosylation", "hydration", "hydrogenation", "hydroxyl_to_amine", "hydroxylation_al", "hydroxylation_ar", "ketone_from_alcohol", "malonylation", "methoxylation_ar", "n_demethylation", "n_methylation", "o_demethylation", "o_methylation", "pentosylation", "prenylation_ar", "propylation", "sulfation"]]\n', 'ranker/meta.json': '{\n  "numeric": [\n    "frag",\n    "neural",\n    "parent_sim",\n    "parent_rank",\n    "z_frag",\n    "z_neural",\n    "z_parent",\n    "rank_frag",\n    "rank_neural",\n    "support",\n    "n_steps",\n    "tan_parent",\n    "heavy",\n    "heavy_parent",\n    "n_cands"\n  ],\n  "categorical": [\n    "rule1",\n    "rule2"\n  ],\n  "categories": {\n    "rule1": [\n      "acetylation",\n      "acid_to_amide",\n      "alcohol_from_ketone",\n      "amide_to_acid",\n      "amine_to_hydroxyl",\n      "aza_ar",\n      "c_demethylation",\n      "c_methylation_al",\n      "c_methylation_ar",\n      "carbamoylation",\n      "carboxylic_from_methyl",\n      "deacetylation",\n      "decarboxylation",\n      "dedeoxyhexosylation",\n      "dehexosylation",\n      "dehydration",\n      "dehydrogenation",\n      "dehydrogenation_ring",\n      "dehydroxylation",\n      "deoxyhexosylation",\n      "depentosylation",\n      "ethylation",\n      "formylation",\n      "glucuronidation",\n      "hexosylation",\n      "hydration",\n      "hydrogenation",\n      "hydroxyl_to_amine",\n      "hydroxylation_al",\n      "hydroxylation_ar",\n      "ketone_from_alcohol",\n      "malonylation",\n      "methoxylation_ar",\n      "n_demethylation",\n      "n_methylation",\n      "o_demethylation",\n      "o_methylation",\n      "pentosylation",\n      "prenylation_ar",\n      "propylation",\n      "sulfation"\n    ],\n    "rule2": [\n      "",\n      "acetylation",\n      "acid_to_amide",\n      "alcohol_from_ketone",\n      "amide_to_acid",\n      "amine_to_hydroxyl",\n      "aza_ar",\n      "c_demethylation",\n      "c_methylation_al",\n      "c_methylation_ar",\n      "carbamoylation",\n      "carboxylic_from_methyl",\n      "deacetylation",\n      "decarboxylation",\n      "dedeoxyhexosylation",\n      "dehexosylation",\n      "dehydration",\n      "dehydrogenation",\n      "dehydrogenation_ring",\n      "dehydroxylation",\n      "deoxyhexosylation",\n      "depentosylation",\n      "ethylation",\n      "formylation",\n      "glucuronidation",\n      "hexosylation",\n      "hydration",\n      "hydrogenation",\n      "hydroxyl_to_amine",\n      "hydroxylation_al",\n      "hydroxylation_ar",\n      "ketone_from_alcohol",\n      "malonylation",\n      "methoxylation_ar",\n      "n_demethylation",\n      "n_methylation",\n      "o_demethylation",\n      "o_methylation",\n      "pentosylation",\n      "prenylation_ar",\n      "propylation",\n      "sulfation"\n    ]\n  },\n  "params": {\n    "objective": "lambdarank",\n    "metric": "ndcg",\n    "eval_at": [\n      1,\n      5,\n      25\n    ],\n    "learning_rate": 0.05,\n    "num_leaves": 15,\n    "min_data_in_leaf": 50,\n    "feature_fraction": 0.8,\n    "bagging_fraction": 0.8,\n    "bagging_freq": 1,\n    "lambda_l2": 1.0,\n    "verbose": -1,\n    "seed": 0\n  },\n  "best_iteration": 35\n}', 'features.py': '"""Reranker features for one query\'s candidate shortlist; shared by rerank/bench.py and the notebook stage.\n\nfeatures(q, keys, smiles, neighbours, scorer, short, order=None) -> ({thr: [row, ...]}, {key: base rank})\n\nneighbours: {thr: [(inchikey14, smiles, spectral similarity), ...] best first}. order: candidate keys in the base\norder (the pipeline\'s list in the notebook); None ranks by z_neural + z_frag over all candidates (the bench).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom rdkit import Chem, DataStructs\n\nimport candidates as C\n\nK_NEIGHBOURS = 30\nNUMERIC = [\'frag\', \'neural\', \'z_frag\', \'z_neural\', \'rank_frag\', \'rank_neural\', \'base_rank\', \'n_short\', \'heavy\',\n           \'sup_max\', \'sup_top1\', \'sup_mean5\', \'sup_wsum\', \'z_sup\', \'best_nb_sim\']\n\n\ndef _rank(v):\n    return (-np.nan_to_num(np.asarray(v, float), nan=-1e9)).argsort(kind=\'stable\').argsort() + 1\n\n\ndef features(q, keys, smiles, neighbours, scorer, short=25, order=None):\n    import frag_rescore as frag\n    if order is None:\n        fs = np.array([np.nan if v is None else v for v in frag.score_molecule(q[\'spectra\'], smiles, cache={})])\n        ns = C.neural_scores(scorer, q, smiles)\n        base = C.zscore(np.nan_to_num(fs, nan=-1.0)) + C.zscore(ns)\n        idx = list(np.argsort(-base, kind=\'stable\')[:short])\n        base_rank = {keys[i]: r for r, i in enumerate(np.argsort(-base, kind=\'stable\'), 1)}\n        fs, ns = fs[idx], ns[idx]\n    else:\n        pos = {k: i for i, k in enumerate(keys)}\n        idx = [pos[k] for k in order[:short]]\n        base_rank = {keys[i]: r for r, i in enumerate(idx, 1)}\n        sm = [smiles[i] for i in idx]\n        fs = np.array([np.nan if v is None else v for v in frag.score_molecule(q[\'spectra\'], sm, cache={})])\n        ns = C.neural_scores(scorer, q, sm)\n    fs = np.nan_to_num(fs, nan=-1.0)\n    sk = [keys[i] for i in idx]\n    mols = [Chem.MolFromSmiles(smiles[i]) for i in idx]\n    fps = [C.FPG.GetFingerprint(m) if m is not None else None for m in mols]\n    zf, zn = C.zscore(fs), C.zscore(ns)\n    rf, rn = _rank(fs), _rank(ns)\n\n    out = {}\n    for thr, nbs in neighbours.items():\n        nfps = [C.FPG.GetFingerprint(m) if (m := Chem.MolFromSmiles(s)) is not None else None for _, s, _ in nbs]\n        sup = []\n        for k, fp in zip(sk, fps):\n            use = [(fp2, sim) for (ik, _, sim), fp2 in zip(nbs, nfps) if ik != k and fp2 is not None][:K_NEIGHBOURS]\n            if fp is None or not use:\n                sup.append((0.0, 0.0, 0.0, 0.0))\n                continue\n            tan = np.array(DataStructs.BulkTanimotoSimilarity(fp, [u[0] for u in use]))\n            sim = np.array([u[1] for u in use])\n            sup.append((float((tan * sim).max()), float(tan[0]), float(tan[:5].mean()),\n                        float((tan * sim).sum() / max(sim.sum(), 1e-9))))\n        sup = np.array(sup)\n        zs = C.zscore(sup[:, 3])\n        best_nb = nbs[0][2] if nbs else 0.0\n        rows = []\n        for j, k in enumerate(sk):\n            rows.append(dict(\n                cand=k, smiles=smiles[idx[j]], frag=fs[j], neural=ns[j], z_frag=zf[j], z_neural=zn[j],\n                rank_frag=int(rf[j]), rank_neural=int(rn[j]), base_rank=j + 1, n_short=len(sk),\n                heavy=mols[j].GetNumHeavyAtoms() if mols[j] is not None else 0, sup_max=sup[j, 0], sup_top1=sup[j, 1],\n                sup_mean5=sup[j, 2], sup_wsum=sup[j, 3], z_sup=zs[j], best_nb_sim=best_nb))\n        out[thr] = rows\n    return out, base_rank\n', 'rerank/ranker.txt': 'tree\nversion=v4\nnum_class=1\nnum_tree_per_iteration=1\nlabel_index=0\nmax_feature_idx=14\nobjective=lambdarank\nfeature_names=frag neural z_frag z_neural rank_frag rank_neural base_rank n_short heavy sup_max sup_top1 sup_mean5 sup_wsum z_sup best_nb_sim\nfeature_infos=[0:1] [-3.9804673194885254:9.1159811019897461] [-4.529852942622024:4.8906235308439099] [-3.9828164689386556:3.8303822272620542] [1:25] [1:25] [1:25] [1:25] [3:149] [0:0.95811279388025883] [0:1] [0.018047619047619052:1] [0:0.99028608829399944] [-3.6161922200894856:4.7999508003935869] [0:0.999999999999996]\ntree_sizes=1735 1743 1744 1745 1733 1733 1739 1749 1748 1749 1748 1743 1742 1741 1748 1748 1747 1753 1752 1750 1749 1759 1750 1757 1742 1744 1762 1756 1753 1758 1748 1769 1758 1765 1748 1756 1752 1763 1755 1768 1758 1758 1761 1751 1744 1763 1755 1745 1749 1754 1749 1749 1754 1762 1754 1751 1757 1753 1754 1757 1756 1771 1753 1757 1741 1760 1748 1734 1765 1765 1742 1761 1751 1763 1758 1738 1773 1753 1771 1760 1757 1755 1736 1758 1738 1751 1740 1743 1758 1756 1744 1746 1752 1752 1744 1762 1744 1747 1758 1755 1753 1755 1749 1735 1756 1765 1748 1752 1735 1753 1759 1743 1761 1759 1739 1759 1761 1737 1751 1763 1740 1768 1747 1734 1761 1744 1755 1750 1757 1747 1742 1776 1753 1752 1779 1731 1758 1745 1768 1748 1751 1746 1741 1752 1747 1746 1732 1772 1746 1760 1761 1739 1753 1751 1745 1748 1750 1751 1752 1772 1751 1754 1756 1738 1744 1741 1747 1749 1751 1758 1751 1761 1749 1760 1750 1761 1764 1766 1749 1764 1755 1753 1764 1735 1728 1754 1763 1745 1743 1745 1754\n\nTree=0\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 2 9 13 13 9 2 3 3 2 13 9 9\nsplit_gain=4801.93 702.322 605.773 288.936 227.514 146.92 124.71 110.997 98.6093 93.5006 54.2542 52.7283 45.8406 37.2706\nthreshold=2.5000000000000004 0.90622825774246907 0.82240565329519633 0.17487998293420839 0.7181456338936627 0.80356500410247722 0.25584294134921287 1.645599366937252 1.4766501342390617 0.87699916622931584 1.4900624917218295 1.4261840188530308 0.13773028815655583 0.4340170232365283\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 6 4 -3 8 9 7 -2 13 -4 -11 -5 -6 -1\nright_child=1 3 5 11 12 -7 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=-0.072607387530977105 -0.091322974866102802 -0.068359388826759057 0.011951533044341586 0.016434672073567382 -0.033271080014919713 0.090463213746942273 -0.04338534386127596 0.017540010572852673 0.03520519347940064 0.032852524721140267 0.087792820512632944 0.070223548967330782 0.058989498848978028 -0.0054160724799476659\nleaf_weight=104.33919293433428 973.3242185076233 83.7103799902834 106.73972450755537 92.486301978351548 14.569688575342296 847.98411024548113 182.41151636821451 23.688564769923687 34.144551169127226 69.085838709026575 137.49312858842313 90.356926812208258 115.75834215059876 25.878140019252896\nleaf_count=1851 20280 1684 831 1236 134 2146 3298 332 219 421 311 601 586 301\ninternal_value=-0.000681361 -0.065199 0.0633932 0.00807716 -0.000633133 0.079627 -0.0817524 -0.0887215 -0.0396507 0.0501047 0.069666 0.0432533 0.0485007 -0.0593975\ninternal_weight=2901.97 1445.98 1455.99 266.554 294.69 1161.3 1179.42 997.013 164.362 313.319 206.579 182.843 130.328 130.217\ninternal_count=34231 27431 6800 3521 3091 3709 23910 20612 2371 1563 732 1837 720 2152\nis_linear=0\nshrinkage=0.05\n\n\nTree=1\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 9 9 13 13 2 2 4 2 4 13 9 3\nsplit_gain=5851.48 2406.81 509.249 414.895 364.392 232.639 227.338 226.647 136.217 131.925 104.112 91.4438 88.7849 86.8105\nthreshold=1.5000000000000002 0.83711243686168813 0.18560323472877255 0.3373762830616705 0.98990312258122526 1.797896805829583 1.6019071606498703 1.3673476462045733 1.5000000000000002 -1.0000000180025095e-35 1.5000000000000002 2.2531977117618109 0.16208072556515951 1.6668156375686918\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 3 10 7 6 -4 13 12 -6 -5 -3 -10 -2 -1\nright_child=1 2 5 9 8 -7 -8 -9 11 -11 -12 -13 -14 -15\nleaf_value=0.0046032232592304843 -0.079434793347377977 0.029489890104237401 0.013527367885694136 -0.047591620223679229 0.083330449099811008 0.058068816860469268 0.066275565093898176 -0.018789084121500921 0.030423162483093364 0.0027149723253143644 -0.051827573877787816 0.081954517585290029 -0.056731581977248896 0.055778438367981444\nleaf_weight=428.75113955885172 1682.3892152197659 44.728958554565907 960.88929153280333 236.60858978144825 1233.821132697165 421.42971028015018 353.15651259198785 207.17590357642621 244.73933956772089 288.39901759428903 295.36396555323154 133.64322206377983 596.74296720698476 101.78952820599079\nleaf_count=1668 17178 75 2342 1176 1202 481 417 983 266 1098 2193 89 4857 136\ninternal_value=-0.000821692 -0.0334832 0.0136535 -0.06044 0.0611128 0.0271389 0.0352478 -0.0689669 0.0752596 -0.0200044 -0.0410776 0.0487919 -0.0735179 0.0145082\ninternal_weight=7229.63 4733.73 1722.41 3011.32 2495.9 1382.32 883.697 2486.31 1612.2 525.008 340.093 378.383 2279.13 530.541\ninternal_count=34161 30383 5091 25292 3778 2823 2221 23018 1557 2274 2268 355 22035 1804\nis_linear=0\nshrinkage=0.05\n\n\nTree=2\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 13 9 2 9 2 13 6 13 8 1 4 4\nsplit_gain=4443.98 2011.28 408.564 379.286 362.299 205.669 200.354 191.861 150.833 99.8863 69.4113 67.7206 65.5463 56.9826\nthreshold=1.5000000000000002 0.83711243686168813 0.98990312258122526 0.15432152518221512 0.69655208401287494 0.25946837494062885 1.3945900949848309 1.9383663450051654 4.5000000000000009 0.19172391214649501 21.500000000000004 5.005284547805787 1.5000000000000002 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 6 12 5 11 -1 8 -5 -6 -10 -2 -3 -11\nright_child=1 3 -4 7 9 -7 -8 -9 10 13 -12 -13 -14 -15\nleaf_value=0.0059883113023627994 -0.080106195674617214 0.023828414534023571 0.07495483289801376 0.028426963868726469 -0.037765951572100713 -0.036147243628281589 0.055221316424925421 0.059114421185198676 -0.049463310800633549 0.045118113572607699 0.0021943231039263014 -0.056061298998360182 -0.054563720530724204 -0.010566645507163354\nleaf_weight=419.5691128987819 1236.097258857917 30.316675566136837 1349.2235619351268 649.41609227377921 330.30988324712962 455.12682955199853 406.71998985856771 332.64212744310498 95.49228941462934 63.706078266724944 199.18034676276147 399.17466874979436 193.68935801647604 157.54560226853937\nleaf_count=1676 14421 63 1614 1355 2844 2881 534 399 603 151 791 3779 1867 1077\ninternal_value=0.000150637 -0.0302356 0.0580006 0.0159829 -0.0564856 -0.0659947 0.0302595 0.0265149 0.0149783 -0.0203925 -0.0146566 -0.0742746 -0.0438955 0.0055978\ninternal_weight=6318.21 4142.7 2175.51 1500.74 2641.96 2090.4 826.289 1276.73 944.089 551.562 294.673 1635.27 224.006 221.252\ninternal_count=34055 30231 3824 5078 25153 21081 2210 3148 2749 4072 1394 18200 1930 1228\nis_linear=0\nshrinkage=0.05\n\n\nTree=3\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 4 7 2 3 1 2 3 10 10 10 12 14\nsplit_gain=3444.39 863.88 557.713 223.777 159.694 158.615 117.781 113.806 106.292 94.5325 94.3465 86.9755 99.9748 72.0091\nthreshold=2.5000000000000004 0.23171031113136512 1.5000000000000002 6.5000000000000009 1.527243955281093 1.0446204841859017 4.1305465698242196 0.73126556468747761 1.2135643508474616 0.2983765383608275 0.45857843137254906 0.45857843137254906 0.28193735001350817 0.62043603938719272\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 5 9 6 -1 -2 8 11 -4 13 -3 -13 -5\nright_child=1 7 3 10 -6 -7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=0.038329963559108317 -0.073773884737241169 -0.04062211060757407 -0.041498645027720944 0.028984052588581923 0.0096968376021522515 0.070126193289919148 -0.046107584450913341 0.024476793927385582 0.031597811338818081 0.0049686276613359562 0.050344450111861727 0.043151097332513336 -0.018333405596336103 -0.011892833288823746\nleaf_weight=664.83318253606558 1099.569488263689 303.44983625970781 197.25582376122475 305.53783008083701 76.551655991002917 972.89764110930264 603.14947702875361 264.76682269573212 112.36978472350165 243.93651999160647 304.81568913348019 83.798044933006167 299.20338246226311 165.43173566740006\nleaf_count=2284 14885 2523 737 967 498 1105 6398 999 320 569 495 196 1478 602\ninternal_value=-0.000581476 -0.0395314 0.0382071 0.0125545 -0.0608318 0.0572499 -0.0640065 -0.00387442 -0.0132902 -0.0158536 0.02871 -0.0206735 -0.00480368 0.014631\ninternal_weight=5697.57 2842.86 2854.71 1216.98 1779.27 1637.73 1702.72 1063.59 798.821 441.192 775.785 686.451 383.001 470.97\ninternal_count=34056 27297 6759 3370 21781 3389 21283 5516 4517 1306 2064 4197 1674 1569\nis_linear=0\nshrinkage=0.05\n\n\nTree=4\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 6 9 9 14 4 13 9 4 0 3 3 3\nsplit_gain=2907.97 1012.11 533.633 349.173 215.313 149.301 139.178 113.995 78.6067 76.5202 75.2816 100.07 70.4738 61.7126\nthreshold=0.90622825774246907 1.5000000000000002 1.5000000000000002 0.3373762830616705 0.204879909209113 0.61729885911214344 1.5000000000000002 1.4630009291942681 0.41994549103866291 2.5000000000000004 0.41030547993300764 1.2501037838947984 1.6668156375686918 1.6668156375686918\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 -2 6 -4 10 -3 9 13 -6 11 -1 -8 -7\nright_child=2 3 4 -5 7 8 12 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.029962187780542094 0.066098750296295125 -0.002360428738365138 -0.019656236689161818 -0.0066387753152612281 0.036087286300743024 -0.047340840746217444 -0.06163892492476869 0.047852067099237477 0.023897003153494328 -0.0038411015821197846 0.051972073087302617 0.053002241973091571 0.0029244195684331436 0.033720828099968451\nleaf_weight=78.318638999015093 1161.0760412905365 110.8898276630789 279.77863397495821 424.92986825481057 208.81400244683027 125.44774882495403 1761.4882232975215 465.46315573714674 102.6739913597703 279.88958799093962 305.33057974278927 66.121167045086622 43.214289104565978 28.072642728686333\nleaf_count=440 1643 477 2107 2381 395 376 22556 790 248 1182 961 110 394 51\ninternal_value=0.000565853 -0.0318384 0.0417804 -0.0476614 0.0188631 0.020614 -0.0567532 0.0301664 -0.00983027 0.0132584 0.0379387 0.00812028 -0.0600922 -0.0323957\ninternal_weight=5441.51 3046.49 2395.02 2340.52 1233.95 705.965 1915.59 954.167 256.194 488.704 449.77 144.44 1804.7 153.52\ninternal_count=34111 27994 6117 25808 4474 2186 23427 2367 675 1577 1511 550 22950 427\nis_linear=0\nshrinkage=0.05\n\n\nTree=5\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 9 13 6 3 13 13 1 14 13 9 14 3 3\nsplit_gain=2438.72 1024.84 353.358 338.387 193.144 155.294 134.268 122.684 106.365 70.9274 67.5039 58.1693 56.829 47.6409\nthreshold=0.9841337858536412 0.2263672111998514 0.76064789851928383 1.5000000000000002 2.1288854818560901 1.732394541732335 0.98990312258122526 5.6900126934051523 0.53439337717742574 1.4261840188530308 0.14527096610254445 0.43606152963336842 1.120602370755815 1.0759893662396698\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 12 6 7 10 8 -1 -2 -4 -5 -9 -3 -10\nright_child=3 2 9 5 -6 -7 -8 11 13 -11 -12 -13 -14 -15\nleaf_value=-0.064025073116049908 0.056051557124999124 -0.032548423551525631 0.014302908023608216 -0.02408737469826934 0.052990467055709194 0.053932629158075077 0.066596071420011643 0.00604938988210201 -0.012865345098156025 0.045694158227686361 0.012965683177370849 -0.042342869427255583 0.013919422173498142 0.03228581841723565\nleaf_weight=1319.410543045029 260.56233851239085 475.58136792993173 471.08866842463613 191.87721086340025 39.865077638998628 173.29918694123626 930.39207071065903 86.13494667597115 117.29330687969923 291.62035315483809 337.89657553751022 219.44025872973725 75.761614148970693 114.6045904904604\nleaf_count=18883 404 4100 1633 1749 136 285 1383 395 293 528 1511 2315 302 186\ninternal_value=0.00049864 -0.0286926 0.00431906 0.0413964 -0.0547353 0.0129937 0.055412 -0.0574091 0.0341902 0.0263494 -0.000474552 -0.0287269 -0.0261495 0.0094913\ninternal_weight=5104.83 2978.9 1314.05 2125.93 1664.85 703.073 1422.85 1624.99 492.46 762.709 529.774 305.575 551.343 231.898\ninternal_count=34103 28292 6563 5811 21729 3545 2266 21593 883 2161 3260 2710 4402 479\nis_linear=0\nshrinkage=0.05\n\n\nTree=6\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 9 13 13 2 3 4 13 1 13 4 14 3 3\nsplit_gain=2161.72 823.296 327.902 324.292 145.495 102.539 98.0668 87.0559 78.6123 76.1271 69.7148 63.8108 61.2996 58.1472\nthreshold=0.9841337858536412 0.18332356235995864 0.98990312258122526 0.73052812899374042 1.7271604568384149 1.0126424540725496 1.5000000000000002 1.4261840188530308 2.8757443428039555 2.2531977117618109 4.5000000000000009 0.5052740492231943 0.52392582359735729 1.1066554537313842\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 4 13 8 -1 -4 -5 -2 -8 -7 -10 -6 -3\nright_child=2 3 6 7 12 10 9 -9 11 -11 -12 -13 -14 -15\nleaf_value=-0.061035579905170591 -0.053666789981487023 -0.035917653230188681 0.065465636910982478 0.006336452621713261 0.0063821088374137608 0.018481288978062008 0.018197813906443788 0.040029709642264745 0.034944281037715692 0.062906703977363657 -0.044187766160275727 -0.0072016700604853523 0.056291638588118656 0.0069132695879437741\nleaf_weight=1200.9444685546914 63.98294763173908 510.91583435353823 827.21581632271409 512.33287306409329 87.045137949287891 69.243983993306756 251.19743228331208 305.79795476607978 126.86069105565548 153.56717309728265 120.36315569211729 300.96303602866828 211.1353458724916 93.409004309214652\nleaf_count=17894 868 5399 1392 2025 450 300 743 596 398 175 1660 1392 366 456\ninternal_value=4.45374e-05 -0.0282964 0.0394676 -0.00154581 0.0142977 -0.05564 0.0555827 0.0189637 -0.00242299 0.0352735 -0.0213243 0.00534803 0.041792 -0.0292969\ninternal_weight=4834.97 2813.01 2021.97 1422.46 789.987 1390.55 1231.98 818.131 491.807 404.765 189.607 427.824 298.18 604.325\ninternal_count=34114 28330 5784 8476 3474 19854 2310 2621 2658 918 1960 1790 816 5855\nis_linear=0\nshrinkage=0.05\n\n\nTree=7\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 13 2 10 13 4 10 13 13 1 2 14 12\nsplit_gain=1865.03 566.24 463.632 166.109 140.199 106.06 83.2647 67.9945 65.8825 59.8886 56.0665 49.8168 48.0055 46.766\nthreshold=2.5000000000000004 0.90622825774246907 1.0652999387949482 1.6903347241130793 0.25472370766488417 1.5007010094932978 1.5000000000000002 0.34228550829127619 1.8602164531472061 -0.66562128948688848 6.2170217037200937 1.645599366937252 0.44746339656424899 0.17056733839577878\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 3 9 10 7 -4 -3 -8 -1 11 -2 -7 -6\nright_child=1 5 6 -5 13 12 8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0248037747034737 -0.062102179325589472 -0.033448723233180294 0.063995914324360401 0.04198326670683597 0.018267983549099379 -0.049608623996162393 0.016404590758968846 0.0079428314561927945 0.054474164687302644 0.0053772675268427871 -0.024247502969961886 -0.0050650236161153064 0.034186481654396243 -0.029668804299106472\nleaf_weight=224.14935901388526 1054.5996568254195 178.3628089511767 723.19721967633814 275.05780924111605 58.193942358717322 17.360644735395908 215.98823412694037 221.37412909511477 241.7225165143609 607.96777274552733 123.71012212289497 39.915880295448005 274.06510280165821 373.32829983904958\nleaf_count=1000 18136 1806 1073 597 438 173 535 793 319 3251 1406 297 720 3633\ninternal_value=-0.00046307 -0.0318335 0.0316315 0.00837398 -0.0477424 0.00615388 0.0534114 -0.0105635 0.0365845 -0.00277268 -0.0564179 -0.0600286 0.0290421 -0.0231769\ninternal_weight=4628.99 2340.91 2288.08 1107.17 1649.75 691.163 1180.91 399.737 457.711 832.117 1218.23 1094.52 291.426 431.522\ninternal_count=34177 27402 6775 4848 23910 3492 1927 2599 854 4251 19839 18433 893 4071\nis_linear=0\nshrinkage=0.05\n\n\nTree=8\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 2 9 10 13 2 6 10 6 10 13 14 10\nsplit_gain=1594.66 760.516 249.228 220.809 112.865 85.7704 84.326 64.5307 58.0392 61.9847 50.05 49.624 45.4467 46.8727\nthreshold=4.5000000000000009 1.1800529563749198 0.82240565329519633 0.19453332722939159 0.56386946386946402 1.1020406375351024 1.8194180785221719 1.5000000000000002 0.27005577005577014 1.5000000000000002 0.95849462365591409 0.55249343846235999 0.48932556351895706 0.16585760517799356\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -2 11 -5 8 -3 9 12 -6 -1 -4 -14\nright_child=3 7 6 5 10 -7 -8 -9 -10 -11 -12 -13 13 -15\nleaf_value=-0.043149899542467715 -0.05689408913355766 0.059674055676487738 0.041675264214881443 -0.032990229264669102 0.014407330264714075 0.0073458376239324621 0.040679966763337939 0.036203825713958769 0.022975024788020864 -0.032583764491898003 -0.083094740730588923 -0.011748307383573589 0.011299449814486634 -0.06423841669511407\nleaf_weight=385.85310838837177 952.55513631924987 761.13664193823934 73.352425338700414 332.75167682673782 216.48683263175189 217.28468765132129 269.50688597094268 486.99425653228536 295.34086419269443 139.472651748918 13.018322080373762 187.87038290547207 72.802453946322203 27.486897978931665\nleaf_count=4008 17668 1141 214 4131 991 777 660 1145 1021 1273 61 915 167 108\ninternal_value=-0.00047746 0.0210018 -0.000954036 -0.0423378 -0.0210674 -0.0170716 0.0174503 0.0505528 0.00711117 -0.00787769 0.00854018 -0.0329055 0.0120397 -0.00983347\ninternal_weight=4431.91 2929.32 1681.19 1502.59 803.229 550.036 877.962 1248.13 608.455 313.114 229.505 573.723 173.642 100.289\ninternal_count=34280 11704 9418 22576 5975 4908 3443 2286 2783 1762 1052 4923 489 275\nis_linear=0\nshrinkage=0.05\n\n\nTree=9\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 9 3 4 10 0 4 9 12 14 10 12 6 3\nsplit_gain=1346.41 510.813 213.982 116.666 94.0942 109.713 78.9077 51.559 53.771 51.0651 50.2452 46.8855 42.5282 38.6312\nthreshold=2.5000000000000004 0.22973180046404876 1.0446204841859017 1.5000000000000002 0.27005577005577014 0.38432741290825173 2.5000000000000004 0.34268005210784097 0.23261107465006284 0.63300027894733968 0.91945945945945962 0.25299979771225373 1.5000000000000002 1.9345448480507716\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 6 4 -4 5 -1 -2 11 -9 12 -6 -3 -7 -5\nright_child=1 7 3 13 10 9 -8 8 -10 -11 -12 -13 -14 -15\nleaf_value=-0.042967205595083775 -0.0092320262567409869 -1.8751134886401294e-05 0.055218651428418676 0.015459247469606502 0.027611587552489443 0.046442609885333289 -0.052002641880829571 0.046379338031630864 0.0039315804619516406 -0.029629805492321533 -0.054935468266248511 -0.04272033142919035 -0.00095509878008456263 0.046592350831854035\nleaf_weight=145.43180234450847 119.40417680237442 172.88990035234019 702.2099674642086 376.64882706943899 459.33828268200159 99.602819796651602 1160.7009370909072 87.021148277446628 491.13368745334446 56.585099441930652 18.297705959528685 101.3301881570369 89.292075203731656 135.42686352133751\nleaf_count=914 1036 1485 1124 1193 1766 584 20813 288 2650 325 84 1035 481 252\ninternal_value=-0.00014639 -0.0280672 0.0284396 0.0419859 0.00947454 -0.00870599 -0.0480235 0.00192458 0.0103897 0.0116836 0.0243411 -0.0158958 0.0241503 0.0237677\ninternal_weight=4215.31 2132.48 2082.83 1214.29 868.548 390.912 1280.11 852.375 578.155 245.48 477.636 274.22 188.895 512.076\ninternal_count=34030 27307 6723 2569 4154 2304 21849 5458 2938 1390 1850 2520 1065 1445\nis_linear=0\nshrinkage=0.05\n\n\nTree=10\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 4 9 2 9 13 9 14 5 13 14 11 2\nsplit_gain=1342.53 474.276 229.356 173.183 130.836 107.916 95.2986 64.0303 63.776 54.55 53.3819 40.7396 41.8382 42.7208\nthreshold=0.98990312258122526 3.5000000000000004 1.5000000000000002 0.15591134728417663 1.6903347241130793 0.25946837494062885 1.9383663450051654 0.3373762830616705 0.48932556351895706 2.5000000000000004 -0.13286243718190949 0.38646208954784628 0.088508261124055423 1.527243955281093\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -2 -4 7 11 -5 8 -1 -6 -11 12 -3 -14\nright_child=2 5 3 6 9 -7 -8 -9 -10 10 -12 -13 13 -15\nleaf_value=-0.0022089485043470312 0.056198483000775248 0.030316695834465997 -0.041387970649141237 0.014013520449915435 0.052606877369735641 -0.018089431963749732 0.048528285265798998 0.007660817707037474 -0.036224083758056649 -0.033906583423218804 0.030401567799212825 -0.056865460654958511 -0.054694264007945484 0.066144458620277263\nleaf_weight=255.7900023041293 688.71730848122388 26.388434024527669 106.34005563333631 584.12288600485772 130.35923722758889 280.65353047801182 304.14638814888895 314.9161763265729 298.0901671575848 49.084596586413682 89.836284858174622 869.83058966207318 66.964580429717898 7.3464696742594233\nleaf_count=1843 1483 133 1345 2111 214 2940 467 1362 2476 250 263 18191 1012 16\ninternal_value=-0.000154956 -0.0242474 0.0340352 0.0186471 -0.000884541 -0.0454806 0.0258728 -0.0103258 -0.0205477 0.0295019 0.00759978 -0.0533746 -0.0229586 -0.0420284\ninternal_weight=4072.59 2389.26 1683.33 994.609 1138.08 1251.18 888.269 868.796 553.88 269.28 138.921 970.53 100.699 74.3111\ninternal_count=34106 28700 5406 3923 6408 22292 2578 5681 4319 727 513 19352 1161 1028\nis_linear=0\nshrinkage=0.05\n\n\nTree=11\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 9 6 10 10 9 3 3 6 0 0 8 3 2\nsplit_gain=1107.88 492.563 142.875 118.993 79.8414 79.6898 77.2986 56.02 49.6066 45.0265 43.7966 40.0388 40.779 39.1345\nthreshold=0.9841337858536412 0.19197000671990455 1.5000000000000002 0.56386946386946402 0.95849462365591409 0.14527096610254445 2.1288854818560901 0.89003831859028237 4.5000000000000009 0.56009724150698792 0.017893471098448719 37.500000000000007 0.86289377661312738 -0.047463665161103223\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 7 8 11 -4 10 9 -3 -2 -1 -5 13 -13\nright_child=2 3 5 4 -6 -7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.020214960116166025 -0.0082040444351776255 -0.0010837729166000719 -0.017567537410126483 0.032063131324352727 -0.073510374293703454 0.023343847741443526 0.038205888750642791 0.048781427924668427 -0.02780402733747063 0.039230203403183485 -0.052554731110797615 -0.06270006276072225 0.036206924761732519 0.0073905242916612532\nleaf_weight=119.6233398441691 85.72733099758625 373.46872588433325 164.00221441080794 337.78216785471886 21.451081858947873 427.31767196860164 25.370632821694016 814.037334298715 323.86887614196166 118.16089328564703 944.70930966991 33.569061510264874 45.836373569443822 46.544575931038707\nleaf_count=1343 287 2014 1781 1158 137 1768 136 1632 3999 265 18794 304 129 341\ninternal_value=-0.00016932 -0.0226459 0.0315615 -0.000293002 0.0187002 0.0119866 -0.0468802 0.0429098 -0.0135157 0.0193432 -0.0489424 0.0231146 -0.000947632 -0.0223896\ninternal_weight=3881.47 2272.22 1609.25 1182.52 485.183 591.32 1089.7 1017.93 697.338 203.888 1064.33 463.732 125.95 80.1136\ninternal_count=34088 28355 5733 8082 2069 3549 20273 2184 6013 552 20137 1932 774 645\nis_linear=0\nshrinkage=0.05\n\n\nTree=12\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 2 4 9 2 9 13 1 9 14 12 3 10 3\nsplit_gain=1066.63 342.526 259.233 162.288 112.2 103.811 97.3632 96.112 44.1049 57.5738 41.0531 39.9598 38.706 38.0161\nthreshold=0.80356500410247722 0.42896023551267498 1.5000000000000002 0.16208072556515951 1.8194180785221719 0.37935766637759955 1.797896805829583 2.9345310926437382 0.3373762830616705 0.48932556351895706 0.11453136591202388 2.1288854818560901 0.95849462365591409 0.81402353687074069\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 -2 10 7 11 12 -3 9 -9 -4 -1 -5 -6\nright_child=2 4 3 6 13 -7 -8 8 -10 -11 -12 -13 -14 -15\nleaf_value=-0.052391544405412299 0.052286532345106621 -0.055347758880337987 0.028117717291669071 0.011117760823532016 0.0050815732481457702 -0.0055751219315134035 0.042545027557286202 0.011944565573108801 0.013612226203038522 -0.030230404469795327 -0.045469918584024743 0.022923936583065192 -0.070220298989367838 0.04925563688409082\nleaf_weight=896.5672655524686 676.55291435495019 107.08693912369199 21.47261125408113 630.87039322499186 84.951196109876037 146.6404579654336 332.47594749834388 126.31153046898544 219.60134226828814 222.01198012195528 129.74073104886338 17.439406208693981 13.986294064670799 113.95361645333469\nleaf_count=18639 1622 2277 99 2610 400 1375 587 676 932 2385 1988 125 64 211\ninternal_value=0.000199795 -0.0255879 0.0278359 0.0131447 -0.00240214 -0.0446715 0.0206069 -0.0121308 -0.00389891 -0.0149462 -0.0349042 -0.050931 0.00924763 0.0305089\ninternal_weight=3739.66 1934.56 1805.1 1128.55 873.917 1060.65 977.333 675.012 567.925 348.324 151.213 914.007 644.857 198.905\ninternal_count=33990 27020 6970 5348 6881 20139 3261 6270 3993 3061 2087 18764 2674 611\nis_linear=0\nshrinkage=0.05\n\n\nTree=13\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 6 9 9 3 2 2 8 10 10 4 3 14\nsplit_gain=931.47 417.94 166.901 104.915 104.52 82.0246 46.7677 46.6692 39.3006 36.8712 38.8324 36.4886 35.4071 48.6082\nthreshold=1.1020406375351024 2.5000000000000004 1.5000000000000002 0.36355675436704565 0.17487998293420839 2.1288854818560901 1.527243955281093 1.9357224602941756 14.500000000000002 0.45857843137254906 0.95849462365591409 1.5000000000000002 1.2501037838947984 0.63300027894733968\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 -2 6 11 7 -3 9 -5 12 -11 -4 -1 -14\nright_child=2 3 4 8 -6 -7 -8 -9 -10 10 -12 -13 13 -15\nleaf_value=-0.022133073295506959 0.052562314129804923 -0.045978024791563724 0.02731571943862381 0.042343141020893094 0.024307640883020159 0.053633533542009074 0.0017113555154284459 0.027918392478251598 -0.015493014355379109 0.013823129084326164 -0.077086968753405372 -0.048751150783848146 0.027560879908852143 -0.039340616501076078\nleaf_weight=302.43383575975895 611.6212822496891 1051.3362128422013 19.361420072615147 32.99567180685699 615.56619206350297 82.034459710121155 53.975855623371899 140.40830753836781 215.93994420813397 285.13903701584786 11.262850016355513 72.560790944844484 95.568140892311931 36.802875358611345\nleaf_count=2507 1200 22353 51 149 1822 150 471 374 1857 1303 44 1168 381 193\ninternal_value=-5.07181e-05 -0.0192049 0.0334503 -0.0370499 0.0168772 0.00614859 -0.0436498 0.00162307 -0.00768826 -0.00345746 0.0101211 -0.0326078 -0.0127158 0.00880487\ninternal_weight=3627.01 2307.9 1319.11 1354.25 707.488 953.65 1105.31 871.615 248.936 731.207 296.402 91.9222 434.805 132.371\ninternal_count=34023 29782 4241 24830 3041 4952 22824 4802 2006 4428 1347 1219 3081 574\nis_linear=0\nshrinkage=0.05\n\n\nTree=14\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 2 13 4 1 9 3 2 9 3 10 10 12 2\nsplit_gain=833.78 300.439 218.114 117.611 100.196 81.0101 77.1161 46.8911 43.2696 40.0108 31.4802 30.7341 30.1812 29.9028\nthreshold=0.82124236120517702 0.69655208401287494 1.9383663450051654 1.5000000000000002 5.5705857276916513 0.18560323472877255 1.7057859673546234 1.992208782912438 0.34268005210784097 1.1066554537313842 0.071796759941089847 0.26713615023474185 0.086958361481737009 1.4900624917218295\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 -2 11 10 7 8 13 -6 -5 12 -1 -3\nright_child=2 6 -4 5 9 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.015935507265406868 0.038248743745345204 -0.030405797161999256 0.050552803981457339 0.034895111599076191 -0.028330721159664136 0.010748835676948967 0.045584382517550537 0.025037414028994148 0.010683917768315879 0.0093801640775834239 -0.034817495498867014 -0.021808642197393081 -0.05815452979468011 0.0020128813610577508\nleaf_weight=46.690882791415788 315.97164137428626 239.21051439037547 573.61204572767019 17.234812805429101 301.21725759399123 611.33226961432956 88.054392367601395 111.0092433616519 159.5530799375847 91.129403974860907 157.35491632670164 80.633295503910631 571.81282972637564 100.88786260643974\nleaf_count=737 1179 3111 1030 73 4554 2494 171 360 642 804 2143 1261 14831 677\ninternal_value=0.000232904 -0.0234866 0.025575 0.0125383 -0.0398631 0.00216622 0.00211592 -0.0042163 -0.0107527 -0.0195701 -0.0277762 -0.051207 -0.0549983 -0.0208113\ninternal_weight=3465.7 1790.2 1675.51 1101.89 1091.48 785.922 698.715 610.661 499.651 392.347 174.59 699.137 618.504 340.098\ninternal_count=34067 27148 6919 5889 22187 4710 4961 4790 4430 5358 2216 16829 15568 3788\nis_linear=0\nshrinkage=0.05\n\n\nTree=15\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 9 13 13 1 6 6 0 14 13 6 9 10 14\nsplit_gain=723.04 293.14 191.098 82.3766 71.5783 60.9808 55.0757 45.751 38.3437 36.6973 33.1908 32.3774 38.6371 30.8023\nthreshold=0.82240565329519633 0.25584294134921287 1.0652999387949482 1.4261840188530308 4.7296948432922372 4.5000000000000009 1.5000000000000002 0.30612415275961918 0.71908998222261677 0.5821657902664904 1.5000000000000002 0.40090617732792916 0.95849462365591409 0.76989662855493035\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 5 11 -1 7 -4 -2 10 -6 -9 -3 -13 -10\nright_child=2 3 6 -5 9 -7 -8 8 13 -11 -12 12 -14 -15\nleaf_value=-0.04921901598117543 -0.017965966752265167 -0.021305855283751403 0.05178584577546657 0.030272219474253362 -0.032846548606427349 -0.031677728764677275 0.024190104881821748 0.037355038356086535 -0.050976588632900222 0.00069417998336552124 0.0069460331605136526 0.0054109724237870627 -0.072553490922784292 0.0072513835451731446\nleaf_weight=645.39759881189093 104.31336045637727 252.63511062832549 521.82512584049255 186.21506993472576 236.16000052075833 88.707457695156336 280.87899310979992 250.15103453397751 26.265929048880935 124.22783301048912 140.45418414659798 352.26926831621677 15.640164302662013 135.05770975863561\nleaf_count=16208 551 2671 1130 425 4687 1746 972 732 151 884 1139 2134 111 613\ninternal_value=0.000169744 -0.0212569 0.0252612 0.00119804 -0.0392517 0.00700744 0.0421718 0.0122769 0.0180149 -0.0213149 0.0264662 -0.0075613 0.00190895 -0.00248465\ninternal_weight=3360.2 1812.55 1547.65 806.76 1005.79 744.95 802.704 656.242 551.929 360.388 390.605 620.545 367.909 161.324\ninternal_count=34154 27120 7034 5341 21779 4932 2102 3186 2635 5571 1871 4916 2245 764\nis_linear=0\nshrinkage=0.05\n\n\nTree=16\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 9 6 3 2 1 2 2 11 3 11 13 2\nsplit_gain=663.543 360.193 155.329 75.0555 54.8629 52.599 36.1416 34.9537 31.9249 38.5761 32.042 34.4302 33.4551 31.7401\nthreshold=1.4261840188530308 2.5000000000000004 0.36355675436704565 4.5000000000000009 2.1288854818560901 1.527243955281093 5.5705857276916513 -0.70703784408564696 1.9357224602941756 0.39959521583407226 1.2501037838947984 0.3066610134778483 -0.084320303758185181 -1.4622577988893621\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 5 -2 8 6 -3 13 9 10 12 -12 -1 -4\nright_child=3 2 7 -5 -6 -7 -8 -9 -10 -11 11 -13 -14 -15\nleaf_value=-0.036178948947446631 0.045247373027964333 -0.047725328886396398 0.033945770010247164 0.0051898322329630014 0.046664424514813604 0.0051747438503086827 -0.025753992709280939 0.0077288142789973995 0.027118524412850654 0.014679031469573651 0.019912458557277499 -0.060274419329028529 -0.0031106588223165834 -0.051425819513002215\nleaf_weight=127.23799318401143 699.46854492835701 726.2938887339551 13.106873646378515 140.70673522166908 88.795818030834198 62.629544490016997 257.69422103883699 272.03894545370713 146.59341142699122 335.72630803845823 131.10837689694017 13.923011168837546 190.044323887676 49.733390925452113\nleaf_count=1086 1570 19217 85 733 191 502 4175 1773 422 1432 629 73 1723 516\ninternal_value=0.000306926 -0.0130097 -0.0297023 0.038553 0.00931232 -0.0391779 -0.0420033 -5.91588e-05 0.00575558 0.00180235 -0.00756002 0.0118545 -0.0164439 -0.0333667\ninternal_weight=3255.1 2414.93 1381.5 840.175 1033.43 1046.62 983.988 334.879 944.633 798.04 462.314 145.031 317.282 62.8403\ninternal_count=34127 31824 26268 2303 5556 23894 23392 2374 5365 4943 3511 702 2809 601\nis_linear=0\nshrinkage=0.05\n\n\nTree=17\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 9 4 0 13 12 13 4 1 9 6 13 12\nsplit_gain=600.821 313.128 88.9145 83.8032 60.7046 57.4689 56.8514 51.4244 35.7044 33.1266 30.5856 34.3555 25.4328 25.9509\nthreshold=1.5007010094932978 5.5000000000000009 0.34755730289246772 1.5000000000000002 0.26806232092754839 2.2531977117618109 0.23261107465006284 0.78838956065807686 1.5000000000000002 3.6929688453674321 0.3373762830616705 1.5000000000000002 0.38580353762997316 0.50270487952149423\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 -1 8 -4 10 -2 -10 11 -5 13 -6\nright_child=5 2 6 7 12 -7 -8 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.02257593390457667 0.046350584448744139 -0.046483101760097192 0.057520856850897799 0.0020687139183180876 0.0014935499628651065 0.053014260934032269 -0.018951316959679703 0.0052027860129846256 -0.037107098702241756 0.020404758720362001 -0.00081344249077859295 -0.032570867494592044 0.033293088793407498 0.0498886264964939\nleaf_weight=85.489812383428216 130.04787555895746 641.80838213698007 29.334315969608724 91.305847773328424 166.69926731940359 360.87219357304275 121.31320524821058 418.71837625373155 27.200661228038371 227.14771582186222 188.63367924978957 328.40314378926996 262.58233984932303 32.237135428935289\nleaf_count=532 382 18492 127 351 1252 641 1314 1864 329 696 1161 5734 1002 151\ninternal_value=-0.000437716 -0.0127686 -0.03838 0.000141556 0.0159332 0.0386823 -0.00377943 -0.0082763 0.0251248 0.0141331 -0.0175472 -0.0250482 0.0230998 0.00954615\ninternal_weight=3111.79 2366.53 792.456 1574.07 547.009 745.268 150.648 1027.06 384.396 254.348 608.343 419.709 461.519 198.936\ninternal_count=34028 31980 19933 12047 2937 2048 1441 9110 1407 1025 7246 6085 2405 1403\nis_linear=0\nshrinkage=0.05\n\n\nTree=18\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 2 13 3 4 2 9 9 14 2 1 3 0 12\nsplit_gain=521.689 194.656 134.568 91.4111 88.2359 57.0838 50.55 50.028 37.9332 45.3076 36.9028 30.2621 32.058 27.1379\nthreshold=0.98990312258122526 -1.0000000180025095e-35 2.2531977117618109 0.54721747536263765 1.5000000000000002 1.6903347241130793 0.34755730289246772 0.18560323472877255 0.45757160205195929 1.527243955281093 4.1850135326385507 2.1288854818560901 0.92210547419691402 0.23261107465006284\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 4 6 -2 11 8 -6 9 -3 -1 12 -5 -9\nright_child=2 3 -4 5 7 -7 -8 13 -10 -11 -12 -13 -14 -15\nleaf_value=-0.052694646338962009 0.039028821655193793 -0.024235025413574876 0.051090478698529285 -0.0090143241165232153 -0.023647756049297239 0.035478221625075211 0.0063674745978793759 0.028764826187816808 -0.039158702581615412 0.043670865259301611 -0.02863456293310377 0.049579460364710753 0.039490750681927041 0.0039279136314099702\nleaf_weight=298.0652201033663 256.05177853070199 88.134336176794022 338.41085918806493 348.50544153572991 126.65318727539852 147.64263857714832 135.30045672785491 154.73427443392575 276.3446159173036 32.921094275079668 352.89895551989321 26.950453300029039 36.75275962613523 375.72965549584478\nleaf_count=9742 1150 1635 621 2774 1608 392 1064 586 6051 129 6522 61 120 1490\ninternal_value=-0.000417466 -0.0180936 0.0242011 -0.00518649 0.0141912 0.00891444 -0.0199951 0.00446838 -0.028964 -0.00556148 -0.0397152 -0.000662866 -0.00429679 0.0112132\ninternal_weight=2995.1 1743.52 1251.58 1092.55 913.169 559.851 532.701 657.117 397.4 121.055 650.964 412.209 385.258 530.464\ninternal_count=33945 28490 5455 12226 4834 3347 8879 3684 7815 1764 16264 2955 2894 2076\nis_linear=0\nshrinkage=0.05\n\n\nTree=19\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 11 4 0 13 12 2 1 8 13 5 4 14\nsplit_gain=504.526 243.319 97.5722 68.8824 44.6862 44.1181 45.5134 39.6506 36.0051 34.7901 32.4033 29.8241 27.3552 24.7568\nthreshold=4.5000000000000009 1.4261840188530308 0.22527488983156077 1.5000000000000002 0.26806232092754839 1.9383663450051654 0.17056733839577878 -0.59120101787752588 5.503817319869996 21.500000000000004 2.4146125192805905 5.5000000000000009 5.5000000000000009 0.44746339656424899\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 12 4 -1 6 -4 -8 -5 -9 -3 -10 -2 -6\nright_child=2 10 5 8 13 -7 7 9 11 -11 -12 -13 -14 -15\nleaf_value=-0.018327767778119362 -0.021891326852811106 0.030792715924132397 0.056051985460185999 -0.020682752818065916 0.047541355197990284 0.033383660905727239 -0.051091789525933975 -0.039789264691544007 -0.0027525705663980753 0.0041473492207965658 0.054926890992793333 0.038760521274208898 -0.048672438597834676 0.016804401509155586\nleaf_weight=78.443045861087739 122.58861940226052 362.66313050407916 18.950444281101227 300.77010332234204 79.723586596548557 49.382601456716657 82.87794303195551 65.809096673037857 499.6986356601119 137.87235139124095 231.5903808651492 46.357550454325974 471.08102668216452 362.14143499359488\nleaf_count=525 2660 1140 98 3835 557 107 1435 1083 3062 1411 433 277 15745 1840\ninternal_value=-0.0008852 0.0135925 -0.0308041 0.00197351 0.0162746 -0.0100243 -0.0171268 -0.0221443 -0.0068222 -0.0101735 0.0402747 0.000836102 -0.0431884 0.0224449\ninternal_weight=2909.95 1961.39 948.562 1367.13 520.308 354.892 305.51 286.559 846.826 203.681 594.254 546.056 593.67 441.865\ninternal_count=34208 11669 22539 10096 2922 4134 4027 3929 7174 2494 1573 3339 18405 2397\nis_linear=0\nshrinkage=0.05\n\n\nTree=20\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 13 2 13 1 4 3 1 0 13 1 2 1\nsplit_gain=421.194 214.967 99.8404 57.6501 51.8242 38.5272 33.1415 28.3197 21.385 26.9799 26.4319 21.1812 23.711 20.8749\nthreshold=4.5000000000000009 1.0652999387949482 0.76064789851928383 1.7271604568384149 2.4146125192805905 3.1612371206283574 1.5000000000000002 2.1288854818560901 8.1648535728454608 0.75456361599240507 0.95559699300162337 0.79792803525924694 -0.22785432879508086 2.2388802766799931\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 13 5 6 -1 -3 8 9 10 -7 -4 -13 -2\nright_child=2 4 11 -5 -6 7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.040912331272597441 -0.053238254017889688 0.041282315621939687 -0.062411582894623001 0.022874024678130447 0.053784947748551006 -0.017895272266441709 0.016730683247948321 0.035759651313368716 0.018646856657144533 0.011240626574660167 0.01310596282442307 -0.016961133023832998 0.010819646891273424 -0.032894257467027344\nleaf_weight=73.023319617845118 188.1985725981649 212.33336716331542 14.210303250700234 207.01878939173184 219.84500772599131 466.17213000613265 393.99004571884871 43.077857660129666 81.272776236757636 139.36738836858422 79.87566726654768 141.15981538547203 166.10005367035046 412.7193680817727\nleaf_count=1416 7233 823 394 744 433 4880 1351 142 367 955 478 1597 1330 11873\ninternal_value=0.000414551 0.0137775 -0.0273278 -0.000808853 0.0329963 -0.00638221 0.0253824 -0.00322172 -0.0054557 -0.00833665 -0.0133448 -0.00481448 -0.00195688 -0.0393436\ninternal_weight=2838.36 1915.98 922.388 1089.81 826.168 882.789 606.323 809.766 766.688 685.415 546.048 321.47 307.26 600.918\ninternal_count=34016 11589 22427 8982 2607 8238 2174 6822 6680 6313 5358 3321 2927 19106\nis_linear=0\nshrinkage=0.05\n\n\nTree=21\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 4 10 4 4 9 14 12 13 3 13 3 3 14\nsplit_gain=408.085 183.981 116.034 67.1546 51.0977 50.5313 46.4419 39.2155 36.9645 32.9049 33.5195 32.6175 28.6443 25.357\nthreshold=1.4261840188530308 5.5000000000000009 0.34228550829127619 1.5000000000000002 3.5000000000000004 0.34755730289246772 0.48847560398550277 0.22992965439201435 2.6133013047935933 1.0922634794594657 0.64551238632825836 1.2501037838947984 2.1288854818560901 0.71127475164422205\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 12 9 -2 6 11 -4 -6 10 13 -5 -3 -1\nright_child=4 2 7 5 8 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=0.0027844205747001708 0.043624868809226836 -0.046478638427301072 0.043156637362754879 -0.010990112907822334 0.0051413080049938055 0.0059848561727467546 -0.031950609427281909 -0.010687298844140778 0.052008445112083183 0.032204745385579818 0.023073115254266353 0.036557058345714426 0.015915748326727357 -0.036977402661462898\nleaf_weight=145.80201510107145 444.49018073268235 488.87650756107178 43.160271279513836 186.66278179781511 195.89360673725605 347.70261945901439 288.61914582457393 143.38830440258607 52.674477159976959 166.82179919676855 149.62747605890036 43.654255230911076 18.652355853002518 54.242096455767751\nleaf_count=1437 1270 17655 275 2299 976 1673 4840 1656 84 529 750 197 127 362\ninternal_value=0.000286698 -0.0107996 -0.0317854 -0.000253175 0.0334817 -0.00875546 -0.0186244 0.00193383 0.0152416 0.0140024 0.00525134 -0.00185895 -0.0441588 -0.00812732\ninternal_weight=2770.27 2077.21 694.077 1383.13 693.058 866.639 518.936 186.549 248.568 516.493 349.672 230.317 507.529 200.044\ninternal_count=34130 31800 19713 12087 2330 9009 7336 1931 1060 3078 2549 2496 17782 1799\nis_linear=0\nshrinkage=0.05\n\n\nTree=22\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 9 0 3 4 9 9 2 13 13 0 2 10\nsplit_gain=368.301 196.407 89.2047 41.179 51.249 40.9394 33.2986 30.3072 22.871 22.2527 21.8528 43.3893 20.73 23.6573\nthreshold=1.5007010094932978 4.5000000000000009 0.40090617732792916 0.61472819196375383 0.80139994038844931 1.5000000000000002 0.17487998293420839 0.72224482408762225 -0.71148459116722751 -0.23488727644015653 0.76064789851928383 0.65980664568139946 0.31817687714608561 0.069605943152454788\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 12 4 7 -2 -7 9 -5 -1 -4 -12 13 -3\nright_child=5 2 10 8 -6 6 -8 -9 -10 -11 11 -13 -14 -15\nleaf_value=-0.038223855395247953 0.049684223968566303 -0.0021960347925435585 -0.020558790856394449 -0.032270562688408326 0.0071767598061057581 -0.025818692021091733 0.027549931862383571 0.041226992951331753 0.014787650733920077 -0.012811052271620361 0.034246731897047709 -0.056414866060344571 -0.023302750625213461 -0.047657211448303177\nleaf_weight=128.26089660800062 237.39762467890978 30.793059276882559 54.673391946591437 26.350196657469496 386.75665339408442 31.096604046411812 351.12764910422266 19.543848484754562 483.2600630777888 263.58334998739883 76.998666936531663 14.94800156354904 157.92573819123209 423.26883236144204\nleaf_count=1871 712 656 861 190 2598 393 945 52 2977 2461 330 141 4229 15716\ninternal_value=-0.000398572 -0.010539 -0.0307603 0.00120559 -0.00588209 0.033381 0.023152 -0.0181467 0.012296 -0.0212055 0.00424683 0.0190593 -0.0391406 -0.0445858\ninternal_weight=2685.98 2066.36 758.608 1307.76 798.145 619.622 382.224 411.388 509.61 391.844 146.62 91.9467 611.988 454.062\ninternal_count=34132 32082 21933 10149 6982 2050 1338 4384 3167 4332 1332 471 20601 16372\nis_linear=0\nshrinkage=0.05\n\n\nTree=23\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 2 9 4 9 11 12 12 9 14 6 9 10\nsplit_gain=310.926 169.373 60.488 57.2396 53.2194 55.6304 46.0338 32.3669 32.352 31.204 60.0801 42.0971 29.446 23.7898\nthreshold=1.4261840188530308 7.5000000000000009 0.82240565329519633 0.41994549103866291 1.5000000000000002 0.15591134728417663 0.19809164072177035 0.098911901734213256 0.19909436529862615 0.36355675436704565 0.69247260929972343 1.5000000000000002 0.37935766637759955 0.95849462365591409\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 -3 -2 -6 7 -1 -8 10 11 -4 -10 -14\nright_child=4 3 9 -5 5 -7 8 -9 12 -11 -12 -13 13 -15\nleaf_value=0.0078900379075134402 0.048173656871363393 -0.043834213409961402 0.023856589628286896 0.0064767169815366891 -0.048490903431155238 0.023658050630505439 0.027747186695704418 -0.035717473567923715 -0.026781786777267991 0.02391980053370401 -0.033994516985562065 -0.0078056380861712491 0.0042610478900317675 -0.067680414992237412\nleaf_weight=51.702765916008502 244.70668078586459 392.83473295986187 232.81327818334103 65.731831790413707 27.816149235703051 378.73203189484775 74.065860545262694 231.25274029141292 137.5942475204356 221.43754346156493 95.882998966379091 189.831123996526 275.94403741927817 10.989164626225827\nleaf_count=797 757 15918 930 661 451 1138 815 5337 1912 920 883 2051 1604 93\ninternal_value=-0.000164339 -0.0100226 -0.00198642 -0.0366241 0.0297782 0.0186147 -0.0116797 -0.0277496 -0.00252264 0.00825356 0.00153012 0.00965061 -0.00785651 0.0012803\ninternal_weight=2631.34 1980.08 1521.51 458.567 651.255 406.548 781.549 282.956 498.593 739.965 518.527 422.644 424.527 286.933\ninternal_count=34267 31921 15342 16579 2346 1589 10558 6134 4424 4784 3864 2981 3609 1697\nis_linear=0\nshrinkage=0.05\n\n\nTree=24\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 11 13 2 10 8 10 8 1 13 12 6 10\nsplit_gain=280.074 154.556 81.224 45.5769 42.16 37.558 33.3849 28.8296 28.5858 24.2266 23.9934 22.4799 19.9235 21.7928\nthreshold=4.5000000000000009 1.134727103335295 0.23171031113136512 2.4146125192805905 1.6903347241130793 0.56386946386946402 44.500000000000007 0.95849462365591409 21.500000000000004 5.6302626132965097 1.9383663450051654 0.16808194036744759 1.5000000000000002 0.24499799919967988\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 9 12 5 -1 8 -7 10 -2 11 -4 -3 -14\nright_child=2 3 6 -5 -6 7 -8 -9 -10 -11 -12 -13 13 -15\nleaf_value=-0.011396364933880609 -0.041170726165875228 0.034799033366859751 0.06071803436525039 0.05169056970440316 0.020285932702913963 0.014778971171121414 -0.061252637977655788 -0.069824860049012727 0.011911238508247553 -0.009582248924510036 0.048077126936892402 -0.034939141108301049 -0.0064486266207398604 0.023161923389536317\nleaf_weight=608.239537985879 410.81842382473405 200.25699511449784 5.6980707440525284 186.83796326071024 210.10994085203856 247.00615836400539 22.597834505140781 9.5150082111358625 193.10902010183781 72.029630780103616 10.536442182958124 88.153183137066662 88.233776642940938 206.18164752982557\nleaf_count=6994 16788 838 19 454 850 1362 459 74 1856 1767 22 1559 470 648\ninternal_value=0.000884731 0.0120673 -0.0235609 0.0306754 0.000251436 -0.00463413 -0.00400802 0.0113818 0.000530913 -0.036488 -0.0204351 -0.0285525 0.0226497 0.0142961\ninternal_weight=2559.32 1756.38 802.943 681.51 1074.87 864.761 320.095 256.521 297.497 482.848 104.388 93.8513 494.672 294.415\ninternal_count=34160 11690 22470 2410 9280 8430 3915 1436 3456 18555 1600 1578 1956 1118\nis_linear=0\nshrinkage=0.05\n\n\nTree=25\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 9 3 9 4 14 10 0 14 3 12 10 9 0\nsplit_gain=243.873 89.3959 80.9956 65.9105 36.3081 46.4146 34.684 30.773 25.4398 24.6168 21.8479 31.3689 25.4951 22.4896\nthreshold=5.5000000000000009 0.34755730289246772 1.9345448480507716 0.3373762830616705 1.5000000000000002 0.69744590828457376 0.95849462365591409 0.34586779907359916 0.48612818605837255 0.93424544978515189 0.51508605832750265 0.68841813923781148 0.48753231203628372 0.30899292712668797\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 3 4 5 7 -5 9 -6 -1 11 12 -3 -14\nright_child=1 10 -4 6 8 -7 -8 -9 -10 -11 -12 -13 13 -15\nleaf_value=-0.042721787179412316 -0.038407309711890381 0.012347420417132535 0.038205832625110216 0.017277529581526754 -8.1735779129761578e-05 -0.035391767113703232 -0.054800295861512652 0.025781937238261079 -0.021991890559345667 0.018207831980176913 -0.034202924082250771 0.056641677553442432 0.01262778922224727 -0.084038780901987103\nleaf_weight=33.391168123111129 473.27887374523561 86.202624301193282 203.310711318627 731.30289347656071 232.62563833873719 49.865102153271437 16.128947306424379 211.52418475458398 306.77748307830188 31.183910372667015 29.953008678741753 29.969810272566974 9.719720898196103 13.86296854354441\nleaf_count=348 18995 766 677 3328 2497 277 117 1380 4856 167 412 111 76 205\ninternal_value=-0.000824989 -0.0272634 0.00854527 0.00478488 -0.0046076 0.00855086 0.015651 0.0165862 -0.0125604 -0.0134693 0.00385762 0.0122 -0.000333726 -0.0453042\ninternal_weight=2459.1 642.987 1816.11 1612.8 865.367 325.964 747.432 276.099 539.403 64.5751 169.708 139.755 109.785 23.5827\ninternal_count=34212 20565 13647 12970 9525 2172 3445 1895 7353 515 1570 1158 1047 281\nis_linear=0\nshrinkage=0.05\n\n\nTree=26\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 9 2 10 10 2 14 6 6 14 12 2 9\nsplit_gain=252.104 155.685 75.1509 35.9682 45.9495 32.4325 27.9943 27.9753 34.5012 25.7287 33.5435 23.6484 34.7141 23.2646\nthreshold=1.797896805829583 3.5000000000000004 0.2627566423052386 1.7271604568384149 0.27326839826839833 0.95849462365591409 0.11095238583044341 0.46321428429942807 12.500000000000002 1.5000000000000002 0.66998646507228099 0.23261107465006284 -0.59120101787752588 0.36355675436704565\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 7 4 9 -6 -2 8 -3 10 -1 -4 -13 -14\nright_child=6 2 11 -5 5 -7 -8 -9 -10 -11 -12 12 13 -15\nleaf_value=0.013841210183730865 0.0027393987338739691 -0.0035688288443919795 0.018820882918355028 0.021680701174347856 0.010667771823301309 -0.073547986689721498 0.041434257421887138 -0.04312020186885504 -0.057891217835173529 -0.024462235369640102 -0.043565036944452872 -0.047150705959244005 -0.041007561287974718 0.0043403240321968394\nleaf_weight=123.06258921790868 54.37546301074326 114.30371860601008 76.852950083557516 235.34041218692437 549.11225098511204 10.686621425673364 338.49134534597397 368.18563330359757 38.340117859654129 222.60698735353071 31.014897484332323 55.857993513578549 32.749516554176807 173.77108318731189\nleaf_count=1081 242 2462 713 948 3048 77 1044 16658 1781 3068 192 929 655 1311\ninternal_value=-0.000513492 -0.00760686 -0.023749 0.00425084 -0.000147774 0.0089318 0.0360992 -0.0356597 -0.0175012 -0.013629 0.00207926 -0.00541118 -0.0125535 -0.00301378\ninternal_weight=2424.75 2031.88 860.061 1171.82 936.483 559.799 392.867 520.829 152.644 376.684 154.077 339.232 262.379 206.521\ninternal_count=34209 32923 24509 8414 7466 3125 1286 20901 4243 4341 1273 3608 2895 1966\nis_linear=0\nshrinkage=0.05\n\n\nTree=27\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 5 13 2 10 2 0 0 11 9 12 2 3 10\nsplit_gain=220.441 75.4217 71.1373 39.5333 40.7591 36.2549 34.9334 33.8899 28.732 24.3204 26.9288 23.6396 21.5422 19.7521\nthreshold=1.1020406375351024 9.5000000000000018 2.4146125192805905 0.82240565329519633 0.45857843137254906 0.11095238583044341 0.30899292712668797 0.1173266999878748 0.2872956082841655 0.18560323472877255 0.23261107465006284 0.31817687714608561 1.4393308112662344 0.26713615023474185\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 4 7 9 -5 -1 -6 -2 -11 13 -9 -3\nright_child=2 11 -4 6 8 -7 -8 12 -10 10 -12 -13 -14 -15\nleaf_value=-0.0015382072601010904 -0.045252533760167052 -0.053843669291800504 0.048466238493767966 -0.022489062811241217 0.05398426227902875 0.019961902272453386 0.012249493550204219 -0.035881116605754652 -0.0022204128051270659 0.03642805520099248 -0.013089115857991352 -0.017887238169561642 -5.2540434417255947e-05 -0.010153698376079354\nleaf_weight=148.47836769116111 32.699273948092014 134.55465993098915 183.86619294993579 87.531499543925747 23.976217268034816 490.36160250008106 396.30481791449711 273.20258260704577 253.10356108820997 39.158824019599706 85.8323192615062 149.08225669153035 49.535922268405557 32.628861205885187\nleaf_count=2927 731 8247 512 531 158 2435 2460 8194 1947 199 412 3855 639 855\ninternal_value=0.000350016 -0.0108017 0.0210941 -0.00519926 -0.0123945 0.0132653 0.00593147 -0.0213277 0.00281972 -0.00755325 0.00259026 -0.032543 -0.0303991 -0.0454279\ninternal_weight=2380.32 1548.4 831.918 1232.13 748.297 648.052 483.836 471.217 277.08 157.69 124.991 316.266 322.739 167.184\ninternal_count=34102 29813 4289 16856 13865 3777 2991 11760 2105 1342 611 12957 8833 9102\nis_linear=0\nshrinkage=0.05\n\n\nTree=28\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 10 2 9 1 3 10 3 9 12 12 7 5 10\nsplit_gain=191.337 95.8036 65.796 67.487 33.8551 31.4773 28.4307 22.7368 21.655 36.6454 20.5896 19.7123 18.9919 18.4216\nthreshold=0.42896023551267498 0.29449472096530926 1.4900624917218295 0.15180890299837504 4.0650434494018564 1.1361560482789164 0.95849462365591409 1.628001827619378 0.51654338434932701 0.23261107465006284 0.095515742534131934 24.500000000000004 13.500000000000002 0.071796759941089847\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 -2 13 -4 7 8 9 -3 -6 -10 -5 -1\nright_child=2 6 5 12 10 -7 -8 -9 11 -11 -12 -13 -14 -15\nleaf_value=-0.0022879711398404055 -0.024587132121848727 0.020270386180174616 0.012403140797965159 0.011852272459715284 0.021177613730052086 0.036897031633918845 -0.075170099466010706 0.040512695808811644 0.024975601813484169 -0.02117516496901025 -0.024325067026616871 -0.052195968945170028 -0.034263214163003368 -0.050069538280928097\nleaf_weight=22.344001636840403 174.21467615617439 75.380487203132361 246.74339890433475 645.81670900247991 26.732076245942153 282.68965301243588 10.97514860704541 35.686411375179887 94.471926959231496 176.24451688607223 258.36330451339018 8.1278368765488249 22.177524912869558 216.34420034463983\nleaf_count=688 3888 846 1416 3845 393 1006 108 113 548 2244 6636 125 504 11621\ninternal_value=-0.000114948 -0.0176826 0.0117323 0.00304087 -0.0317416 0.0255265 0.000719619 0.00304258 -0.000835773 -0.00872804 -0.0199994 0.0184173 0.0102723 -0.0456248\ninternal_weight=2296.31 924.67 1371.64 842.209 523.784 529.433 400.886 389.911 354.225 251.625 285.095 102.6 667.994 238.688\ninternal_count=33981 23322 10659 8237 19338 2422 3984 3876 3763 3090 7029 673 4349 12309\nis_linear=0\nshrinkage=0.05\n\n\nTree=29\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 13 6 4 2 13 9 1 9 12 11 12 10 7\nsplit_gain=213.262 77.8548 60.3717 35.1828 29.7851 31.7408 27.8554 26.3438 24.2969 23.4193 22.5355 22.4125 21.5396 20.6243\nthreshold=0.70774219324661491 2.4146125192805905 5.5000000000000009 3.5000000000000004 1.3438334896653068 -1.9353917026821741 0.3373762830616705 4.1850135326385507 0.14768624399293936 0.71946760054043668 0.2585977747180786 0.086958361481737009 0.64899964899964913 8.5000000000000018\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 4 -2 5 -1 -7 11 -5 12 -8 -4 -10 -12\nright_child=1 -3 7 8 -6 6 10 -9 9 -11 13 -13 -14 -15\nleaf_value=0.050489162023232961 0.01671918504015665 0.045202804274194326 0.01141527025319811 -0.030186173704677046 0.007419656020028898 -0.027227251225806388 0.045315812360164048 -0.018642072942872966 -0.0015438161000840291 -0.072711013436571284 0.0013691214988051648 -0.051559716764955681 0.028001104861470741 -0.040224411779826434\nleaf_weight=16.939617186319083 615.36031764792278 173.7544834446162 15.047072116984053 63.848058132454753 215.76042620837688 281.75589782145107 21.005420090630651 162.43173883669078 269.24087153561413 8.9159437678754312 124.4177687028423 169.72735589905642 78.989886786788702 38.166228916496038\nleaf_count=100 3020 522 529 1868 1287 6141 103 5609 2099 80 975 11067 322 334\ninternal_value=4.04152e-05 0.0143245 -0.0164944 0.00910887 -0.00799827 -0.0148969 -0.0173803 -0.0335037 -0.00202825 0.00307585 -0.00220236 -0.0463975 0.00521891 -0.00858107\ninternal_weight=2255.36 1210.11 1045.25 1036.36 698.045 482.285 465.345 347.206 420.995 357.147 183.589 184.774 348.231 162.584\ninternal_count=34056 7911 26145 7389 8940 7653 7553 17205 4369 2501 1412 11596 2421 1309\nis_linear=0\nshrinkage=0.05\n\n\nTree=30\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 9 4 4 3 13 1 3 3 2 14 9 14 8\nsplit_gain=169.15 86.1591 75.9202 39.4293 38.2559 44.902 24.8427 24.8041 22.9493 19.9275 19.5785 19.5483 18.2241 17.662\nthreshold=1.4630009291942681 0.36355675436704565 4.5000000000000009 1.5000000000000002 1.2135643508474616 -0.094644560114769946 7.6704530715942392 1.8740213748638983 -0.80624206413687138 2.2500374604554767 0.4243920088484317 0.13773028815655583 0.38993833693644864 17.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -2 5 13 -6 12 -5 -7 -10 -9 -4 -1\nright_child=3 -3 7 8 6 9 -8 11 10 -11 -12 -13 -14 -15\nleaf_value=-0.0036962131227179802 0.045960507978114067 0.0078669223468548089 -0.0060287851564819576 -0.050270877947383233 0.021691070363051811 -0.0043256204394393274 -0.049963620278367538 -0.029306591931887463 -0.0395356523621831 0.037516303193963568 0.020919189969371137 0.052812151044644054 -0.038787325787474305 -0.038322170141511597\nleaf_weight=48.432428650092334 160.37795720249414 585.84181504696608 48.485777516849339 11.768450346775351 148.1370725245215 295.20196642540395 12.222284627147017 12.97198924329132 13.117963405791668 30.47997157368809 298.94948538718745 14.177339983172713 365.05395707418211 155.79184148798231\nleaf_count=631 716 3847 1243 119 746 3670 84 188 134 108 1177 64 18554 2978\ninternal_value=-0.000324004 -0.00768725 -0.0157442 0.0257447 -0.00537746 -0.0118506 0.0159539 -0.0319331 0.0156222 -0.000306858 0.0182598 0.0139285 -0.0349703 -0.0301685\ninternal_weight=2201.01 1716.8 1130.95 484.214 690.266 529.906 160.359 440.689 323.836 325.682 312.067 27.1493 413.54 204.224\ninternal_count=34259 32113 28266 2146 8217 7387 830 20049 1430 3778 1311 252 19797 3609\nis_linear=0\nshrinkage=0.05\n\n\nTree=31\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 10 13 3 12 10 10 4 11 3 12 12 8\nsplit_gain=157.366 117.521 56.4741 38.5037 38.9259 28.0953 24.3732 26.9959 21.3236 22.4827 23.7837 19.1886 17.4659 15.8778\nthreshold=7.5000000000000009 1.9383663450051654 0.31929669030732866 0.64551238632825836 2.1288854818560901 0.50270487952149423 0.95849462365591409 0.54890004782400781 6.5000000000000009 0.30198526726146607 0.76111247319296804 0.60314213570565234 0.4035268289642307 16.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -2 4 5 8 7 -5 9 10 -1 -4 -9 -11\nright_child=2 -3 11 6 -6 -7 -8 12 -10 13 -12 -13 -14 -15\nleaf_value=-0.016234243131387289 -0.042391866189002425 0.036781754179204942 0.0056287783887742715 0.0017810100210070249 0.038918077444705684 0.014187038430144631 -0.065658188138806362 0.038231479520567907 -0.043120023328313384 0.0094494188609367605 0.0083462813502980134 -0.072662659611891695 0.011888776642742712 -0.038007641100001086\nleaf_weight=236.6867256249534 269.25030499976128 253.77948806248605 104.61835761927068 485.23667300213128 40.343237597495317 94.206418932415545 10.137097314000128 97.891890941187739 60.069816254661418 20.704553581774235 167.11985965259373 7.4526016106829038 177.05057198787108 105.38645856548101\nleaf_count=5322 15394 921 1408 3560 161 596 85 422 2559 161 1408 169 745 1255\ninternal_value=-0.000957352 0.00539322 -0.0300169 4.33275e-05 -0.00822521 -0.0110578 0.00782084 0.00888566 -0.0151061 -0.0118552 -0.00606583 -0.000174105 0.0213725 -0.030202\ninternal_weight=2129.93 1748.61 381.321 1494.83 724.517 684.174 770.316 760.179 589.967 529.898 403.807 112.071 274.942 126.091\ninternal_count=34166 17195 16971 16274 11462 11301 4812 4727 10705 8146 6730 1577 1167 1416\nis_linear=0\nshrinkage=0.05\n\n\nTree=32\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 13 2 1 9 1 10 10 12 12 12 3 12 8\nsplit_gain=155.77 70.5529 59.0311 41.3448 31.7163 29.0105 21.7432 21.9606 29.5718 21.0888 19.2173 17.2156 17.1657 32.7507\nthreshold=0.70774219324661491 2.2531977117618109 0.84069566187459877 5.5705857276916513 0.14768624399293936 4.3248009681701669 0.86240703177822864 0.54890004782400781 0.20050230359448049 0.084535385028355392 0.71946760054043668 1.6668156375686918 0.54803933842734343 30.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 3 9 5 -2 7 8 -6 -1 -9 12 13 -4\nright_child=1 -3 11 -5 6 -7 -8 10 -10 -11 -12 -13 -14 -15\nleaf_value=0.010451581911547556 -0.032997165719197134 0.041501699599348803 0.00014581555188714212 -0.010729005952192148 0.023717774729618182 0.013150707651298558 -0.028905324862080292 0.026892905943239544 -0.002791507553294271 -0.040857665259156593 -0.024259108583309555 0.030161607151464521 0.031344264079889543 -0.045694629496139082\nleaf_weight=21.179538387106732 83.375914417672902 181.14786933362484 268.75733106571715 290.96030882978812 158.37415815610439 56.377585069276392 31.315590278245509 279.61965657584369 308.97414487879723 278.81496197683737 18.8897505402565 43.425574526190758 32.12969365157187 44.569505362771451\nleaf_count=629 2006 640 3254 5340 1132 522 218 1134 2150 16263 93 167 115 306\ninternal_value=0.000274821 0.0130238 -0.0142713 -0.0242134 0.00747945 -0.0144196 0.0113319 0.0130127 0.00622337 -0.0372126 0.0235858 0.000860326 -0.00289941 -0.00649944\ninternal_weight=2097.91 1118.07 979.837 590.955 936.927 139.753 797.173 765.858 467.348 299.995 298.509 388.882 345.457 313.327\ninternal_count=33969 7895 26074 22232 7255 2528 4727 4509 3282 16892 1227 3842 3675 3560\nis_linear=0\nshrinkage=0.05\n\n\nTree=33\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 0 3 11 11 12 4 13 13 12 11 0 0\nsplit_gain=120.649 82.4134 27.7619 33.7781 26.7238 24.2002 24.1385 20.5127 19.7328 17.4801 17.0187 19.1555 17.2809 15.4634\nthreshold=1.797896805829583 4.5000000000000009 0.61869443822210857 0.80139994038844931 0.23171031113136512 0.41449514314371977 0.090997256007901164 1.5000000000000002 0.76064789851928383 -0.23488727644015653 0.18854389541763747 0.19200570707404305 0.13680055063109184 0.44584947358274835\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 9 6 10 -3 -2 -6 -1 11 -5 -12 -7\nright_child=7 4 -4 5 8 13 -8 -9 -10 -11 12 -13 -14 -15\nleaf_value=-0.032503636168867055 0.046212405759396093 0.015615929821294953 0.011646448744031104 -0.0040970854727538772 -0.024692883800344401 0.036843253895881702 -0.034233533656714506 0.018961222289061618 0.0048660463339698637 -0.0079318614582947027 0.030858237435773851 0.035379892073803577 -0.038288440890851941 -0.00012000150489856963\nleaf_weight=100.38995397253893 106.75684358179569 25.82108001539018 462.06811878480949 129.99526332016103 98.922593895578757 82.96418126206845 303.94418417755514 200.00148813193664 129.19815916009247 256.49582938221283 10.269429301843045 39.175837272778153 48.316433328203857 42.66601849347353\nleaf_count=1855 525 844 3339 1377 2724 343 17692 756 1061 2584 65 265 498 230\ninternal_value=-0.000282503 -0.00541043 0.00212013 -0.00408488 -0.0212106 0.00686395 -0.0302948 0.0285644 -0.00800376 -0.0149153 -0.0028667 0.00519899 -0.0258534 0.0243881\ninternal_weight=2036.99 1730.23 1172.34 710.273 557.886 353.387 329.765 306.758 228.121 356.886 227.757 169.171 58.5859 125.63\ninternal_count=34158 32877 10556 7217 22321 2778 18536 1281 3785 4439 2205 1642 563 573\nis_linear=0\nshrinkage=0.05\n\n\nTree=34\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 6 14 2 14 1 9 10 9 11 0 4 11\nsplit_gain=114.662 102.565 42.1502 37.3574 34.5043 20.682 20.581 19.5288 16.4699 15.631 20.2455 14.7764 13.776 13.7265\nthreshold=0.3373762830616705 1.4900624917218295 8.5000000000000018 0.67885129756991658 1.2413453555885363 0.52298926960539249 6.7416048049926767 0.11777184785235929 0.061381475667189961 0.1648877196759799 0.16821326847638382 0.064529627235824158 15.500000000000002 0.26948978220549369\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 -3 13 7 8 -1 -4 -7 -11 -12 -8 -2\nright_child=4 3 6 -5 -6 9 12 -9 -10 10 11 -13 -14 -15\nleaf_value=-0.015562767583124488 0.031421295061329013 0.02277250750185255 0.019795640261307677 -0.02754355935155119 0.030885295908238044 -0.037806702678767456 -0.043250929161698926 0.0081113307167776406 -0.044515139268172882 0.020021011603734991 0.027016635032633265 -0.025673953143566942 0.047764924778016776 0.0057535005224722225\nleaf_weight=172.64197519095615 56.500756189227104 253.30535883735865 9.935467027127741 42.27832201635465 227.61428626673296 99.142870924668387 5.5079338550567654 173.90075604687445 198.56152696092613 36.098684927681461 13.812313579022883 146.89262748078909 10.743718519341199 563.67477276688442\nleaf_count=3975 374 1476 305 346 964 3789 190 1971 13733 524 149 2287 185 3994\ninternal_value=0.000300349 -0.00989176 -0.0185564 0.015507 0.0142749 -0.012003 -0.0371545 -0.00369357 -0.0413707 -0.0216878 -0.0134176 -0.0210062 0.0161992 0.00813877\ninternal_weight=2010.61 1162.82 867.238 295.584 847.79 642.489 224.749 346.543 208.497 295.946 196.804 160.705 16.2517 620.176\ninternal_count=34262 28930 27108 1822 5332 12695 14413 5946 14038 6749 2960 2436 375 4368\nis_linear=0\nshrinkage=0.05\n\n\nTree=35\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 2 9 9 0 0 9 10 4 14 12 10 14 10\nsplit_gain=103.783 54.3033 43.3907 37.4088 30.6829 19.5472 22.5845 17.7865 17.0959 16.4242 25.068 16.083 15.0671 32.9797\nthreshold=7.5000000000000009 0.82240565329519633 0.40774619116223271 0.3373762830616705 0.34007656527049129 0.12042922364702495 0.13211613819948531 0.083722741433021816 1.5000000000000002 0.86194163353498943 0.51508605832750265 0.45857843137254906 0.85443103819729782 0.55753467843631788\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -2 5 -3 6 7 -1 12 10 -4 -11 -6 -14\nright_child=2 4 9 -5 8 -7 -8 -9 -10 11 -12 -13 13 -15\nleaf_value=0.017318886301291792 -0.034490152383830179 -0.0082616859990626412 0.047081877637347537 0.0063192528935825058 0.029257297770380625 -0.023711912752230693 0.023422003493731559 -0.039242208274214742 0.009506544724817368 -0.075544591810042749 -0.041719466222301806 0.0034344843003947184 -0.029851405427933533 0.039695393356972128\nleaf_weight=20.343554484425113 266.39549415849615 133.17641931446269 34.830859653186053 385.50187043659389 327.04575842386112 252.70026481803507 56.712264301488176 39.897513633593917 339.12577513162978 8.0711728474125284 9.3707674131728691 21.995087110437453 35.043392213992774 31.349452678114176\nleaf_count=236 16222 1008 188 2463 1832 6698 601 1095 2874 173 157 249 200 107\ninternal_value=-5.12211e-05 0.00522473 -0.0250963 -0.0045716 0.0137651 -0.0159158 0.000987464 -0.0201705 0.0177753 0.00860951 0.0277495 -0.0195164 0.0248604 0.00308903\ninternal_weight=1961.56 1620.9 340.663 755.155 865.741 369.654 116.953 60.2411 732.564 74.2679 44.2016 30.0663 393.439 66.3928\ninternal_count=34103 17114 16989 11093 6021 8630 1932 1331 5013 767 345 422 2139 307\nis_linear=0\nshrinkage=0.05\n\n\nTree=36\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 4 10 12 2 3 0 14 8 9 4 1 10 12\nsplit_gain=110.642 57.5937 73.9835 34.3755 30.8123 29.0124 40.7452 21.7032 20.2775 18.85 18.2487 16.8619 15.0029 32.2539\nthreshold=1.9383663450051654 5.5000000000000009 0.34228550829127619 0.23261107465006284 -0.74960001319577663 0.80139994038844931 0.58752226578811262 0.39728985235740838 19.500000000000004 0.204879909209113 1.5000000000000002 4.528681993484498 0.54890004782400781 0.1974095365040042\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 7 -4 -5 6 -1 11 -6 10 -2 -3 13 -7\nright_child=9 2 3 4 8 12 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.015447875404662154 0.044070378386522363 -0.023496171209231642 0.04417058131307848 -0.052461366955678636 -0.056565716821203677 0.016161297838785154 0.0097783783065920164 -0.039144864099717638 0.012783597508620875 0.036647555337581385 -0.043882050535885057 0.03905349594515499 0.022343363314273876 -0.013893212089333355\nleaf_weight=415.18959604343399 8.5827722242102045 29.967817681841552 43.972828471567482 27.530461094575003 10.757939395494757 223.6068363385275 259.24869970942382 268.55177598469891 101.44720171927474 209.50148929422721 14.430926918052135 15.525304698850958 164.55132699664682 147.20623992849141\nleaf_count=6365 49 1190 306 548 196 1656 3003 16943 992 815 150 148 737 1052\ninternal_value=-0.000337231 -0.00474912 -0.0190466 0.0061409 -0.00605532 0.00114421 -0.00575098 -0.0337534 0.00569362 0.0319446 -0.0106116 -0.00176917 0.00982513 0.00422491\ninternal_weight=1940.07 1707.56 497.753 183.708 139.736 1209.8 674.438 314.045 112.205 232.515 23.0137 45.4931 535.364 370.813\ninternal_count=34150 33136 20323 2042 1736 12813 9368 18281 1188 1014 199 1338 3445 2708\nis_linear=0\nshrinkage=0.05\n\n\nTree=37\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 10 4 10 13 3 14 1 6 8 4 1 14 2\nsplit_gain=107.251 44.8487 56.4188 27.6959 23.5274 21.9787 21.6598 21.8279 21.2786 20.1494 18.3784 21.1026 18.0457 15.4883\nthreshold=1.797896805829583 0.45857843137254906 5.5000000000000009 0.95849462365591409 -0.067632415460588013 -0.053930193617201173 0.31542235760988979 5.5705857276916513 7.5000000000000009 19.500000000000004 1.5000000000000002 1.4491258263587954 0.69247260929972343 0.89013838570432136\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -3 5 -1 -4 -8 -2 -9 13 -12 -7 -6\nright_child=8 3 6 -5 10 12 7 9 -10 -11 11 -13 -14 -15\nleaf_value=-0.034159559699103739 0.032103764936665662 0.0080045127426794701 0.031833943345320762 -0.058862186467506884 -0.033750854923454324 0.0041743593216570177 -0.04049787510087987 -0.035362742300105811 -0.022971714310526539 0.0047129448929627317 -0.061670949889419881 -0.00091516590459927126 -0.037009004656644105 0.021224344258166963\nleaf_weight=96.077538561425172 258.67374265077524 554.87700659385882 13.783177336212246 14.938569960184394 13.082445529289542 131.6659970759647 204.59326477907598 57.29435663158074 18.113314472138882 67.450254141353071 14.050506816711275 284.38404177583288 32.285307061392814 153.172336691292\nleaf_count=2905 1124 3756 244 154 609 1431 15379 1736 160 1619 612 2912 503 949\ninternal_value=-0.000298715 -0.00516864 -0.0112093 0.00615145 -0.00328864 -0.0153017 -0.0278807 -0.0304686 0.0284298 -0.0138284 0.003452 -0.003972 -0.00411067 0.0167225\ninternal_weight=1914.44 1637.65 1067.84 569.816 724.718 260.029 343.121 329.338 276.787 124.745 464.689 298.435 163.951 166.255\ninternal_count=34093 32809 28899 3910 9921 4839 18978 18734 1284 3355 5082 3524 1934 1558\nis_linear=0\nshrinkage=0.05\n\n\nTree=38\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 13 2 10 0 3 12 10 14 4 4 11 4 0\nsplit_gain=90.8445 38.9989 34.933 25.6316 26.2584 22.919 22.0734 21.169 26.9619 27.3132 17.3359 16.2952 23.7592 21.2689\nthreshold=1.0278617765834659 2.6133013047935933 0.18991715370570753 0.21473779385171793 0.1173266999878748 2.1288854818560901 0.10285253332280199 0.28281378178835115 0.88527816776246016 6.5000000000000009 2.5000000000000004 0.5538054284498054 5.5000000000000009 0.70084736685794613\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 7 3 6 -5 11 -1 8 9 -2 -9 13 -13 -4\nright_child=1 -3 5 4 -6 -7 -8 10 -10 -11 -12 12 -14 -15\nleaf_value=-0.0022528756981329435 0.011577801043950503 0.045452252503546696 -0.014426333791303154 0.023247650942349856 -0.016731706684065994 0.038115547571732328 -0.039080762662030422 0.029375864750290848 -0.070329791145926385 -0.031807431903369321 0.0077731538435872093 0.002819505105606912 0.060895733201412963 0.0064119148057611422\nleaf_weight=52.362819037749432 157.30954517261125 85.75659183319658 375.77786083542742 52.685693907667883 174.41092044347897 33.153333796188235 183.47841566533316 156.10376750398427 12.898762209224513 46.063314230879769 229.67805586825125 119.10875280108303 19.689294792246073 180.70443781360518\nleaf_count=1910 1213 379 7073 827 3982 109 12684 795 105 1060 1150 543 91 2143\ninternal_value=-0.000360029 0.0140966 -0.00871102 -0.019435 -0.00739537 -0.00187836 -0.0309481 0.00956229 -0.00295291 0.00164363 0.0165679 -0.00383741 0.0114346 -0.00766024\ninternal_weight=1879.18 687.81 1191.37 462.938 227.097 728.434 235.841 602.053 216.272 203.373 385.782 695.28 138.798 556.482\ninternal_count=34064 4702 29362 19403 4809 9959 14594 4323 2378 2273 1945 9850 634 9216\nis_linear=0\nshrinkage=0.05\n\n\nTree=39\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 10 10 10 0 1 11 14 10 13 11 8 0\nsplit_gain=90.6779 44.9022 40.1328 24.6225 24.6571 20.1861 19.0352 20.4349 18.0412 17.7772 16.8696 16.5037 16.4184 15.8221\nthreshold=2.2531977117618109 5.5000000000000009 0.34228550829127619 0.95849462365591409 0.56386946386946402 0.68049333853542271 5.0947940349578866 0.10742844234870796 0.78323413965962441 0.28281378178835115 0.62172772998707737 0.14479877069305611 42.500000000000007 0.75456361599240507\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 6 4 5 -1 7 -3 -7 11 -6 -10 -4 -11\nright_child=-2 2 12 -5 10 8 -8 -9 9 13 -12 -13 -14 -15\nleaf_value=-0.007471628047769097 0.037283169859802269 -0.0024632935225614655 0.0094075682771529515 -0.063575676072151907 -0.0027970236841421124 0.017858773950904482 -0.0098897164446836092 -0.045363299432937501 0.037587639798002892 -0.038029475252829688 0.020826573631001238 -0.059398062219931214 -0.042090214167954276 0.016667073194501666\nleaf_weight=598.29824827203993 148.37994914874434 34.020776284043677 134.32542989053763 13.395473920274524 109.81617430504411 185.77962203766219 93.840589419472963 152.72230313910404 4.6657372270710793 16.031016156077385 239.81123700784519 22.573555928654969 16.4819630025886 58.244612010661513\nleaf_count=8465 655 2157 1509 121 899 1681 3283 13010 49 93 1116 281 442 317\ninternal_value=-4.94228e-05 -0.00336689 -0.0172536 0.00144087 0.0021968 -0.00224408 -0.0284021 -0.0376025 0.00863057 -0.00826717 0.0134197 -0.0420429 0.00353911 0.00451346\ninternal_weight=1828.39 1680.01 431.391 1248.62 1235.22 885.593 280.584 186.743 287.295 101.515 349.627 27.2393 150.807 74.2756\ninternal_count=34078 33423 20401 13022 12901 10886 18450 15167 2421 740 2015 330 1951 410\nis_linear=0\nshrinkage=0.05\n\n\nTree=40\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 2 1 9 4 12 9 13 2 9 13 1 12 12\nsplit_gain=77.3989 48.2524 28.2975 23.3559 21.2011 18.3927 20.6406 17.7132 17.6116 28.6456 16.2767 13.351 13.0844 12.3714\nthreshold=1.9383663450051654 -1.0000000180025095e-35 0.75381749868392955 0.14768624399293936 3.5000000000000004 0.23261107465006284 0.51654338434932701 2.6133013047935933 1.527243955281093 0.34268005210784097 -1.9353917026821741 5.6900126934051523 0.10178222564395091 0.65834879259986268\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 11 -2 -5 -7 -6 9 10 -4 12 -1 -8\nright_child=4 2 8 5 7 6 13 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.007510223661964837 0.03948122797835471 -0.05220478790576092 0.04813619764623956 0.0056292363515214189 -0.010040806209629903 -0.031516380470719599 0.018534866755895763 0.040589412213764636 0.012805493940402621 0.0072187806486347875 -0.012591968651642547 -0.0029492296168142241 -0.047929012191365777 -0.036427598431478754\nleaf_weight=24.959417012752965 130.74643107503653 24.029854585125577 10.365243412088601 111.15909916796954 52.587634233757854 122.80252197908703 41.615113630890846 24.527192048728466 264.40730879455805 428.30750145530328 420.2952274432173 28.812508681148756 109.5106908918242 12.492300929501651\nleaf_count=1385 673 1662 131 3095 253 2170 328 79 1599 2739 9132 791 9830 163\ninternal_value=-0.00133813 -0.00507477 0.00037723 -0.0189048 0.0272888 -0.0102815 -0.0202543 0.00637613 0.00154743 -0.00192889 -0.0110222 -0.0339631 -0.0405376 0.0054144\ninternal_weight=1806.62 1598.76 1147.41 451.352 207.861 288.069 176.91 77.1148 1123.38 858.968 430.66 163.283 134.47 54.1074\ninternal_count=34030 33025 15263 17762 1005 5756 2661 332 13601 12002 9263 12006 11215 491\nis_linear=0\nshrinkage=0.05\n\n\nTree=41\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 2 10 10 3 10 10 8 3 10 14 0 12 0\nsplit_gain=65.5416 32.7681 30.0995 23.2815 18.2781 16.4637 16.1669 15.7018 13.6191 24.1635 22.0392 13.8732 13.6179 12.4233\nthreshold=7.5000000000000009 1.992208782912438 0.31929669030732866 0.17089216944801031 1.0296437521702668 0.16585760517799356 0.95849462365591409 35.500000000000007 0.77268779575498192 0.56386946386946402 0.63300027894733968 0.94501246039486408 0.13314182280750156 0.21110451021020937\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -2 5 13 -1 8 -4 12 10 -10 -12 -5 -3\nright_child=2 4 7 6 -6 -7 -8 -9 9 -11 11 -13 -14 -15\nleaf_value=-0.0079062618143672277 -0.034243590492558984 -0.050968927964086898 0.0063865685878530809 -0.047444918371666228 0.038249406379582893 -0.063688485818811799 -0.047093748711378731 -0.056158039535072979 0.017625831230618862 0.024017732736251962 -0.014870993743306488 0.033366270047879718 0.00093106286867098798 0.012462414952282234\nleaf_weight=343.88355407916242 188.05539910728112 7.6988543625921002 90.89633187383879 14.051684224279596 96.590419753920287 12.790194929111747 13.759470635559408 10.265019872691481 124.93537156796083 213.38428614847362 149.05236417448032 15.603638740256427 433.5734229182126 68.115511649288237\nleaf_count=6879 15320 50 1146 315 569 318 127 391 1182 960 1332 65 4907 388\ninternal_value=-0.000721662 0.00350129 -0.0224648 0.000818215 0.0239368 -0.0100791 0.0048549 -0.00044752 0.00565553 0.0112928 0.00186616 -0.010126 -0.00068991 0.00544136\ninternal_weight=1782.66 1493.44 289.217 1321.03 172.405 356.674 964.36 101.161 950.601 502.976 289.591 164.656 447.625 75.8144\ninternal_count=33949 17092 16857 16085 1007 7197 8888 1537 8761 3539 2579 1397 5222 438\nis_linear=0\nshrinkage=0.05\n\n\nTree=42\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 0 3 9 14 12 12 1 13 13 10 10 0\nsplit_gain=82.9078 49.2015 27.5402 19.8873 21.3671 28.2738 23.6287 16.3729 15.8765 15.8161 15.3288 13.7441 29.3636 13.7699\nthreshold=0.14288093717637959 1.6940571358511212 0.70626634622016105 1.8740213748638983 0.40090617732792916 0.60756002454153013 0.81420074388251518 0.169348039313127 5.7287018299102792 2.9496234745417209 1.4630009291942681 0.86240703177822864 0.78979857050032498 0.95397106958637368\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 8 4 5 7 11 -2 -1 -3 -7 12 13 -6\nright_child=1 9 -4 -5 6 10 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.029311396081722965 0.025452425132436559 0.019259341286281761 0.0094241239704492084 0.03362778627579345 0.0049316239056521587 -0.022253273251024427 -0.05055756279360793 -0.0031644873407372779 -0.0043072532659432011 0.051014545127325076 0.022332181251455599 -0.021805316485440485 0.046744379367547655 0.056458342420512508\nleaf_weight=306.74389535561204 67.094406092073768 207.87569974409416 71.981471018400043 43.685450448654592 365.73381574591622 246.33939375288901 16.122229467146099 187.26344194705598 80.466313941287808 48.244128361344337 20.116406159475446 32.138873399700969 51.392712125554681 12.459687694907187\nleaf_count=16579 1005 1064 1321 194 2501 6300 127 2709 1328 249 132 274 264 37\ninternal_value=-0.00063212 0.00582688 -0.0188693 0.000993779 -0.000465975 -0.00746459 0.00716159 0.00445418 -0.0241398 0.0254161 -0.0188163 0.0092819 0.0116529 0.0067736\ninternal_weight=1757.66 1298.47 459.192 1042.35 998.661 520.814 477.847 254.358 387.21 256.12 266.456 461.725 429.586 378.194\ninternal_count=34084 14856 19228 13543 13349 10146 3203 3714 17907 1313 6432 3076 2802 2538\nis_linear=0\nshrinkage=0.05\n\n\nTree=43\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 13 14 6 14 13 12 13 12 9 6 0 0\nsplit_gain=73.1243 49.9991 41.7829 23.4195 21.5114 17.4071 19.1009 16.0837 14.9198 13.756 14.1678 13.591 16.7848 14.4382\nthreshold=0.3373762830616705 1.4900624917218295 1.6428137117518207 0.69247260929972343 8.5000000000000018 0.52298926960539249 0.043496472104380256 0.23525212531660353 -1.9353917026821741 0.20949214218116877 0.22395486572525211 2.5000000000000004 0.44018376613338656 0.077505217748931474\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 7 -3 5 8 -7 -2 -1 10 11 12 -8 -13\nright_child=2 3 -4 -5 -6 6 9 -9 -10 -11 -12 13 -14 -15\nleaf_value=0.050573688772938219 0.027579469181727517 0.016893028889017305 0.034266710159470866 -0.028876385439491416 -0.031058968190704795 -0.039392401867471422 -0.037684646229794019 0.0028198041605007403 -0.0048701222880293436 -0.028407745329466064 0.025803518006550043 0.031027858192411252 0.037811784984451945 -0.054534089138429102\nleaf_weight=11.656712427036835 73.906192713417113 210.9272611127235 164.4254968133755 31.269679523073137 166.52326562540838 85.613641954492778 10.714773776009681 530.04451889870688 293.98325935163302 60.558962117182091 36.812441233545542 5.5061463450547334 18.943188392557204 22.947374824609142\nleaf_count=100 512 1497 815 319 14306 4064 135 3969 5801 940 329 94 141 1008\ninternal_value=0.00052418 -0.00870691 0.0119992 0.0108894 -0.015367 -0.0105396 -0.0205556 0.00589026 -0.00259753 -0.00999798 0.00192067 -0.0133892 0.010197 -0.0374854\ninternal_weight=1723.83 955.457 768.376 242.197 713.26 546.737 241.097 603.951 305.64 155.483 94.9239 58.1115 29.658 28.4535\ninternal_count=34030 28734 5296 1816 26918 12612 6711 4481 5901 2647 1707 1378 276 1102\nis_linear=0\nshrinkage=0.05\n\n\nTree=44\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 13 3 4 10 4 7 10 10 12 4 3 3 1\nsplit_gain=65.9034 33.0902 26.6528 24.7104 18.8572 23.4549 16.0447 13.1088 14.9265 12.3084 12.0578 11.8255 11.7349 11.1203\nthreshold=0.74504825313243128 2.6133013047935933 2.1288854818560901 1.5000000000000002 0.27326839826839833 6.5000000000000009 24.500000000000004 0.86240703177822864 0.8034250585480095 0.18854389541763747 5.5000000000000009 0.3784319597864772 1.8740213748638983 6.8055505752563485\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 3 -1 5 -2 -5 8 9 -6 -8 -9 -7 -12\nright_child=1 -3 -4 6 7 12 10 11 -10 -11 13 -13 -14 -15\nleaf_value=0.0025636252021386452 0.0037405335585802067 0.04318999638668844 0.034735684952914722 -0.011838949747710877 0.03474050938647099 -0.039924871220053393 -0.012512888396115302 -0.067588000617592528 0.045787372424767769 0.0091968239988113737 -0.04834403045025245 0.00085576143488185039 0.03645200840843462 0.0019482969258766613\nleaf_weight=226.66012059035711 220.85087960865349 68.047089592088014 31.61401049932465 421.20967752783326 52.153041060082614 49.57446521113161 57.932582281529903 8.0434442758560198 34.228438323363662 435.23132180399261 82.077389251615386 20.805384458974004 4.8065304462797931 12.580101740313692\nleaf_count=2130 2037 386 199 12342 351 1887 1731 68 169 2471 9853 145 41 398\ninternal_value=0.000506222 0.00992849 -0.00961412 -0.0114069 0.00713841 -0.00353934 -0.0169203 0.012474 0.0142965 0.0119957 -0.0307784 -0.0198523 -0.0326379 -0.0417096\ninternal_weight=1725.81 893.741 832.074 800.46 825.694 275.232 573.8 550.462 521.613 487.384 152.59 28.8488 54.381 94.6575\ninternal_count=34208 7555 26653 26454 7169 3965 24324 3204 2991 2822 11982 213 1928 10251\nis_linear=0\nshrinkage=0.05\n\n\nTree=45\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 2 11 9 10 14 14 9 10 3 3 2 10\nsplit_gain=54.546 31.9036 29.9149 21.7367 17.6499 17.6112 15.4229 14.1316 16.1654 15.8377 13.5877 13.0258 14.872 12.4642\nthreshold=0.14396824730298804 1.1354444923191898 1.9357224602941756 0.097696728138885289 0.40090617732792916 0.95849462365591409 0.63441062498529421 0.66998646507228099 0.62109193738564106 0.71631706333198875 0.20027498381202996 1.8740213748638983 0.9841337858536412 0.39228096301811871\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 -1 6 7 11 -6 9 -9 -3 12 -2 -8\nright_child=2 10 -4 -5 5 -7 13 8 -10 -11 -12 -13 -14 -15\nleaf_value=0.0011438636684537758 -0.0080166357470509621 -0.016699984461900338 0.030007893853309153 -0.032678784575482761 0.026929958894559444 -0.049913836551481378 -0.0069591182281476503 0.0032633727281678663 0.02065643813197679 -0.039209681688631419 0.013848495467766304 0.03871964387193963 0.014686687234498617 -0.034488521527567088\nleaf_weight=60.370317904627882 180.95428623305634 56.199556020088494 109.11911162687466 220.72701853275066 104.33900297945365 11.358834186568854 204.55053579807281 268.46833773539402 135.44962127832696 22.894822244532406 99.166707409312949 23.769386432599276 118.51491194684058 50.720468695973977\nleaf_count=2136 3431 1008 688 15484 752 157 5365 1806 681 138 783 110 917 625\ninternal_value=-0.000249944 -0.0154153 0.00513901 -0.0254365 0.00269371 0.00916436 -0.00337964 0.0105192 0.00645383 -0.000196785 0.00276227 0.0038727 0.000987179 -0.0125422\ninternal_weight=1666.6 436.464 1230.14 281.097 1121.02 542.511 578.51 531.152 426.813 291.363 155.366 323.239 299.469 255.271\ninternal_count=34081 19411 14670 17620 13982 3534 10448 3377 2625 1944 1791 4458 4348 5990\nis_linear=0\nshrinkage=0.05\n\n\nTree=46\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 13 14 1 14 12 10 2 9 3 2 2 3\nsplit_gain=60.9853 37.7024 32.0456 30.5178 24.9843 19.4885 13.6267 12.363 12.1968 11.8092 11.6931 10.7207 10.6106 14.1559\nthreshold=0.33195118872252999 1.4900624917218295 1.8602164531472061 0.6948810236403612 3.1265689134597783 0.48932556351895706 0.15980846598300777 0.077221228384019097 -0.71148459116722751 0.069631293796656252 0.12223764733466212 2.7774364689331477 0.42896023551267498 1.2135643508474616\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 6 11 7 12 -2 9 -8 -1 -9 -3 13 -6\nright_child=2 3 -4 -5 5 -7 8 10 -10 -11 -12 -13 -14 -15\nleaf_value=-0.054073578688778337 0.04776568069229814 0.010967311837078187 0.034000288505839207 -0.040947289934591784 -0.017003544306827455 -0.017659966184606701 -0.015112830463239064 -0.049696054317092259 0.0072310205311625388 0.025244967663427327 -0.016067980181385926 0.045245141620588396 0.014316927755838088 0.020587762970428598\nleaf_weight=5.6008224748075035 17.996540009742603 177.7637909273617 121.48116962658241 26.20593532291241 98.336482286802493 310.93377108068671 67.79209557804279 82.784619595331606 547.09330733097158 15.242613816517403 39.255632842192426 25.413117239251733 94.493002524133772 32.55831486498937\nleaf_count=397 125 1350 590 307 2429 10786 842 9169 3864 456 2180 171 910 495\ninternal_value=8.40109e-05 -0.00863646 0.0105849 0.00886183 -0.0145504 -0.00956551 0.00604373 -0.0330647 0.00474704 0.00243158 -0.0390977 0.0154553 0.00162988 -0.00756801\ninternal_weight=1662.95 908.588 754.363 229.383 679.205 536.322 632.882 142.884 614.885 20.8434 122.04 203.177 225.388 130.895\ninternal_count=34071 28650 5421 1828 26822 14620 4831 12202 4706 853 11349 1521 3834 2924\nis_linear=0\nshrinkage=0.05\n\n\nTree=47\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 3 13 0 10 14 6 0 13 13 10 10 3 9\nsplit_gain=52.8559 28.3014 26.7153 19.1456 18.5196 14.5787 13.2128 15.9356 17.9236 13.1483 14.0388 36.2409 17.1266 14.0883\nthreshold=0.70774219324661491 2.1288854818560901 2.9496234745417209 0.93488923097199472 0.27326839826839833 0.86829526068985119 4.5000000000000009 0.54674929111464277 1.3408171756243705 1.732394541732335 0.71631706333198875 0.78193473193473206 -0.18361356884518543 0.56746260328383402\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 -1 5 -2 9 -8 -9 10 13 -12 -13 -6\nright_child=2 -3 -4 -5 6 -7 7 8 -10 -11 11 12 -14 -15\nleaf_value=-0.014425398828399098 -0.0021620280407344777 0.035627044536009268 0.048030500304755364 0.028761148336500737 0.0098010727181422324 -0.050253448400588069 0.0074255442507075997 -0.068083452708957909 0.0025506411518162056 0.030729800630900129 -0.045179166094201202 -0.040519551050385877 0.033393095870341796 0.039692139550186231\nleaf_weight=696.6455557710724 268.17425281053875 30.470797828864306 37.394095478579402 25.738723011920229 181.06107234302908 15.750920661259441 95.88861536514014 16.36074236780405 18.463488214416429 100.6081323036924 35.549996830988675 8.7184694269672018 41.488502978812903 49.802647126838565\nleaf_count=25802 4019 198 247 261 1107 216 807 252 110 511 185 44 238 167\ninternal_value=-0.00115157 -0.0108424 0.00724431 -0.012849 0.00535542 -0.00499694 0.010719 -0.00313473 -0.0316081 0.0150563 0.00999493 -0.0072186 0.0200175 0.0163926\ninternal_weight=1622.12 752.855 869.261 722.384 831.867 283.925 547.942 130.713 34.8242 417.229 316.621 85.757 50.207 230.864\ninternal_count=34164 26261 7903 26063 7656 4235 3421 1169 362 2252 1741 467 282 1274\nis_linear=0\nshrinkage=0.05\n\n\nTree=48\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 6 2 14 13 0 9 2 8 13 9 2 9\nsplit_gain=61.1009 29.3922 19.6852 17.1412 15.8386 14.6805 13.8041 12.7675 13.402 12.2774 10.9792 10.565 21.755 12.3794\nthreshold=0.14288093717637959 2.9496234745417209 10.500000000000002 0.17496785352589664 0.39563421294144213 1.732394541732335 0.70626634622016105 0.047794029506707726 -0.16167587380505435 64.500000000000014 -1.7266965431618808 0.62109193738564106 -0.26652199078508504 0.67046756849715294\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 4 9 7 -5 -6 -1 -9 10 -2 -12 13 -13\nright_child=1 -3 -4 5 6 -7 -8 8 -10 -11 11 12 -14 -15\nleaf_value=-0.034266723373097704 0.04932748288704418 0.046913635713727149 -0.042119075186543176 0.0043296667862681368 -0.024887038435306881 0.022107853360031238 0.0041577192219368021 -0.029247610402744241 0.022908873285418758 -0.060762205690712813 -0.0086207100173491327 0.01904302484504021 0.054743333735612326 -0.06187321497890913\nleaf_weight=16.229979375377297 8.3598186918534321 39.125608043046668 66.574643192871008 671.70809849826037 202.34844658715883 139.59861002350226 50.884598810574971 14.265793866245074 64.491714803269133 8.848590762820093 305.95344432373531 8.9934904612600821 19.280116231180727 8.1978853014297766\nleaf_count=249 130 263 9081 6242 7894 759 871 447 786 105 7083 63 91 107\ninternal_value=-0.000846239 0.00483202 -0.017383 0.00338708 -0.0125467 0.00741215 -0.0190574 0.00482811 0.0132132 -0.00568854 -0.00413052 -0.00557546 0.019519 -0.0208229\ninternal_weight=1624.86 1210.07 414.795 1170.94 348.221 811.307 253.233 94.9875 78.7575 359.633 350.785 342.425 36.4715 17.1914\ninternal_count=34171 14843 19328 14580 10247 7001 8765 1482 1233 7579 7474 7344 261 170\nis_linear=0\nshrinkage=0.05\n\n\nTree=49\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 10 3 9 0 10 1 1 1 14 3 1 13\nsplit_gain=48.2428 28.4412 20.4003 19.6904 19.1115 17.0066 16.0168 11.919 15.0076 20.0282 16.5921 14.7431 12.1012 11.8983\nthreshold=1.4630009291942681 5.5000000000000009 0.78193473193473206 0.59546621855130788 0.204879909209113 0.77947462504345733 0.82793451758969006 7.3815495967864999 6.130833864212037 5.8755407333374032 0.93244939128669302 1.3544876714586842 8.1648535728454608 1.0652999387949482\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 6 -2 7 -4 8 9 10 11 -1 -9 -10\nright_child=4 -3 3 -5 -6 -7 -8 12 13 -11 -12 -13 -14 -15\nleaf_value=-0.012900273972224124 -0.010883572794322496 -0.017804272300563473 0.034041366791889141 0.040158573858168775 0.022739189965276213 0.011416013427121396 -0.032587532552288551 -0.029506196395564916 0.00099856187489639537 -0.048075403385825853 0.035443306961736268 0.016351192950419688 0.0055411354300844108 0.028966019610372151\nleaf_weight=278.46401249035262 49.633743765531108 301.88727418181952 13.778627861291168 56.378834614064544 264.7794126374647 172.68711280403659 22.21417454816401 90.269854260841385 156.31139682466164 29.107882965821773 22.100342060322873 50.040435692411847 33.346716051455587 49.289150747470558\nleaf_count=5589 647 19991 100 260 1493 1423 234 864 1775 411 259 586 225 250\ninternal_value=-9.72341e-05 -0.00442272 -0.000261103 0.0219643 0.0174138 -0.00261202 -0.0068502 -0.00604036 -0.00304534 -0.00892875 -0.00555123 -0.00840835 -0.0200833 0.00781102\ninternal_weight=1590.29 1275.88 973.989 92.3716 314.413 881.617 35.9928 708.93 585.313 379.713 350.605 328.504 123.617 205.601\ninternal_count=34107 31967 11976 594 2140 11382 334 9959 8870 6845 6434 6175 1089 2025\nis_linear=0\nshrinkage=0.05\n\n\nTree=50\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 2 14 2 1 12 11 11 12 6 6 8 8 7\nsplit_gain=41.5243 34.5491 23.446 19.4876 19.4327 23.1414 15.559 19.6197 14.1313 12.8046 12.0323 16.598 20.4611 15.6591\nthreshold=0.27326839826839833 0.81060921052942725 0.6948810236403612 1.992208782912438 5.2616169452667245 0.086958361481737009 0.15970234850245438 0.200564726431503 0.18523167472669072 1.5000000000000002 4.5000000000000009 23.500000000000004 44.500000000000007 12.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 9 6 5 -1 -2 -8 -9 -3 -10 13 -13 -12\nright_child=3 2 -4 -5 -6 -7 7 8 10 -11 11 12 -14 -15\nleaf_value=0.020828729664144355 0.061719554484834294 0.020244158485110419 -0.024375180508389403 0.036924605464099813 -0.0058879701775139404 -0.03503636939305553 -0.040496742685488404 0.031661870214852554 0.0084209593919007241 -0.0008107843058600228 -0.068108683908553722 0.014836544016117359 -0.044345966387936586 -0.012221552785701542\nleaf_weight=19.991337577463128 11.408819802571086 134.16932128136978 63.301583114138339 53.746464031748474 157.48479882406536 180.257551333023 21.77855659252964 51.506092873867601 573.07851514266804 154.87114431429654 15.076354776043443 96.310792067553848 16.19731780141592 59.904142519400921\nleaf_count=701 114 992 1068 350 3976 16538 383 505 3725 2666 307 1120 450 1088\ninternal_value=0.000869264 -0.00816212 0.00295082 0.0080038 -0.0190853 -0.0293834 0.00612339 0.0052897 0.00656592 0.00899878 0.00482706 -0.00614445 0.00600067 -0.0242073\ninternal_weight=1609.08 710.076 352.342 899.007 357.734 200.249 845.261 833.852 812.073 289.04 760.567 187.489 112.508 74.9805\ninternal_count=33983 25941 4726 8042 21215 17239 7692 7578 7195 3658 6690 2965 1570 1395\nis_linear=0\nshrinkage=0.05\n\n\nTree=51\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 9 4 3 9 10 12 9 9 0 13 9 8 13\nsplit_gain=40.5589 25.8311 24.6799 18.772 17.6663 17.415 16.0084 16.9244 14.7391 13.1497 12.061 12.0021 20.0889 16.4721\nthreshold=1.4261840188530308 0.40090617732792916 5.5000000000000009 1.7569128067320097 0.18560323472877255 0.95849462365591409 0.20050230359448049 0.14396824730298804 0.16956570461989892 0.68049333853542271 2.9496234745417209 0.33195118872252999 21.500000000000004 0.94092064921267249\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 -4 -2 -3 7 9 -5 -1 -6 12 -8 -14\nright_child=4 5 3 8 10 -7 11 -9 -10 -11 -12 -13 13 -15\nleaf_value=-0.011831482367039764 -0.012302674402895048 0.0083201898092740735 -0.027410367449765147 -0.025038645503632259 0.016813274958312984 -0.056667756722973174 -0.050465640493898178 0.015561293350794869 0.056668777516194535 0.016092649192028668 0.047652068288647216 0.00063044765364976205 -0.02745777515793528 0.0191451640890151\nleaf_weight=189.06977429584367 46.06136226374656 404.53931323345751 202.43355964799412 8.9531173193827289 248.77199966646731 9.5885565197095257 45.436466710467357 152.21280276746256 11.737928725313393 53.243787844083272 35.913825999945402 77.391650877194479 45.82036881800741 30.962510988581926\nleaf_count=4791 633 2905 18163 271 1389 130 790 1945 68 878 254 705 847 208\ninternal_value=0.000697782 -0.00348006 -0.00863162 -0.0227711 0.0162157 0.0066826 -0.0032923 0.00254928 0.0217892 -0.00565476 0.0208568 -0.0147932 -0.0244856 -0.00866072\ninternal_weight=1562.14 1231.39 817.262 223.125 330.747 414.128 594.137 394.526 20.691 242.314 284.686 199.611 122.219 76.7829\ninternal_count=33977 31701 28666 18502 2276 3035 10164 7614 339 5669 1643 2550 1845 1055\nis_linear=0\nshrinkage=0.05\n\n\nTree=52\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 6 10 14 12 10 2 11 13 3 2 12 3 3\nsplit_gain=39.5412 27.0822 12.8014 18.6434 15.7419 13.7763 13.5261 10.8897 10.8509 16.1435 12.6769 9.43693 9.37479 9.03635\nthreshold=2.4146125192805905 4.5000000000000009 0.27326839826839833 0.70000416581427871 0.21141684650071227 0.95849462365591409 0.93387620018493878 0.094034202108957757 1.0424343345234035 1.7057859673546234 1.5610522555580648 0.09978491514898781 1.2501037838947984 -0.20861508743145712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 6 -5 8 12 -8 -4 10 13 -9 -1 -10\nright_child=-2 -3 5 4 -6 -7 7 11 9 -11 -12 -13 -14 -15\nleaf_value=-0.018949677480376563 0.036496721245013325 -0.011828280498244121 0.003373770443750289 -0.0096411895255419846 -0.057487345143381469 -0.044907958064484395 -0.037432303196804753 0.057417846974089526 -0.023622259437139002 -0.019224732683581474 0.049892548185825669 0.0093104766091600469 0.0085097267658858968 0.019191099381896271\nleaf_weight=98.988524062093347 72.651431130710989 416.8148706521315 368.34338423190638 53.405279504717328 24.728020864073187 11.189691603649406 10.890362261096014 9.8076156107708794 12.42401381395757 26.704975349828601 32.424336994532496 199.27153406431898 44.407136753899977 173.97854570671916\nleaf_count=2914 510 22388 2976 904 305 104 128 52 65 183 179 1942 608 846\ninternal_value=0.000737799 -0.00103697 0.00318742 -0.00332617 -0.0253189 0.00778609 0.00145937 0.00923245 0.00881806 0.0169387 0.0214194 0.0118297 -0.0104459 0.0162267\ninternal_weight=1556.03 1483.38 1066.56 441.498 78.1333 625.065 363.365 219.97 613.875 245.532 218.827 209.079 143.396 186.403\ninternal_count=34104 33594 11206 6853 1209 4353 5644 2122 4249 1273 1090 1994 3522 911\nis_linear=0\nshrinkage=0.05\n\n\nTree=53\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 6 13 14 13 11 2 9 3 14 2 11 13 1\nsplit_gain=42.9657 23.7007 16.7687 16.0796 13.7287 12.8192 12.4397 12.7963 13.978 11.476 11.2485 14.4267 11.7074 11.4348\nthreshold=0.34755730289246772 7.5000000000000009 1.8602164531472061 0.63300027894733968 -0.084320303758185181 0.30198526726146607 1.4304554939972705 0.10965635606201694 1.4393308112662344 0.81108005967984309 -0.70703784408564696 0.33846321047115019 0.87952391891555892 7.1329500675201425\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 6 -5 9 7 -1 -9 -2 -7 13 -13 -12\nright_child=2 -3 -4 4 -6 10 -8 8 -10 -11 11 12 -14 -15\nleaf_value=-0.017046495843413126 0.045506071524014431 -0.025272552594859291 0.027579976944844029 -0.03669272571167586 -0.0069499367133032458 -0.017448752608540646 0.011880961501342172 -0.0030261613822109494 0.030547548720730591 0.0037661775289912318 -0.0053168332638475436 -0.0008800036619574773 0.015493892756017279 -0.083135307720244855\nleaf_weight=128.54390722315293 35.204486779868603 155.00763499629102 103.98768570809625 56.627250693854876 120.68941837176681 57.744905672036111 169.24883943935856 175.45628730603494 36.620105085195974 30.566244638059288 12.39704525191337 208.32805409352295 228.11943426216021 6.7749675358645609\nleaf_count=4221 254 15876 559 2102 1840 726 1431 3384 294 271 141 1763 1253 76\ninternal_value=-0.000123303 -0.0076771 0.00918706 -0.00367716 -0.016601 0.00584264 0.000841051 -0.00466323 0.00288731 0.0264548 0.00315557 0.00579944 0.00769405 -0.0355742\ninternal_weight=1525.32 842.193 683.123 687.186 177.317 579.135 509.869 340.62 212.076 65.7707 513.364 455.62 436.447 19.172\ninternal_count=34191 29148 5043 13272 3942 4484 9330 7899 3678 525 3959 3233 3016 217\nis_linear=0\nshrinkage=0.05\n\n\nTree=54\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 4 13 13 3 14 1 10 12 13 0 0 3\nsplit_gain=39.8237 26.4646 22.0503 13.6328 13.6118 12.8315 11.3536 10.901 10.1421 9.11774 14.4136 12.6498 12.0005 11.03\nthreshold=0.13963014728278042 1.8194180785221719 12.500000000000002 2.6133013047935933 -0.19863109897944198 2.1288854818560901 0.44746339656424899 5.7287018299102792 0.096022128556375139 0.15344145005079826 0.73052812899374042 0.97043570271994106 0.11449036280221957 -0.75305914588649314\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 5 8 -3 6 -1 -8 -2 -10 11 12 -11 -12\nright_child=1 4 -4 -5 -6 -7 7 -9 9 10 13 -13 -14 -15\nleaf_value=-0.001314025611737387 -0.028383973203657993 -0.018562840245684904 -0.045919664082425385 0.035193565823118084 0.033500273356109118 0.034491595628943134 -0.027261096868474263 0.004491303880272959 0.02031267577896944 0.018940074230461235 0.027860034343357421 0.062112826717858109 -0.011469126506875798 0.0038031812604377227\nleaf_weight=149.48421629180666 26.41426758072339 13.59060722356662 45.345254358835518 30.16926059674006 105.32154085440561 15.184439014643429 125.16366646051756 34.073489612899721 65.600550075585488 35.418082805059385 51.55279844161123 5.5230370005592695 302.5269969466608 516.73499738483224\nleaf_count=3128 593 94 7132 158 752 107 7876 692 1711 876 377 30 6925 3817\ninternal_value=0.000630304 0.00520823 -0.0136381 0.00262679 0.0274447 -0.00899137 -0.0112345 -0.0204811 0.00161391 0.00245282 0.00114696 -0.00692739 -0.00823564 0.00603063\ninternal_weight=1522.1 1152.85 369.251 1033.94 118.912 323.906 308.721 159.237 1003.77 977.356 911.756 343.468 337.945 568.288\ninternal_count=34268 15333 18935 14487 846 11803 11696 8568 14329 13736 12025 7831 7801 4194\nis_linear=0\nshrinkage=0.05\n\n\nTree=55\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 2 12 1 14 10 1 2 1 1 8 3 0 1\nsplit_gain=35.945 28.1191 15.6625 14.7827 16.8951 12.4843 16.3496 12.2537 12.1269 16.0496 11.6246 11.4646 19.9473 13.1773\nthreshold=0.29449472096530926 0.25897856863174212 0.76309033109884206 0.6179969608783723 0.72190845238709189 0.083722741433021816 4.1850135326385507 1.2413453555885363 5.0610318183898935 5.503817319869996 14.500000000000002 0.031554964581337788 0.11122955540524941 6.0034096240997323\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 7 -3 -5 11 -7 8 -2 -10 -6 -1 13 -13\nright_child=2 3 -4 4 10 6 -8 -9 9 -11 -12 12 -14 -15\nleaf_value=0.023878309849246951 0.022229533228275981 -0.048848332115615824 -0.025184713726508698 0.0078155200029737925 0.00818013971648336 -0.043958040301266557 -0.01471529772736596 0.021385700463685424 -0.035213156993349988 0.0053919566047008413 -0.029488566957283496 -0.040171123327911916 -0.073613122785444837 0.045799125301563187\nleaf_weight=27.592092190170661 101.15925159887411 13.407343547034541 31.944902236107737 334.10027976997662 30.694358501117676 73.989636366401101 136.57355840830132 181.51886288018432 24.675987510709092 467.57879434141796 58.447632568189874 6.4065617282176452 10.004551183432339 10.319920315872876\nleaf_count=1203 1197 1165 274 5645 370 10657 5610 1131 428 4280 1417 173 416 63\ninternal_value=0.00143177 -0.00684068 0.00862378 0.000962978 0.00265231 -0.0196598 -0.0251501 0.0100481 0.00654994 0.00328922 -0.0165712 0.00169123 -0.0212458 0.0124623\ninternal_weight=1508.41 701.536 806.878 436.65 423.242 264.886 210.563 774.933 593.414 492.255 89.142 54.3231 26.731 16.7265\ninternal_count=34029 26719 7310 8597 7432 18122 16267 7036 5905 4708 1787 1855 652 236\nis_linear=0\nshrinkage=0.05\n\n\nTree=56\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 1 14 0 10 10 3 14 10 12 1 8 10\nsplit_gain=30.4604 17.6483 14.7432 15.5531 13.0258 12.9276 14.0996 11.6845 19.5536 15.9199 14.6523 12.9671 13.6743 10.9704\nthreshold=0.14396824730298804 12.500000000000002 4.3886544704437265 0.44322655514882026 0.72432681855990089 0.95849462365591409 0.78193473193473206 1.4766501342390617 0.5721442419320123 0.26401991614255771 0.15520902028626146 4.909909963607789 43.500000000000007 0.86240703177822864\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 -1 6 7 11 -9 10 -10 -2 -13 -8\nright_child=5 -3 -4 -5 -6 -7 13 8 9 -11 -12 12 -14 -15\nleaf_value=-0.011047727594609993 0.011707183519352959 -0.042411633085573351 0.001997872084485645 -0.033771716160252915 0.04789639139506404 -0.045282004836259965 0.030428031006759684 0.037631981529518549 0.038763667345062619 0.015481467732592048 -0.038255125664706104 -0.0016616173871398356 -0.029159999746083895 -0.0054445807992113456\nleaf_weight=72.353293522726744 170.99431351292878 44.795242737716762 154.3695881178719 88.074106981308432 9.7504003474023175 12.568258835934101 79.079985976452008 56.443218361353502 6.7656081046443424 71.629851804231293 31.557172351982445 602.439708439284 47.892617502715439 28.671186396386474\nleaf_count=2095 4196 7337 3015 6818 151 168 510 400 61 481 320 7719 528 258\ninternal_value=-0.000911766 -0.0133269 -0.00919454 -0.0192996 -0.00355558 0.00323484 0.00383226 0.00194882 0.0139888 0.00162197 -0.024018 -0.000504454 -0.00372835 0.0209205\ninternal_weight=1477.38 369.343 324.547 170.178 82.1037 1108.04 1095.47 987.722 166.396 109.953 38.3228 821.327 650.332 107.751\ninternal_count=34057 19416 12079 9064 2246 14641 14473 13705 1262 862 381 12443 8247 768\nis_linear=0\nshrinkage=0.05\n\n\nTree=57\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 2 11 0 1 9 1 10 2 6 3 1 10\nsplit_gain=34.7926 19.2773 16.5235 13.9359 18.5536 14.7197 11.7628 11.6746 10.9823 9.5585 8.99339 8.46928 8.28629 8.6412\nthreshold=0.14288093717637959 12.500000000000002 1.992208782912438 0.15784628527155695 0.70626634622016105 6.130833864212037 0.62109193738564106 1.4491258263587954 0.050316455696202543 2.2500374604554767 7.5000000000000009 -1.7381005412235642 1.750018358230591 0.16287683031869082\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 6 4 7 9 8 11 -2 -5 -10 -1 -9 -14\nright_child=2 -3 -4 5 -6 -7 -8 12 10 -11 -12 -13 13 -15\nleaf_value=0.021217339558743272 -0.066823866941537247 -0.045643511669624624 0.026754598030417073 -0.04539495837645692 0.027197874237519598 0.017338227762851908 0.015890230854663481 0.03638559504564156 0.0026272714380898496 0.0065118261211255316 -0.011484549603468985 -0.055776034391362209 -0.013000777081890728 0.018995199638120706\nleaf_weight=3.7865638341172625 5.0800331914797416 41.747483844403177 72.198962089139968 56.161566979659256 39.59080205304781 14.379787027603014 135.35297305183485 10.154661622887941 783.93201425415464 10.223298479802905 130.87055923009757 19.126848767511547 138.27301836729748 23.960672314628027\nleaf_count=242 73 7258 623 2799 822 243 769 369 8326 66 5089 2498 4225 726\ninternal_value=-0.00052994 -0.0141013 0.00378095 -0.00979865 -0.00359363 -0.0275804 0.00218526 -0.00994231 0.000153618 -0.0374233 0.000598115 -0.0426973 -0.00536746 -0.0081878\ninternal_weight=1484.84 357.405 1127.43 315.657 234.893 80.7647 1055.24 195.302 919.883 66.3849 914.803 22.9134 172.388 162.234\ninternal_count=34128 19248 14880 11990 8882 3108 14257 8060 13488 2865 13415 2740 5320 4951\nis_linear=0\nshrinkage=0.05\n\n\nTree=58\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 4 10 1 14 13 12 10 13 14 1 13 9\nsplit_gain=27.104 19.0546 18.3374 13.8327 11.5254 12.7498 13.8175 11.1623 16.199 14.4478 12.1833 11.8068 13.4821 12.5134\nthreshold=0.14288093717637959 1.8602164531472061 12.500000000000002 0.28281378178835115 4.7296948432922372 0.44746339656424899 -1.9353917026821741 0.20050230359448049 0.35329985652797707 -1.6392762945591308 0.82407444526602891 4.2128593921661386 0.90622825774246907 0.36355675436704565\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 7 4 -3 5 -1 -7 10 -9 -10 -2 12 13 -11\nright_child=1 3 -4 -5 -6 6 -8 8 9 11 -12 -13 -14 -15\nleaf_value=-0.0015098730829744111 0.016571578789418128 -0.0014660460364064594 -0.042924746878656134 0.031202369815508529 0.0041928096145470292 0.047156183500527749 -0.032126716435468265 -0.015748498313490716 0.053530051091977809 -0.069439839863034869 -0.0122127761214431 -0.00047178806999042855 0.052001907424180552 0.024480470692948866\nleaf_weight=90.060004473198205 197.35821457603015 46.829864226281643 39.484825839463156 103.55661176983267 131.1333955078735 4.9774630904430515 91.748254778067349 149.15703055029735 12.654441921738906 4.592490949551574 44.35388375542243 518.52815037814435 22.20572928711772 8.7792698508128506\nleaf_count=2485 3771 360 7270 644 2409 146 6946 3411 83 132 1711 4512 85 113\ninternal_value=0.000740083 0.00460262 -0.0112139 0.021087 -0.00715323 -0.0150987 -0.0276061 0.0019946 -0.00114536 0.00271878 0.0112611 0.00146404 0.0289195 -0.0103634\ninternal_weight=1465.42 1108.02 357.404 150.386 317.919 186.786 96.7257 957.629 715.917 566.76 241.712 554.106 35.5775 13.3718\ninternal_count=34078 14822 19256 1004 11986 9577 7092 13818 8336 4925 5482 4842 330 245\nis_linear=0\nshrinkage=0.05\n\n\nTree=59\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 10 11 10 12 0 3 10 9 9 0 12 10 12\nsplit_gain=24.8909 28.147 16.7116 15.116 17.902 14.5247 14.6288 20.574 14.4719 12.2156 12.2667 10.8429 10.6187 10.1109\nthreshold=0.89013838570432136 0.62279616455304676 0.094034202108957757 0.86240703177822864 0.76309033109884206 0.080737732012600816 1.8740213748638983 0.17089216944801031 0.050041143743288476 0.11248255864287728 0.026851214599446126 0.17131079051116874 0.76728257406817468 0.09978491514898781\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 -2 4 12 9 7 8 -7 -1 11 -11 -3 -4\nright_child=2 3 13 -5 -6 6 -8 -9 -10 10 -12 -13 -14 -15\nleaf_value=-0.01387574655424054 -0.047857074345320202 0.0093307116947258482 0.049792258302562564 -0.027785117185893229 -0.057032334627738046 0.029945909447239266 0.022770996730925933 -0.010741167653128762 -0.037864080188294569 0.019757166950469012 0.035922325075247716 -0.037366751690316426 0.035779935494752411 0.0070821571064134175\nleaf_weight=55.772603633988183 12.663629091461187 123.38358537759632 13.221420703222973 23.15970223583281 7.4549580342136315 7.6605044680181882 22.869068666594103 332.82298293334316 127.96662703499896 17.768388933967803 38.294036305509508 13.935031502041964 54.52312245965004 600.21365751512349\nleaf_count=2479 204 885 88 323 78 261 270 9326 11997 391 854 309 359 6231\ninternal_value=-0.000635815 -0.00633401 0.00687628 0.00951557 0.0143001 -0.0116965 -0.0155712 -0.0174875 -0.0338429 0.0034433 0.0172439 -0.00572611 0.0175913 0.00808223\ninternal_weight=1451.71 825.611 626.099 208.521 185.362 617.089 491.319 468.45 135.627 125.77 69.9975 31.7034 177.907 613.435\ninternal_count=34055 27532 6523 1645 1322 25887 21854 21584 12258 4033 1554 700 1244 6319\nis_linear=0\nshrinkage=0.05\n\n\nTree=60\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 2 11 10 3 3 6 10 2 11 13 11 14 12\nsplit_gain=27.2993 17.3176 14.7586 15.2539 11.5757 11.0735 15.4807 10.4324 14.7817 13.3177 11.1045 12.4963 9.8255 9.53676\nthreshold=2.2531977117618109 -0.71148459116722751 0.5538054284498054 0.86240703177822864 -0.27674685557969109 0.24873812587180019 1.5000000000000002 0.45857843137254906 -0.9771914392159412 0.088508261124055423 1.2465570421923056 0.26948978220549369 0.66998646507228099 0.45977311129625353\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 5 4 -4 6 -3 9 -9 -1 -11 12 -12 -7\nright_child=-2 2 3 -5 -6 13 -8 8 -10 10 11 -13 -14 -15\nleaf_value=0.025668699216182662 0.026797120728338238 -0.045447253116183708 0.040394790530141042 -0.020702122869268896 0.01029389021602623 0.0030971484341210942 -0.0065976295498394082 0.01781761955003933 -0.033702273451772631 -0.039881401042222722 -0.01471924058419135 -0.057162794544481678 0.054133289418163436 -0.024002961226159351\nleaf_weight=10.90177489974303 88.339449791237712 27.132391044404358 37.498531500808895 32.795409167185426 195.68091602157801 554.72123629273847 297.32681119357585 40.37270632898435 19.983805851312354 78.43290999281453 10.115243249572819 6.4418384437449268 8.9556043439079058 33.481401066295803\nleaf_count=494 644 927 201 284 1200 8473 13682 433 267 6954 219 57 40 270\ninternal_value=2.65766e-05 -0.00173872 0.000446596 0.010788 0.0152864 -0.00257567 -0.00997603 -0.0163589 0.000488313 -0.0251406 -0.0306632 -0.0018887 0.0186998 0.00151645\ninternal_weight=1442.18 1353.84 1178.64 265.975 233.179 912.662 324.459 175.204 60.3565 114.847 103.946 25.5127 19.0708 588.203\ninternal_count=34145 33501 25037 1685 1401 23352 14609 8464 700 7764 7270 316 259 8743\nis_linear=0\nshrinkage=0.05\n\n\nTree=61\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 13 2 14 13 1 12 12 12 2 1 14 2 13\nsplit_gain=20.7398 21.2683 18.9584 17.3369 21.6574 12.158 13.0183 11.8475 10.0994 10.3798 11.2066 14.3906 9.75727 10.9281\nthreshold=0.27326839826839833 2.2531977117618109 0.25897856863174212 0.80688273093955121 0.86578312661731183 5.3563439846038827 0.18969223474415661 0.2020546430968527 0.17993568848336086 2.1520662742152665 5.7287018299102792 0.5321233765890433 -0.61466903587674659 -1.7266965431618808\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 7 5 8 -5 -1 -7 -2 9 10 11 -4 13 -9\nright_child=1 -3 3 4 -6 6 -8 12 -10 -11 -12 -13 -14 -15\nleaf_value=-0.028955709044056383 0.019634098870496303 0.035202359294973833 -0.0039784674052782774 -0.0061418343049550695 -0.077875762028233514 0.0062343009996767432 -0.032043143317441049 0.050625630761756926 0.015558498370447328 0.020606005091254934 0.013315041208874624 -0.038198038594589909 0.0028927840172663145 -0.019872880404340262\nleaf_weight=124.6208376888826 91.803491753176786 53.715257741976529 105.03467859938974 34.973871301030158 14.171253694221376 70.901996690954547 31.188538020593114 4.9521592403762034 114.74485637509497 39.523676141747274 49.163635099423118 42.589357722987188 574.84863100550137 79.624410797725432\nleaf_count=14647 1220 392 2786 1046 151 2047 1010 44 1534 297 552 1961 4941 1396\ninternal_value=-0.000204494 0.00510348 -0.0070171 -0.000462421 -0.0279672 -0.0185349 -0.00565681 0.00290738 0.00345654 -0.00246057 -0.00717412 -0.0140414 0.00055252 -0.0152017\ninternal_weight=1431.86 804.944 626.913 400.201 49.1451 226.711 102.091 751.229 351.056 236.311 196.788 147.624 659.425 84.5766\ninternal_count=34024 7993 26031 8327 1197 17704 3057 7601 7130 5596 5299 4747 6381 1440\nis_linear=0\nshrinkage=0.05\n\n\nTree=62\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 1 13 14 2 13 12 1 12 6 13 7 1\nsplit_gain=27.5188 13.6651 16.0664 15.0405 14.6331 12.3535 10.9969 10.0958 9.6901 10.806 20.5345 9.65782 11.7843 10.4182\nthreshold=0.33195118872252999 2.1520662742152665 2.1380437612533574 -0.45797181309695573 0.49463494509828093 1.9357224602941756 1.4630009291942681 0.2436581872418776 3.6326419115066533 0.20050230359448049 17.500000000000004 1.1020406375351024 17.500000000000004 4.2128593921661386\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -3 -4 7 8 -2 -6 -10 -11 12 13 -9\nright_child=5 3 4 -5 6 -7 -8 11 9 10 -12 -13 -14 -15\nleaf_value=-0.031211149863918059 0.020228892432409752 -0.036203807854776886 0.0023777900659250656 0.025796181397831543 -0.045070601026944351 0.03511637737735137 0.0055564127485198053 0.041770493748152226 -0.007041950804934762 -0.036998347094770366 0.047662653896403048 0.010649840184159445 -0.026590872973273089 -0.0014085937241589232\nleaf_weight=73.145414095633896 94.130445038434118 11.021814689971505 261.32378110039281 53.14461139170453 27.963998949242523 35.612214125692844 62.466231617610902 13.760842836461959 168.88444186854758 74.458012818475254 7.0570658693322903 217.08506192965433 44.419602622045204 259.79324834863655\nleaf_count=8664 736 108 5764 430 3739 282 495 119 6665 2497 297 1188 816 2256\ninternal_value=-0.00050646 -0.00713851 -0.0092375 0.0147544 -0.00652097 0.00687008 -0.0133309 0.00521815 -0.017574 -0.0143362 -0.0291801 0.00254457 -0.00300505 0.000907712\ninternal_weight=1404.27 739.465 675.299 64.1664 602.154 664.801 340.83 629.189 278.364 250.4 81.5151 535.059 317.974 273.554\ninternal_count=34056 28659 28121 538 19457 5397 13693 5115 13198 9459 2794 4379 3191 2375\nis_linear=0\nshrinkage=0.05\n\n\nTree=63\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 13 12 9 3 9 12 9 14 3 0 3 1\nsplit_gain=27.8495 17.2674 15.6019 17.4404 23.3716 16.4839 10.2733 9.7399 9.50324 11.8645 12.571 12.0077 8.16194 7.69091\nthreshold=0.13963014728278042 5.5000000000000009 1.8602164531472061 0.19909436529862615 0.34755730289246772 -0.64598341937415527 0.22067448407602921 0.086958361481737009 0.1898944789484063 0.70300088204528965 0.96710619023857913 0.78438036099062902 -0.98679632137124029 3.2570905685424809\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 5 8 12 -4 -3 -5 10 -10 13 -2 -11\nright_child=2 7 6 4 -6 -7 -8 -9 9 11 -12 -13 -14 -15\nleaf_value=-0.0053557318597694569 -0.0017257233140939487 0.015811591639238308 -0.019311844918762547 -0.050529921621515018 0.0034213591531874923 0.019263585779554192 0.024460636127621765 -0.03713204160341578 -0.017204781292272136 0.040861423337630154 0.016982765780069978 0.0044448297030964283 -0.053404715282470996 -0.054107987649735616\nleaf_weight=233.93953524398967 25.239796302252216 9.2089058987912704 14.26030903495848 18.524011208326556 511.87337759276852 199.55829880773672 124.75921519729309 77.545730345271295 69.857534323236905 1.5664743351226196 42.373159536160529 14.682172458618878 10.001037088630254 33.661635465250583\nleaf_count=5800 956 573 145 786 4210 4184 845 12556 1541 33 478 269 537 1177\ninternal_value=0.000420372 -0.0124759 0.00430682 0.00195561 -0.00204057 0.013701 0.0198647 -0.031396 -0.0174493 -0.0133835 -0.00426114 -0.0334097 -0.0174607 -0.0488737\ninternal_weight=1387.05 320.694 1066.36 927.337 692.538 234.799 139.02 86.7546 180.665 162.141 112.231 49.9103 35.2408 35.2281\ninternal_count=34090 18929 15161 14171 8494 5677 990 13129 4284 3498 2019 1479 1493 1210\nis_linear=0\nshrinkage=0.05\n\n\nTree=64\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 3 2 0 0 9 12 9 13 14 9 9 14 3\nsplit_gain=22.2864 17.6795 13.1375 15.1171 13.5303 11.3135 11.846 12.8439 10.9591 13.6231 9.78338 13.1408 13.1004 9.72504\nthreshold=0.34268005210784097 2.1288854818560901 0.92110015028812986 0.34007656527049129 0.1173266999878748 0.2529894131731712 0.24693037208688387 0.28664751606416011 1.5007010094932978 0.66417691709368565 0.58595927517517843 0.51654338434932701 0.88961174169059232 0.81402353687074069\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -4 13 -6 -7 -8 -2 -10 11 12 -11 -1\nright_child=8 -3 3 -5 5 6 7 -9 9 10 -12 -13 -14 -15\nleaf_value=-0.011258452449768362 0.0035619061214079845 0.02794436764079854 -0.021530581690180423 0.0053090005650787853 -0.02681312176543529 0.011564465479053619 0.010303900774656552 -0.05594486178418781 0.044789859871305099 0.019348441719178537 0.029770885511663178 -0.052968612879950673 -0.035356616333124334 0.017068152633616312\nleaf_weight=75.526752782549011 502.07992269599345 36.508297148160636 64.31130487867631 265.69225075954455 182.23369860934326 47.172101348260185 12.074287002556959 16.572531506535597 37.217397572472692 49.410699452739209 39.269091427326202 9.4400275573134405 13.009272394003345 49.16050413105404\nleaf_count=4414 4244 364 785 3732 17071 832 331 556 337 260 151 69 80 854\ninternal_value=0.000340989 -0.00553409 -0.00728568 2.92412e-05 -0.0135763 -0.02009 -0.00379193 -0.0286161 0.00710683 0.0190005 0.0100411 -0.00100118 0.00756935 -4.3168e-05\ninternal_weight=1399.68 749.252 712.743 330.004 382.74 258.053 75.8189 28.6468 650.426 148.346 111.129 71.86 62.42 124.687\ninternal_count=34080 28939 28575 4517 24058 18790 1719 887 5141 897 560 409 340 5268\nis_linear=0\nshrinkage=0.05\n\n\nTree=65\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 12 14 3 2 13 12 13 3 3 13 2 13\nsplit_gain=18.2604 15.6928 10.5963 13.0687 14.9048 11.4487 11.0033 10.6738 9.83936 8.94383 11.3515 8.92735 11.6812 8.07694\nthreshold=0.13963014728278042 1.875273743832488 0.098911901734213256 0.44746339656424899 1.0126424540725496 1.4900624917218295 1.5007010094932978 0.094075162804246951 -0.13286243718190949 1.8740213748638983 1.8062975497470057 1.4261840188530308 -0.12823173267677504 1.0424343345234035\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 9 7 13 6 -6 -5 -1 -3 10 11 -2 -13 -4\nright_child=1 8 3 4 5 -7 -8 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.0058171077461230472 -0.0015625155186152371 -0.021345225191752807 0.0029580892561128901 -0.041045317511102762 -0.020977009467586599 0.028365702619366242 0.017741583136986448 0.039122100537378655 0.029299124100406321 0.023151963674239124 -0.049490653164473403 -0.012439089921524503 0.017390127655492006 -0.033951955189654021\nleaf_weight=40.66100473163533 739.03338553820504 10.189411491155623 102.2182681284612 88.439041406934848 29.066761729540303 18.306211538612843 8.1850453135557455 18.427987244038377 77.876118823420256 43.854556498583406 10.311191888991742 41.628977385349572 144.07898720167577 16.312432125792839\nleaf_count=1474 12906 83 3463 11548 1209 143 142 506 686 281 61 281 879 466\ninternal_value=2.2509e-05 0.00317161 -0.0104022 -0.0146992 -0.0247672 -0.00171742 -0.0359347 0.00861584 0.0232656 0.001342 0.000295458 0.000903452 0.0106727 -0.00236309\ninternal_weight=1388.59 1066.97 321.617 262.528 143.997 47.373 96.6241 59.089 88.0655 978.907 935.053 924.741 185.708 118.531\ninternal_count=34128 15177 18951 16971 13042 1352 11690 1980 769 14408 14127 14066 1160 3929\nis_linear=0\nshrinkage=0.05\n\n\nTree=66\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 10 13 1 0 11 6 11 3 3 13 4 0 0\nsplit_gain=23.2984 18.0251 17.3825 13.691 13.3074 12.8061 11.9115 11.5573 16.7273 15.1277 14.3423 12.2353 11.2363 12.7746\nthreshold=0.22287581699346407 0.95849462365591409 1.6428137117518207 5.4778068065643319 0.87645216386918368 0.15730752734423484 8.5000000000000018 0.61813055473851874 1.195786293646284 1.2501037838947984 -0.65713924291875048 10.500000000000002 0.39959369704351672 0.84793958240236689\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 5 4 -1 -2 -4 8 10 -9 11 -7 -10 -14\nright_child=1 -3 6 -5 -6 7 -8 9 12 -11 -12 -13 13 -15\nleaf_value=-0.017201057468033631 0.042214403775598264 -0.054717876047524178 0.026163859936203439 0.0018359772440026318 0.031001633434256833 -0.041015145475885681 -0.022173448407695747 0.017290971402634629 0.040839308197465195 -0.033083527344385549 -0.0032691403103495328 0.035854712365439245 -0.012517644878432483 0.031254941689813125\nleaf_weight=293.4550535446615 19.007175987004302 11.36548599693924 133.73660384910181 202.3921543530887 14.155173195409587 42.571282177115791 13.320344656007363 159.17415991798043 30.45778067514766 15.508941405918447 354.76482504379237 5.0973133326042435 41.750092240865342 26.34925749246031\nleaf_count=19550 340 164 912 3789 284 845 101 1122 269 75 5742 242 457 103\ninternal_value=0.000183462 0.00523583 0.00610932 -0.00826157 -0.0148897 0.00278987 0.0216653 0.00162008 -0.00224532 0.0126557 -0.00677066 -0.0322275 0.0161152 0.00462622\ninternal_weight=1363.11 853.103 841.738 510.002 307.61 694.681 147.057 675.674 500.991 174.683 402.433 47.6686 98.5571 68.0993\ninternal_count=33995 10372 10208 23623 19834 9195 1013 8855 7658 1197 6829 1087 829 560\nis_linear=0\nshrinkage=0.05\n\n\nTree=67\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 9 9 1 4 12 14 3 1 14 2 2 9 1\nsplit_gain=19.7436 14.6509 13.846 11.7229 10.0598 11.0666 11.099 9.41686 12.0808 9.10881 9.07658 8.62662 11.4579 9.53734\nthreshold=1.4630009291942681 0.18560323472877255 0.62109193738564106 0.75381749868392955 6.5000000000000009 0.45977311129625353 0.78323413965962441 1.195786293646284 7.4220299720764169 0.77466088087221119 2.1520662742152665 1.645599366937252 0.09151038969615162 6.0034096240997323\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 11 3 -1 5 -3 -7 -6 -9 -4 -5 12 -2 -14\nright_child=1 4 9 10 7 6 -8 8 -10 -11 -12 -13 13 -15\nleaf_value=-0.039085142541657333 0.023708539617664039 0.031162377113474662 -0.034345865669347178 -0.0051094019254837733 0.01973713874281607 0.036502202647272712 -0.014943400710675606 -0.03710222994964401 0.024094422130611366 0.018925779057413841 0.016645245599139803 0.016215018974433823 -0.074948884630728643 0.0046488419557581444\nleaf_weight=23.404784543265123 5.7864375014323768 119.68910662725102 7.8590152901597312 925.36422985757235 31.421175302006304 14.678005701862274 30.661666326690465 23.750762697309256 11.136871255934237 93.21337386360392 49.601040609646589 12.014910373836754 12.848473277408631 5.0138555727899066\nleaf_count=4015 123 956 92 26856 170 155 129 127 57 549 362 122 323 44\ninternal_value=-9.49792e-05 0.0120749 -0.00305875 -0.00484782 0.0166746 0.0232496 0.00213961 0.000208348 -0.0174399 0.0144877 -0.00398672 -0.0174024 -0.0344469 -0.0535443\ninternal_weight=1366.44 267.001 1099.44 998.37 231.338 165.029 45.3397 66.3088 34.8876 101.072 974.965 35.6637 23.6488 17.8623\ninternal_count=34080 2206 31874 31233 1594 1240 284 354 184 641 27218 612 490 367\nis_linear=0\nshrinkage=0.05\n\n\nTree=68\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 2 4 3 13 1 12 14 1 8 10 12 0 0\nsplit_gain=16.5425 19.2379 11.4563 10.2707 9.582 8.91046 9.46974 19.2877 8.72303 17.4937 9.48033 8.65213 8.74007 11.2255\nthreshold=0.22696969696969699 1.875273743832488 8.5000000000000018 1.9345448480507716 1.5940256006610307 0.6179969608783723 0.23261107465006284 0.7618343974420706 5.0610318183898935 19.500000000000004 0.1952621250350435 0.60314213570565234 0.68049333853542271 0.54674929111464277\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 11 5 4 -4 -1 8 -8 9 -7 -11 12 13 -2\nright_child=1 -3 3 -5 -6 6 7 -9 -10 10 -12 -13 -14 -15\nleaf_value=-0.040577784772311415 0.0033432724446996105 0.029355028509298737 -0.023658195800831051 0.023098378282187994 -0.079854707888689438 -0.04747873271236791 -0.0048775201158570642 -0.081588128844637786 0.0074720235749869014 0.0015175901920487418 -0.043798670975270508 -0.012130339962140305 0.011914387474101715 -0.016804863118291588\nleaf_weight=15.382647857710252 363.65478222881211 65.711839292664081 69.208643370569916 9.4225656967610103 7.8676787858130401 23.34128541033715 30.991806098725647 10.20975182391703 179.57820579010877 149.14045195335348 11.504593228746669 101.99908478325233 207.48100888880435 84.303821359470021\nleaf_count=1643 5261 532 12154 155 160 1048 479 125 2738 4925 539 870 2204 1263\ninternal_value=-0.000760122 0.00361397 -0.00785984 -0.0243248 -0.0303438 -0.00442038 -0.00294964 -0.0253693 -0.000346281 -0.00797641 -0.00197861 0.00134504 0.00345735 -0.000477502\ninternal_weight=1329.8 823.151 506.648 86.4989 77.0763 420.149 404.766 41.2016 363.565 183.986 160.645 757.439 655.44 447.959\ninternal_count=34096 10130 23966 12469 12314 11497 9854 604 9250 6512 5464 9598 8728 6524\nis_linear=0\nshrinkage=0.05\n\n\nTree=69\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 9 11 2 11 12 9 4 12 9 12 12 14 13\nsplit_gain=25.4687 17.4618 14.9067 14.4003 14.9912 19.3268 17.3076 11.2885 11.7648 11.3798 8.90884 8.43592 7.9292 12.9016\nthreshold=0.27326839826839833 0.47907313190699835 0.14883297416184757 1.3438334896653068 0.22527488983156077 0.18523167472669072 0.39497491498587228 6.5000000000000009 0.084535385028355392 0.044272729873579268 0.60314213570565234 0.76309033109884206 0.79707212829417917 1.2227218491770515\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 -2 4 -4 -6 10 9 -9 -1 -7 -8 13 -11\nright_child=2 -3 3 -5 5 6 11 8 -10 12 -12 -13 -14 -15\nleaf_value=-0.050377358372927755 0.068833332457679364 -0.08000445768743096 -0.03191308343307888 0.019061238079882664 0.042312516626998364 -0.0096284954053720423 0.0089614898233117257 0.036745798774997777 -0.023614073991654366 -0.0040150374261004535 -0.063478468108556535 -0.022341138238304123 -0.020623170539006874 0.017876214837239408\nleaf_weight=11.234120009932665 8.3918436875101161 7.2365672010928384 30.015709278057329 153.38179392251186 29.795590998604894 147.71427175024292 371.93896965566091 7.7728073475882402 119.67850259403349 297.0878839805373 7.1584146772511295 21.900580410379916 47.011484686605399 85.935194732621312\nleaf_count=174 59 76 722 1079 301 2789 2793 439 15520 7691 85 224 1245 909\ninternal_value=0.000257973 -0.00768783 0.00620137 0.0054222 0.00195877 0.00376824 0.00160208 -0.00662884 -0.0196755 -0.0028246 -0.0125088 0.00716864 -0.00146835 0.000930252\ninternal_weight=1346.25 575.957 770.297 761.905 608.524 578.508 548.712 568.72 127.451 441.269 154.873 393.84 430.035 383.023\ninternal_count=34106 26054 8052 7993 6914 6192 5891 25978 15959 10019 2874 3017 9845 8600\nis_linear=0\nshrinkage=0.05\n\n\nTree=70\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 4 14 10 13 9 1 11 11 3 13 3 3 5\nsplit_gain=17.8865 16.6328 16.472 13.6672 11.9226 10.7329 10.4278 10.7642 13.1677 15.1191 14.4697 11.8811 22.7798 12.2151\nthreshold=0.56386946386946402 5.5000000000000009 0.31542235760988979 0.95849462365591409 1.8602164531472061 0.044272729873579268 8.4597792625427264 0.54136429796339314 0.48746413200477029 0.55974167934101127 1.0826423419160627 1.2935414034959523 1.528347149318972 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 -3 4 6 -1 7 8 9 10 -2 13 -13 -9\nright_child=3 2 -4 -5 -6 -7 -8 11 -10 -11 -12 12 -14 -15\nleaf_value=-0.047110593154736993 -0.045387647306771459 0.041386338339726166 -0.017396187794413025 -0.044454548717943104 0.034124211172227879 0.00098913027704623227 0.037897426965768879 0.029248653149754545 -0.039212861288174208 0.022508418178521263 0.018090690935593554 -0.086478573629819255 0.017192643987342653 0.0037321063368709905\nleaf_weight=10.803038839483632 22.498302653897554 11.577754299389197 250.88796076732979 10.773355692392213 45.24963363702409 665.63709849391307 24.301841100677848 82.235429826658219 25.42353497305885 40.328703222097829 13.819432935677467 7.7918825601227608 12.67862717900425 108.23052185541019\nleaf_count=155 327 256 19485 164 319 11259 122 566 248 266 62 69 55 791\ninternal_value=-0.000248691 -0.00399827 -0.014656 0.00868923 0.0102978 0.000152549 0.00701013 0.00449898 -0.00877402 0.0016975 -0.0213957 0.0109327 -0.0244586 0.0148443\ninternal_weight=1332.24 938.906 262.466 393.331 382.558 676.44 337.308 313.006 102.07 76.6464 36.3177 210.936 20.4705 190.466\ninternal_count=34144 31155 19741 2989 2825 11414 2506 2384 903 655 389 1481 124 1357\nis_linear=0\nshrinkage=0.05\n\n\nTree=71\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 11 0 2 13 10 13 10 10 11 11 13 10\nsplit_gain=21.6209 12.1328 10.3563 12.5064 9.04949 8.63445 9.70067 8.25719 7.22119 7.3118 7.99615 15.3885 13.1636 11.9889\nthreshold=0.13963014728278042 12.500000000000002 0.15784628527155695 0.70626634622016105 2.7774364689331477 0.51474928523531993 0.15424266455194294 2.9496234745417209 0.050316455696202543 0.95849462365591409 0.54136429796339314 0.48746413200477029 -0.67432252158699446 0.13660287081339714\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 -4 6 -5 8 -2 10 11 12 13 -10\nright_child=7 -3 4 5 -6 -7 -8 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.0080959771989678776 -0.052739655936941222 -0.041997800919181948 -0.029275114484047411 0.025594607159099964 0.049781398960816035 0.058723677510686204 -0.043390502366810899 0.03428882509681596 0.030140484625416408 -0.033654132127027073 0.010935208855351094 -0.02418393368856497 0.0055174622524233208 -0.02796272319464041\nleaf_weight=173.41624399539432 4.7702496063429853 28.466021236672532 64.005703770264518 17.058391992759429 2.9450175534002474 11.142149476567281 6.115259420592337 20.626288459287025 9.7885481980338209 12.168492682045324 283.14180251013022 54.893148316419683 580.82013508932141 52.492644210869912\nleaf_count=7997 82 7155 2907 563 17 123 115 250 319 169 2035 642 10073 1599\ninternal_value=0.000226236 -0.0114738 -0.00816967 -0.00261376 -0.0251162 0.0245356 0.00634835 0.00371581 0.0030503 0.003371 0.003864 0.000984023 0.00316649 -0.0184988\ninternal_weight=1321.85 303.149 274.683 207.732 66.9507 34.3158 23.1737 1018.7 998.075 993.305 981.136 697.994 643.101 62.2812\ninternal_count=34046 18877 11722 8798 2924 801 678 15169 14919 14837 14668 12633 11991 1918\nis_linear=0\nshrinkage=0.05\n\n\nTree=72\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 12 9 13 14 12 1 6 2 14 8 1 5 13\nsplit_gain=22.4126 13.5648 15.0257 12.7186 10.8964 12.3881 10.589 9.8416 9.58924 8.54926 8.36259 10.9168 9.18657 11.0324\nthreshold=0.13963014728278042 0.20050230359448049 0.34755730289246772 1.5401119998885948 0.44746339656424899 0.11196642838265915 6.0601730346679696 1.5000000000000002 1.527243955281093 0.46321428429942807 19.500000000000004 6.2170217037200937 2.5000000000000004 -0.61770373433649828\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 7 9 8 -1 -6 -7 -2 -4 -3 11 -9 -12 -14\nright_child=1 2 3 -5 5 6 -8 10 -10 -11 12 -13 13 -15\nleaf_value=-0.0011932380041861663 0.031535650240197645 0.0065138785169172944 -0.00045343053458843662 0.020208657355070086 0.0020194742480733122 -0.036139003695227136 0.0062509293209904934 -0.027205507150475351 0.027724431525166179 -0.016672838601638179 -0.0085245932388381129 0.011004898504656057 -0.031321969589339617 0.03176058919973495\nleaf_weight=140.3446913396765 63.014236096525565 53.53684413433075 422.50116115971468 113.53332864283584 43.60267675225623 91.563997426172136 17.20103496190859 29.544322597619612 31.514512300491333 148.50328698789235 24.780214789556339 47.242351978900842 6.855542905163019 72.682425611332292\nleaf_count=4496 819 622 3760 771 2922 10888 560 1238 157 3911 363 634 581 2301\ninternal_value=0.000894619 0.00441469 0.00115053 0.00531877 -0.0112695 -0.0204823 -0.0294392 0.0146583 0.00155911 -0.010527 0.00864507 -0.00385762 0.0177973 0.026002\ninternal_weight=1306.42 1013.71 769.589 567.549 292.712 152.368 108.765 244.119 454.016 202.04 181.105 76.7867 104.318 79.538\ninternal_count=34023 15157 9221 4688 18866 14370 11448 5936 3917 4533 5117 1872 3245 2882\nis_linear=0\nshrinkage=0.05\n\n\nTree=73\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 9 11 9 6 6 1 5 14 12 3 5 10 10\nsplit_gain=13.2398 19.2587 13.6854 11.0901 10.2933 9.88171 15.8073 9.7437 8.79623 9.72157 9.34889 10.2279 9.3855 20.5748\nthreshold=0.27326839826839833 0.47907313190699835 0.14883297416184757 0.40090617732792916 3.5000000000000004 12.500000000000002 6.8055505752563485 19.500000000000004 0.83454127259551958 0.19909436529862615 1.2501037838947984 1.5000000000000002 0.061381475667189961 0.10045871559633029\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -2 5 -5 8 -7 -4 10 -10 11 -1 -13 -14\nright_child=2 -3 7 4 -6 6 -8 -9 9 -11 -12 12 13 -15\nleaf_value=-0.034599904502923086 0.067706180262634336 -0.085381626103936625 0.0039629322845993039 -0.017114361634732688 0.05364502827482015 -0.039153546431003071 0.055197972399731191 -0.061334947884074925 -0.0095299874641524198 -0.065677110254502499 0.0099111881980306418 0.023586441089891476 -0.038516150255584046 -0.00095453809069208492\nleaf_weight=28.683193209813908 7.3871225216425946 6.5373723581433287 731.67538727170904 7.2360129030421403 13.938976839184759 38.785536965937354 4.1144251283258191 4.7607612149440675 23.940326305339113 10.680557384679558 114.45103812922025 27.368700200458989 41.872309591621161 244.02531848437502\nleaf_count=869 54 77 7792 75 89 9297 107 223 1498 355 1831 405 1899 9572\ninternal_value=-0.000186863 -0.00597581 0.00418632 -0.0048888 0.0297835 -0.00631698 -0.0290533 0.00345821 -0.00427553 -0.0282089 -0.00240232 -0.00654056 -0.00386947 -0.00657085\ninternal_weight=1305.46 561.634 743.823 555.096 21.175 533.921 42.9 736.436 491.021 34.6209 456.401 341.95 313.266 285.898\ninternal_count=34143 26074 8069 25997 164 25833 9404 8015 16429 1853 14576 12745 11876 11471\nis_linear=0\nshrinkage=0.05\n\n\nTree=74\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 9 0 10 13 11 4 0 13 10 10 0 1 13\nsplit_gain=17.7389 16.6421 15.3009 13.8368 11.7972 12.0683 11.5442 8.38869 13.469 8.55333 7.94679 14.1236 10.6243 10.0286\nthreshold=0.2208547266686802 0.4636603360598463 0.35111134286233475 0.95849462365591409 2.2531977117618109 0.15970234850245438 7.5000000000000009 0.12042922364702495 -1.1518156456185185 0.064309346567411094 0.76728257406817468 0.83912025482839525 5.3209125995635995 -0.66562128948688848\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 6 -2 8 9 -1 13 12 -12 -7\nright_child=3 -3 -4 -5 -6 10 -8 -9 -10 -11 11 -13 -14 -15\nleaf_value=0.011684663079437036 0.055441260656465319 -0.085463612107220546 0.00035204454305408768 -0.051588733675546963 0.026894106486107728 -0.020683949270456473 -0.03308472815959608 -0.029286212521261484 -0.00083871372325566244 -0.070954172135223384 -0.02600764975433479 -0.044929817632091147 0.025114690216558752 0.0015525376889122587\nleaf_weight=3.8621215634047976 17.755840924859516 5.9121400201693168 274.80607258291275 10.155799460131673 53.802258618408814 54.618778399657458 3.8712885729910331 88.091213490988594 90.573003354205866 11.266859999333972 10.629214749671517 8.3452761140651983 89.642445000470616 572.9392759015609\nleaf_count=84 208 56 11707 161 446 1190 194 6888 4136 614 117 72 624 7583\ninternal_value=-0.00071023 -0.00839761 -0.00724538 0.00373353 0.00448842 0.00286432 0.0388331 -0.0179653 -0.00834451 -0.0504417 0.0017564 0.0141778 0.0194922 -0.000412617\ninternal_weight=1296.27 474.511 468.599 821.76 811.604 757.802 21.6271 193.793 105.702 15.129 736.175 108.617 100.272 627.558\ninternal_count=34080 23485 23429 10595 10434 9988 402 11722 4834 698 9586 813 741 8773\nis_linear=0\nshrinkage=0.05\n\n\nTree=75\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 11 11 6 12 9 14 8 9 1 4 9 2 1\nsplit_gain=16.3097 9.59124 11.8915 11.1951 8.36361 8.0689 10.2788 14.5525 12.4094 12.4926 14.4613 10.8073 10.4259 9.88058\nthreshold=0.33195118872252999 0.26948978220549369 0.19809164072177035 14.500000000000002 0.81420074388251518 0.62109193738564106 0.88527816776246016 33.500000000000007 0.55319063900854715 7.5042240619659433 3.5000000000000004 0.53930446639708307 -0.8023925232996747 7.541351556777955\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 4 5 6 11 8 9 10 -8 12 -3 -10\nright_child=1 3 -4 -5 -6 -7 7 -9 13 -11 -12 -13 -14 -15\nleaf_value=-0.0057858883641840062 -0.018032793815381472 -0.021493358639016157 0.037058786312919825 -0.039361372936611308 -0.03474802837987747 0.016947325851673533 0.016993806747030171 -0.068710162393572527 0.00013940729079960033 -0.04986458661104301 -0.037938380322554803 -0.017007704690834388 0.013002848795770756 0.059385320853282346\nleaf_weight=671.24710316155688 11.905344179831443 22.984002486104146 43.395060915965587 13.881998790893702 12.223399253562091 119.67842709133402 31.746139861643314 8.940773696638642 19.532052165828649 19.220948377856985 17.902021567802876 40.991335097933188 263.50194783566985 10.004986146464942\nleaf_count=28643 142 297 379 309 120 696 193 115 124 171 263 336 2277 34\ninternal_value=-0.000354006 0.00537982 0.0250888 0.00346272 0.00457918 0.0055071 0.0023267 -0.0110347 -0.00515605 -0.0167304 -0.0031715 0.00672681 0.0101702 0.0214951\ninternal_weight=1307.16 635.908 55.3004 580.608 566.726 554.503 434.824 107.347 98.4061 68.8691 49.6482 327.477 286.486 29.537\ninternal_count=34099 5456 521 4935 4626 4506 3810 900 785 627 456 2910 2574 158\nis_linear=0\nshrinkage=0.05\n\n\nTree=76\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 3 2 12 9 9 10 2 0 11 12 0 10 14\nsplit_gain=16.9246 14.3187 13.073 10.2194 9.00529 9.50024 9.1167 8.82187 17.2063 16.8747 13.2554 8.64251 11.1039 10.9091\nthreshold=0.21076774505070017 0.43422984344691146 1.8194180785221719 0.81420074388251518 0.47091134847787214 0.35770511525393117 0.19410230401391107 0.92110015028812986 0.038183356974087791 0.094034202108957757 0.10626745092758898 0.026851214599446126 0.071796759941089847 0.43606152963336842\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 3 -3 -2 5 6 7 8 11 -9 -11 12 -1 -14\nright_child=1 2 -4 -5 -6 -7 -8 9 -10 10 -12 -13 13 -15\nleaf_value=0.03034030146413412 -0.0039336353562167753 0.0060766230199895965 0.032575086352355731 -0.070094594479674824 -0.080951723527883573 0.023041227001457932 -0.035833678369620009 -0.051152483916272411 -0.023289753655229901 0.045666205775784383 0.00035736475287907916 0.055192630272775635 0.0047241322301437268 -0.053844832050873939\nleaf_weight=18.962970635038801 271.09305060777115 497.145671686274 50.469670408871025 4.9714152459055176 3.2176943896338335 23.263808894203976 28.857819870230742 13.479491787962614 148.03045428488986 16.991024086979451 157.25282965588849 7.4869274631200815 21.510412467760034 11.480347046395762\nleaf_count=222 5826 4892 468 72 40 223 1596 208 16088 180 2729 364 321 970\ninternal_value=-0.000362648 0.00389823 0.00857372 -0.00537376 -0.0081439 -0.00744077 -0.00916331 -0.00713026 -0.0142103 0.000728441 0.00501094 0.00841219 0.000755802 -0.0166416\ninternal_weight=1274.21 823.68 547.615 276.064 450.534 447.316 424.052 395.194 207.471 187.723 174.244 59.4407 51.9537 32.9908\ninternal_count=34199 11258 5360 5898 22941 22901 22678 21082 17965 3117 2909 1877 1513 1291\nis_linear=0\nshrinkage=0.05\n\n\nTree=77\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 13 2 11 12 11 13 3 0 13 13 2 12 8\nsplit_gain=14.955 10.3703 5.78067 5.74019 20.781 7.74553 7.45466 11.7099 12.0238 10.5339 8.31448 9.54267 7.46775 7.40622\nthreshold=0.75381749868392955 2.2531977117618109 2.5476400730133029 0.22964059561341896 0.18095615427408129 0.15730752734423484 -1.7266965431618808 1.7057859673546234 0.27133502011801675 -0.65713924291875048 1.6428137117518207 -0.047463665161103223 0.50270487952149423 18.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 -1 5 -5 -2 -6 9 -9 12 -11 -12 -8 -7\nright_child=1 -3 -4 4 6 13 7 8 -10 10 11 -13 -14 -15\nleaf_value=-0.046219329518953078 0.0021830341914530582 0.020636284355271262 0.033697679553446956 0.040656127274549682 0.039621204793268021 -0.02575206522093933 -0.033917924810713061 0.035169133445116289 -0.047615351234624077 0.0020372790321480038 -0.014849441189521851 0.029891442837836225 0.010000870503943035 -0.0046537314322057603\nleaf_weight=20.346524968030284 276.40576999232871 69.776094783097506 1.8128361962735642 35.234370796242729 11.797364439698866 57.098703497234965 31.115003958606394 4.7164064440876272 20.731041181599721 543.82771957985824 14.525908156996591 54.297357880976051 13.30988574400544 148.59269568722812\nleaf_count=4096 12687 636 9 421 118 1890 823 43 220 6344 100 383 139 6313\ninternal_value=0.00131433 0.00203247 -0.0385087 0.000944984 0.00374055 -0.00328451 0.00181143 0.00110088 -0.0315225 0.00241031 0.00412667 0.020371 -0.0208292 -0.0106068\ninternal_weight=1303.59 1281.43 22.1594 1211.65 729.555 482.097 694.321 682.523 25.4474 657.076 612.651 68.8233 44.4249 205.691\ninternal_count=34222 30117 4105 29481 8591 20890 8170 8052 263 7789 6827 483 962 8203\nis_linear=0\nshrinkage=0.05\n\n\nTree=78\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 13 10 11 0 1 10 0 2 6 2 1 10\nsplit_gain=14.6128 10.2062 10.5268 8.68706 12.2057 11.1034 9.95575 8.88279 8.20111 10.7451 9.68635 10.2979 9.66411 10.2701\nthreshold=0.06393129226250184 0.14676584033514908 2.2531977117618109 0.19900990099009905 0.13928496946513533 0.0015187125643835173 6.7117965221405038 0.11443452380952382 0.36320357302763279 1.0000000180025095e-35 1.5000000000000002 0.040681031834896166 6.5555982589721689 0.32742946708463955\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 3 -3 4 5 -2 -7 -6 10 -10 -5 12 -12 -14\nright_child=1 2 -4 8 7 6 -8 -9 9 -11 11 -13 13 -15\nleaf_value=-0.025419472042141192 0.03611224026876305 0.0028128144812783456 0.024421045987028992 0.050546829682035867 -0.0026833487974706844 -0.022885602266772883 0.036538604857817132 -0.045642601356204654 -0.0047425004428104907 -0.052205515293019603 0.01183073232589015 0.048094500919503846 -0.067917394253202715 -0.0070753653442051055\nleaf_weight=52.356955771741923 14.260305872710886 787.38410351992934 59.750522846006788 10.983533188700674 17.19541940864292 39.774821950020851 7.5879507804056621 40.121096813396434 102.68838790760492 12.533745737659045 68.967933790787356 10.880480278166941 7.1516094896942368 48.712322750594467\nleaf_count=2050 184 12274 598 148 956 6010 154 5554 2582 324 2219 141 247 639\ninternal_value=0.000196765 0.00130874 0.00436389 -0.00548042 -0.0166249 -0.00109035 -0.0128066 -0.0330215 -0.000375631 -0.0103102 0.00744448 0.00361188 -0.000616672 -0.0159217\ninternal_weight=1280.35 1227.99 847.135 380.858 118.94 61.6231 47.3628 57.3165 261.918 115.222 146.696 135.712 124.832 55.8639\ninternal_count=34080 32030 12872 19158 12858 6348 6164 6510 6300 2906 3394 3246 3105 886\nis_linear=0\nshrinkage=0.05\n\n\nTree=79\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 3 4 8 1 12 10 14 10 5 7 3 3\nsplit_gain=17.4778 9.71802 9.79126 8.02032 8.07672 7.50016 7.80476 10.1542 12.1661 9.29988 11.8227 12.3392 8.60398 8.42659\nthreshold=0.40090617732792916 0.78438036099062902 1.8740213748638983 4.5000000000000009 20.500000000000004 0.6179969608783723 0.2436581872418776 0.21076774505070017 0.52298926960539249 0.1822134387351779 3.5000000000000004 7.5000000000000009 1.0126424540725496 1.4393308112662344\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 -3 -5 -1 7 8 9 10 11 -7 -13 -8\nright_child=-2 3 -4 4 -6 6 13 -9 -10 -11 -12 12 -14 -15\nleaf_value=-0.039091569163353455 0.0087380338343107048 0.015118297389602115 0.016570069353539787 -0.064922775550660852 0.014880063087915946 0.0029652702895192639 -0.020596121116797702 0.0076724551535987361 -0.019959545551103422 -0.040782422772480176 0.013785535698744418 -0.075647161935679441 -0.017377821948688185 0.015008931045840002\nleaf_weight=16.849308386008488 483.95364726858679 98.039877738454379 45.932973554823548 5.7200108541874197 5.4254561464185826 53.193798873398919 130.32431240318692 150.38939636058058 127.77197837248968 12.201236304565098 103.96588412286656 8.8547315226460359 20.193205650750315 18.293129401979968\nleaf_count=3556 3703 1273 576 357 333 1052 2722 3877 9918 843 4925 195 573 178\ninternal_value=0.00125244 -0.00329733 -0.00549867 0.0104973 -0.0280492 -0.00710084 -0.00617762 -0.00304279 -0.00799152 -0.000222871 0.00263844 -0.0114495 -0.0370666 -0.0161425\ninternal_weight=1281.11 797.155 687.97 109.185 11.1455 642.037 625.188 476.57 326.181 198.409 186.208 82.2417 29.0479 148.617\ninternal_count=34081 30378 28415 1963 690 27839 24283 21383 17506 7588 6745 1820 768 2900\nis_linear=0\nshrinkage=0.05\n\n\nTree=80\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 2 10 12 12 12 9 10 2 2 13 9 13\nsplit_gain=14.4635 13.0843 11.4751 10.1828 10.3017 10.0919 8.72154 10.8854 8.07377 7.93667 7.81747 7.60322 7.53944 10.9799\nthreshold=0.13963014728278042 0.70626634622016105 1.9357224602941756 0.12712842712842715 0.16039590971387854 0.10548238379079276 0.15520902028626146 0.16956570461989892 0.029776858282843214 0.89013838570432136 1.2665638635973151 -1.6828132015762167 0.62109193738564106 1.6428137117518207\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 10 8 7 -5 -1 -9 -2 -3 -8 -14\nright_child=2 11 -4 6 -6 -7 12 9 -10 -11 -12 -13 13 -15\nleaf_value=-0.065719059402007493 -0.015585994664110855 0.063558607161651556 0.022091980332434421 0.050690788008188638 -0.036325568262482476 -0.02189992344800977 -0.0013209575715130368 0.017577098110434144 0.0051551015045309713 -0.036300076164197531 0.042848548033073114 0.005404711139535931 0.0062222889019194622 0.050812474544646657\nleaf_weight=3.2791848437627769 14.274629132472912 5.6683440050110212 65.647246085805818 19.00104914084659 34.746722143900115 169.21164506968489 690.78128978988389 26.246368920605164 64.944439856495592 8.1349823620112129 8.185083107557146 37.324472437147051 100.71475936379284 15.076828819699584\nleaf_count=56 665 52 703 739 776 14960 10747 563 2558 146 59 1270 670 71\ninternal_value=-0.000800021 -0.0107947 0.00205987 0.000603539 -0.0196374 -0.0153874 0.00197126 0.0213521 0.000848356 0.00416354 0.00662826 0.0143424 0.000663703 0.0124136\ninternal_weight=1263.24 280.428 982.809 917.162 57.2064 237.435 859.955 53.3824 68.2236 34.3814 22.4597 42.9928 806.573 115.792\ninternal_count=34035 18896 15139 14436 1500 17574 12936 1448 2614 709 724 1322 11488 741\nis_linear=0\nshrinkage=0.05\n\n\nTree=81\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 14 9 9 7 0 3 0 1 6 13 13 0 1\nsplit_gain=20.307 8.99832 7.6712 9.88937 16.7457 10.3672 13.0287 10.9822 10.7198 9.47397 8.45015 8.3668 9.78145 13.1139\nthreshold=0.13963014728278042 0.52298926960539249 0.56746260328383402 0.52668805818070152 10.500000000000002 0.097762431807039349 0.98248835939454404 0.29703713332023246 5.1478576660156259 2.5000000000000004 0.89202833493713396 1.2227218491770515 0.93488923097199472 7.8242692947387704\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 5 9 6 8 -7 -2 -5 -10 12 13 -9\nright_child=2 -3 -4 4 -6 7 -8 11 10 -11 -12 -13 -14 -15\nleaf_value=-0.0073699725708400851 0.029582533458084877 -0.02795652840210085 0.011974246266133444 0.01640230043201208 -0.068444265280261027 -0.014142749778391864 0.041014035637827975 -0.00025060095133202435 0.0099972912421231273 -0.044850314650832321 -0.03909349238798631 0.011214810795451649 0.026037465815587187 -0.02911167549585322\nleaf_weight=215.17296404275112 20.836274120985763 70.003246630192734 178.90469543868676 20.957174647133801 12.697345300111918 116.96471268724417 35.111798465950415 279.32431650957733 16.556554483191576 7.8609452275559297 16.883544319542125 186.98402093333425 27.900121331447735 44.818369219196029\nleaf_count=10174 558 8744 1179 122 159 2615 254 6999 454 106 142 1821 265 535\ninternal_value=-0.000804365 -0.0125035 0.00265944 0.000529163 -0.0229275 0.0018646 0.0177396 -0.000322308 0.00221361 -0.00124984 -0.0152036 0.00269726 -0.00184571 -0.00431841\ninternal_weight=1250.98 285.176 965.8 786.895 41.5155 745.38 89.3882 655.992 54.2764 28.8181 33.4401 539.027 352.043 324.143\ninternal_count=34127 18918 15209 14030 387 13643 1408 12235 1154 228 596 9620 7799 7534\nis_linear=0\nshrinkage=0.05\n\n\nTree=82\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 3 12 0 11 1 8 8 14 3 3 4 8 2\nsplit_gain=9.96974 14.4452 13.8601 13.6031 11.316 18.9387 10.4907 10.6746 9.43214 8.52008 13.1354 8.02378 8.05326 7.73627\nthreshold=0.61813055473851874 1.2501037838947984 0.58502746877936629 0.76680442430599671 0.39959521583407226 6.5163369178771982 15.500000000000002 17.500000000000004 0.563846381449853 1.2719125853272131 1.4766501342390617 5.5000000000000009 17.500000000000004 -0.19577275780615164\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 5 6 -3 -8 -7 -2 -11 12 13 -5\nright_child=9 4 -4 11 -6 8 7 -9 -10 10 -12 -13 -14 -15\nleaf_value=-0.0068025285879195381 0.014715996224987066 0.063397152180193636 -0.052227234377951952 0.075596386750061043 0.025457274333738174 0.011324641103114123 -0.070283612804808951 0.013372342744467981 -0.028655761937009434 -0.063030892677426981 0.012247614500084348 -0.017683022992032119 0.0029880114369303925 0.021018563280798565\nleaf_weight=659.59321916619956 170.48076822597068 9.304961600340901 14.051260650157927 6.8124168636277309 72.814339716685936 19.824963763472624 2.9836817588657132 96.967258932476398 51.910249802051112 8.4865894711110759 14.231772608589379 21.025287145224866 64.220355359080713 49.564383512479253\nleaf_count=26921 1288 41 110 40 485 236 56 1852 625 64 117 791 1056 393\ninternal_value=0.000453778 -0.0014381 -0.00468107 -0.00378317 0.00895637 0.00221451 0.0152678 0.0102036 -0.0176029 0.010876 -0.017345 0.0102283 0.0151999 0.0288162\ninternal_weight=1262.27 1069.07 815.267 801.216 253.805 180.991 109.256 99.9509 71.7352 193.199 22.7184 141.622 120.597 56.3768\ninternal_count=34075 32606 29311 29201 3295 2810 1949 1908 861 1469 181 2280 1489 433\nis_linear=0\nshrinkage=0.05\n\n\nTree=83\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 11 12 1 1 13 4 10 10 4 2 10 1\nsplit_gain=12.4232 9.29995 10.7135 10.1689 11.2189 8.88829 8.30499 8.72083 8.0413 9.8548 12.5962 12.6868 12.4286 12.1629\nthreshold=0.40090617732792916 1.1354444923191898 0.091690551211511276 0.30090049693865728 6.4586071968078622 6.5163369178771982 -1.9353917026821741 8.5000000000000018 0.95849462365591409 0.56386946386946402 22.500000000000004 0.9651609729322409 0.51136950904392775 7.6236207485198983\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 6 -6 7 -1 9 10 11 12 13 -2\nright_child=8 2 -4 -5 5 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=0.061341748173276638 0.00095015201337555035 -0.050743552507217959 0.00486580382764159 -0.024716424528609258 0.055586016280640041 0.0034901823395086305 -0.012188429774796027 -0.041325705613740639 -0.035587662293611606 0.012573679132217458 0.05763086292254252 0.012193824883966316 -0.049695750427750339 -0.052982423688070372\nleaf_weight=6.5724736995762196 69.042982026934624 7.9599778883857644 261.39884460764006 79.884645008831285 7.8168271088507018 117.49416186404414 283.81530659134296 2.0508212183776777 10.420925279613583 286.20586773671675 7.2690954515710464 61.378854955313727 21.059067072346807 11.312113912776111\nleaf_count=120 749 112 3450 1588 136 2270 22314 361 145 1961 49 412 260 114\ninternal_value=-0.000523579 -0.00443548 0.00304085 -0.00848025 -0.00532385 0.00715424 -0.0106769 0.0351679 0.00590129 0.00692463 -0.00260037 -0.00562527 -0.0164237 -0.00720021\ninternal_weight=1233.68 766.993 269.359 497.634 417.75 125.311 292.439 8.62329 466.689 456.268 170.062 162.793 101.414 80.3551\ninternal_count=34041 30351 3562 26789 25201 2406 22795 481 3690 3545 1584 1535 1123 863\nis_linear=0\nshrinkage=0.05\n\n\nTree=84\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 10 9 7 9 1 5 0 0 3 13 14 3\nsplit_gain=17.459 10.0969 10.6798 13.6299 12.5058 14.384 11.5987 11.4226 9.75218 17.1292 9.68567 9.65171 8.54285 8.3104\nthreshold=0.40090617732792916 0.86240703177822864 0.54890004782400781 0.49756835111409253 8.5000000000000018 0.58595927517517843 7.1975305080413827 2.5000000000000004 0.37759757228191332 0.097762431807039349 1.2135643508474616 1.5401119998885948 0.97688709391677009 1.120602370755815\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=10 2 3 7 -5 -6 -7 13 9 -9 -1 12 -4 -2\nright_child=1 -3 11 4 5 6 -8 8 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0074104528319002359 -0.038058008276352959 -0.017731086022335595 0.01404130164570889 0.011352276626859323 -0.078912356089360702 -0.043933625679749434 0.042896965476028774 0.054325866234720292 0.032148037247495605 -0.041124631800466729 0.0057284411776113812 0.034677135833804391 -0.022232749339252796 0.0058483085090949138\nleaf_weight=580.65133936735947 16.592689699027687 34.413050442468375 207.10781429335475 15.405148982070385 13.441948327235879 7.9114998031873238 5.7839086819440126 6.120690229348841 51.388674209592864 12.905087654478846 184.02156110716169 54.338112853467464 16.716830712743104 27.25098699843511\nleaf_count=27417 161 392 1430 122 202 123 18 35 424 188 2989 338 105 183\ninternal_value=0.000412777 0.00799406 0.0100654 -0.000374212 -0.0242043 -0.044075 -0.00683924 0.00863164 0.0209928 -0.00923829 -0.00424517 0.0159304 0.0112453 -0.0112462\ninternal_weight=1234.05 469.376 434.963 156.801 42.5425 27.1374 13.6954 114.258 70.4145 19.0258 764.673 278.163 223.825 43.8437\ninternal_count=34127 3721 3329 1456 465 343 141 991 647 223 30406 1873 1535 344\nis_linear=0\nshrinkage=0.05\n\n\nTree=85\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 2 2 6 0 13 13 12 0 3 6 0 9\nsplit_gain=12.5127 8.32849 8.16559 10.7461 11.174 9.28779 11.2019 7.46294 7.40913 12.0376 10.8573 8.66905 8.60273 7.52217\nthreshold=0.36355675436704565 0.77947462504345733 0.89013838570432136 0.76098550396778364 1.5000000000000002 0.3742019489740443 0.83711243686168813 1.1800529563749198 0.30604502668814709 0.76189201172894638 0.89003831859028237 2.5000000000000004 0.13376347902723115 0.15591134728417663\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 -2 -5 6 -4 -6 9 13 -10 12 -12 -1\nright_child=2 -3 5 4 7 -7 -8 -9 10 -11 11 -13 -14 -15\nleaf_value=-0.0087034753664458845 0.0042852860858109045 0.0098800397748159859 -0.038060624161836885 -0.077766756225463501 -0.031361694452074838 0.019003135865982384 0.020830167452630718 0.030308473085009859 -0.039781843472303013 -0.066174904540281534 0.058226825817664044 -0.049724967562217726 -0.0093265733289941838 0.0031161717266080038\nleaf_weight=285.74460718257615 312.3487157120835 96.557144242629874 16.222474342212085 7.1170296594500568 11.248133238404991 169.26368483586702 14.316053924616424 7.2168778097257009 35.048805552825797 6.6902977265999644 7.4271493444684955 4.6999063657131037 10.166663289302958 252.96366842287534\nleaf_count=19742 2985 1900 99 31 107 1253 114 49 890 279 26 117 146 6459\ninternal_value=0.000723133 -0.00368456 0.00645135 0.00170086 -0.0288283 0.0144375 -0.0106683 -0.00693969 -0.00587033 -0.0040505 -0.0228131 0.00442659 0.0207886 -0.00315778\ninternal_weight=1237.03 699.298 537.733 337.931 25.582 199.802 30.5385 18.465 602.741 545.399 57.3425 22.2937 17.5938 538.708\ninternal_count=34197 29559 4638 3172 187 1466 213 156 27659 26480 1179 289 172 26201\nis_linear=0\nshrinkage=0.05\n\n\nTree=86\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 13 0 2 0 2 2 0 12 13 0 2 10 10\nsplit_gain=10.451 10.9056 14.0155 15.4932 15.0856 13.8557 11.4426 8.189 8.07116 8.41327 7.76339 12.5481 11.9468 8.33647\nthreshold=0.27326839826839833 0.94092064921267249 0.2350987293572229 0.63732844452779402 0.93488923097199472 1.1354444923191898 -0.035031313975078582 0.26806232092754839 0.17056733839577878 -1.7266965431618808 0.19520940339182061 -0.21702817875146707 0.66877637130801693 0.91945945945945962\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 6 7 -5 8 -4 -2 -10 11 12 -3 -12\nright_child=1 10 4 5 -6 -7 -8 -9 9 -11 13 -13 -14 -15\nleaf_value=-0.005551310731708916 0.058763392389892304 0.036919893424176065 -0.048213948325896705 -0.062029051384332637 0.030463381323450407 0.013000942003734106 0.050588967701544196 -0.0092852670156911635 0.055994154180068201 -0.012330351240535819 0.014083977498799466 -0.035274139320061446 -0.034272928735701445 -0.040077085267813096\nleaf_weight=512.97867642853089 5.6514485313091418 17.867618187563497 13.42629898607265 9.8001049201702717 22.383271229220554 13.6597778275609 28.166838761389954 235.59602227795403 4.341504684649407 27.815374255587813 303.4569000499323 22.305728352861479 7.6607167324982575 6.3333358163945368\nleaf_count=25942 99 110 216 120 128 75 261 3568 33 733 2459 168 60 79\ninternal_value=-0.000106088 0.00378407 -0.00234697 0.014716 -0.00800304 -0.0195966 0.0270275 -0.0115686 0.00862318 -0.00169529 0.00995955 -0.00864821 0.0150693 0.0128513\ninternal_weight=1231.44 718.465 360.841 89.435 271.406 23.4599 65.9752 249.022 37.8083 32.1569 357.624 47.8341 25.5283 309.79\ninternal_count=34051 8109 5233 1321 3912 195 1126 3784 865 766 2876 338 170 2538\nis_linear=0\nshrinkage=0.05\n\n\nTree=87\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 12 9 9 0 13 9 11 11 2 0 2 6\nsplit_gain=11.5553 8.40055 8.67112 10.1384 9.18194 13.5268 11.7586 10.2721 9.64728 9.06521 8.65837 11.6251 13.6692 9.63644\nthreshold=0.15180890299837504 2.1372617647300372 0.20050230359448049 0.41422431494474371 0.24355386799887715 0.077505217748931474 1.5940256006610307 0.20238627472324786 0.30198526726146607 0.26167757102841144 1.0929501479680097 0.31246881819115052 0.71416809583600083 7.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 4 10 5 7 -7 -2 9 -6 11 12 13 -4\nright_child=1 -3 3 -5 8 6 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0087699552982127421 0.05011388039600017 0.018868321258040294 0.020582095689487913 0.0036418656735598856 0.017569685929744373 -0.0014615332376203142 -0.060161994349036053 -0.03240119769657962 -0.031386551230227856 0.05752796335188079 0.0040638740682868157 -0.023918173248202716 -0.058397076015105256 -0.0292389732188401\nleaf_weight=299.84560559137753 18.580748120395587 71.359128369484097 46.569338506727945 382.45731200627051 55.747530870808987 81.82648308914213 8.5315295066684467 3.9379388603847465 6.6202282826416186 18.818050393136218 87.259933716151863 123.00066138706461 7.1914187390357247 11.263350336696019\nleaf_count=20288 263 707 733 3149 799 3206 81 61 123 156 839 3090 98 651\ninternal_value=-0.000258909 0.002512 0.00112124 -0.00162209 0.0103799 0.00111333 -0.00760185 0.03492 0.0230935 0.0282818 -0.00892209 -0.0149383 0.00215306 0.010547\ninternal_weight=1223.01 923.164 851.805 657.742 194.063 112.877 90.358 22.5187 81.1858 74.5656 275.285 188.025 65.0241 57.8327\ninternal_count=34244 13956 13249 8560 4689 3611 3287 324 1078 955 5411 4572 1482 1384\nis_linear=0\nshrinkage=0.05\n\n\nTree=88\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 0 2 5 13 3 1 1 12 2 10 3 1 10\nsplit_gain=12.6352 9.56456 10.623 9.57521 9.92385 9.44562 9.2775 9.60112 12.8275 9.98848 13.6535 20.2877 9.91419 9.88392\nthreshold=2.0353064537048344 0.68049333853542271 1.0000000000000004 20.500000000000004 -0.31824396407007477 1.8740213748638983 6.2170217037200937 4.9366118907928476 0.17993568848336086 1.2665638635973151 0.073606811145510848 0.30771841529455551 4.7296948432922372 0.65733855185909984\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 3 -3 5 -5 6 7 8 9 10 11 -2 -12 -9\nright_child=1 2 -4 4 -6 -7 -8 13 -10 -11 12 -13 -14 -15\nleaf_value=-0.022368938157696642 0.050014279030958009 -0.0012477104175168917 0.016490946115109605 -0.044053817040174435 0.054498762496308674 0.018555895982526608 0.0015399420711218949 -0.023681673124120742 0.01236379825711971 0.0066560165592597729 -0.040720218621653169 -0.063156524943128775 0.021057291238554657 0.0090953696170091042\nleaf_weight=59.236870050568541 11.990114559099313 158.45261719304835 179.27656346018193 2.3626226948035773 12.348427018514482 48.832103101885878 384.56855250406079 140.30610647043795 86.254619251354598 42.750288610463031 55.516957150481176 4.7370244444173286 6.7164866755483663 26.918855437543243\nleaf_count=8323 311 2879 1689 561 376 572 5055 4009 1959 486 6933 144 435 242\ninternal_value=-4.06552e-05 0.00111682 0.00818938 -0.00179084 0.0368745 -0.00253898 -0.00391742 -0.00950041 -0.00230436 -0.0127155 -0.0231556 0.0162102 -0.0338254 -0.0183827\ninternal_weight=1220.27 1161.03 337.729 823.302 14.711 808.591 759.759 375.19 207.965 121.711 78.9606 16.7271 62.2334 167.225\ninternal_count=33974 25651 4568 21083 937 20146 19574 14519 10268 8309 7823 455 7368 4251\nis_linear=0\nshrinkage=0.05\n\n\nTree=89\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 3 4 2 11 2 0 13 0 13 0 11 2 2\nsplit_gain=12.2699 8.72943 7.83583 10.5641 8.97912 7.50621 10.9311 7.63697 7.2069 6.61243 7.43015 6.23589 5.74045 5.61394\nthreshold=0.22527488983156077 2.1288854818560901 20.500000000000004 -0.59120101787752588 0.2585977747180786 1.0000000000000004 0.12042922364702495 -1.3103852186368081 0.032573743160524575 -1.5946450474823883 0.30024322335876469 0.15195027894806232 0.81060921052942725 1.2413453555885363\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 -2 6 -1 9 -7 12 -11 -12 -8 -5\nright_child=2 -3 -4 13 -6 8 7 -9 -10 10 11 -13 -14 -15\nleaf_value=-0.00029957044020814569 0.025417973287893077 0.022537394580983281 0.031537853394849845 0.001960870112396988 -0.02163189474271204 -0.052109316893374717 -0.050769635196605593 -0.025557695786881874 0.0011377415265483115 -0.028906722114281969 0.077042204213127907 -0.035212651148040855 0.038184751457722738 0.012471943743813944\nleaf_weight=93.106596380363044 11.077788781185516 23.267776993219741 23.395805204403587 467.30006553593557 66.127185748075135 5.5835381909273556 4.9983025269466479 142.39310361527896 181.99735112197232 2.6259012468799465 6.0649857511743903 0.89045350180822413 1.7544839952024629 173.71438810322434\nleaf_count=4945 298 281 372 5693 1431 58 787 14984 2954 250 348 89 54 1493\ninternal_value=-0.000320044 -0.00670146 0.00366489 0.00271434 -0.0146422 -0.00829736 -0.0139199 -0.0218574 -0.000715131 0.0100131 0.0352433 0.0600513 -0.0257136 0.00482435\ninternal_weight=1204.3 462.682 741.615 718.219 77.205 439.415 251.834 158.727 187.581 16.3341 9.58134 6.95544 6.75279 641.014\ninternal_count=34037 24750 9287 8915 1729 24469 21457 16512 3012 1528 687 437 841 7186\nis_linear=0\nshrinkage=0.05\n\n\nTree=90\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 6 0 12 13 9 3 14 5 9 13 4 9 13\nsplit_gain=13.776 11.3761 12.5037 8.59307 10.2283 9.79386 10.3059 9.19128 9.32808 9.02031 9.69677 9.56621 8.3317 7.94565\nthreshold=0.078907978039233523 3.5000000000000004 0.080737732012600816 0.11520419728812863 -1.4653257879145714 0.34755730289246772 1.4393308112662344 0.56650164769769262 1.5000000000000002 0.11777184785235929 1.4630009291942681 2.5000000000000004 0.23963038819386376 1.8602164531472061\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -3 4 -2 6 -5 9 -9 11 -11 -8 -10 -7\nright_child=3 2 -4 5 -6 13 7 8 12 10 -12 -13 -14 -15\nleaf_value=-0.03106727347362517 -0.0288959179002376 0.023740470221552987 -0.028201353578013323 -0.0070515055404453729 0.030614802283051187 0.0037472266265552363 0.025505105957892651 -0.024126541202463783 -0.041864543910276351 0.053678173836311349 -0.00062384966637275049 -0.059933193666171572 0.041614358949818214 0.021929971822704852\nleaf_weight=57.866620140237501 7.7074622514774083 26.80972792468674 18.865673995896941 406.73801708788233 46.198011304397369 490.75376325694378 6.633640135638414 25.543964529118966 2.9756575180508653 26.264626016840339 11.737737015122546 4.7986217768921051 12.402973213233052 67.585888817731757\nleaf_count=1155 360 1010 3263 20118 1818 4455 75 349 168 223 200 240 118 551\ninternal_value=0.000815376 -0.0165374 0.00214197 0.00244846 0.0217346 0.00144379 -0.00365699 0.0115332 -0.00594183 0.0258302 0.03732 -0.0122933 0.0238919 0.00598348\ninternal_weight=1212.88 103.542 45.6754 1109.34 53.9055 1055.43 497.095 90.3572 40.9226 49.4346 38.0024 11.4323 15.3786 558.34\ninternal_count=34103 5428 4273 28675 2178 26497 21491 1373 635 738 423 315 286 5006\nis_linear=0\nshrinkage=0.05\n\n\nTree=91\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 3 11 14 3 11 10 1 10 10 10 3 3\nsplit_gain=12.6051 8.47931 9.9436 14.0617 12.9378 12.8394 10.9406 9.76237 9.59286 9.65543 9.45045 10.127 12.9496 11.2018\nthreshold=0.34755730289246772 1.1020406375351024 -0.27674685557969109 0.30198526726146607 0.74852742620451984 -1.1150774321726502 0.45963358344632416 0.31400560224089641 8.0035862922668475 0.55753467843631788 0.8034250585480095 0.71631706333198875 0.076001442663896454 1.3303911279833078\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 5 4 -4 -2 -7 -6 9 -3 11 12 -5 -14\nright_child=1 8 3 10 7 6 -8 -9 -10 -11 -12 -13 13 -15\nleaf_value=-0.0042629246055763707 -0.020035398346686176 0.0056316235053756498 0.062093799765309848 -0.054241713956136986 -0.055719534963140409 -0.0012033944901156507 0.045393500869024501 0.022417739160019549 -0.013663775357355638 0.028182017313920713 0.014608077823014112 -0.033667776885133484 0.0034555913817885479 -0.0392205095897653\nleaf_weight=631.65985747766535 16.628351166378707 89.786476516630501 13.783423645887526 11.792563869268632 4.8118692967109409 19.251013752538711 35.169975335709751 11.810921022086402 27.247299700044096 101.20402386866044 39.162266766186804 41.495172419119626 126.21910008531995 16.492714828811586\nleaf_count=28964 268 699 116 296 80 246 314 101 220 658 360 310 1227 102\ninternal_value=0.000564128 0.00605714 0.00107005 -0.00338771 0.0280618 0.017548 0.0291859 -0.00207928 0.0137094 0.0176654 -0.00755571 -0.0120359 -0.00604707 -0.00171493\ninternal_weight=1186.52 554.855 336.617 265.568 30.4062 71.0493 54.421 16.6228 218.238 190.991 235.162 196 154.504 142.712\ninternal_count=33961 4997 3420 2592 297 828 560 181 1577 1357 2295 1935 1625 1329\nis_linear=0\nshrinkage=0.05\n\n\nTree=92\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 14 8 1 11 14 3 2 0 10 1 3 13\nsplit_gain=11.7485 7.76823 7.7076 6.27444 6.80088 8.02236 5.98082 6.09435 5.95568 8.76499 5.919 5.83602 11.294 7.26651\nthreshold=0.13963014728278042 1.9383663450051654 0.52298926960539249 19.500000000000004 5.1478576660156259 0.11369092552193359 0.50648204120676144 -1.4292755085845681 -1.1825784024014352 0.026851214599446126 0.10982310372554276 3.7816778421401982 -1.2772396259111036 -0.5540010917208652\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 8 6 4 -4 -6 7 10 9 -2 -1 12 -10 -14\nright_child=1 -3 3 -5 5 -7 -8 -9 11 -11 -12 -13 13 -15\nleaf_value=0.054086579375453296 -0.055064686218319059 0.015341290755212431 -0.045359166983089221 -0.038190394431102323 -0.038727747564242881 0.030638835425929163 0.024128111307949382 -0.010235301590805726 -0.048700171942411906 0.024753993402331389 -0.015113505139433912 -0.0021257404119969076 -0.046637273547660771 0.027952862980886901\nleaf_weight=6.517511614438261 2.8234969502082086 91.912101489841007 6.6278987695332061 37.485526250660769 4.9613318722113062 13.102750154386738 14.758669336428282 178.71663160118624 5.2643672467675051 39.048723443876952 4.6470382230036194 714.84595218244067 2.6241978160687713 44.675759129720973\nleaf_count=167 85 895 1408 6697 219 368 1264 8293 337 1087 445 10736 343 1639\ninternal_value=-0.0010824 0.00164841 -0.0102776 -0.0255558 -0.00563492 0.0105551 -0.00553384 -0.00795366 7.62614e-05 0.0182129 0.0264087 -0.000935741 0.0149852 0.0229347\ninternal_weight=1168.01 901.195 266.817 62.1775 24.692 18.0641 204.64 189.881 809.282 41.8722 11.1645 767.41 52.5643 47.3\ninternal_count=33983 15122 18861 8692 1995 587 10169 8905 14227 1172 612 13055 2319 1982\nis_linear=0\nshrinkage=0.05\n\n\nTree=93\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 12 13 0 11 0 6 9 6 7 12 4 9\nsplit_gain=13.7772 16.2815 12.8098 9.62671 7.51911 7.94389 7.20991 14.2091 11.5284 13.8991 10.0452 8.9572 8.21309 7.12356\nthreshold=0.2529894131731712 0.093854865771812096 0.20050230359448049 2.2531977117618109 0.93488923097199472 0.31152380752121994 0.080737732012600816 7.5000000000000009 0.11905835224947654 3.5000000000000004 5.5000000000000009 0.09978491514898781 12.500000000000002 0.34755730289246772\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 -2 -3 4 13 -6 8 -8 9 10 -1 -9 -11 -4\nright_child=1 2 3 -5 5 -7 7 11 -10 12 -12 -13 -14 -15\nleaf_value=-0.0027597765215542323 -0.071354346842608302 0.02511046577103522 -0.012879527218576353 0.025411066074279912 0.056155443610564383 0.0076668281246524113 -0.0049060167614252657 0.021330585762334316 0.027094764463451593 0.021475426257683523 -0.062184607149855679 -0.040289875665097445 -0.049192248115634551 0.0028279079214277919\nleaf_weight=14.040525817079471 5.8592223467421709 77.553709879633971 84.911553749407176 46.070072817848995 11.35148182348348 27.984660012181848 320.48421374290774 6.1069534435955566 38.958156330190832 32.727024930805783 13.61555699497694 46.142890017479658 3.7260217735019969 450.67247325868811\nleaf_count=255 126 958 1989 439 96 162 9570 889 1173 1093 212 11634 1196 4190\ninternal_value=0.000901545 0.0053385 0.00608287 0.00367001 0.00188674 0.0227048 -0.0056646 -0.00889668 0.00599712 -0.00704283 -0.0331647 -0.0328224 0.0131316 0.000318287\ninternal_weight=1180.2 704.403 698.544 620.99 574.92 39.3361 475.801 372.734 103.067 64.1091 27.6561 52.2498 36.453 535.584\ninternal_count=33982 7960 7834 6876 6437 258 26022 22093 3929 2756 467 12523 2289 6179\nis_linear=0\nshrinkage=0.05\n\n\nTree=94\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 0 3 0 5 0 13 5 5 10 3 9 13\nsplit_gain=15.7343 7.13239 8.33003 6.60084 6.01441 8.24218 7.2613 12.3299 12.5206 9.04927 8.65942 8.20522 7.50053 8.69922\nthreshold=0.13963014728278042 0.70626634622016105 0.63573588568815487 2.6316286664499118 0.020884627018111463 1.5000000000000002 0.097762431807039349 0.54008126682696778 2.5000000000000004 4.5000000000000009 0.61311827956989251 0.98248835939454404 0.60276660801060655 1.6428137117518207\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 5 -2 7 9 11 -6 -10 -9 -8 -14\nright_child=3 -3 -4 -5 6 -7 12 8 10 -11 -12 -13 13 -15\nleaf_value=-0.012015815451110828 0.027230715560916594 0.0090186203050047053 -0.049547860249529611 0.05167889537674835 0.058899331431848417 -0.037600240735789804 -0.00075414772334615309 -0.069050547907035062 0.037709039900456764 -0.0032255620587456043 -0.026469489348784271 -0.00064524393283532774 0.0064224359891836107 0.040821729607401547\nleaf_weight=212.18208451529063 5.7247932218015221 36.084162958810339 15.061971727162016 5.7558661554940036 21.080037129344422 18.686108662979677 670.03765777514491 7.1203774744644788 19.883759175630988 7.8439917505747863 6.2272821180522433 9.5683834669180197 108.073140734341 21.288809527177364\nleaf_count=16595 57 1277 969 72 248 378 12709 43 185 459 73 92 756 110\ninternal_value=-0.000708575 -0.0114363 -0.0147097 0.00243493 0.00206114 -0.0219229 0.00275759 0.0178847 0.000654531 0.0425067 0.0219912 -0.0320845 0.00137971 0.0123532\ninternal_weight=1164.62 263.328 227.244 901.29 895.534 24.4109 871.123 71.7238 42.7998 28.924 26.111 16.6888 799.4 129.362\ninternal_count=34023 18841 17564 15182 15110 435 14675 1100 393 707 258 135 13575 866\nis_linear=0\nshrinkage=0.05\n\n\nTree=95\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 10 13 14 9 11 11 11 8 13 10 3 9\nsplit_gain=11.5168 10.3769 8.19168 7.54326 9.38311 12.4602 10.0263 14.5187 16.7192 11.0196 9.52309 9.13465 8.89896 7.32198\nthreshold=0.078907978039233523 3.5000000000000004 0.91945945945945962 -0.92626928906613226 0.47258077750084743 0.18332356235995864 0.70419784427315657 0.56629291394080161 0.47815812583848105 40.500000000000007 0.18047575133005703 0.68841813923781148 1.4016324554707726 0.48753231203628372\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 13 -3 -5 -6 7 8 -7 11 -10 -9 -12 -2\nright_child=1 3 -4 4 5 6 -8 9 10 -11 12 -13 -14 -15\nleaf_value=-0.01603083935061873 0.0035514648543167535 0.019543415661525378 -0.035598038832675732 0.012986558020904294 -0.033334432903276014 -0.0023789012449448111 -0.036768992656109521 0.0080420060720958458 0.011663874647762732 -0.034805046608104621 -0.063427250589355846 0.051654049370467185 0.011338430326428423 0.016113214773775263\nleaf_weight=97.721891296256217 479.33305892543285 30.23320005088317 10.731990862637756 55.384336730057839 44.794592781989195 190.90065756010881 22.413255682215095 25.977724105818197 6.974964838474988 7.0662552234716705 24.030815753387291 22.055165786761791 4.4049756764434269 152.41779522725847\nleaf_count=5416 6963 2730 126 2592 9521 4509 286 186 124 141 305 92 37 951\ninternal_value=0.000314936 0.00181196 0.00584367 -0.00415297 -0.00597014 -0.00901029 -0.00532644 -0.00270083 -0.00831391 0.0200912 -0.0393656 0.0287123 -0.0518536 0.00660282\ninternal_weight=1174.44 1076.72 642.483 434.236 404.003 348.618 303.824 281.411 226.311 55.0991 35.4108 48.0329 28.4358 631.751\ninternal_count=33979 28563 8040 20523 17793 15201 5680 5394 4975 419 466 278 342 7914\nis_linear=0\nshrinkage=0.05\n\n\nTree=96\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 0 3 0 6 0 3 6 2 3 13 3 3 10\nsplit_gain=6.98193 9.46517 8.96306 8.4715 8.08335 7.65652 7.52551 7.18755 8.15736 8.26459 9.94792 7.51257 9.62896 9.55143\nthreshold=6.0601730346679696 0.68049333853542271 -0.79381492413181876 0.16027408678569108 11.500000000000002 0.055526473498762462 -1.0819198756096169 7.5000000000000009 -1.5396058991564723 -0.75305914588649314 1.5007010094932978 -0.50900795747247185 -0.28772920750134479 0.054669987546699876\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 -2 5 -4 -3 11 -9 -10 -11 -7 -13 -14\nright_child=2 6 4 -5 -6 7 -8 8 9 10 -12 12 13 -15\nleaf_value=-0.008482047986119878 -0.064479760043939849 -0.037527911177505653 0.021973191054196043 -0.0081327750977348151 0.027988084502980681 -0.034724699758900357 0.010952738673420523 0.043948375394799205 0.055525360080755065 -0.06047174902152868 0.021503046217242916 0.041449862551856509 -0.062750798713642142 0.0029380411471752414\nleaf_weight=422.96581434951077 7.3883481866214415 7.654779333563055 47.007593843154609 34.514689616800752 33.048574136191746 12.227395331254227 109.92365341447294 2.9279108238406506 1.466556484811014 16.501776376011549 4.1409829510375848 15.136018977034835 4.6031432496383777 453.35340769746108\nleaf_count=22854 125 529 456 572 793 148 2588 38 6 688 27 148 57 5035\ninternal_value=-0.000829112 -0.00499678 0.00273594 -0.0193392 0.00433272 0.00288247 0.00750661 0.00108455 -0.0245124 -0.035088 -0.0437937 0.00245276 0.00348485 0.00214248\ninternal_weight=1172.86 540.544 632.316 41.903 590.413 557.365 117.578 510.357 25.0372 22.1093 20.6428 485.32 473.093 457.957\ninternal_count=34064 25971 8093 697 7396 6603 3117 6147 759 721 715 5388 5240 5092\nis_linear=0\nshrinkage=0.05\n\n\nTree=97\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 9 4 9 14 10 2 4 13 11 12 11 10 0\nsplit_gain=10.9443 6.4404 7.88485 7.03123 10.9022 11.4902 8.47957 11.7312 7.89158 9.5612 12.2238 7.44169 9.94644 9.15449\nthreshold=0.086085996167950277 0.089449220420208878 5.5000000000000009 0.60276660801060655 0.88677025781813368 0.64031250000000017 -1.0670243939590576 5.5000000000000009 1.6428137117518207 0.60405659051977689 0.54803933842734343 0.32124879495568431 0.26401991614255771 0.58448447051057995\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 4 -3 6 -6 11 9 10 -5 12 -8 -13\nright_child=1 3 -4 8 5 -7 7 -9 -10 -11 -12 13 -14 -15\nleaf_value=-0.012816135324044671 0.053118647134986097 0.0029484599208493465 -0.03950218096522308 0.0016213690338404128 0.028996394927220188 -0.068651435970495014 -0.016389931367127065 -0.043726686723456545 0.040189971576038458 0.018179293059228996 -0.066017985789690661 -0.054612471815172162 0.033750322056846992 -0.0046269803454528887\nleaf_weight=125.1902147123692 10.878377168788573 777.40911595214857 2.075276126190146 34.251079688081518 12.425469328358302 7.5869633723050347 16.858484515018063 23.45193167803518 21.176085329381749 74.243934109341353 7.2842278415337196 11.366613929159938 21.443664428661577 36.169791397347581\nleaf_count=7427 248 21384 699 264 504 65 540 1728 124 485 59 118 187 293\ninternal_value=0.00110306 0.00276279 0.0365125 0.00230935 0.000708501 -0.0126781 -0.00865809 -0.0131632 0.0128258 0.00751956 -0.011514 -0.00440557 0.0118259 -0.0174581\ninternal_weight=1181.81 1056.62 12.9537 1043.67 906.712 129.303 121.716 109.29 136.955 115.779 41.5353 85.8386 38.3021 47.5364\ninternal_count=34125 26698 947 25751 24819 3435 3370 2866 932 808 323 1138 727 411\nis_linear=0\nshrinkage=0.05\n\n\nTree=98\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 4 0 13 10 9 14 10 10 8 4 9 13\nsplit_gain=10.6528 7.90361 7.865 6.61172 14.4672 13.158 6.67626 6.89858 15.3262 7.3066 6.62411 7.08693 7.57907 10.1042\nthreshold=0.14288093717637959 1.9357224602941756 12.500000000000002 0.097762431807039349 0.78838956065807686 0.69732205778717427 0.62109193738564106 0.88527816776246016 0.67725134408602161 0.091372912801484255 31.500000000000004 3.5000000000000004 0.48753231203628372 1.3179902501680307\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 -1 4 -2 -6 7 9 10 -5 11 -9 13 -13\nright_child=1 -3 -4 6 5 -7 -8 8 -10 -11 -12 12 -14 -15\nleaf_value=-0.0049831732067847313 0.03430679336858717 0.021989600739046572 -0.036433464586025276 -0.035954687746682916 0.0096371974381266705 -0.049485306390463196 0.01250039802767933 0.0087149113613951053 -0.078938753028669262 0.0025226087181183391 -0.038557828804766853 -0.021573629050768241 -0.05715382646377537 0.036524221755211667\nleaf_weight=249.17541952060856 47.477221330649627 54.921038583386689 20.661833541529631 11.643122976587618 41.657395232934505 11.080126562155781 106.88621355232317 60.280994862376247 6.8821002130862317 513.29243975595455 13.496944188198539 28.131134543516964 8.4624800432939065 9.0952123077586275\nleaf_count=12083 1029 679 7278 355 368 80 738 576 48 9039 250 1521 147 54\ninternal_value=0.00119405 0.00377315 -0.007517 0.00258339 0.0145869 -0.00347413 0.000979652 -0.000927353 -0.0113678 0.00160244 -0.00685229 -0.00249139 -0.0171467 -0.00679491\ninternal_weight=1183.14 913.306 269.837 858.385 100.215 52.7375 758.171 651.284 126.349 524.936 119.467 105.97 45.6888 37.2263\ninternal_count=34245 14884 19361 14205 1477 448 12728 11990 2596 9394 2548 2298 1722 1575\nis_linear=0\nshrinkage=0.05\n\n\nTree=99\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 12 13 7 3 2 13 11 1 1 11 13 9\nsplit_gain=11.0637 7.1321 8.27601 6.71402 7.25786 6.13144 7.76767 9.59039 7.68004 8.1193 10.1712 7.21853 8.63248 6.18824\nthreshold=0.13963014728278042 0.68049333853542271 0.098911901734213256 -1.1739033568228121 3.5000000000000004 -0.17251423663552959 1.645599366937252 0.21562793959290846 0.090395177304369301 6.4876372814178476 6.130833864212037 0.24258405726642648 -0.71075444278156408 0.071372763252211563\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 -4 6 -5 -8 -7 10 11 12 -10 -14\nright_child=-2 -3 3 5 -6 8 7 -9 9 -11 -12 -13 13 -15\nleaf_value=0.0059843820699727376 0.0027180179440420157 0.010107117558780418 0.023838933139145129 -0.042332587029721626 -0.056880903078263524 0.034136213265324337 0.04129899184709139 -0.044245515484854926 0.02023188662381914 -0.04871388559214973 0.046068808863654824 0.058173416800086232 0.0096516550007646716 -0.029724196307721801\nleaf_weight=46.585643431826611 920.43687107716687 40.390291522679036 2.733282612636688 31.244745276795584 16.561595704682986 9.1350944552978017 6.6980682852445153 4.7146067544817924 13.801010580951688 11.867991981474914 7.500271263124886 2.5961328635748933 11.811946795147383 51.398348983209871\nleaf_count=1800 15328 1607 47 6764 1555 147 110 68 855 309 172 24 581 4825\ninternal_value=0.000155515 -0.00899558 -0.012587 -0.0176795 -0.0448351 -0.0139317 -0.0297756 0.00524276 -0.00746461 -0.0116073 -0.00605615 -0.0114777 -0.0145414 -0.0223304\ninternal_weight=1177.48 257.039 216.649 170.063 19.2949 150.768 42.6574 11.4127 108.111 98.9757 87.1077 79.6074 77.0113 63.2103\ninternal_count=34192 18864 17257 15457 1602 13855 6942 178 6913 6766 6457 6285 6261 5406\nis_linear=0\nshrinkage=0.05\n\n\nTree=100\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 12 3 14 9 3 2 13 10 5 10 3 3\nsplit_gain=13.1015 12.0012 6.9553 5.72451 5.70619 7.5873 7.54354 7.59819 6.5905 6.902 10.6881 7.79505 7.13104 9.43721\nthreshold=0.2529894131731712 0.096022128556375139 0.76309033109884206 2.1288854818560901 0.51157670266878574 0.086085996167950277 1.2935414034959523 1.1354444923191898 -1.8686458876557555 0.11558704453441297 11.500000000000002 0.12104562104562107 -1.4292755085845681 0.86289377661312738\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 -3 4 5 6 8 -8 -1 12 11 -11 -10 -14\nright_child=1 2 -4 -5 -6 -7 7 -9 9 10 -12 -13 13 -15\nleaf_value=0.035906386046592333 -0.052428210013930665 0.0064472084336436855 -0.019049616579267056 0.018066539045583072 -0.013940374510370973 0.0044821593581359691 -0.073991790316451908 -0.00050083055739164604 0.031291050159839871 0.057379358607548593 -0.049108284146313325 0.0020687764014605064 -0.046898399048634257 0.0033422338650006158\nleaf_weight=7.1098680171417064 8.09010383102577 660.31453527187114 26.924026932334527 22.409259571461007 188.6094054615678 156.1180031886397 8.1772112799226289 5.7247445592656723 4.377987806859891 7.4117885752784778 6.8615302140678969 26.576787339479779 28.033734066877514 13.598475141101515\nleaf_count=218 143 7544 256 314 15311 5299 270 60 125 146 1224 1434 1568 203\ninternal_value=0.000304707 0.00467468 0.00542209 -0.00608854 -0.00732171 -0.0025582 -0.0126949 -0.0457931 -0.00736777 -0.0112783 0.00367136 0.0154254 -0.0243471 -0.0307947\ninternal_weight=1170.34 695.329 687.239 475.009 452.6 263.99 107.872 13.902 93.9702 86.8603 40.8501 33.9886 46.0102 41.6322\ninternal_count=34115 7943 7800 26172 25858 10547 5248 330 4918 4700 2804 1580 1896 1771\nis_linear=0\nshrinkage=0.05\n\n\nTree=101\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 1 1 3 11 0 2 2 9 14 9 7 9 5\nsplit_gain=7.91439 7.87615 14.8797 11.9778 10.5328 9.04987 7.55809 6.90597 10.4341 7.61881 9.03852 6.7517 6.29362 6.99535\nthreshold=0.13963014728278042 6.130833864212037 6.5555982589721689 0.98248835939454404 0.16091067892009897 0.87645216386918368 1.527243955281093 -0.65268298548172743 0.044272729873579268 0.44322655514882026 0.11777184785235929 5.5000000000000009 0.070846804972665336 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -3 12 5 7 -6 -1 -9 10 11 -10 -4 -14\nright_child=-2 2 3 -5 6 -7 -8 8 9 -11 -12 -13 13 -15\nleaf_value=-0.033240423591026518 0.0021710465227154315 0.041972181306027243 -0.024855850712916287 -0.04863076920704984 -0.050991089585734367 0.048392139032862075 -0.0074263703471112623 -0.060073956550993284 0.038448268793221468 -0.012628294907525062 0.066167750266533973 -0.0036621019650128414 0.060226878393611183 -0.014354323240181967\nleaf_weight=23.385059359236038 905.97939347031206 20.244324267783668 6.2512462763115826 12.211697810125768 25.120328475946735 6.2615564274601629 16.717731697601266 6.9008928525727233 10.946072510327211 66.143084290069964 5.7658908230951047 46.172321892336186 7.5579790314659459 4.4573496174707543\nleaf_count=4303 15208 379 63 234 3713 164 209 191 263 7220 68 1782 75 176\ninternal_value=-3.07547e-05 -0.00773698 0.00978341 -0.0122512 -0.012048 -0.00628827 -0.0341631 -0.00872585 -0.00429642 -0.000874146 0.0114933 0.00493056 0.0133313 0.0335822\ninternal_weight=1164.11 258.136 50.7226 30.4783 207.413 165.575 41.8381 159.313 135.928 129.027 62.8843 57.1184 18.2666 12.0153\ninternal_count=34048 18840 927 548 17913 13991 3922 13827 9524 9333 2113 2045 314 251\nis_linear=0\nshrinkage=0.05\n\n\nTree=102\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 12 0 2 13 7 13 2 5 13 0 13 14 13\nsplit_gain=12.1638 7.84219 7.54567 8.4803 8.98033 10.3033 6.97527 7.16209 6.83714 10.6215 7.9053 16.1389 12.5599 12.3572\nthreshold=0.14288093717637959 0.098911901734213256 0.92210547419691402 1.875273743832488 0.14585447041112107 12.500000000000002 1.0000000000000027 1.0000000000000022 5.5000000000000009 -1.4061108993616001 0.094366192105706528 0.56606542793081605 0.43853240193351967 1.3654625160993905\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 8 5 -5 -4 -8 9 -2 11 -11 -13 -14\nright_child=2 -3 6 4 -6 -7 7 -9 -10 10 -12 12 13 -15\nleaf_value=0.0072531172336965273 0.040017151061698703 -0.014349053151899572 0.033519748910025425 0.023849227356182192 0.027158303640592508 -0.064684543572384304 -0.033643552139145501 0.018162903711273898 0.0070914186954712166 0.039910669494888668 -0.0067826507650657611 0.035982634800260378 -0.048691114821095255 0.012737934916730011\nleaf_weight=52.348028445499949 12.825871009146793 201.43408406934395 35.166654413565993 5.9033686779439432 52.820086646592245 5.4542679083533585 9.2578565981239098 18.080654244869947 224.21947127763997 27.982460040831938 460.34232499262725 10.973961297422646 19.00041621865239 13.282780611421911\nleaf_count=1989 141 17275 340 36 623 68 57 152 6491 359 6179 64 202 89\ninternal_value=-0.000241991 -0.00988216 0.00249856 0.00123019 0.0185464 -0.0204611 0.0191126 5.11443e-05 -0.000237871 -0.00326397 -0.00438143 0.0110141 -0.00815811 -0.0237931\ninternal_weight=1149.09 253.782 895.31 832.805 64.1777 11.3576 62.5052 27.3385 768.627 544.408 531.582 71.2396 43.2572 32.2832\ninternal_count=34065 19264 14801 14252 727 104 549 209 13525 7034 6893 714 355 291\nis_linear=0\nshrinkage=0.05\n\n\nTree=103\nnum_leaves=15\nnum_cat=0\nsplit_feature=0 4 6 8 10 5 0 2 2 13 13 7 13 13\nsplit_gain=11.4971 11.2311 10.2367 10.1721 10.0599 9.95294 10.0052 9.80121 15.1117 8.22812 8.39744 7.04077 12.2809 6.8076\nthreshold=0.79302386496143196 1.5000000000000002 4.5000000000000009 14.500000000000002 0.54890004782400781 1.5000000000000002 0.89370886943599614 -0.20714541043013943 1.1771025309410685 1.2227218491770515 -1.7266965431618808 5.5000000000000009 0.97345993649914986 -0.019136545515088907\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 3 5 -2 -1 6 -3 -7 11 10 -6 -9 -13 -5\nright_child=1 2 -4 13 9 7 -8 8 -10 -11 -12 12 -14 -15\nleaf_value=-0.0051238701310976978 -0.017611583716657025 -0.046218545298175318 -0.028705594817091104 -0.0026498108800243527 0.051934277898429415 0.056803468172267085 0.018504361555889474 -0.05480027064274158 0.036498524707209767 0.020415469528858515 -0.00063211859350217382 0.02838480791116086 -0.032700719655651127 0.042204424199500545\nleaf_weight=654.04225244252302 10.169617657084016 13.224617515224965 20.808386070420966 9.9622530537890253 6.8699927199631921 12.260854911059139 9.5506094144657236 6.4888522792607537 28.135479587246664 72.923396271537058 218.84807862178423 16.263904906576503 14.720885606948285 51.115742607740685\nleaf_count=28714 90 95 885 105 34 98 60 58 234 596 2059 191 95 668\ninternal_value=0.000602175 0.0117074 0.00240083 0.0273193 -0.00165403 0.00905122 -0.0194408 0.0175253 0.00944254 0.00593687 0.00118981 -0.011292 -0.000751971 0.0349635\ninternal_weight=1145.38 192.701 121.454 71.2476 952.684 100.645 22.7752 77.87 65.6091 298.641 225.718 37.4736 30.9848 61.078\ninternal_count=33982 2579 1716 863 31403 831 155 676 578 2689 2093 344 286 773\nis_linear=0\nshrinkage=0.05\n\n\nTree=104\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 12 10 10 0 0 12 12 2 10 1 11 0 0\nsplit_gain=6.64314 11.4761 7.05886 5.58161 7.07521 7.46002 7.74995 9.37663 6.3006 7.00209 6.48855 6.05351 6.93809 5.84666\nthreshold=0.22527488983156077 0.17056733839577878 0.079077644606565925 0.069605943152454788 0.95397106958637368 0.4870485253493006 0.19488605834408215 0.1840815982052029 1.5610522555580648 0.12018518518518519 5.1763627529144296 0.13033950224217439 0.49597849638769437 0.50466499360108685\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 -3 -1 5 -5 7 8 11 10 -10 -7 -13 -8\nright_child=1 2 -4 4 -6 6 13 -9 9 -11 -12 12 -14 -15\nleaf_value=0.011104709041598694 0.034509138555411406 -0.058324635517189805 0.0023739307157149576 -0.011901131508809053 -0.063555287397485616 0.015493730666905146 0.041686577332348075 0.045189687720431845 -0.047487270855414113 0.024842031100725304 0.021213076975505219 0.041364175016975174 -0.021492317317257686 -0.034102800869071352\nleaf_weight=48.993370559459436 27.187057209928753 3.8247217206517226 681.15047048004635 215.46551539102802 4.4005148939322671 20.182364569496713 2.0531629385659498 13.182303274355943 6.6391968860407342 41.336710695235524 5.4448038500268012 3.8945756368193534 48.592113475555379 19.091018738763523\nleaf_count=1215 343 72 8834 16310 103 1813 38 362 160 463 49 144 3474 671\ninternal_value=0.00028004 0.00324006 0.00195048 -0.00462671 -0.00667726 -0.00584451 0.00231385 0.00663979 0.00228566 0.0151722 -0.0172769 -0.00726486 -0.0161421 -0.0251933\ninternal_weight=1141.44 712.162 684.975 429.276 380.282 375.882 160.416 139.272 126.09 53.4207 12.084 72.6691 52.4867 21.1442\ninternal_count=34051 9249 8906 24802 23587 23484 7174 6465 6103 672 209 5431 3618 709\nis_linear=0\nshrinkage=0.05\n\n\nTree=105\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 13 12 11 13 13 12 13 7 11 12 5 11 14\nsplit_gain=10.1002 7.11185 7.01347 13.9668 9.90826 7.63404 7.42602 7.82282 7.38668 7.05722 9.12081 14.2353 11.9686 10.4246\nthreshold=0.41449514314371977 -1.3103852186368081 0.24044446096226554 0.22527488983156077 -0.29908239299858858 0.50005730055654296 0.16279637679596423 -0.13286243718190949 15.500000000000002 0.20966882216503246 0.17993568848336086 2.5000000000000004 0.15366261807246603 0.76989662855493035\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 3 9 -4 8 -5 -8 -6 10 -1 -12 -13 -14\nright_child=1 -3 4 6 5 -7 7 -9 -10 -11 11 12 13 -15\nleaf_value=-0.0070635516597345729 0.036831575076838922 0.0048525681247023065 -0.046831512022102346 0.04167757876674745 0.027572344527732558 -0.01335197515921425 -0.025120740329851635 0.013256129938206442 -0.030636205779573562 -0.032120208538874755 -0.02643232253501281 0.069690092477165219 -0.0077951486194585903 0.04515277229477202\nleaf_weight=328.10748399816657 17.211348920827731 401.92924607102759 17.535020217357669 18.691004205669742 22.982854776724711 126.98070013933466 14.323078563233137 101.22420440972201 6.1563415544806039 22.10184724064311 16.084857759822626 11.219054524757665 23.994959637093416 13.935100066875746\nleaf_count=21611 128 3833 723 233 484 1593 574 1608 382 984 312 113 1208 371\ninternal_value=8.2916e-05 0.00625028 -0.00349448 -0.000719806 -0.0122229 -0.00806247 0.0132421 0.00832426 0.014666 -0.00525216 -0.00366482 0.0132783 0.0265422 0.0123175\ninternal_weight=1142.48 419.141 723.337 549.682 173.655 156.12 134.238 115.547 29.1392 415.443 393.341 65.234 49.1491 37.9301\ninternal_count=34157 3961 30196 27014 3182 2459 2415 2182 866 24599 23615 2004 1692 1579\nis_linear=0\nshrinkage=0.05\n\n\nTree=106\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 2 1 5 3 0 3 5 2 5 7 13 1 0\nsplit_gain=7.82365 10.5409 10.4252 8.95865 8.83385 23.1532 12.242 11.5216 12.6702 11.3645 8.73499 12.2307 8.43441 9.50672\nthreshold=2.5000000000000004 -0.75842404078536507 7.5042240619659433 2.5000000000000004 0.87699916622931584 0.40198683571299992 0.63743079856948504 9.5000000000000018 2.7774364689331477 15.500000000000002 5.5000000000000009 1.3179902501680307 1.3719071149826052 0.080737732012600816\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 -3 5 9 7 -7 -9 -1 -8 -12 -5 -14\nright_child=1 3 -4 12 -6 6 10 8 -10 -11 11 -13 13 -15\nleaf_value=-0.03849691860736279 -0.019020653924787793 0.025702602150383943 0.020971473497110635 -0.025815482325329236 0.00040693031973644051 0.015675214008786874 0.014616885736602823 -0.065485101252249808 0.029876358484677675 0.040938181084311617 -0.052429701235251737 0.0077798298816696048 0.025163361766759752 0.0026853031136963236\nleaf_weight=75.420682557261898 81.933495603050687 52.654193971626228 19.402045105409343 21.485713885296718 325.27807676000521 88.409691258362727 12.284781091380863 8.7659331891918537 4.6816427810117593 3.9899015044793478 24.846530802489724 12.174065564293413 53.197429706571711 358.74216903155684\nleaf_count=1207 6946 916 307 4778 3487 1465 125 146 35 15 202 80 1406 12935\ninternal_value=-0.000657917 0.00336292 -0.0112335 0.00642273 -0.00490548 -0.0123693 -0.000832741 0.00905857 -0.0325159 -0.0340463 -0.0210405 -0.0329462 0.00402676 0.005642\ninternal_weight=1143.27 587.415 101.336 486.08 555.851 230.573 151.163 101.857 13.4476 79.4106 49.3054 37.0206 433.425 411.94\ninternal_count=34050 27288 7253 20035 6762 3275 2053 1646 181 1222 407 282 19119 14341\nis_linear=0\nshrinkage=0.05\n\n\nTree=107\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 1 9 1 3 10 0 7 13 13 3 10 9 13\nsplit_gain=11.3513 13.1729 12.8191 10.82 11.9157 10.3899 9.91685 11.1199 9.06235 7.63295 6.97721 11.634 8.08319 7.76188\nthreshold=4.3886544704437265 4.2128593921661386 0.18560323472877255 4.81640648841858 -0.88828135172906986 0.25472370766488417 0.28111133719388409 23.500000000000004 0.21562793959290846 0.33548239013603998 -0.52058204253280704 0.71631706333198875 0.37935766637759955 0.90622825774246907\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 6 -5 -3 -2 -8 -4 -1 12 13 -10 -12\nright_child=3 5 8 4 -6 -7 7 -9 10 -11 11 -13 -14 -15\nleaf_value=-0.0044017127516555038 -0.0070390713972765199 -0.0086384001907459183 -0.032290643603964303 -0.020169474273925687 0.0026258112962223813 -0.080871429865623456 0.047610222258953304 -0.0077398735541009834 0.0087369399771454869 -0.02827737386552507 -0.036047765714410547 -0.065412148140171739 0.076133792087619084 0.029617033637863738\nleaf_weight=79.714711317647016 20.741086447174894 9.452066896628823 8.4232289767824096 60.761543831933523 798.06760169724294 10.185219641803995 35.556990169250639 11.733150498897883 8.4869607113651018 57.276204321038676 4.5713378647051313 3.7955594032537183 9.0785426599904877 24.646903761895373\nleaf_count=12413 806 648 707 1388 12181 181 676 532 184 3844 182 47 40 291\ninternal_value=0.000117491 -0.0101881 -0.00615348 0.00252344 0.000991479 -0.0482066 0.0215684 0.0340014 0.0131825 -0.0145168 0.0212346 0.00720481 0.0457949 0.0184905\ninternal_weight=1142.49 215.631 195.993 926.86 858.829 19.6373 68.0312 47.2901 59.0025 136.991 50.5793 33.0138 17.5655 29.2182\ninternal_count=34120 18537 17708 15583 13569 829 2014 1208 1451 16257 744 520 224 473\nis_linear=0\nshrinkage=0.05\n\n\nTree=108\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 1 8 1 3 3 6 8 13 1 0 10 3 5\nsplit_gain=10.6818 5.72859 6.6476 5.86247 5.42588 9.11954 8.24794 8.90176 8.14996 8.52266 8.06301 7.76251 6.86296 6.83389\nthreshold=0.13671566500256818 7.4631829261779794 16.500000000000004 5.0947940349578866 0.80139994038844931 0.83769965641148536 1.5000000000000002 56.500000000000007 0.68358916659085278 7.3424916267395028 0.13376347902723115 0.78979857050032498 0.20027498381202996 6.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 6 -6 7 8 9 11 -7 -2 -10 -12\nright_child=4 -3 -4 -5 5 10 -8 -9 12 -11 13 -13 -14 -15\nleaf_value=-0.035181546183136507 -0.037001053056772724 -0.052881410131357255 -0.011357886700790383 0.022295319305301778 0.037872124750613197 0.023203990091321503 0.0012020697867818127 0.039024333505219257 -0.041816683514329586 -0.082723878847242721 0.0008883126845665297 0.045107700000672483 0.012870961454274833 0.051513834912295657\nleaf_weight=4.3673359556123641 13.526176492683595 6.3988980570575213 201.87477416016191 26.40317334397696 21.298755323805381 50.92871790760546 454.839771484425 5.0215252768248311 7.4558140216395286 8.7412340808659774 315.46538037699065 2.6916187703609458 16.914583476260304 5.8110557740437789\nleaf_count=384 503 88 17324 611 198 461 9948 77 78 143 3941 59 167 77\ninternal_value=-9.52081e-05 -0.0094706 -0.00805536 0.0132869 0.00239524 0.00679699 -0.0010131 -0.0192377 -0.0258261 -0.0453328 0.00492206 -0.0215453 -0.00484881 0.001961\ninternal_weight=1141.74 239.044 232.645 30.7705 902.695 393.504 509.191 54.351 49.3294 24.959 372.205 16.2178 24.3704 321.276\ninternal_count=34059 18407 18319 995 15652 4677 10975 1027 950 705 4479 562 245 4018\nis_linear=0\nshrinkage=0.05\n\n\nTree=109\nnum_leaves=15\nnum_cat=0\nsplit_feature=0 5 11 13 14 1 0 2 9 13 0 14 12 7\nsplit_gain=7.03265 8.06371 9.34184 7.49908 6.33335 6.07296 6.32521 15.452 8.56895 13.3339 11.6687 8.18347 8.68891 7.29926\nthreshold=0.81608219810852256 4.5000000000000009 0.13136041233026699 0.46279908428335076 0.91574102062695573 0.75381749868392955 0.094366192105706528 0.63732844452779402 0.11905835224947654 0.4271357933468744 0.032573743160524575 0.63873343348423661 0.14420199433537964 12.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 -2 -3 4 -4 -1 7 8 13 10 -10 12 -9 -7\nright_child=1 2 3 -5 -6 6 -8 11 9 -11 -12 -13 -14 -15\nleaf_value=-0.031766108876705342 0.016444948425080645 0.052499039818304255 0.028322752064256437 -0.03367797282815569 -0.044732926555678627 0.0050978746318156125 -0.0027435616659830285 -0.037932436391142646 0.0074566742805636929 0.0075223423206619627 0.066442771459401431 -0.06240770844842064 0.023507241469425996 -0.053868779508201792\nleaf_weight=15.201571996221899 119.39597707070061 4.5183864973369046 11.259523363260085 23.938687612535432 2.9522079151356584 38.749211059781373 790.12879206152502 10.1066635909956 12.975411051607805 52.051005803485168 24.220200832616683 8.5767444367520493 10.985214900225399 5.0492930745385811\nleaf_count=3975 1217 146 373 274 177 873 23710 109 454 628 634 86 95 1325\ninternal_value=0.000206443 0.00981304 -0.00869602 -0.0170996 0.0112038 -0.00141031 -0.000893219 0.00805382 0.0153362 0.0241447 0.0465999 -0.0240384 -0.00631741 -0.0027508\ninternal_weight=1130.11 162.065 42.6688 38.1504 14.2117 968.044 952.843 162.714 133.045 89.2466 37.1956 29.6686 21.0919 43.7985\ninternal_count=34076 2187 970 824 550 31889 27914 4204 3914 1716 1088 290 204 2198\nis_linear=0\nshrinkage=0.05\n\n\nTree=110\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 14 10 10 11 0 10 11 6 12 10 9 13 9\nsplit_gain=9.48183 9.80356 12.2018 9.38107 10.9478 8.42334 6.88175 12.1587 10.4831 6.57863 6.4652 8.56903 6.38969 6.25766\nthreshold=0.56746260328383402 0.83454127259551958 0.63188142770719913 0.84050262102991069 0.40762895036110941 0.93488923097199472 0.53208647906657525 0.40762895036110941 2.5000000000000004 0.27326717664988259 0.78979857050032498 0.53930446639708307 -0.084320303758185181 0.40090617732792916\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 5 4 -2 6 7 8 9 -3 11 -1 -10 -12\nright_child=3 2 -4 -5 -6 -7 -8 -9 12 -11 13 -13 -14 -15\nleaf_value=-1.8725563694671915e-05 -0.034546375675754817 -0.016201243175739063 -0.075407473333855599 -0.019008278632687507 0.019149253153162397 0.023432538039847031 0.009830801511007967 -0.051051868064502848 -0.024934318416004686 -0.06364266571700046 -0.012173195319914905 -0.051479050901186429 0.010866836912521563 0.030251147444441975\nleaf_weight=745.15189375182672 9.284935877658425 22.18048373478814 6.4665574622340491 21.768033103086054 137.92058812489267 16.136839206563309 24.902064239722677 22.484322364674881 15.918552705297769 10.708818006562067 11.207248618127776 7.1783844497986129 46.349227943894221 31.73261141916737\nleaf_count=27402 104 284 64 267 835 166 211 365 2395 110 123 79 1253 343\ninternal_value=0.000128224 -0.00179651 -0.0128429 0.0110143 0.0155522 -0.00983697 -0.0137403 -0.0187706 -0.0106914 -0.0330704 0.000510644 -0.000577419 0.00146494 0.0191534\ninternal_weight=1129.39 960.417 165.147 168.974 147.206 158.68 142.543 117.641 95.1571 32.8893 795.27 752.33 62.2678 42.9399\ninternal_count=34001 32795 4848 1206 939 4784 4618 4407 4042 394 27947 27481 3648 466\nis_linear=0\nshrinkage=0.05\n\n\nTree=111\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 3 1 10 1 2 10 10 10 1 10 10 9\nsplit_gain=9.61236 5.53909 7.47468 10.0985 8.99864 7.91416 8.48664 6.69928 10.0928 5.4535 5.28594 5.10008 13.2778 8.24235\nthreshold=0.56746260328383402 1.2413453555885363 -0.50900795747247185 7.0667347908020028 0.62279616455304676 4.4676787853240976 0.9841337858536412 0.61311827956989251 0.4770979020979022 0.8034250585480095 6.1600956916809091 0.17089216944801031 0.16585760517799356 0.44731785518021011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=11 2 10 5 -5 -4 7 8 -7 -6 -2 12 13 -1\nright_child=1 -3 3 4 9 6 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0043796796530242762 0.055831850301954825 0.027578755442998516 -0.068130799024731528 0.046631914367955281 -0.011654452076491869 0.021788153136293086 -0.06121981118251734 0.013813285665014986 -0.063779543736510944 0.023782549690164422 4.4193268788213944e-05 0.00011227936802276084 -0.054618611141093398 -0.075628439523476657\nleaf_weight=294.83071153945639 10.521474558860065 36.48244964890182 5.2524679630296296 15.198761980514972 25.687239071819931 5.2577836308628303 5.4544664958957574 28.438373085577041 7.2111076996661723 17.320331266149879 6.7266370379365972 650.97952804326633 13.373161017552773 3.1214683200232676\nleaf_count=17186 68 191 45 62 179 77 55 233 78 140 68 14939 859 37\ninternal_value=-0.000500187 0.0106687 0.00563622 0.000705799 0.0148643 -0.01524 -0.00793585 0.000460195 -0.0287592 0.00283313 0.0352698 -0.00240753 -0.00765992 -0.00537659\ninternal_weight=1125.86 163.551 127.069 109.821 58.2063 51.6142 46.3617 40.9073 12.4689 43.0076 17.2481 962.305 311.325 297.952\ninternal_count=34217 1196 1005 869 381 488 443 388 155 319 136 33021 18082 17223\nis_linear=0\nshrinkage=0.05\n\n\nTree=112\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 6 12 12 12 6 1 1 13 10 14 13 10\nsplit_gain=11.7751 7.82116 7.22989 6.97314 6.83264 5.93932 10.4066 5.90603 6.40324 6.13064 5.83655 6.68031 9.65627 9.49311\nthreshold=0.13963014728278042 0.079077644606565925 3.5000000000000004 0.15520902028626146 0.15728995790988926 0.76309033109884206 1.5000000000000002 4.1305465698242196 5.8755407333374032 0.87952391891555892 0.094782214156079861 0.44225003866361096 0.59759845839878112 0.10403138528138529\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 7 -5 -6 -7 -2 9 -9 -3 12 13 -12\nright_child=3 10 -4 4 5 6 -8 8 -10 -11 11 -13 -14 -15\nleaf_value=-0.012101243794239165 -0.015909561611810605 -0.041692099905191973 0.024958607674665265 -0.048333459796049624 0.0032409813969018508 0.024026114400013152 -0.04302956012634071 0.048455146499587862 0.0054571293084244767 0.00038544313953289513 -0.057459658330964003 -0.019697103412227367 -0.023683450944862208 0.01905122866836019\nleaf_weight=23.33266502461629 10.388182971844797 16.114173958390893 27.889100271553616 5.6732156628131625 784.52629977167817 7.8548498181626227 16.532285593915731 27.404481842946552 28.430795904365368 8.6746892463415843 3.4591258931031907 95.191456129548897 26.898013416866888 47.224437206503353\nleaf_count=390 893 1649 1354 200 12663 59 196 633 573 95 198 12098 928 2253\ninternal_value=0.000761837 -0.00904093 0.00816855 0.00341519 0.00206192 0.00247461 -0.021336 0.0179121 0.0235183 0.0372196 -0.0137262 -0.0108929 2.28259e-05 0.0128187\ninternal_weight=1129.59 240.109 51.2218 889.485 814.587 808.913 24.3871 74.8981 64.51 36.0792 188.887 172.773 77.5816 50.6836\ninternal_count=34182 18870 1744 15312 13118 12918 255 2194 1301 728 17126 15477 3379 2451\nis_linear=0\nshrinkage=0.05\n\n\nTree=113\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 14 10 1 2 13 1 1 13 1 3 3 14 2\nsplit_gain=12.6886 7.44916 11.2889 10.7004 8.08953 9.08058 7.63918 7.08896 6.62215 6.14451 6.12464 10.0475 6.78926 8.58022\nthreshold=0.41994549103866291 0.6656636044143921 0.58195386702849394 5.7287018299102792 1.4900624917218295 -1.3839347461794167 5.3563439846038827 4.557345151901246 0.22749536674849544 3.8759006261825566 -0.17251423663552959 -0.34594005640884345 0.92717729594341447 1.3673476462045733\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 10 4 5 -1 7 9 -8 -6 11 12 13 -3\nright_child=-2 2 -4 -5 6 -7 8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=0.020074870238523043 0.0070774918567524563 0.011274455969793424 -0.063697520487762588 0.0071150326285181727 0.0089842835396665346 -0.015432838870504652 -0.075482809974805443 0.034886588833291259 0.0056227421881325677 -0.035210779079821328 -0.01458416104979744 0.038613445325732187 -0.046761873556122741 -0.059188216156517592\nleaf_weight=18.950010204513092 381.82886174088344 31.414673181177932 9.0490845353924652 232.8917191454384 44.409476662112866 196.99924996582558 4.0525623802095634 21.497541557997465 4.7587625680025667 8.513437823392449 151.61814377003611 14.789780723469446 7.9496718871778276 3.9857266243197946\nleaf_count=1180 3283 2865 49 3855 696 15403 39 197 76 129 4653 582 912 142\ninternal_value=-0.000332605 -0.00410511 -0.0118484 -0.000904553 -0.00715015 -0.0122388 0.00604115 0.0113743 -0.0355713 0.00135371 -0.00931994 0.00442267 -0.00784986 0.00193299\ninternal_weight=1132.71 750.88 218.807 532.073 299.181 215.949 83.2318 74.4205 8.81132 52.9229 209.758 58.1399 43.3501 35.4004\ninternal_count=34061 30778 9203 21575 17720 16583 1137 1022 115 825 9154 4501 3919 3007\nis_linear=0\nshrinkage=0.05\n\n\nTree=114\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 9 14 10 10 1 3 3 11 13 11 1 1\nsplit_gain=7.56633 6.16403 9.45172 15.5909 10.5642 12.9781 7.97122 7.8867 10.6834 5.99702 6.41153 5.97631 11.3199 16.1429\nthreshold=0.13963014728278042 1.2413453555885363 0.58595927517517843 0.70740888062883378 0.61311827956989251 0.23562458691341706 7.1009039878845224 1.2935414034959523 0.4581604925069665 0.56629291394080161 1.1800529563749198 0.27328108044628058 4.972569704055787 5.5705857276916513\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 11 3 -3 5 -5 7 8 -7 -4 -11 -2 -13 -14\nright_child=1 2 9 4 -6 6 -8 -9 -10 10 -12 12 13 -15\nleaf_value=-0.0088854679774140813 0.0060372659056326687 0.014830796695044335 0.056931295004025773 -0.049902085806220652 -0.068020226822846919 0.0256558069997391 -0.022457017128302368 0.053525064400941041 -0.050160853528644259 0.038067245031970072 -0.012305982162707695 0.015676999252110187 -0.039892669397127545 -0.0042556269949423301\nleaf_weight=232.47699661272418 199.41217270241032 114.36969558126293 13.321699891239403 13.018023737356996 7.4020096557214847 13.089435354573657 16.003161868313327 10.255741414148359 5.9329410856589666 12.635426601395013 11.068467008764854 60.470835598476697 33.859495591255836 363.95055370603222\nleaf_count=18857 7109 1321 69 258 60 106 123 62 76 46 70 926 552 4428\ninternal_value=-0.000888266 0.00124472 0.00852938 0.0037241 -0.0155424 -0.0078448 0.00506333 0.0203506 0.000684913 0.0311869 0.0149996 -0.00116915 -0.0043112 -0.00738129\ninternal_weight=1107.27 874.79 217.097 180.071 65.7013 58.2993 45.2813 29.2781 19.0224 37.0256 23.7039 657.693 458.281 397.81\ninternal_count=34063 15206 2191 2006 685 625 367 244 182 185 116 13015 5906 4980\nis_linear=0\nshrinkage=0.05\n\n\nTree=115\nnum_leaves=15\nnum_cat=0\nsplit_feature=0 11 4 12 5 10 12 13 5 0 3 12 11 0\nsplit_gain=8.91623 11.1971 9.07219 7.80068 7.3301 7.32798 7.07625 7.22303 6.82776 6.23925 6.05305 6.46698 8.89178 6.60896\nthreshold=0.77366324840924527 0.5538054284498054 13.500000000000002 0.53142246749396205 3.5000000000000004 0.096022128556375139 0.76309033109884206 0.020971208253095851 1.5000000000000002 0.76189201172894638 1.4766501342390617 0.19488605834408215 0.23171031113136512 0.020884627018111463\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 3 5 -5 -2 -3 -8 -7 10 11 12 -1 -13\nright_child=2 6 -4 4 -6 8 7 -9 -10 -11 -12 13 -14 -15\nleaf_value=-0.0047053373045965758 -0.028707154004435895 0.011097145239784563 -0.057913822748895998 -0.028074351469634759 0.022847820954740913 -0.0026324472291687183 0.020270872023158373 -0.03553943428045106 0.02255441232084604 -0.045046008056291095 0.0070747321808682434 -0.056967045365430985 0.022856615523847056 -0.011031923300347579\nleaf_weight=291.29344804213179 8.247292750340419 184.47139773925301 3.9450030881271223 21.237907728063877 9.4777869535610062 33.736479356884956 7.9312965990975526 16.284234814113006 128.230037238769 8.8197469580627494 91.058054273598827 7.1515330733382134 31.523929433809826 259.13034873917786\nleaf_count=21065 197 1534 212 166 56 592 57 174 1754 235 1579 183 522 5581\ninternal_value=-0.000222592 -0.00237256 0.00916334 0.0107606 -0.0121366 0.0149392 0.00774954 -0.0171809 0.0173242 -0.00544527 -0.00486638 -0.006721 -0.00195157 -0.0124741\ninternal_weight=1102.54 897.664 204.875 200.93 30.7157 170.214 208.687 24.2155 161.967 688.977 680.157 589.099 322.817 266.282\ninternal_count=33907 30930 2977 2765 222 2543 1765 231 2346 29165 28930 27351 21587 5764\nis_linear=0\nshrinkage=0.05\n\n\nTree=116\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 11 11 13 13 1 11 2 13 4 6 2 3 11\nsplit_gain=9.29379 5.86382 7.84445 9.63217 10.7571 9.38452 9.28062 13.3924 7.76179 8.15541 7.73057 7.49603 8.58444 10.2092\nthreshold=0.22527488983156077 0.26571371385901138 0.43007633641122361 -0.048213762945273554 0.23860957374911687 5.0610318183898935 0.47815812583848105 -0.26652199078508504 -0.16146052683522655 6.5000000000000009 1.5000000000000002 1.3438334896653068 -0.23339967496118191 0.56629291394080161\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 3 -3 -5 -6 7 -4 9 -8 -9 12 -10 -14\nright_child=1 2 6 4 5 -7 8 10 11 -11 -12 -13 13 -15\nleaf_value=-0.006277784825482878 0.01589950879859315 -0.032241159790096473 -0.030946041065508051 0.03572892689660797 0.018018731936122551 -0.010740908114980116 0.0085391478556158355 0.0017367705137184498 0.017041201229941207 0.055668427391509012 0.046623078191035561 0.016202363364777001 -0.027121131605746047 -0.0015397984491689655\nleaf_weight=415.16086380089837 79.052311106861453 28.960700739145977 8.4669264456024376 16.428808554832358 33.974794221940101 150.06702691852115 34.87162965349853 13.205629232106729 37.773973306640983 11.699836735846473 34.845638335798867 53.233782878727652 53.345977121847682 138.61034903116524\nleaf_count=24863 1870 1010 201 480 416 1824 331 186 332 135 265 393 637 1222\ninternal_value=-0.000367892 0.0031687 0.00151043 -0.00579631 -0.00183492 -0.00536309 0.00585658 0.0243913 0.00261354 0.0213004 0.0345742 -0.000526213 -0.00445603 -0.00875274\ninternal_weight=1109.7 694.537 615.485 229.431 200.471 184.042 386.054 56.5182 329.536 46.5715 48.0513 282.964 229.73 191.956\ninternal_count=34165 9302 7432 3730 2720 2240 3702 652 3050 466 451 2584 2191 1859\nis_linear=0\nshrinkage=0.05\n\n\nTree=117\nnum_leaves=15\nnum_cat=0\nsplit_feature=12 9 10 9 7 10 7 11 12 9 4 10 3 12\nsplit_gain=6.44774 8.68957 8.47334 10.4643 14.3313 10.2511 9.75741 10.9828 9.35845 8.61151 11.5014 8.4199 8.60037 12.9144\nthreshold=0.76309033109884206 0.40090617732792916 0.54890004782400781 0.49756835111409253 8.5000000000000018 0.86240703177822864 10.500000000000002 0.30198526726146607 0.31228991981132515 0.41994549103866291 4.5000000000000009 0.8034250585480095 -0.2217883105887852 0.45977311129625353\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 6 -5 11 7 -3 -7 10 -8 12 13 -4\nright_child=-2 2 5 4 -6 8 9 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.0024189542459362064 -0.023324183348536861 0.03017476611728357 -0.0090040732059476925 0.011354499181064327 -0.055359537431280283 0.032460134508830195 0.031475541524029871 -0.031061721831572436 -0.046761967898389135 0.026521810699724466 -0.059871455478697204 0.041668461086560311 0.0066687099977302561 0.05143691859462815\nleaf_weight=685.85268751226249 27.070333516108803 8.9965361701324564 12.582978433463721 13.242659019306304 18.442547471495345 4.4014526905957636 5.4242656426504237 27.255931050749496 12.065036195795981 47.651011686655693 6.6327024176716805 25.231585955363698 174.98112116975244 26.733242633752525\nleaf_count=30394 260 61 111 125 331 33 46 315 199 443 104 218 1263 170\ninternal_value=0.000331554 0.000951542 0.00696775 -0.00353768 -0.0279825 0.0121994 0.00473909 -0.015463 -0.02494 0.0170577 -0.0195125 0.0148456 0.0115086 0.0323495\ninternal_weight=1096.56 1069.49 383.641 127.646 31.6852 255.995 95.9604 36.2525 16.4665 59.708 12.057 239.529 214.297 39.3162\ninternal_count=34073 33813 3419 1425 456 1994 969 376 232 593 150 1762 1544 281\nis_linear=0\nshrinkage=0.05\n\n\nTree=118\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 3 12 9 11 12 9 6 10 11 6 2 11 5\nsplit_gain=9.48849 8.12852 7.33273 8.12605 8.1262 10.4421 9.08163 9.32619 7.66619 9.14946 8.51916 7.5762 7.76601 7.52147\nthreshold=0.40090617732792916 1.2135643508474616 0.38810479972336492 0.38985550194288415 0.22964059561341896 0.18854389541763747 0.3373762830616705 17.500000000000004 0.17792998477929986 0.43007633641122361 5.5000000000000009 1.1122279291289965 0.25428458928435943 9.5000000000000018\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 -1 -6 7 8 10 11 -7 -10 -13 -8\nright_child=-2 -3 -4 -5 5 6 13 -9 9 -11 -12 12 -14 -15\nleaf_value=-0.0091239724718103054 0.0055213592535755945 0.0056147280289128715 -0.026159769696707576 -0.055952061972204974 0.029682143440555961 -0.018144600376330585 0.02392921959344109 0.049340243017150176 -0.043582809737772184 0.016760540412118826 0.057650670098525064 0.04536653739295346 -0.024858895417743207 -0.037863271143078567\nleaf_weight=311.74334918690511 405.76573702064343 162.76943744612799 45.330257852911018 6.7958053421461946 32.250290068608592 7.1669175291899574 38.642374979215674 4.9234175511955973 37.319289688777644 11.324410567758603 5.9664177511876915 4.788702629623006 11.31839056801982 4.7143269664375103\nleaf_count=22739 3677 3038 739 112 449 144 461 234 1681 189 117 62 161 146\ninternal_value=-0.000529799 -0.0041168 -0.00715456 -0.00529732 -0.00444795 0.0047535 -0.0018021 -0.0114716 -0.0158926 -0.0229215 0.0179316 -0.031486 -0.00254913 0.0165082\ninternal_weight=1090.82 685.053 522.284 476.954 470.158 158.415 126.164 82.8075 77.8841 64.7508 13.1333 53.4264 16.1071 43.3567\ninternal_count=33949 30272 27234 26495 26383 3644 3195 2588 2354 2093 261 1904 223 607\nis_linear=0\nshrinkage=0.05\n\n\nTree=119\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 0 6 1 5 10 1 10 14 12 4 12 12 10\nsplit_gain=5.29558 6.07667 4.77188 8.04047 9.11155 11.8464 11.4317 10.6782 11.6224 9.70682 8.14874 8.93906 7.94454 7.10525\nthreshold=1.3719071149826052 0.52976267587003434 2.5000000000000004 6.9686598777771005 5.5000000000000009 0.72783389450056124 6.6454498767852792 0.68841813923781148 0.91020246897107404 0.50270487952149423 13.500000000000002 0.38121896056391874 0.13638406002247264 0.95849462365591409\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 13 4 5 9 7 8 10 -4 -6 -12 -10 -2\nright_child=2 -3 3 -5 6 -7 -8 -9 12 -11 11 -13 -14 -15\nleaf_value=-0.041092035794680778 -0.0018455555004724508 0.006093072602565466 -0.0091460587142222511 0.013265483432685035 0.014272918519371881 -0.051598559164069739 -0.033067476827636812 0.045856548355531213 0.038634816277515326 0.02927022356088077 -0.031379336087538227 0.043869453335735782 -0.055810274908350159 -0.055833396580406217\nleaf_weight=15.103950187789453 516.82839827392309 11.443130943203867 133.40578730691777 167.89820535026956 146.08180365697626 13.656013261992483 15.566030015586874 18.019362221937627 2.0176408363186047 17.752751993946731 19.159192034800075 3.9847402997547752 10.311617353778272 5.1690981227438888\nleaf_count=4536 6501 1400 5288 2229 6938 137 465 122 343 113 4871 86 874 72\ninternal_value=0.000376236 -0.02127 0.000932724 0.00418861 0.000153336 -0.00866399 0.00691694 0.0101849 0.00641266 -0.00447156 0.00990087 -0.0171432 -0.0386157 -0.00248591\ninternal_weight=1096.4 26.5471 1069.85 547.853 379.955 164.815 215.14 199.574 181.555 151.159 169.226 23.1439 12.3293 521.997\ninternal_count=33975 5936 28039 21466 19237 5538 13699 13234 13112 5401 11895 4957 1217 6573\nis_linear=0\nshrinkage=0.05\n\n\nTree=120\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 10 3 3 14 4 5 9 5 7 2 1 9 5\nsplit_gain=6.36297 7.51382 6.04774 4.97575 4.49794 6.5922 6.99988 6.74265 6.81265 5.41687 4.55442 4.30173 6.74532 4.70129\nthreshold=0.10135754907356449 0.088748241912798884 1.2501037838947984 2.6316286664499118 0.52810343530048642 9.5000000000000018 8.5000000000000018 0.061164077610165714 4.5000000000000009 24.500000000000004 1.7271604568384149 0.6179969608783723 0.064824694063463792 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -3 11 5 6 7 -1 -9 -7 -6 12 -2 -13\nright_child=3 2 -4 -5 10 9 -8 8 -10 -11 -12 13 -14 -15\nleaf_value=-0.02696301775954555 0.026097805158695003 -0.05098417901179525 0.0027856158527406038 0.041049308388633361 -0.038014562636680328 0.060537882402442483 -0.065205470099587159 0.0004117287390069987 0.051530008280269893 -0.033206155074209841 0.014626424078835083 -0.0051518087327052549 -0.052743420089640961 0.0032842284095309809\nleaf_weight=13.104733956075505 3.241896315084885 16.941399991301299 7.2400487472186787 6.8680432939436278 14.056858374773585 6.5102995332708806 3.23428228891862 23.91317135994905 7.9684368565212926 1.252693946000363 5.0354780243942505 208.97126860992284 7.5624025509387138 773.36393165399204\nleaf_count=268 250 1718 136 109 1444 181 236 409 168 405 56 3196 2497 22948\ninternal_value=0.00023949 -0.01177 -0.0354139 0.00144207 -0.003789 0.00343697 -0.00373834 0.00200266 0.0143667 0.0433476 -0.0240939 0.00112716 -0.0288798 0.00148622\ninternal_weight=1099.26 99.2574 24.1814 1000.01 75.076 55.9836 48.2206 44.9863 31.8816 7.76299 19.0923 993.139 10.8043 982.335\ninternal_count=34021 5021 1854 29000 3167 1667 1081 845 577 586 1500 28891 2747 26144\nis_linear=0\nshrinkage=0.05\n\n\nTree=121\nnum_leaves=15\nnum_cat=0\nsplit_feature=13 11 11 12 12 0 0 6 12 11 8 8 0 0\nsplit_gain=5.60625 6.08687 5.34663 10.9765 8.37669 8.68218 6.31206 6.72439 18.907 15.6028 9.52766 11.5669 6.29399 8.44179\nthreshold=2.2531977117618109 0.30198526726146607 0.56629291394080161 0.56553704453902076 0.76309033109884206 0.47174594007001402 0.77366324840924527 1.5000000000000002 0.19909436529862615 0.15366261807246603 21.500000000000004 37.500000000000007 0.1173266999878748 0.18350562776239354\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 3 6 -4 -6 7 8 9 -1 -10 -12 -9 -14\nright_child=1 -3 4 -5 5 -7 -8 12 10 -11 11 -13 13 -15\nleaf_value=-0.01922580061605772 -0.009949208090842784 0.028099869084703008 0.0093307282160340187 -0.062519708624134487 -0.046677727282707887 0.0074357595236367703 0.0071452837750281273 0.0080892682453140539 -0.0055764702449200227 0.020667542639073142 -0.058745416678662558 0.0085225716474330721 -0.029142568040399216 -0.0024664759967364235\nleaf_weight=47.880519243422896 14.629231482045723 32.860563366324641 201.6779852565378 6.6416649704333386 13.578094602562489 14.600338939577339 144.26811059947067 126.54886818082741 20.341275853104889 48.161310187657364 35.790185241843574 7.4511983019765458 31.441508864758362 348.88079349095642\nleaf_count=761 231 415 1658 72 169 88 2492 5402 242 680 393 142 2483 18883\ninternal_value=-0.000205994 0.0164154 -0.000974659 -0.00286943 0.00574669 -0.0193455 -0.00230437 -0.00435746 -0.013273 0.000786015 -0.0341924 -0.0472234 -0.00152918 -0.00474245\ninternal_weight=1094.75 47.4898 1047.26 817.405 229.856 28.1784 810.764 666.496 159.624 96.0418 63.5827 43.2414 506.871 380.322\ninternal_count=34111 646 33465 31550 1915 257 31478 28986 2218 1441 777 535 26768 21366\nis_linear=0\nshrinkage=0.05\n\n\nTree=122\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 0 0 9 4 2 0 2 4 0 0 9 9\nsplit_gain=8.52713 6.98406 5.22905 8.5819 11.9315 8.46757 7.63002 10.5276 9.63505 7.51035 21.5428 10.9332 7.58899 8.15605\nthreshold=0.13963014728278042 0.17963123019302799 0.1173266999878748 0.30024322335876469 0.72224482408762225 2.5000000000000004 0.81060921052942725 0.026851214599446126 -0.28572182230510362 3.5000000000000004 0.23197765418400104 0.2744976418651715 0.2529894131731712 0.30014869815283679\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 6 4 5 -4 7 8 -2 11 12 -7 -11 -14\nright_child=2 -3 3 -5 -6 9 -8 -9 -10 10 -12 -13 13 -15\nleaf_value=-0.0055739603538598609 -0.037954923396308432 -0.029913848791079913 -0.033671597367385668 0.0025576333971085957 0.04751680436908045 -0.0076000143862415747 -0.020992053647968158 0.027023624328094617 0.02097134814407035 -0.044313574250763682 -0.05438217473757017 0.067650571422112851 0.061567373673093555 0.0030745032049724233\nleaf_weight=195.66358038536055 13.488917129623587 33.867758222171688 40.398084974265657 611.97542222114134 7.1321223219856611 12.423959873325659 14.547682443168013 68.836207272062893 12.411723380995682 5.575441902561578 21.15625146907405 6.7169687674613661 6.2867171714315182 33.384551976807415\nleaf_count=15591 367 3262 434 10783 31 155 145 1137 139 646 791 43 105 480\ninternal_value=-0.000745448 -0.00927954 0.00155555 5.13736e-05 -0.0114074 -0.0150923 0.0117168 0.0169064 -0.00998726 -0.0060306 -0.0139749 0.0208548 0.00568599 0.0136298\ninternal_weight=1083.87 229.531 854.334 745.05 133.074 125.942 109.285 94.7368 25.9006 85.5439 66.403 19.1409 45.2467 39.6713\ninternal_count=34109 18853 15256 13468 2685 2654 1788 1643 506 2220 2022 198 1231 585\nis_linear=0\nshrinkage=0.05\n\n\nTree=123\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 14 13 10 10 0 13 1 0 4 9 1 1 4\nsplit_gain=8.38935 6.22786 5.64821 5.54105 4.8533 4.81887 10.762 4.63422 4.50898 7.05068 7.58231 6.75166 10.5191 9.30545\nthreshold=0.40090617732792916 0.57478362346458833 2.2531977117618109 0.84050262102991069 0.56386946386946402 0.077505217748931474 0.59759845839878112 0.6179969608783723 0.26806232092754839 11.500000000000002 0.69450985341528104 4.972569704055787 5.503817319869996 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 5 7 -4 6 -3 -1 9 10 -7 -10 13 -13\nright_child=1 2 4 -5 -6 8 -8 -9 11 -11 -12 12 -14 -15\nleaf_value=-0.03140945917444303 0.028001410713127169 0.052830480098578797 -0.00010315007037336381 -0.058245660828122929 0.048934408373520838 -0.025562829351880506 -0.010380572937233358 -0.0021112801349350671 0.025123904723721231 0.031767195694364807 0.021281805415336825 0.016467005452717651 0.0039492956381951428 -0.05934916105704055\nleaf_weight=12.787168821625526 28.742687256424688 13.28324444871396 8.3760987464338559 3.525369922630488 11.674019378842784 45.709942027926445 12.216250671073793 660.0639893330881 32.054883936420083 7.5680345420259973 9.7950704526156169 5.4182027317583588 226.39933278446551 11.644129948224871\nleaf_count=3712 316 82 71 55 112 444 89 26662 294 119 48 31 1962 146\ninternal_value=0.000338296 0.00594651 0.00422786 -0.00308886 0.0294169 0.00276395 0.0232985 -0.00271386 0.00115342 -0.010802 -0.0170657 0.00391949 0.00103631 -0.0356947\ninternal_weight=1089.26 412.882 384.139 676.377 20.0501 364.089 25.4995 672.851 338.59 63.073 55.505 275.517 243.462 17.0623\ninternal_count=34143 3714 3398 30429 183 3215 171 30374 3044 611 492 2433 2139 177\nis_linear=0\nshrinkage=0.05\n\n\nTree=124\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 2 11 11 13 14 14 12 9 3 3 7 1 12\nsplit_gain=6.32141 4.92992 5.08127 10.4955 5.1066 4.66751 4.47179 6.92038 6.88119 10.6387 8.68293 7.92761 8.32785 7.21032\nthreshold=0.34268005210784097 1.9357224602941756 0.19809164072177035 0.26948978220549369 -0.18966173231886244 0.89338524034174294 0.48216822135602816 0.26410036853271063 0.2627566423052386 -1.2956154205787347 -1.0000184455921113 17.500000000000004 5.60058617591858 0.19194349112342055\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 2 4 5 -2 -4 -1 8 9 -8 -11 12 -10 -12\nright_child=1 -3 3 -5 -6 -7 7 -9 11 10 13 -13 -14 -15\nleaf_value=0.001476755321337212 0.011433276137904477 0.026945115136725834 0.045030487574080943 0.001742374928129523 -0.063924404650399266 0.0018551026512029841 -0.0552619358969672 -0.026175756494994984 0.030718503505874014 0.034412673901980197 -0.0032928004622301481 0.029720592181819314 -0.022695982646957453 -0.025503514189628179\nleaf_weight=245.02816041307233 3.3253634660504758 21.397622309392318 20.157503250287849 458.8003224512795 4.5796234231675035 9.0157704630400968 10.604502355301518 41.917375094955787 9.5791869753738847 11.530792033092437 156.71653585115928 27.302382043562829 22.666149750584736 46.80578892439371\nleaf_count=8585 38 277 176 4564 77 104 2001 1361 187 1454 12359 514 424 1960\ninternal_value=-1.00222e-05 0.00399158 0.00294888 0.00362485 -0.0344999 0.0321912 -0.00362849 -0.00744537 -0.00461068 -0.00859717 -0.00605095 0.0103887 -0.00638134 -0.00850051\ninternal_weight=1089.43 517.276 495.879 487.974 7.90499 29.1733 572.151 327.123 285.205 225.658 215.053 59.5477 32.2453 203.522\ninternal_count=34081 5236 4959 4844 115 280 28845 20260 18899 17774 15773 1125 611 14319\nis_linear=0\nshrinkage=0.05\n\n\nTree=125\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 9 7 11 2 11 12 5 3 5 13 5 13 1\nsplit_gain=8.71458 6.35426 9.35662 5.68698 11.8909 6.18608 9.14376 8.19545 8.14047 9.70938 5.84536 5.1953 5.15936 4.93079\nthreshold=0.43007633641122361 0.48753231203628372 11.500000000000002 0.47815812583848105 -0.26652199078508504 0.56629291394080161 0.48187617214593886 3.5000000000000004 -0.1186150853325887 4.5000000000000009 0.68358916659085278 1.5000000000000002 0.83711243686168813 4.0054044723510751\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 13 12 4 -2 6 8 -8 -5 11 -6 -10 -3 -1\nright_child=3 2 -4 5 10 -7 7 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.009504840214722211 -0.029607530441795388 -0.036746924549057619 -0.054731542151505844 0.035153743394281997 0.0038560634874507901 0.0085013871975038839 -0.011925953566527023 -0.070002770578065573 -0.021532244262638747 -0.052122445832184165 0.042563573631150355 0.017565422832806384 0.014989812832595026 0.00079421390625310456\nleaf_weight=148.08618236965413 8.7651196776423586 6.0804222570732263 13.13699473813176 14.842967846547252 13.97144114715047 234.91246330901049 21.131693712552082 7.8532893436495206 11.61129461636301 8.2860253936378268 33.030264438944869 25.19171459111385 14.094946804805657 526.51088349825295\nleaf_count=16459 209 72 218 173 231 1969 213 147 167 146 247 213 92 13552\ninternal_value=0.000730045 -0.0025472 -0.0235382 0.00683242 0.0214396 0.00425865 -0.00692025 -0.0294713 0.00429082 -0.00641131 0.0313776 0.00498688 -0.00160155 -0.00147734\ninternal_weight=1087.51 707.909 33.3124 379.596 55.7668 323.829 88.917 28.985 59.932 45.089 47.0017 36.803 20.1754 674.597\ninternal_count=34108 30393 382 3715 687 3028 1059 360 699 526 478 380 164 30011\nis_linear=0\nshrinkage=0.05\n\n\nTree=126\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 10 3 1 10 4 5 9 1 1 9 6 11 2\nsplit_gain=8.00058 9.2637 17.1319 8.82112 6.0026 5.6275 5.33149 4.86044 5.51685 6.39282 4.90563 4.66495 8.47065 6.78276\nthreshold=0.17089216944801031 0.17963123019302799 -0.25339303029263582 2.8454167842864995 0.16585760517799356 1.5000000000000002 6.5000000000000009 0.44731785518021011 5.4778068065643319 4.7620620727539071 0.27364141710690554 5.5000000000000009 0.11851485802112142 -0.71148459116722751\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 5 7 -3 -4 8 9 10 -1 13 -13 -10\nright_child=1 3 6 -5 -6 -7 -8 -9 11 -11 -12 12 -14 -15\nleaf_value=-0.0078696362039960954 -0.060350358792987405 0.014328974421216716 0.019653877800674176 0.0019788424490868638 -0.042257733497362109 -0.049095843075548172 0.06877432214420591 -0.065518420808244549 -0.037959852264030119 -0.029701104323945845 0.039265356919502697 -0.050957418311445693 0.03964398274679734 0.0035109202078261723\nleaf_weight=124.2214681263722 3.5776111629020333 4.1785891233012107 11.690491331901287 742.68551381486031 10.943010245180629 12.750003536129951 11.795029227141638 2.3961401795968404 10.122112034820018 35.113332083790738 4.7912996142404145 2.1774136933963772 15.305547570344059 85.927905322401784\nleaf_count=13245 556 99 425 12452 862 2184 373 39 191 1411 73 239 674 1292\ninternal_value=-0.000432423 0.00218859 0.0304003 0.00114483 -0.00750122 -0.0335143 0.0461247 -0.00599761 -0.00525068 -0.0110864 -0.00583058 0.00320875 0.0262136 -0.00120553\ninternal_weight=1077.68 786.677 27.0631 759.614 290.998 16.9286 23.4855 280.055 277.659 164.126 129.013 113.533 17.483 96.05\ninternal_count=34115 16089 1354 14735 18026 2283 798 17164 17125 14729 13318 2396 913 1483\nis_linear=0\nshrinkage=0.05\n\n\nTree=127\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 14 6 1 1 3 1 2 3 3 13 11 11 0\nsplit_gain=9.29295 7.51242 7.11981 6.36021 6.83843 6.717 6.17775 7.04909 8.23063 5.99816 5.26304 5.11688 8.96744 4.70178\nthreshold=0.12552521008403364 0.52810343530048642 4.5000000000000009 5.423553705215455 6.7740724086761483 -1.0952598965141904 1.9646461606025698 1.875273743832488 0.16816061193025691 1.4393308112662344 2.2531977117618109 0.13779101521697718 0.14134251545640727 0.077505217748931474\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 5 -5 -1 7 8 -2 -3 11 -8 -13 -14\nright_child=6 9 -4 4 -6 -7 10 -9 -10 -11 -12 12 13 -15\nleaf_value=0.03112259197671409 -0.055884363123393156 -0.034424793329198476 0.015498726873609711 0.020052293447221519 -0.024923632710356598 -0.03012714746019975 0.017700983566042463 0.01988496678169031 0.022992638021768153 0.0075694878478142812 0.020920213750878103 -0.054848980030732047 0.013398588411301489 0.00046884734703659395\nleaf_weight=4.2070897030062033 13.433657564433821 41.39154043975941 34.366527313060942 23.172132027859334 12.017507584765552 40.261694707049173 49.156566917095915 6.4812276827869928 3.6800506071231203 10.237134328519458 38.330514538101852 6.0651746282528611 76.92999664520903 720.03702278508717\nleaf_count=74 4087 4751 3019 288 146 1171 2233 106 409 197 534 374 1602 15103\ninternal_value=0.000377665 -0.0104822 -0.00323964 -0.0114157 0.00442843 -0.0237754 0.00235497 -0.0223723 -0.0385901 -0.0261124 0.0030345 0.00220644 0.00123713 0.00173216\ninternal_weight=1079.77 165.654 114.025 79.6584 35.1896 44.4688 914.114 23.5949 17.1137 51.6287 890.519 852.189 803.032 796.967\ninternal_count=34094 9646 4698 1679 434 1245 24448 4602 4496 4948 19846 19312 17079 16705\nis_linear=0\nshrinkage=0.05\n\n\nTree=128\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 10 0 6 2 1 6 13 9 13 11 9 1 2\nsplit_gain=7.97278 8.39014 6.5253 8.05305 7.13782 6.30661 6.44094 8.5015 7.84872 7.5019 7.52624 7.26257 7.10502 10.3124\nthreshold=0.84050262102991069 0.78193473193473206 0.81608219810852256 4.5000000000000009 1.5610522555580648 0.79792803525924694 2.5000000000000004 1.732394541732335 0.50674003195298434 -2.0214381760953946 0.090395177304369301 0.67046756849715294 6.8997397422790536 0.24498431757668332\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 -4 -5 -1 7 11 -9 -8 -11 -7 13 -12\nright_child=-2 -3 3 4 -6 6 9 8 -10 10 12 -13 -14 -15\nleaf_value=-0.034872360336769763 -0.022750824683736768 0.018782811357931958 0.014084471466541335 -0.032847485687023573 0.043071952698318178 -0.0079822515712625504 -0.046085406270534438 0.0034676243872611063 0.056993673194676769 0.028612129503008173 -0.012408019183510307 -0.044891485406203448 0.011525773694046901 0.0059095187501145147\nleaf_weight=13.890139698025449 38.669186985469423 55.750297444348689 116.98127075965749 16.478866585472137 2.861061447125393 324.12439102330245 6.985810152051271 36.44166112219682 7.4773341635591342 25.594452076948073 146.00750721119039 12.969869327731429 118.59383598563727 160.07481981759338\nleaf_count=4079 511 450 1325 746 29 4564 406 527 45 822 13651 76 1908 4908\ninternal_value=-0.000749346 8.6383e-05 -0.000986012 0.00913093 -0.0200509 -0.00261324 -0.00203779 -0.00683056 0.0136465 0.00196202 0.00280793 -0.00953093 0.0011892 -0.00284027\ninternal_weight=1082.9 1044.23 988.481 136.321 19.3399 852.16 838.27 381.013 43.919 457.256 450.271 337.094 424.676 306.082\ninternal_count=34047 33536 33086 2100 775 30986 26907 5212 572 21695 21289 4640 20467 18559\nis_linear=0\nshrinkage=0.05\n\n\nTree=129\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 1 5 10 13 2 5 0 12 0 3 5 10 3\nsplit_gain=6.60673 5.86646 7.59548 14.7243 14.7218 13.4138 8.8974 9.06038 9.59812 8.18172 9.77448 7.86023 8.25599 7.23888\nthreshold=1.992208782912438 1.4491258263587954 5.5000000000000009 0.20766968880176431 1.1576931101042465 1.0573530789184751 16.500000000000004 0.5250290166790027 0.3524848567250895 0.017893471098448719 0.064246569147159552 4.5000000000000009 0.78979857050032498 -0.9999468844480115\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 13 9 6 7 -6 -9 10 -4 -5 -13 -3\nright_child=-2 2 4 11 5 -7 -8 8 -10 -11 -12 12 -14 -15\nleaf_value=-0.026051478966925806 0.015353428492093752 0.016110576095016513 0.053516689992630075 0.0034889329862210043 -0.053227371600975952 0.018507461939561536 0.02133131063507876 -0.046619963897856352 0.025020559407153271 0.0098390611819558337 -0.048828778621270941 -0.028600276814158939 0.030969849598810439 -0.017177644168683951\nleaf_weight=21.525854506315227 66.335644100210629 17.104761019232683 18.697272697081647 455.50350522328517 26.345751831511734 23.056938258814625 7.5039618575829072 7.2685644248849739 9.7983126249164325 195.63597965256486 1.8596233400239714 34.686177978044725 6.0938185777049503 187.20302373362938\nleaf_count=6108 989 448 410 5317 526 179 151 150 55 12831 187 695 51 5960\ninternal_value=0.000187631 -0.000820366 -0.000246463 -0.00306366 0.00654267 -0.0125939 -0.026763 -0.0353693 -0.00638186 0.0131184 0.0424227 0.00159219 -0.0191704 -0.0143258\ninternal_weight=1078.62 1012.28 990.758 700.591 290.166 73.9735 50.9166 43.4126 17.0669 216.193 20.5569 496.284 40.78 204.308\ninternal_count=34057 33068 26960 12471 14489 1061 882 731 205 13428 597 6063 746 6408\nis_linear=0\nshrinkage=0.05\n\n\nTree=130\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 1 7 1 11 1 0 7 7 9 10 1 0 0\nsplit_gain=6.67823 6.17467 8.8785 6.96734 7.01826 5.66919 9.99304 7.55292 6.05966 6.85988 5.41119 5.31545 10.1267 8.63674\nthreshold=4.0979936122894296 4.2128593921661386 15.500000000000002 4.3886544704437265 0.20355288168979549 4.81640648841858 0.49286160918825045 9.5000000000000018 23.500000000000004 0.11975557264378354 0.27608892921960076 5.503817319869996 0.71078138323361617 0.45712332234835967\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 4 -3 6 7 -5 10 -10 -8 12 13 -7\nright_child=1 3 -4 5 -6 11 8 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.0094903481001146952 0.052015301382824765 0.0056632390194967692 -0.033694270875437221 0.021629165346786265 -0.060148642009723599 -0.0082031027161543237 0.009411216049526671 -0.020444764009515046 0.049993279121174877 -0.035194656528596942 0.071771125699216967 0.0017361270881643702 0.018089883084917897 -0.043720709683569212\nleaf_weight=171.5347945780959 11.495200973193276 6.8915824215364401 3.2208310481801155 19.567198544391431 8.3089539167849562 63.058013973364723 5.1595400169317207 21.158824071884737 3.1552374725142718 4.4815540295821847 12.338053650571963 706.10158192280505 24.093765712605091 22.834376683080336\nleaf_count=17167 214 551 309 382 235 1911 272 996 89 181 110 10596 419 745\ninternal_value=-0.000469386 0.00123612 0.0323061 0.000691077 -0.0318032 0.00128651 0.0152839 -0.000196055 0.0394143 0.00171505 0.0548858 0.000139558 -0.0100336 -0.0180399\ninternal_weight=1083.4 911.865 14.716 897.149 15.2005 881.948 65.8604 40.726 25.1344 7.63679 17.4976 816.088 109.986 85.8924\ninternal_count=34177 17010 523 16487 786 15701 2030 1378 652 270 382 13671 3075 2656\nis_linear=0\nshrinkage=0.05\n\n\nTree=131\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 0 2 9 2 2 12 11 1 6 10 10 1\nsplit_gain=10.9028 7.37023 6.62219 10.3224 9.72861 9.67871 8.89682 7.8802 7.36054 7.16238 6.93454 6.55677 9.63466 7.12076\nthreshold=0.078907978039233523 0.86240703177822864 0.1173266999878748 1.0372423979668091 0.090459380569532843 0.79404142972014002 -0.8023925232996747 0.14605449746596447 0.15627841297233167 5.3563439846038827 7.5000000000000009 0.69732205778717427 0.68841813923781148 6.1893992424011239\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 5 4 -4 6 -2 -5 9 -9 -10 12 13 -6\nright_child=1 -3 3 7 11 -7 -8 8 10 -11 -12 -13 -14 -15\nleaf_value=-0.017856790254268451 -0.0039191905018684551 -0.022131896929289727 -0.047862861491962838 0.026464465293636264 -0.013047253009806121 -0.018339414038261712 0.025759978920146056 -0.00031797242006754398 0.0073914592712767749 -0.059938916746541539 -0.050610210992747623 0.0088287511622835834 -0.070975003402339129 0.00042962538696848644\nleaf_weight=83.367234615976486 35.331221931653999 32.678061638958752 11.59341184977893 43.94113126724551 165.54169107660346 22.009326441329904 85.447381739832053 12.313054897924305 243.78992279613885 7.5374892812687895 4.2758622132532755 89.674191121011972 4.6392657125834367 238.05245158360049\nleaf_count=5437 1801 415 2200 931 12459 228 2279 299 2760 103 186 799 52 4159\ninternal_value=-0.000597784 0.000862084 0.00166355 -6.92498e-05 -0.00445185 0.0115623 0.0171171 0.00708222 0.00381673 -0.0247457 0.00619251 -0.00334714 -0.00603679 -0.00511675\ninternal_weight=1080.19 996.824 964.146 821.358 509.501 142.788 120.779 311.857 267.916 19.8505 248.066 497.908 408.233 403.594\ninternal_count=34108 28671 28256 23948 19669 4308 4080 4279 3348 402 2946 17469 16670 16618\nis_linear=0\nshrinkage=0.05\n\n\nTree=132\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 10 3 10 13 3 1 9 13 10 9 13 10 9\nsplit_gain=7.50636 6.19742 9.05795 8.9486 5.31107 6.05889 7.19519 10.7156 6.37166 5.6245 5.10197 4.99892 4.97753 9.9751\nthreshold=0.34268005210784097 0.55753467843631788 0.72538182894604841 0.45857843137254906 0.87952391891555892 -0.20861508743145712 5.1478576660156259 0.18560323472877255 2.0190639871957083 0.029776858282843214 0.30964888337946778 1.0000000000000007 0.39228096301811871 0.44731785518021011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 9 6 7 8 11 -1 -4 -6 13 -2\nright_child=12 2 10 -5 5 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.052188577280340598 0.0231286964746273 -0.056080106278944844 0.022833878510726088 0.025840166716632196 0.0010638878874919828 -0.0064622648237219575 -0.056755940477974887 0.023564163810181133 0.032086176207578011 -0.0002654221194827977 -0.03756814502463493 -0.063263628542570394 0.0013713443777778555 -0.012898942379022466\nleaf_weight=4.3014948155177981 72.813170039793476 12.927963977330366 8.8642506719334033 23.920304372964893 3.9345399005105781 130.26865198233281 13.657766164280472 11.346640007221138 2.2188778835115945 322.52119895449869 4.4167073844000688 11.546103209417195 426.2621646244661 25.239888592157513\nleaf_count=115 842 297 90 411 215 2562 137 139 29 24133 67 611 4098 299\ninternal_value=-0.000555906 -0.00463292 -0.0279077 -0.00341719 -0.00486626 -0.011924 -0.028056 -0.0131212 -0.0366416 -0.00110592 0.00152255 -0.0478417 0.00372093 0.0138182\ninternal_weight=1074.24 549.924 26.2089 523.716 499.795 172.973 42.7039 29.0462 17.6995 326.823 13.281 15.4806 524.315 98.0531\ninternal_count=34045 28806 454 28352 27941 3693 1131 994 855 24248 157 826 5239 1141\nis_linear=0\nshrinkage=0.05\n\n\nTree=133\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 11 1 7 2 13 2 14 6 14 9 14 9 4\nsplit_gain=5.44995 7.11475 10.3537 10.5969 7.40287 6.72353 11.6563 6.33224 5.98696 6.90084 5.90843 6.47586 4.97214 4.87747\nthreshold=0.13963014728278042 0.16091067892009897 6.130833864212037 15.500000000000002 2.1520662742152665 0.77517283292263039 1.1771025309410685 0.37609602239627654 4.5000000000000009 0.5321233765890433 0.24637228930449964 0.63300027894733968 0.10965635606201694 20.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 4 -4 -3 7 13 -2 -1 12 11 -8 -10 -7\nright_child=5 2 3 -5 -6 6 10 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.0084927731048648671 -0.036037373235541237 -0.045653265231736101 0.045532066360951168 -0.057675985200625363 0.010815365224997667 -0.0084660584951780343 0.0061647916227321213 0.0080070794377826975 0.010866196135374984 -0.028522250157223861 0.015295477563619079 -0.041691150005492984 0.05396201461445746 0.018949312804394238\nleaf_weight=119.05596136944223 7.3720333059318355 30.933648993446695 8.1857370682992023 2.4677911242470136 6.6806366684613741 313.94214921152161 21.646083733532581 367.37883147831599 44.649944617005531 8.6243396042336808 111.96045001782477 9.3519236396532488 7.0702231459799796 16.178379619959742\nleaf_count=3474 181 3882 141 114 71 3373 312 9845 5257 5449 1190 112 486 225\ninternal_value=0.00020788 -0.0066435 -0.0234751 0.0187271 -0.0356036 0.00205335 -0.00190895 0.00704704 -0.00200991 0.0107103 0.00997377 -0.00912479 0.0176693 -0.00706933\ninternal_weight=1075.5 227.668 48.2678 10.6535 37.6143 847.83 473.079 374.751 179.4 60.3445 142.958 30.998 51.7202 330.121\ninternal_count=34112 18874 4208 255 3953 15238 5212 10026 14666 11192 1614 424 5743 3598\nis_linear=0\nshrinkage=0.05\n\n\nTree=134\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 9 0 12 12 14 12 11 13 3 3 3 3 11\nsplit_gain=7.63497 7.0976 6.99521 7.80063 11.0739 8.60175 8.1007 13.3342 8.02129 12.8532 8.43825 16.3175 7.32084 8.81871\nthreshold=0.40090617732792916 0.38985550194288415 0.77947462504345733 0.19488605834408215 0.17993568848336086 0.77218019103003688 0.17056733839577878 0.22964059561341896 1.0000000000000007 1.0000000180025095e-35 0.33109536440571641 0.76111247319296804 0.61547820836038591 0.16449293709486093\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 6 12 7 8 10 -10 -1 -12 13 -6\nright_child=-2 -3 -4 -5 5 -7 -8 -9 9 -11 11 -13 -14 -15\nleaf_value=0.0092837561973243121 0.0048394218240733998 -0.03218643783210412 0.010338280291324291 -0.012811781196014144 0.039244187165898038 0.057241039061029078 -0.029406467911389834 0.040339842195940334 -0.06353507667761582 -0.0025246787466166342 -0.045049529348236708 -0.00033406714189793693 0.026975302463749459 -0.047145468757470305\nleaf_weight=111.09805823141141 398.25426768622128 20.48157383420039 84.861605979065644 202.20701394478237 3.405912784292016 11.950280196469974 27.172690804163722 17.027547204241273 12.976944913469195 22.805261248824536 24.779001899034483 97.892871403022582 19.195578782193479 8.5063838860514789\nleaf_count=13245 3688 277 1978 5323 83 336 1506 229 520 559 2163 3057 383 789\ninternal_value=-0.00062737 -0.00390681 -0.00295873 -0.00499214 -0.000537275 0.0229422 -0.00382217 -0.00130251 -0.00407243 -0.0257768 -0.000654589 -0.00965756 0.00839375 -0.0213189\ninternal_weight=1062.61 664.361 643.879 559.018 356.811 43.0582 313.752 286.58 269.552 35.7822 233.77 122.672 31.1079 11.9123\ninternal_count=34136 30448 30171 28193 22870 1591 21279 19773 19544 1079 18465 5220 1255 872\nis_linear=0\nshrinkage=0.05\n\n\nTree=135\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 12 3 1 4 3 6 6 12 12 12 8 13 5\nsplit_gain=4.95199 6.51605 5.44138 8.86769 8.7847 7.29102 6.7917 5.26349 5.40324 7.92686 7.78123 8.67266 7.52854 5.8394\nthreshold=0.34755730289246772 0.76309033109884206 -0.75305914588649314 4.9366118907928476 3.5000000000000004 -0.91484210894904117 2.5000000000000004 14.500000000000002 0.23525212531660353 0.24693037208688387 0.22295095288921266 22.500000000000004 2.2531977117618109 7.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 -2 5 6 -5 8 10 -10 11 -4 13 -11\nright_child=1 -3 7 4 -6 -7 -8 -9 9 12 -12 -13 -14 -15\nleaf_value=-0.0019052249700602334 0.053829000087209249 -0.018390787005572527 -0.020116581646179715 0.033467808941717317 -0.034732519803472077 0.060090063274395616 -0.023426393205209419 -0.038617646420447099 -0.046692152951750006 -0.00037457164667726857 0.052844725920449122 0.031643997158908248 0.031003706176423784 0.01791336502096148\nleaf_weight=570.35427298694049 14.180456604575737 27.407068465021439 14.436091399868021 9.148526952601971 8.2331998017616552 7.5409942986443665 9.8797907466068846 5.9970681892009443 6.8080115350894621 295.72524253022857 12.808393063256515 16.096865119587164 23.521862371300813 50.200061911717057\nleaf_count=28979 149 269 180 99 196 30 86 195 86 2650 75 203 299 527\ninternal_value=0.00128121 0.00489823 0.00627903 0.0218967 0.00774558 0.0216909 0.00423274 0.00443443 0.0051401 0.00318542 0.0216546 0.00730955 0.0042281 0.00232333\ninternal_weight=1072.34 501.984 474.577 48.983 34.8025 26.5693 19.0283 425.594 419.597 376.255 43.3413 30.533 369.447 345.925\ninternal_count=34023 5044 4775 560 411 215 185 4215 4020 3562 458 383 3476 3177\nis_linear=0\nshrinkage=0.05\n\n\nTree=136\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 4 1 13 7 2 13 7 2 13 10 6 11\nsplit_gain=5.53731 7.63736 7.1358 9.61718 9.39662 11.0255 9.68523 13.554 9.00052 13.0731 13.6312 8.80487 6.63508 6.54652\nthreshold=2.5000000000000004 0.12043646866653869 3.5000000000000004 8.4597792625427264 -0.73432211481373177 8.5000000000000018 -0.63656542347250999 -0.35267747499651753 7.5000000000000009 -0.58190388732679377 -0.028659912574114984 0.7071068124474349 4.5000000000000009 0.75405830993331013\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 4 8 -2 6 -6 -8 9 12 -11 -9 -4 -10\nright_child=2 -3 3 -5 5 -7 7 11 13 10 -12 -13 -14 -15\nleaf_value=-0.023568600985755472 -0.021197240165952203 -0.0018501450697290098 0.0085295677683235038 0.051137495840159308 0.032299882449433617 0.023860804697526206 0.044005355173345015 -0.040397268097957614 -0.0040065378425301057 -0.032583704034513931 0.057961911352186458 0.013992928297932137 -0.042298778662696102 -0.049938161047020577\nleaf_weight=43.490104824624723 19.200268887114362 451.71899735517945 21.422630836867032 7.7658042565453789 13.896466428355778 114.75873353642965 7.3522714673017608 27.672101373289479 301.05091827053548 4.4180963370163253 21.386011646187399 9.4757685377262515 8.0255801412931742 6.9755712116602799\nleaf_count=906 756 5906 516 54 155 1800 187 478 22149 222 178 58 552 120\ninternal_value=5.19327e-05 -0.00380115 0.00343939 -0.000617463 0.0112276 0.0149241 -0.00274408 -0.0141569 -0.00186117 0.0165412 0.0418217 -0.0265202 -0.00625704 -0.00520494\ninternal_weight=1058.61 495.209 563.4 371.045 192.356 173.155 58.3966 44.5001 363.279 55.2523 25.8041 37.1479 29.4482 308.026\ninternal_count=34037 6812 27225 23791 3434 2678 878 723 23737 1468 400 536 1068 22269\nis_linear=0\nshrinkage=0.05\n\n\nTree=137\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 3 2 1 12 12 12 13 4 10 13 5 2 13\nsplit_gain=7.49662 6.58096 5.84086 6.73428 5.54651 6.8499 5.49136 7.41156 6.96917 6.69268 7.60227 7.44565 5.2579 6.22003\nthreshold=0.078907978039233523 0.98248835939454404 1.992208782912438 1.6472240686416628 0.11520419728812863 0.12224019378306555 0.10082637781458172 -0.77065017158148164 11.500000000000002 0.10403138528138529 -1.9353917026821741 3.5000000000000004 -0.70026333225044357 0.60754552369044912\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 -2 -5 -6 8 -8 9 10 -1 -11 13 -12\nright_child=2 -3 -4 4 5 -7 7 -9 -10 11 12 -13 -14 -15\nleaf_value=0.042874592146567594 -0.032526978796780633 -0.03428159575565503 0.01717086762868383 0.020881789116105571 -0.044915234345313915 0.0013956195572484165 -0.062346114641886656 -0.0060038022818488775 0.06146524117250915 0.061305594666855978 -0.041499184671195184 -0.020480404613308353 -0.049332181359574401 0.066600023368244984\nleaf_weight=3.1990324771613841 13.542926721338516 24.51831828635477 60.326937446428929 35.439606685104081 7.0599925268361394 858.75328370404532 6.4490643243225341 28.211955829632643 4.5490517205944334 6.2134944052668271 1.6375405988073937 3.8865135951418779 11.05640043056337 1.7693284652195869\nleaf_count=66 4583 664 905 1495 775 20828 1095 2345 470 70 141 149 456 10\ninternal_value=0.000943836 -0.0126623 0.00223209 0.00122927 0.001772 0.000966612 -0.00435963 -0.0179412 0.0103108 8.32554e-05 -0.0182054 0.0308243 -0.0336143 0.0170147\ninternal_weight=1066.61 91.4907 975.123 914.796 901.253 865.813 66.9724 34.661 32.3114 27.7623 17.6623 10.1 14.4633 3.40687\ninternal_count=34052 5466 28586 27681 23098 21603 4802 3440 1362 892 673 219 607 151\nis_linear=0\nshrinkage=0.05\n\n\nTree=138\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 14 12 0 10 10 13 10 10 8 13 14 10\nsplit_gain=9.0822 6.39926 6.74573 12.9059 6.80218 7.26255 11.7474 6.15873 5.21591 8.79405 7.1842 4.22915 10.6535 14.1206\nthreshold=0.35770511525393117 0.77947462504345733 0.69247260929972343 0.24992060306850863 0.73012711180625955 0.40930735930735934 0.32742946708463955 1.4630009291942681 0.14158805031446545 0.23348909657320874 11.500000000000002 0.4271357933468744 0.88527816776246016 0.42280606717226438\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 5 6 7 8 -4 10 -10 12 13 -2\nright_child=11 -3 3 -5 -6 -7 -8 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.0028300414902020254 0.055932502937468577 0.0096698579812734831 -0.044469760723026941 -0.057335396126905053 -0.065136753926230459 -0.066085711299751965 0.034034647518614444 0.023383422703862927 0.041743274935342793 -0.044597473358249234 -0.010366614612051111 0.0027222412580598081 -0.0076155621689065443 0.010547778646644239\nleaf_weight=403.12417895897306 23.023495553759858 76.299456094639027 13.455315042438086 15.766496175201608 4.1524323502526377 3.7238282132893792 16.572986325365491 9.5057732227724028 8.4731458823662269 11.370543739525599 21.034170864066255 355.74937216361286 40.485475192544982 62.107069515157491\nleaf_count=20772 220 1840 2427 626 265 52 199 84 102 699 2282 3073 633 827\ninternal_value=0.000446117 -0.00374533 -0.00577856 -0.0170659 -0.00931297 -0.00582511 -0.00225701 -0.0120582 -0.0185693 -0.00918563 0.00547476 0.00552394 0.0133743 0.0233289\ninternal_weight=1064.84 583.478 507.179 104.055 88.2882 84.1358 80.4119 63.8389 54.3332 40.8779 29.5073 481.365 125.616 85.1306\ninternal_count=34101 29348 27508 6736 6110 5845 5793 5594 5510 3083 2384 4753 1680 1047\nis_linear=0\nshrinkage=0.05\n\n\nTree=139\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 0 0 10 12 11 10 0 10 13 11 9 12 6\nsplit_gain=7.79848 18.671 7.75806 14.9525 9.15186 8.38258 9.04782 6.77167 6.52548 11.9964 6.79387 6.21843 6.32165 5.92682\nthreshold=1.5000000000000002 0.35111134286233475 0.77366324840924527 0.78979857050032498 0.54803933842734343 0.16449293709486093 0.12018518518518519 0.077505217748931474 0.10135754907356449 -1.4061108993616001 0.070065725961223194 0.070846804972665336 0.10779725792516721 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 11 3 5 13 6 -3 -2 9 -9 -11 -1 -13 -4\nright_child=7 2 4 -5 -6 -7 -8 8 -10 10 -12 12 -14 -15\nleaf_value=-0.071511088119313698 0.01265383397368172 -0.026081894968107835 0.011709515182672737 0.048158074648396304 -0.028835647353166689 -0.021410070028074314 0.026079740661705902 0.03581897070480599 0.0013380768324986396 0.021086151017594462 -0.039945442381567386 0.022094917286394098 -0.031344415908473533 0.049845924198224208\nleaf_weight=8.0571900401264411 125.90308628641651 11.947665261686778 45.93183945043711 9.960916036856359 10.228933875681831 83.42445273057092 22.393765632383293 7.34353291287698 634.02933401554947 4.7112698679120504 29.653061477201845 5.5821671406738451 44.4782163745258 12.482572622364385\nleaf_count=132 4099 300 797 164 91 1247 351 569 22505 168 2840 53 593 78\ninternal_value=-0.000365942 -0.00797381 -0.000554625 -0.00782489 0.0128915 -0.0129257 0.00770777 0.00205617 6.58356e-05 -0.0188528 -0.0312185 -0.0326087 -0.0250694 0.0205607\ninternal_weight=1056.13 254.488 196.37 127.727 68.6433 117.766 34.3414 801.64 675.737 41.7079 34.3643 58.1176 50.0604 58.4144\ninternal_count=33987 3806 3028 2062 966 1898 651 30181 26082 3577 3008 778 646 875\nis_linear=0\nshrinkage=0.05\n\n\nTree=140\nnum_leaves=15\nnum_cat=0\nsplit_feature=3 8 13 13 1 9 10 7 0 2 3 13 11 4\nsplit_gain=6.44586 13.5255 9.28637 10.6118 7.71138 12.4864 6.53794 6.26735 6.21864 8.1436 11.3781 8.89461 5.72401 5.57103\nthreshold=-1.0000184455921113 32.500000000000007 -0.37871335444295678 1.0120710860654987 6.130833864212037 0.037728545846711044 0.064309346567411094 5.5000000000000009 0.68049333853542271 0.73126556468747761 -0.79381492413181876 1.2227218491770515 0.1833038021485047 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -4 5 -2 -7 -3 -8 10 -10 -12 -5 -14\nright_child=4 7 3 12 -6 6 8 -9 9 -11 11 -13 13 -15\nleaf_value=-0.0079267636777222497 -0.074038591633957609 0.0047236666714795235 0.048744434407220302 -0.042008581393124945 0.0020658117680712753 0.019738357058822181 -0.010627092267522047 -0.054902223362356707 0.046002974258906655 0.011602487970922054 -0.063724499501474977 0.021060464622823738 0.047831584958873595 -0.021809827005416421\nleaf_weight=17.863657794876417 5.8116824928438282 8.0796622380148637 33.93360684293657 4.7559041051499653 553.70618606454809 22.82430852654943 310.27762520228134 8.3248532257130119 3.671142492414222 73.749593338405248 11.470999851815575 3.5651555883232495 5.8491160573903453 4.1627247127471492\nleaf_count=2842 103 142 1814 158 7503 568 18114 318 110 1281 1028 62 49 88\ninternal_value=-0.000706325 0.0125931 0.0225931 0.033722 -0.00183812 -0.00684239 -0.0057533 -0.0269508 -0.00724331 0.00410461 -0.0245431 -0.0435618 -0.00169913 0.019525\ninternal_weight=1068.05 82.9695 66.565 48.7014 985.077 431.371 425.559 16.4045 402.735 92.4569 18.7073 15.0362 14.7677 10.0118\ninternal_count=34180 5411 4951 2109 28769 21266 21163 460 20595 2481 1200 1090 295 137\nis_linear=0\nshrinkage=0.05\n\n\nTree=141\nnum_leaves=15\nnum_cat=0\nsplit_feature=0 4 10 14 8 4 1 12 10 9 10 10 9 13\nsplit_gain=6.51523 9.5854 12.3373 8.50784 10.974 7.44984 7.31646 6.60961 5.85983 7.86949 5.76372 5.71696 5.67449 5.35732\nthreshold=0.71999515207696929 1.5000000000000002 0.11272900561262313 0.73142348189180872 44.500000000000007 13.500000000000002 1.3719071149826052 0.10082637781458172 0.158804331701528 0.4636603360598463 0.73598008385744251 0.12864214992927867 0.070846804972665336 0.80356500410247722\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -4 8 10 -3 11 -5 -10 12 -8 -2 -11\nright_child=5 6 3 4 -6 -7 7 -9 9 13 -12 -13 -14 -15\nleaf_value=-0.042943092549197676 -0.036630204951893057 -0.030351882981662921 0.0019994972994039299 0.015327063434777112 0.030096434794844792 -0.05367387400729752 0.025468209953353839 0.0002647141049024707 -0.070841317946946508 -0.048906566717809907 -0.0136840177588472 -0.044568074199966348 0.012414612237865897 0.0045734923277478317\nleaf_weight=27.153110011015087 5.118922555964672 17.403758247073711 93.162927178200334 4.3948894168715915 6.8228595284745088 3.9346324785874449 39.114979935919109 593.21833025157321 13.540392117691228 7.4649791140109327 25.418816133576911 2.2415713883674462 212.79024079596275 10.135672772245014\nleaf_count=471 172 5520 1662 52 32 339 1195 20503 185 64 281 133 3394 107\ninternal_value=0.000172925 -0.0019859 -0.012804 -0.00649507 -0.024793 0.00726492 0.000728807 0.00162764 -0.0358671 -0.0433441 0.00846596 0.0207096 0.0111004 -0.0195189\ninternal_weight=1061.92 814.653 162.675 135.522 42.3588 247.263 651.979 634.575 35.5359 31.141 243.328 41.3566 217.909 17.6007\ninternal_count=34110 29924 2573 2102 440 4186 27351 21831 408 356 3847 1328 3566 171\nis_linear=0\nshrinkage=0.05\n\n\nTree=142\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 11 11 1 11 11 3 1 14 5 6 11 1 5\nsplit_gain=7.36407 12.9641 6.4383 9.6956 8.88922 7.83148 6.04313 8.32344 7.58201 5.88752 5.40485 9.7465 5.69765 5.30231\nthreshold=0.13671566500256818 0.16091067892009897 0.15730752734423484 4.81640648841858 0.12185857541662339 0.13286447314631505 -1.7381005412235642 2.0353064537048344 0.37337387399022032 11.500000000000002 4.5000000000000009 0.15572983134335905 6.3072609901428232 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 -2 -5 -6 9 8 -8 -1 11 -7 -12 -9\nright_child=2 -3 -4 4 5 10 7 13 -10 -11 12 -13 -14 -15\nleaf_value=-0.036258142838413873 0.041439526922602471 -0.034792431061828326 -4.283045939035385e-05 -0.043842870975278031 0.049002019887495307 -0.044481606855651397 0.010609861734102157 -0.016459278037754634 -0.048439549751554623 0.060333401920616758 -0.026296328707730517 0.04281926314845852 0.06182492965607337 0.0049146032586949578\nleaf_weight=1.3389207754080406 23.172113622014876 39.120745587541023 769.52307945086432 7.7029944903915739 9.280786965799054 10.5489937546663 7.243522112828944 38.318498545981129 17.573281860662973 5.5786532611164148 2.1798025636380762 3.5274906083941451 4.0179471027804539 109.75267881622494\nleaf_count=157 1141 4010 13559 259 128 221 364 857 5411 476 165 21 77 7235\ninternal_value=-0.000953634 -0.00908876 0.00120062 0.0167777 0.00075744 0.0134362 -0.00333485 -0.0052624 -0.0314609 0.0394196 -0.00438262 -0.0212154 0.0314844 -0.00068993\ninternal_weight=1048.88 218.926 829.953 60.4301 37.258 29.555 179.806 172.888 24.8168 6.91757 20.2742 14.0765 6.19775 148.071\ninternal_count=34081 18510 15571 2012 871 612 14500 13867 5775 633 484 242 242 8092\nis_linear=0\nshrinkage=0.05\n\n\nTree=143\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 12 4 6 0 4 12 13 4 0 0 1 13 4\nsplit_gain=5.17972 7.6993 5.88998 5.41156 4.73407 7.20959 6.89845 6.70801 6.46389 14.7954 10.0915 11.7593 10.6116 11.864\nthreshold=8.2524304389953631 0.76309033109884206 4.5000000000000009 1.5000000000000002 0.71999515207696929 5.5000000000000009 0.38121896056391874 -0.34448158223808972 2.5000000000000004 0.34856411803995579 0.36320357302763279 7.723942995071412 1.0120710860654987 14.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 -3 8 -6 7 -7 9 -1 12 -12 13 -10\nright_child=1 3 -4 -5 5 6 -8 -9 10 -11 11 -13 -14 -15\nleaf_value=-0.02309700491503212 0.010916113058639522 0.011194718969660232 0.049420585576984324 -0.058767324574237147 0.008346004364144375 0.019221299900434398 0.012438750284947623 -0.057989964651239803 0.016046779235331023 8.8631862667377771e-05 -0.0048927034504728803 -0.052581569748249381 -0.0089100020559059832 -0.028857903053070941\nleaf_weight=113.61100842416636 40.214306080597453 4.7783593218773586 12.58918770146556 4.7845802837982774 192.71941842146043 3.1138108473533075 9.2943543065339309 11.235730843058263 185.6521088424106 172.14482892538945 190.87001614220571 12.893902377458287 79.794650174917479 15.059083533760711\nleaf_count=1681 325 30 84 75 2763 452 109 641 8090 2981 10551 259 1322 4725\ninternal_value=-0.000503655 0.0133545 0.0208441 -0.0260588 -0.0013925 0.00512669 -0.0203875 -0.0410746 -0.00322855 -0.00917793 0.000293707 -0.00815244 0.00643629 0.0125513\ninternal_weight=1048.76 62.3664 52.8035 9.56294 986.389 216.363 23.6439 14.3495 770.026 285.756 484.27 203.764 280.506 200.711\ninternal_count=34088 514 409 105 33574 3965 1202 1093 29609 4662 24947 10810 14137 12815\nis_linear=0\nshrinkage=0.05\n\n\nTree=144\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 3 3 5 13 10 3 10 0 8 13 11 8 6\nsplit_gain=5.17362 5.2645 6.49347 4.79728 8.33704 5.17035 4.30407 4.05729 6.39198 10.7373 5.6161 4.17484 6.47237 9.45305\nthreshold=0.044272729873579268 1.0446204841859017 0.82635962692040243 22.500000000000004 -0.31824396407007477 0.23117408906882594 1.9345448480507716 0.86240703177822864 0.39078968243383 30.500000000000004 0.95559699300162337 0.56629291394080161 41.500000000000007 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 6 -5 -6 7 11 10 -10 -9 -2 -13 -14\nright_child=3 -3 -4 4 5 -7 -8 8 9 -11 -12 12 13 -15\nleaf_value=-0.036072101779156478 -0.0022105423456930322 -0.073286994209543335 0.036659771615614727 -0.043622377494032867 0.056554774747539394 -0.033969698015624453 0.014982941352298546 -0.06686318041735978 -0.049252696144494068 0.023771733752705186 0.0036091389417759207 0.012720081386835676 0.011701847456340959 -0.026651867789037421\nleaf_weight=7.1278073843277507 752.45615244056535 3.7165774870882151 3.9684054860845208 2.464313806285646 10.397968230121473 1.1539621567935689 42.10441501319292 8.1970865405164641 6.5769857431296286 13.998448944534173 3.9397039581090203 123.7615822974476 32.356233524624258 30.112275803927332\nleaf_count=237 29228 38 23 1042 779 105 736 144 117 113 40 944 202 398\ninternal_value=-0.000500097 -0.0288823 -0.00918027 -6.32529e-05 0.0279909 0.0455261 -0.000478437 -0.00116332 -0.018206 -0.000771611 -0.0454539 -0.000550652 0.00613206 -0.00691478\ninternal_weight=1042.33 14.8128 11.0962 1027.52 14.0162 11.5519 1013.5 971.398 32.7122 20.5754 12.1368 938.686 186.23 62.4685\ninternal_count=34146 298 260 33848 1926 884 31922 31186 414 230 184 30772 1544 600\nis_linear=0\nshrinkage=0.05\n\n\nTree=145\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 12 11 8 4 4 1 5 10 2 8 9 9 7\nsplit_gain=5.45302 10.884 8.03065 12.0979 5.56014 4.82327 4.28219 4.08683 5.05787 4.52538 4.2778 3.98355 9.50793 9.76315\nthreshold=0.22527488983156077 0.17056733839577878 0.20655146208973843 14.500000000000002 2.5000000000000004 1.5000000000000002 4.3248009681701669 1.5000000000000002 0.20280062671367025 -0.30148834743280445 20.500000000000004 0.55319063900854715 0.52668805818070152 10.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 4 -1 -4 6 -5 -2 -7 -9 -10 -11 12 -3 -14\nright_child=1 11 3 5 -6 7 -8 8 9 10 -12 -13 13 -15\nleaf_value=-0.0024878973817479364 -0.041487040272101636 0.00076973819731521035 -0.083297796047525061 -0.051466915369148637 0.05029292207577394 0.034870321279844449 0.024632550260914758 -0.052349067200748556 -0.04890306895321847 -0.035518242244246705 0.035986838324380092 0.0083129340808338649 -0.0074692816388343287 -0.074920282671668087\nleaf_weight=362.85622780652557 2.7928251181729156 448.55139711291849 6.6708079645759417 4.8723807220812914 16.703937478945591 5.1086536748334792 5.9130729637108743 4.4993247559832552 2.0445708577972317 1.8647409450204486 7.8631558472188772 156.72224976017606 15.446135135134684 7.4352573482319704\nleaf_count=23404 52 7388 147 124 156 64 138 382 248 112 253 1292 148 122\ninternal_value=-4.5045e-05 0.00275859 -0.00467007 -0.0279378 0.0342038 -0.0113306 0.00133254 -0.000292946 -0.0127125 0.00534881 0.0202469 0.00143424 -0.000865257 -0.0316067\ninternal_weight=1049.34 653.565 395.78 32.9236 25.4098 26.2528 8.7059 21.3804 16.2718 11.7725 9.7279 628.155 471.433 22.8814\ninternal_count=34030 9296 24734 1330 346 1183 190 1059 995 613 365 8950 7658 270\nis_linear=0\nshrinkage=0.05\n\n\nTree=146\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 9 4 2 3 8 13 13 2 1 12 3 6 4\nsplit_gain=4.8913 5.36721 5.23166 6.07848 4.83295 10.3396 6.65169 7.08301 6.44121 7.25841 6.72896 9.50854 7.17055 5.11799\nthreshold=0.43007633641122361 0.48753231203628372 3.5000000000000004 1.0000000000000022 -0.46303995471989462 73.500000000000014 1.134727103335295 0.69510616476850862 0.027188914327749458 5.8755407333374032 0.51508605832750265 0.9983310878911571 3.5000000000000004 13.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 -3 6 8 7 -2 9 -6 12 -12 -11 -14\nright_child=4 2 -4 -5 5 -7 -8 -9 -10 10 11 -13 13 -15\nleaf_value=-0.0014700396286751176 0.0097834409155217704 0.015793451155014623 -0.048067982544035513 -0.039569940440080512 -0.05004663117435898 -0.060745875150454813 -0.019208991195799545 0.050624754787445991 0.0074536029938422612 -0.024550843476181781 -0.054994681736822454 0.011914191431690755 0.034909732840692202 -0.016295335243323411\nleaf_weight=641.66277368278497 27.027813457534648 11.868291292339569 11.330443440238012 7.0933403871022156 8.7586342081194726 5.3684742050245395 8.940647882409392 17.174925527069718 236.83035712566925 10.048890647711231 11.006151142995806 8.9860700806602818 20.125688230386007 5.6915262655820689\nleaf_count=30017 433 94 193 89 198 48 53 113 1919 196 362 60 183 94\ninternal_value=6.026e-05 -0.0024585 -0.0226804 -0.00586216 0.00475594 0.00231743 0.0185315 0.0264212 0.00363766 -0.0102488 -0.00323799 -0.0257857 0.0096889 0.0234346\ninternal_weight=1031.91 671.955 30.2921 18.9616 359.959 306.816 53.1434 44.2027 301.447 64.617 55.8583 19.9922 35.8661 25.8172\ninternal_count=34052 30393 376 183 3659 3060 599 546 3012 1093 895 422 473 277\nis_linear=0\nshrinkage=0.05\n\n\nTree=147\nnum_leaves=15\nnum_cat=0\nsplit_feature=12 9 5 9 1 11 12 0 12 12 6 0 14 1\nsplit_gain=6.0405 7.24109 5.13612 6.11173 7.94723 8.07774 5.58426 5.04076 6.46325 10.463 9.78829 10.0542 7.87405 7.79673\nthreshold=0.76309033109884206 0.40090617732792916 17.500000000000004 0.30014869815283679 2.2098360061645512 0.13629513085848707 0.14083392609291573 0.78438036099062902 0.19488605834408215 0.17993568848336086 6.5000000000000009 0.10775215889566821 0.74127221739390092 6.0601730346679696\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 6 -6 -4 8 9 10 13 -12 -11 -1\nright_child=-2 -3 3 -5 5 -7 -8 -9 -10 12 11 -13 -14 -15\nleaf_value=-0.015206096043647453 -0.024159858694300998 0.0061714683334257951 0.021081235661159548 -0.053738222945676331 -0.040085828744138879 0.049616543862095269 -0.037049738408996967 0.0081087851264108942 -0.012327748300176829 0.004572017622783915 0.034451012695873563 -0.0061887906606439506 0.049277633144091837 0.0043375868228349322\nleaf_weight=159.58206747193617 24.316837366786785 367.68690888083074 7.4927595170574914 2.0565220780554219 2.1106639396384734 19.061058715746185 7.1088138023551437 78.213486710810685 206.70302339576301 25.083116352539946 28.519639389392978 30.885984931290295 15.468144220940301 74.258959596219938\nleaf_count=6179 266 3479 1989 121 550 1251 1600 1659 5147 1010 2386 7061 281 1130\ninternal_value=-3.28222e-05 0.000562998 -0.00258254 0.0150663 0.0203755 0.0392692 -0.00778073 -0.00368421 -0.00540255 -0.00109013 -0.00440358 0.013569 0.0224004 -0.00900773\ninternal_weight=1048.55 1024.23 656.544 37.8298 35.7733 21.1717 14.6016 618.714 540.501 333.798 293.247 59.4056 40.5513 233.841\ninternal_count=34109 33843 30364 5511 5390 1801 3589 24853 23194 18047 16756 9447 1291 7309\nis_linear=0\nshrinkage=0.05\n\n\nTree=148\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 0 0 12 13 13 11 1 0 11 12 2 10\nsplit_gain=9.73583 19.5846 8.85982 8.54186 7.19732 6.2581 9.26628 5.74992 5.31756 5.8725 5.15485 5.57028 4.51119 4.29949\nthreshold=0.078907978039233523 1.5000000000000002 0.41325796971748735 0.53664329701096924 0.10082637781458172 -1.4653257879145714 -0.49248288101553256 0.076998033793758983 0.6179969608783723 0.90109853348056423 0.11851485802112142 0.12570481843286005 1.992208782912438 0.10652482269503548\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -4 5 -3 7 13 9 -2 -6 -12 -10 -7\nright_child=8 4 3 -5 10 6 -8 -9 12 -11 11 -13 -14 -15\nleaf_value=-0.077951612540652152 -0.063138334285369413 0.046903301338450581 0.042349864036002161 -0.043568621298582973 -0.044369766809324807 -0.028787090864192878 0.025477209567103615 -0.066782746446678795 0.001209592415622264 0.039042584979835593 0.024075498003720761 -0.023383613849068719 0.015566538965320544 0.062401219925915075\nleaf_weight=13.612010441487657 4.9235127925621773 9.9301000095147156 3.3162238185759625 8.1246357419295219 10.315565080192753 2.116086137051751 13.730931660855278 5.1443872833551723 876.649198512363 1.1010964206652714 9.8655157290886581 13.367320037363241 57.377431973407511 1.2476273484062401\nleaf_count=183 2286 385 57 155 1399 192 229 532 25418 16 674 1530 889 24\ninternal_value=0.000267793 -0.0152778 -0.051963 -0.0172622 -0.000723389 0.0159236 0.000667653 -0.0378401 0.0017835 -0.0415637 -0.0166849 -0.00306886 0.00210725 0.0115845\ninternal_weight=1030.82 90.7704 25.0529 11.4409 65.7175 32.1691 22.239 8.5081 940.051 6.02461 33.5484 23.2328 934.027 3.36371\ninternal_count=33969 5360 395 212 4965 1362 977 748 28609 2302 3603 2204 26307 216\nis_linear=0\nshrinkage=0.05\n\n\nTree=149\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 8 9 3 10 10 12 1 1 1 3 1 9 10\nsplit_gain=6.54366 7.99231 7.74423 6.92135 6.44251 7.1766 4.61417 4.96741 7.79073 10.1294 7.76325 6.75044 10.6662 10.5928\nthreshold=0.86240703177822864 30.500000000000004 0.62109193738564106 0.80139994038844931 1.0000000180025095e-35 0.029776858282843214 0.76309033109884206 6.1893992424011239 5.1241953372955331 5.2616169452667245 1.6668156375686918 6.5163369178771982 0.047794029506707726 0.24264069264069268\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 4 -3 -1 -6 7 8 -7 -10 -11 13 -13 -9\nright_child=1 3 -4 -5 5 6 -8 11 9 10 -12 12 -14 -15\nleaf_value=0.066638397702129912 -0.057402366308907132 -0.028068855407977496 0.014756557423036104 0.026814134761967662 -0.047362670477643518 0.0014586530071661438 -0.038910195376757974 0.038602593136863317 -0.04313280967185551 -0.011639953136836613 0.022281860433265757 -0.073412794957624566 0.0020793498361524119 0.00025172489543699012\nleaf_weight=2.5719467252492896 9.6281276418594661 12.079036314273255 92.080096001154743 9.2609376342734304 7.1464412089408134 283.57074678862773 6.6419206354767075 32.19267539086286 21.804753815886215 126.95702985818934 18.486780976847513 3.7385314684361211 372.27898119036399 39.380664407537552\nleaf_count=4 176 209 628 51 136 21597 73 673 666 3534 339 23 5241 662\ninternal_value=0.000310325 -0.0219611 0.00101645 -0.00411708 -0.000381226 -0.000642953 -0.00022218 0.000106783 -0.00359382 -0.0121181 -0.00720558 0.00383378 0.00113604 0.0177956\ninternal_weight=1037.82 30.9681 1006.85 21.34 914.77 912.199 905.052 898.41 450.819 167.249 145.444 447.591 376.018 71.5733\ninternal_count=34012 436 33576 260 32948 32944 32808 32735 26136 4539 3873 6599 5264 1335\nis_linear=0\nshrinkage=0.05\n\n\nTree=150\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 0 9 12 0 2 13 1 2 0 3 2 1 13\nsplit_gain=5.5363 12.3293 10.5555 12.1302 8.00823 10.6259 10.9067 8.92227 9.56989 8.26014 10.0172 7.50694 7.36728 7.18203\nthreshold=1.5000000000000002 0.26477995581497177 0.078907978039233523 0.25299979771225373 0.080737732012600816 -0.3306222700042753 0.87952391891555892 1.9646461606025698 1.0372423979668091 0.68049333853542271 0.67240384858860802 0.71416809583600083 3.1265689134597783 -0.77065017158148164\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -4 5 -2 11 -6 12 10 13 -7 -9 -10\nright_child=4 -3 3 -5 7 6 -8 8 9 -11 -12 -13 -14 -15\nleaf_value=-0.066775649950309851 -0.0020682823227663526 -0.0014515828801336546 0.0097869765224902691 -0.049242675280816661 -0.029814806749327245 0.046708772904221853 -0.0038044023263541014 -0.036053373947221704 -0.046210423036766583 0.025123664326751394 -0.015269960069996738 -0.0068250699862750083 -0.0017373831633321294 0.024580218754398114\nleaf_weight=12.751178708858786 63.955733894981677 202.80463359759597 20.794344239868224 13.64813649188727 24.178975527483999 41.879652437884943 21.392845813708846 15.190148708457853 3.0158000086667007 63.250314156888635 43.499400433560368 7.4509477510582647 466.04436980401078 39.075134042039281\nleaf_count=141 3281 3265 290 132 6628 666 179 3602 140 737 623 103 13719 577\ninternal_value=-0.000392453 -0.00686099 -0.0295943 -0.0143334 0.0016634 0.0127185 0.0259328 -0.000628257 0.000537239 0.0115735 0.0013856 0.038648 -0.00289308 0.0185534\ninternal_weight=1038.93 249.998 47.1937 34.4425 788.933 134.679 70.7234 654.254 630.075 148.841 85.5903 49.3306 481.235 42.0909\ninternal_count=34083 3828 563 422 30255 4229 948 26026 19398 2077 1340 769 17321 717\nis_linear=0\nshrinkage=0.05\n\n\nTree=151\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 9 14 8 11 11 2 9 3 0 14 8 14 9\nsplit_gain=8.70072 5.64902 13.8544 9.50514 4.37915 5.20323 4.99924 4.11646 3.88713 5.96384 4.7578 7.1142 4.33651 3.94672\nthreshold=0.40090617732792916 0.44731785518021011 0.88677025781813368 18.500000000000004 0.45026034298148981 0.47815812583848105 1.4637192056758757 0.38985550194288415 0.16816061193025691 0.13680055063109184 0.74852742620451984 34.500000000000007 0.95323867913113813 0.47091134847787214\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 2 -2 -4 -3 -6 8 -1 9 -7 11 -11 -12 -10\nright_child=1 4 3 -5 5 6 -8 -9 13 10 12 -13 -14 -15\nleaf_value=-0.002795311155349825 0.024493818479609627 -0.008770039855962761 0.007300543488769508 -0.070871346626804374 0.031079241614316768 -0.03502762039858922 0.022864778434429467 -0.026751367294540793 -0.038264183124543466 -0.053336307741899791 0.035078731220650486 0.016947490227357278 -0.010128926555908004 -0.0019563809509162205\nleaf_weight=630.07577353158194 81.050750429509208 65.811172306828666 8.4828703202656452 5.9294811249710611 17.681325473822653 5.1899820743128648 27.890393887413666 17.478997269179672 6.9780662929406381 4.3850755812600246 28.359849031781778 9.9408905331511033 6.0033878125250331 123.56200639647432\nleaf_count=30215 721 744 75 141 145 66 172 260 85 107 238 145 76 1030\ninternal_value=7.3181e-05 0.0059513 0.0164608 -0.0273723 0.00251555 0.00576888 0.00352519 -0.00348219 0.000492855 0.0116684 0.0172506 -0.00664228 0.0271177 -0.00417337\ninternal_weight=1038.82 391.265 95.4631 14.4124 295.802 229.991 212.31 647.555 184.419 53.8792 48.6892 14.326 34.3632 130.54\ninternal_count=34220 3745 937 216 2808 2064 1919 30475 1747 632 566 252 314 1115\nis_linear=0\nshrinkage=0.05\n\n\nTree=152\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 10 10 6 12 1 1 3 3 0 3 1 1 9\nsplit_gain=6.01311 3.97152 9.95164 8.76284 6.85204 4.98389 7.98318 4.34496 4.28438 4.67892 4.232 4.11085 5.77073 4.3598\nthreshold=8.2524304389953631 0.10135754907356449 0.088748241912798884 9.5000000000000018 0.12994831829542161 7.1009039878845224 5.7881557941436776 2.6316286664499118 1.2501037838947984 0.94501246039486408 1.1361560482789164 4.81640648841858 4.0650434494018564 0.057128223454471175\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 5 6 11 -3 9 -4 -6 12 13 -1\nright_child=-2 7 8 -5 10 -7 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.057849883575985361 0.013969416066782642 -0.00097913317694022905 -0.055413003008188459 0.033963544184543039 -0.055542934013159496 -0.048979765953052957 0.031982583269673674 0.039556115568488349 -0.00074599895646619671 0.041741187577644695 -0.0045477568828779106 -0.043815165883029736 0.034738821416657094 0.00036684983447248258\nleaf_weight=3.7469602229903165 63.767736751819029 849.58430814265375 17.450306259674115 12.841223817969874 9.9721949910745042 3.6647400758229187 16.24753127223812 5.6629526809556401 5.7687734929786521 0.62262733711395313 6.771054141514468 6.2211721035127985 6.8213547284467486 9.9326504131786333\nleaf_count=186 527 28338 1749 1401 368 57 171 110 128 9 104 178 153 520\ninternal_value=-0.000775467 -0.00177328 -0.0111471 -0.00216037 -0.00989347 0.000164798 0.00537479 -0.000664849 -0.0386329 -0.0500529 -0.0363388 -0.0113733 5.13554e-05 -0.0184338\ninternal_weight=1019.08 955.308 100.061 76.2189 63.3777 46.6344 42.9697 855.247 23.8417 18.0729 16.7432 26.7221 20.501 13.6796\ninternal_count=33999 33472 5024 3138 1737 1265 1208 28448 1886 1758 472 1037 859 706\nis_linear=0\nshrinkage=0.05\n\n\nTree=153\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 13 2 9 11 3 3 10 2 2 11 4 5 11\nsplit_gain=7.95191 5.56614 6.39886 6.27738 8.70923 6.97417 11.9749 7.37925 5.53198 5.51414 5.43376 6.77205 5.07662 4.43611\nthreshold=0.10045871559633029 -0.81745190390253553 -0.70703784408564696 0.066175762005596286 0.13979671616874659 1.2135643508474616 -0.043823199286241683 0.079077644606565925 -0.11376473053622037 1.0733414474692362 0.08687741371186751 6.5000000000000009 3.5000000000000004 0.13190698099906675\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 12 4 -4 6 7 8 -5 -7 11 -1 -3 -2\nright_child=13 2 3 5 -6 9 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.043747424620425385 0.012781232893828771 0.027141512838101706 -0.075071191841546384 -0.042305558452854365 0.037304291824684123 -0.027063324957478908 -0.059530869163042933 -0.052372179302834047 0.04614222834653519 0.033927900185877762 0.020138965713236798 0.030186372079650265 -0.040885030457017142 0.00032078083920289609\nleaf_weight=8.7100477657804749 76.970242566758316 9.5602303691048274 10.522782008876673 1.4086354552127884 1.4034100351855148 5.2629551420395719 16.214465064578697 2.9432122568268815 7.2504902218861336 8.2256823678035271 18.894410886354308 3.7124117838575321 2.7418260178310438 852.11927077852852\nleaf_count=254 5339 154 422 164 10 91 762 633 142 100 1277 387 500 23957\ninternal_value=-6.07622e-05 -0.0136189 -0.0218045 -0.0292149 -0.0599845 -0.0191228 -0.033054 0.00573566 0.0288636 0.00990512 0.00365487 -0.0210497 0.0100463 0.0013657\ninternal_weight=1025.94 96.8506 65.5337 53.2316 11.9262 41.3054 27.8168 11.6023 8.65913 13.4886 31.3169 12.4225 12.3021 929.09\ninternal_count=34192 4896 2978 2324 432 1892 1701 939 306 191 1918 641 654 29296\nis_linear=0\nshrinkage=0.05\n\n\nTree=154\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 2 8 10 12 12 5 1 2 14 9 9 4 4\nsplit_gain=5.51797 5.44625 6.02343 6.65013 5.89749 7.15194 8.14171 7.20535 5.64158 5.3397 5.50522 4.67589 4.46447 4.45229\nthreshold=-0.71148459116722751 -1.4103809358165493 20.500000000000004 0.088748241912798884 0.12376771494778609 0.169348039313127 1.5000000000000002 3.5973261594772343 -1.9338071638265355 0.93244939128669302 0.20238627472324786 0.53930446639708307 20.500000000000004 13.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 9 -4 -5 7 -7 -6 -1 -3 -11 -8 -2 -9\nright_child=12 2 3 4 5 6 11 13 -10 10 -12 -13 -14 -15\nleaf_value=-0.040792711932903951 0.00097203606740513 -0.048773736105906122 0.031517394093339028 -0.053843774210787038 -0.039815511418605148 0.015823054777765005 -0.039441844020991323 0.058795100825338266 0.01772473338177117 0.059690350684249695 -0.034242498370958649 0.0061430092318409107 0.043472356622481501 -0.015312932164904173\nleaf_weight=3.9138307941029806 887.6820322425101 18.423868392663284 8.2522119165732857 7.1170188425421648 2.6136259640679773 12.710812904442717 26.173635077815561 7.8218435782327997 25.316378074119712 1.9028055799717538 2.4214732359105255 6.7837479698937377 5.223270643495197 2.1700376646876975\nleaf_count=394 25144 1695 506 919 1294 146 1182 239 1325 76 133 128 457 368\ninternal_value=-0.00011485 -0.00988351 -0.0155812 -0.00838928 -0.0138243 -0.00811077 -0.0172926 0.02398 0.00879922 -0.0375299 0.0105385 -0.0301543 0.00126901 0.0427714\ninternal_weight=1018.53 125.621 96.3911 73.6429 65.3907 58.2737 45.6682 12.6055 29.2302 22.7481 4.32428 32.9574 892.905 9.99188\ninternal_count=34006 8405 6686 4782 4276 3357 1456 1901 1719 1904 209 1310 25601 607\nis_linear=0\nshrinkage=0.05\n\n\nTree=155\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 3 13 9 2 14 14 8 14 1 1 1 2 2\nsplit_gain=5.79612 7.90092 9.03164 7.08742 6.57822 6.7683 6.28345 5.74045 5.70862 5.3654 9.12147 5.31105 5.19629 5.16633\nthreshold=0.078907978039233523 1.0000612945049567 0.33548239013603998 0.052192916052207806 0.70710678118654757 0.46321428429942807 0.4265400345375237 23.500000000000004 0.50924783191920786 1.750018358230591 1.4491258263587954 0.6179969608783723 0.29151341194208508 1.0000000180025095e-35\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 13 8 6 -6 -3 -5 10 -4 -2 -9 -1\nright_child=11 7 9 4 5 -7 -8 12 -10 -11 -12 -13 -14 -15\nleaf_value=0.041313668451520597 -0.044655384789171158 -0.064800200106324382 -0.049319704494111355 -0.055010115896003464 -0.0047797407229778618 -0.044747579065084919 0.062752790170181053 -0.044269780774916133 0.014549044836372843 -0.049990543305215088 0.062187251708452233 0.0013897411717068999 0.015747704967230023 -0.021238410155509833\nleaf_weight=16.475366160942944 5.3057984150909752 9.5489578446722607 2.6470309806863979 8.5219429566168419 6.6018946323893015 3.0326810079204725 5.9147700013854774 5.9105739054357382 3.7623603910051306 12.284211909260192 2.7071594570588786 924.48124740812455 6.750760853959946 3.4104652253809027\nleaf_count=405 2337 245 274 2011 187 272 72 247 1066 341 44 26234 107 140\ninternal_value=-8.38167e-05 -0.0123063 -0.00335302 0.00802386 -0.00823281 0.0131204 0.0294145 -0.0373734 -0.0342145 -0.0329115 0.00797388 0.00107929 -0.0134594 0.0300826\ninternal_weight=1017.36 87.5682 65.3579 47.7195 27.8336 15.5493 12.5167 22.2103 12.2843 17.6384 5.35419 929.787 12.6613 19.8858\ninternal_count=33982 5411 4812 4153 3608 531 259 599 3077 659 318 28571 354 545\nis_linear=0\nshrinkage=0.05\n\n\nTree=156\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 6 4 1 2 5 1 4 2 12 13 5 11 8\nsplit_gain=4.47221 9.01619 6.71964 6.30064 4.92501 4.20753 5.55372 8.27325 8.22058 7.56446 8.44053 5.08682 6.6748 5.24822\nthreshold=0.61813055473851874 5.5000000000000009 12.500000000000002 8.3477363586425799 0.73126556468747761 5.5000000000000009 1.3719071149826052 1.5000000000000002 1.5610522555580648 0.26016240578361016 -0.067632415460588013 4.5000000000000009 0.09665283648278454 25.500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 3 4 -2 -3 11 8 9 -7 10 -8 13 -13 -1\nright_child=1 2 -4 -5 -6 6 7 -9 -10 -11 -12 12 -14 -15\nleaf_value=0.00088616339101434908 0.015739125933396325 -0.057637698340822031 0.022986575692783399 -0.013451892497247776 0.01406714748861642 -0.048457879083360265 -0.052202931209163889 0.0083807325742302607 0.014010845706981257 -0.059954510920144116 0.010473004860694336 0.027113115957295982 -0.02567845093204222 -0.0096912454802215841\nleaf_weight=337.95780966188977 122.69664039695635 10.341506768949332 4.5547533100470892 20.943480812595226 2.6025617411360136 11.158438025007856 6.0829959474504021 234.8597628897478 8.6586296666791878 7.1129449419677249 21.329149077879265 6.0399849284221991 45.60455193695816 178.46997442315478\nleaf_count=6425 934 241 88 147 29 5287 172 14054 227 93 193 138 1780 4229\ninternal_value=-0.000360883 0.00725526 -0.0256955 0.0114193 -0.0432457 -0.00179925 0.00310309 0.00500905 -0.0218017 -0.0175174 -0.00478316 -0.00429607 -0.0191066 -0.0027809\ninternal_weight=1018.41 161.139 17.4988 143.64 12.9441 857.274 289.202 269.385 19.8171 34.5251 27.4121 568.072 51.6445 516.428\ninternal_count=34037 1439 358 1081 270 32598 20026 14512 5514 458 365 12572 1918 10654\nis_linear=0\nshrinkage=0.05\n\n\nTree=157\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 13 3 7 1 3 9 9 7 3 1 9 0 0\nsplit_gain=5.6762 5.44317 5.77069 8.29533 5.86694 7.10236 6.20013 5.76161 9.35632 5.61325 5.49356 5.2084 6.67807 7.65011\nthreshold=0.58595927517517843 -1.4274874728908729 -1.7381005412235642 9.5000000000000018 0.6179969608783723 0.20027498381202996 0.064824694063463792 0.49756835111409253 12.500000000000002 -0.64598341937415527 7.6704530715942392 0.40090617732792916 0.2744976418651715 0.12709845052112426\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 3 -3 5 -4 -6 11 -9 -1 -11 -8 13 -13\nright_child=-2 2 4 -5 6 -7 7 8 -10 10 -12 12 -14 -15\nleaf_value=-0.015594774263386527 0.008852702673595822 -0.041752922091469316 -0.061060275133603008 0.044559766317429146 -0.023370147093701027 0.041539754273912016 -0.0032665479108049157 -0.0050944922764678763 -0.044324244061168236 0.031355553959778334 -0.040866701131102003 0.011010891231914129 0.012129641185935174 -0.03514448566527334\nleaf_weight=10.424415698027586 128.38755871227477 2.7005281180900047 7.3586244693124154 11.891846428192364 35.858686146970285 1.362109513942414 550.29755998279506 60.953271626145579 19.393963468493894 29.142929924364584 2.031110791489481 17.047505976166576 128.09105834452203 17.147566430736333\nleaf_count=938 1047 132 3001 723 1327 70 23347 581 334 971 25 129 1277 250\ninternal_value=-0.000937165 -0.00235188 -0.00322042 0.026933 -0.00377729 -0.0424102 -0.00332017 -0.00238515 -0.0149921 0.0150972 0.0255261 -0.00094464 0.00690053 -0.0124753\ninternal_weight=1022.09 893.701 852.103 14.5924 837.51 8.72073 828.79 792.931 80.3472 41.5985 31.174 712.584 162.286 34.1951\ninternal_count=34152 33105 31171 855 30316 3071 27245 25918 915 1934 996 25003 1656 379\nis_linear=0\nshrinkage=0.05\n\n\nTree=158\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 1 13 11 12 3 7 11 12 1 9 5 11\nsplit_gain=4.59439 9.77042 7.33831 10.9623 6.80866 7.76333 6.27561 9.29962 7.65441 7.91024 12.62 7.24529 5.95724 8.88971\nthreshold=2.5000000000000004 0.11982348494640736 5.9661550521850595 -0.17027254964923733 0.13190698099906675 0.14808445788931066 1.0446204841859017 12.500000000000002 0.17042593809859161 0.26016240578361016 7.0667347908020028 0.28151257257619622 1.5000000000000002 0.72946590788707044\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -4 5 -2 7 12 -8 10 11 -10 13 -3\nright_child=4 6 3 -5 -6 -7 8 -9 9 -11 -12 -13 -14 -15\nleaf_value=-0.042352178405596436 0.010077629442541555 -0.024733955523392095 0.036436792373447754 -0.039279580238803569 0.00091876884352537166 0.054275150180985145 0.024703224416144216 -0.025378579091470913 -0.015269996386760657 0.0084175674797418287 -0.052597280502929777 0.030557248281626839 0.0038158753835482828 0.03220902480867259\nleaf_weight=25.708430426959239 68.287051749946841 42.998531879333314 8.8300337162800151 8.3060150626115483 462.82911832259924 10.791690317229948 40.017631816299399 50.608103252016008 23.526131761725992 101.32008186070016 14.929240028490311 12.303630008886101 141.29081411232619 7.2286242919508359\nleaf_count=649 7493 1222 92 121 19458 352 504 625 417 1024 193 176 1613 94\ninternal_value=0.000102556 -0.00347188 -0.0259673 -0.000405903 0.0032499 0.0167116 -0.00119771 -0.00653981 0.00553468 0.000362296 -0.0155688 0.00086909 -0.00145575 -0.01607\ninternal_weight=1018.98 477.067 42.8445 17.136 541.908 79.0787 434.223 242.126 192.097 152.079 50.759 35.8298 191.518 50.2272\ninternal_count=34033 6730 862 213 27303 7845 5868 3554 2314 1810 786 593 2929 1316\nis_linear=0\nshrinkage=0.05\n\n\nTree=159\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 3 11 0 12 3 3 3 11 6 13 9 12 2\nsplit_gain=4.66037 5.30955 4.04274 8.64027 6.43349 7.97976 12.9317 10.3363 6.84418 10.522 6.82186 6.70352 9.14181 8.86035\nthreshold=14.500000000000002 2.6316286664499118 0.5538054284498054 0.71999515207696929 0.2199457879862228 0.40026107269265482 -0.88828135172906986 0.50270858258986306 0.25428458928435943 2.5000000000000004 0.27731332837924033 0.13963014728278042 0.12422395354678313 1.7586767363381492\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 8 6 -6 -7 10 -10 -1 12 -12 -13\nright_child=-2 -3 -4 -5 5 7 -8 -9 9 -11 11 13 -14 -15\nleaf_value=0.0033928176986878309 -0.015261381216550615 0.043010474747913932 0.0053723313391770833 0.0073532627095770605 0.0075263920758809472 0.033929511507747312 -0.038218855757705124 -0.0099600629237253015 -0.0084695785995749871 0.038601265327004232 -0.0013535340343250322 -0.0086717708942181896 -0.039132157246169801 0.037012329589540822\nleaf_weight=153.30245757583361 53.404628003969265 6.0518833606038234 220.80432444380131 173.79463773855241 21.13901163610717 14.245717029203659 53.603899037952942 111.51177013809502 21.971191452117637 24.089248891628813 27.412391625686723 75.137274192522455 37.759576628708601 11.33620872866595\nleaf_count=11323 8374 110 1985 3401 589 178 1320 1722 322 284 423 1458 2362 124\ninternal_value=-0.00102835 -0.000214875 -0.000536491 -0.00234024 -0.00540269 -0.0125241 -0.0253525 -0.00475983 -0.00131081 0.0164452 -0.00403773 -0.0115225 -0.0235024 -0.00232817\ninternal_weight=1005.56 952.16 946.108 725.303 551.509 200.5 74.7429 125.757 351.008 46.0604 304.948 151.645 65.172 86.4735\ninternal_count=33975 25601 25491 23506 20105 3809 1909 1900 16296 606 15690 4367 2785 1582\nis_linear=0\nshrinkage=0.05\n\n\nTree=160\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 5 6 14 11 2 4 6 12 4 2 4 14 14\nsplit_gain=4.21896 3.96202 6.09514 5.44811 3.45407 8.16213 9.47737 8.31284 7.83894 5.54337 5.76463 5.39881 6.7424 7.2278\nthreshold=0.75381749868392955 22.500000000000004 20.500000000000004 0.51317108302686154 0.60405659051977689 -0.18458186875235524 5.5000000000000009 4.5000000000000009 0.56553704453902076 5.5000000000000009 -0.67291701482661315 1.5000000000000002 0.95059642799092414 0.9056780897954958\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 4 -3 8 9 7 11 -2 10 -6 12 13 -7\nright_child=1 3 -4 -5 5 6 -8 -9 -10 -11 -12 -13 -14 -15\nleaf_value=-0.027939188080527252 -0.00046852793147783892 0.067401106047943041 -0.043866476103949642 -0.0051563316988792584 0.015604223326388606 -0.0069976542181333749 0.036904656543592942 -0.043654919565776135 -0.036687121165904252 -0.050130559508598821 -0.045836441914293466 0.017038439350841705 -0.050031784631598124 0.048011019364222481\nleaf_weight=12.512332992182563 799.82471132773208 4.147793831456509 6.9024173192083245 4.9372245153899712 12.450676745735111 32.018853668821976 27.738695608451962 7.2471008757129303 14.224473860696888 8.3348453298676741 4.3325279913842705 68.757246446562931 6.0748892487026742 6.2942992206662884\nleaf_count=4105 25462 232 1893 712 140 415 178 85 164 235 63 378 41 30\ninternal_value=-0.000186633 0.000186954 -0.000129217 0.0313686 0.000220636 0.0066101 0.0110929 0.00489084 -0.00114564 -0.0192395 -0.0019421 0.00835547 -0.00517355 0.00303085\ninternal_weight=1015.8 1003.29 994.201 9.08502 987.298 173.249 148.131 120.392 814.049 25.1181 16.7832 113.145 44.388 38.3132\ninternal_count=34133 30028 29084 944 27191 1565 1127 949 25626 438 203 864 486 445\nis_linear=0\nshrinkage=0.05\n\n\nTree=161\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 13 9 12 7 3 3 2 10 1 9 10 2 2\nsplit_gain=6.20789 7.59886 7.0062 12.5884 11.8665 7.33774 8.90602 5.63602 5.8985 5.59217 7.23505 5.44044 5.16981 7.74181\nthreshold=0.2208547266686802 1.1020406375351024 0.18560323472877255 0.2199457879862228 16.500000000000004 0.60489904380701975 0.76111247319296804 -0.77669855346550165 0.18655367231638423 6.0601730346679696 0.36355675436704565 0.19744964262508125 -0.35060192286674868 -0.099602736678978523\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 5 4 -4 -3 -7 -5 -9 11 -11 12 -1 -14\nright_child=-2 2 3 7 -6 6 -8 8 -10 10 -12 -13 13 -15\nleaf_value=-0.019835315955837943 0.0031900010743923096 -0.057454046619736132 -0.02301186708634435 0.021925358125459326 0.03900111603084383 0.046619340092075218 -0.031322747950362433 -0.022381076724983923 -0.081205310008251222 0.0046824072455816145 0.05256282239768334 -0.037879084183534459 0.039908490591241824 -0.0025453202758757469\nleaf_weight=37.325850920085941 650.4081655043592 17.010966530709993 11.914579572505319 3.3426229981705573 18.499954723753035 4.0738394626387189 12.425930022029204 14.079387846344618 5.9773679291829458 81.409145141471527 7.7716905944980672 11.284364769369857 10.649669115489813 136.372406087532\nleaf_count=8851 10593 798 129 34 148 66 342 225 87 1969 70 1402 1871 7585\ninternal_value=0.000244562 -0.00489812 -0.0177113 -0.00630401 0.014749 -0.0353166 -0.0105143 -0.0331516 -0.042936 -0.000921315 0.00939155 -0.00564639 -0.00347963 0.000778694\ninternal_weight=1022.55 372.138 87.3246 53.8139 30.4145 33.5107 16.4998 23.3994 20.0568 284.813 89.1808 195.632 184.348 147.022\ninternal_count=34170 23577 1829 623 277 1206 408 346 312 21748 2039 19709 18307 9456\nis_linear=0\nshrinkage=0.05\n\n\nTree=162\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 13 3 11 8 4 14 11 1 2 14 8 11 14\nsplit_gain=5.29733 7.98454 9.9938 7.38596 6.87737 9.02295 8.11753 6.44665 6.00194 7.65127 9.2395 11.2957 6.31304 5.52087\nthreshold=2.5000000000000004 -0.97342025210221095 -1.2019034506619326 0.08687741371186751 48.500000000000007 3.5000000000000004 0.87333819865304119 0.60405659051977689 7.7722408771514901 0.34469705618441188 0.64632445513156267 19.500000000000004 0.24770048612669873 0.98620005756189977\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 -2 -1 5 6 8 -8 9 12 -11 -12 -3 -5\nright_child=1 4 -4 13 -6 -7 7 -9 -10 10 11 -13 -14 -15\nleaf_value=-0.039989770872082289 -0.030619676293926257 0.0064341366428993787 0.028840447524307478 -0.0023730918782177823 -0.021083300793989571 -0.0016135413715444029 0.044959704856829354 -0.0093473143310924458 -0.035854077254976344 0.012054078796194918 0.011319686765770429 -0.072789796630362405 0.051791762396966548 -0.036835387386988425\nleaf_weight=13.232664202390877 7.4063835899214601 15.810086825487817 48.410695963613762 412.18642000742693 32.58642076770775 361.27832385537477 27.224488652816945 6.3961843550205222 6.8931652165483674 32.885735708099674 6.9896955185104144 7.6320892646253924 14.523632113530764 10.967726260423659\nleaf_count=280 1125 500 3157 6424 528 20476 249 40 89 751 102 193 133 100\ninternal_value=-0.000413763 0.00276699 0.0205506 -0.00455095 0.000792824 0.00231987 0.0142395 0.0346565 0.00582878 0.00992791 -0.00266816 -0.0344319 0.029111 -0.00335106\ninternal_weight=1004.42 568.037 55.8171 436.387 512.22 479.633 118.355 33.6207 84.7344 77.8412 47.5075 14.6218 30.3337 423.154\ninternal_count=34147 27343 4282 6804 23061 22533 2057 289 1768 1679 1046 295 633 6524\nis_linear=0\nshrinkage=0.05\n\n\nTree=163\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 9 9 5 9 8 5 1 13 0 5 9 5 8\nsplit_gain=5.479 7.32367 7.47213 9.03978 7.24702 7.0248 6.65503 6.29931 6.88968 8.80234 5.86661 9.60644 7.91512 7.65052\nthreshold=0.45857843137254906 0.48753231203628372 0.41994549103866291 6.5000000000000009 0.21201994795255166 30.500000000000004 4.5000000000000009 7.9458193778991708 1.5401119998885948 0.1919612135732755 12.500000000000002 0.62109193738564106 7.5000000000000009 44.500000000000007\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 5 -2 -4 -3 -1 9 -9 12 -12 -6 -14\nright_child=4 6 3 -5 10 -7 -8 8 -10 -11 11 -13 13 -15\nleaf_value=-0.0025072902930816576 0.047946487906971376 -0.0068601293646307114 -0.022367763892255841 0.044286722536531126 0.0017095260563724209 0.03947922805946804 -0.066767812321464809 0.0035208630779612875 0.023786526483912263 -0.064636609055242364 -0.066545844328588191 0.018935983796892118 0.031172202625701362 -0.028835760840323833\nleaf_weight=527.67797060945486 8.3054770784219709 10.820635251468049 14.365340048330838 17.516551412758417 337.05314896226628 5.5562374126166096 7.4670996420318252 7.1790208097081694 4.484350736020132 12.383819287177174 6.5305782534414885 5.2046479843556881 40.224221946671605 5.2993667551781973\nleaf_count=28638 100 129 189 139 3559 42 159 54 58 231 185 60 338 151\ninternal_value=-0.000196603 -0.00319273 -0.00219634 0.0191264 0.00432053 -0.00405577 -0.0335146 -0.00367521 -0.0281802 -0.0406698 0.00328266 -0.0301241 0.00438325 0.0237171\ninternal_weight=1010.07 607.451 589.163 37.4381 402.617 19.9216 18.2877 551.725 24.0472 19.5628 394.312 11.7352 382.577 45.5236\ninternal_count=34032 29639 29351 370 4393 231 288 28981 343 285 4293 245 4048 489\nis_linear=0\nshrinkage=0.05\n\n\nTree=164\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 5 3 7 7 2 13 10 1 13 6 0 14 0\nsplit_gain=5.52041 6.16513 8.72157 9.37844 7.77396 11.8157 6.20944 5.75562 4.84959 4.84526 8.01598 9.44307 6.35333 7.09103\nthreshold=0.40090617732792916 1.5000000000000002 0.86289377661312738 4.5000000000000009 8.5000000000000018 1.4130234980658136 1.5401119998885948 0.82793451758969006 7.4631829261779794 1.0000000000000007 1.5000000000000002 0.052430497782015556 0.6656636044143921 0.73012711180625955\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 2 3 -2 8 6 -6 9 -4 -1 12 -12 -11 -14\nright_child=1 -3 4 -5 5 -7 -8 -9 -10 10 11 -13 13 -15\nleaf_value=-0.00037993377405244058 0.020549973203230581 0.0071036266756040389 -0.0061611813553316425 -0.072169700572146639 -0.062157396499653565 0.020587223270319682 0.0018338352127209613 -0.058046768677550854 0.029896532232876499 0.018113380930775972 0.014771066991696136 -0.02097499536235084 -0.053676192784039373 0.013636699635892877\nleaf_weight=428.63202076070775 3.3105432195588937 298.62719131098129 16.320159990165848 8.4538382561877352 12.599982734303927 10.385831125604453 5.1783083223272106 3.8543132401537141 19.894526886520907 36.448376757325605 20.92418258991529 124.73651844284905 6.8022723868489248 7.2039781301282346\nleaf_count=26543 167 2797 228 55 115 131 61 70 161 709 275 2469 70 83\ninternal_value=-0.000929932 0.00386698 -0.00875443 -0.0465122 -0.00124881 -0.0205597 -0.0444135 -0.00379143 0.0139182 -0.00336448 -0.00985252 -0.0157744 0.00721792 -0.0204529\ninternal_weight=1003.37 374.77 76.1432 11.7644 64.3788 28.1641 17.7783 628.602 36.2147 624.747 196.115 145.661 50.4546 14.0063\ninternal_count=33934 3715 918 222 696 307 176 30219 389 30149 3606 2744 862 153\nis_linear=0\nshrinkage=0.05\n\n\nTree=165\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 6 11 11 1 0 1 6 12 1 5 4 13 8\nsplit_gain=4.55594 7.66872 9.00154 7.87982 7.11905 6.80592 6.45654 9.94205 6.05214 5.64981 7.68424 9.93758 10.8674 5.87832\nthreshold=6.8997397422790536 12.500000000000002 0.22964059561341896 0.27328108044628058 6.6454498767852792 0.81608219810852256 7.0373966693878183 7.5000000000000009 0.23261107465006284 6.1893992424011239 5.5000000000000009 4.5000000000000009 -0.40581029540031788 36.500000000000007\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 5 9 8 -5 -8 -4 -1 11 13 -13 -11\nright_child=1 -3 3 6 -6 -7 7 -9 -10 10 -12 12 -14 -15\nleaf_value=-0.0022260293061954562 -0.015671441883852926 0.040786612962626242 0.027463929270004919 0.025716390218968117 -0.018315652524655123 0.056481011064112012 0.0034389428582111063 -0.052065754856940365 -0.028803948336172964 0.017068540712959945 0.026217767641433527 0.036387894261459858 -0.053734046492057347 -0.034807658378557177\nleaf_weight=468.83129560024827 49.092260605364572 13.777076997212133 14.777626681898257 30.208518902305514 56.282940911507467 13.10930740716867 234.32018426019931 7.3550660455948664 5.9276157663553013 43.232193097413983 44.093730102234986 3.6288344752683797 12.889486207364824 5.2855906147742635\nleaf_count=26648 916 225 158 300 1046 68 2419 276 152 595 897 101 305 95\ninternal_value=0.00106663 0.00547986 0.00399806 0.0071976 -0.00150255 0.0296049 0.0043125 0.00154214 0.0107703 0.000163545 0.0103564 -0.000631661 -0.0329886 0.0108283\ninternal_weight=1002.81 368.568 354.791 305.698 634.244 33.8145 271.884 241.675 20.7052 577.961 109.13 65.0361 16.5183 48.5178\ninternal_count=34201 4514 4289 3373 29687 378 2995 2695 310 28641 1993 1096 406 690\nis_linear=0\nshrinkage=0.05\n\n\nTree=166\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 6 12 10 10 5 6 3 14 0 2 11 12 7\nsplit_gain=5.04505 8.90147 6.30831 6.88937 8.7895 6.98225 6.81316 6.18333 5.64433 7.38569 5.24521 4.87017 7.64219 6.48628\nthreshold=6.8997397422790536 12.500000000000002 0.65834879259986268 0.68841813923781148 0.71631706333198875 5.5000000000000009 2.5000000000000004 -0.91484210894904117 0.64909264460070693 0.65980664568139946 -0.999999999999999 0.34360627172897679 0.30090049693865728 11.500000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 7 3 -1 -5 6 8 10 -6 -10 -2 13 -13 -9\nright_child=1 -3 -4 4 5 -7 -8 11 9 -11 -12 12 -14 -15\nleaf_value=-0.0043592773902334271 0.0099873697089417745 0.041238737755201593 -0.038162904182565877 0.048022923375097532 0.028629921694342739 0.031114139747313627 -0.043419957566574074 0.0046788801319967736 -0.047580639580415168 0.022509851976854938 -0.050033598245162236 0.036441360426858006 0.0049459510779011019 -0.02042872464522974\nleaf_weight=559.76281609149555 5.2626234628260127 14.402218260802327 11.735595368896609 13.005992302205412 11.804564805468546 14.386990198865531 9.3372963201254588 65.932328953465912 7.4444900829112139 5.9544953806325793 9.0266907031182182 20.228481630329043 214.08870258514071 40.884968255399144\nleaf_count=28662 86 233 139 113 206 99 104 858 94 61 133 171 2149 931\ninternal_value=-0.000613108 0.00402067 -0.00332059 -0.00260265 0.0130909 0.00302941 -0.00921477 0.00240108 0.00463083 -0.0170324 -0.028721 0.00378484 0.00780824 -0.00503155\ninternal_weight=1003.26 369.826 633.432 621.697 61.9338 48.9278 34.5408 355.424 25.2036 13.399 14.2893 341.134 234.317 106.817\ninternal_count=34039 4561 29478 29339 677 564 465 4328 361 155 219 4109 2320 1789\nis_linear=0\nshrinkage=0.05\n\n\nTree=167\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 5 13 0 0 10 8 4 12 13 11 12 0 4\nsplit_gain=6.00876 6.57912 8.0261 8.50111 6.29515 6.6663 9.93613 5.20105 7.02643 8.4479 4.32436 6.12684 6.59169 6.04095\nthreshold=0.6179969608783723 17.500000000000004 -0.31824396407007477 0.67106743739858599 0.77947462504345733 0.72783389450056124 42.500000000000007 5.5000000000000009 0.38121896056391874 -1.1518156456185185 0.61813055473851874 0.76309033109884206 0.45951410741920368 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 4 -3 -4 10 7 -7 -6 9 -9 13 -12 -13 -2\nright_child=1 2 3 -5 5 6 -8 8 -10 -11 11 12 -14 -15\nleaf_value=-0.036175477359741241 -0.0088806342838683126 -0.022824853518540661 0.045521336999391745 -0.023421234151412333 0.016058883802097867 -0.046537293185156335 0.025144783162687132 0.06062161311811403 0.022429451177924269 -0.056586646805889998 0.011300648667621253 -0.045778855461687737 0.010737760239652708 0.00087154284591748309\nleaf_weight=9.8312278640696587 264.68464746188693 7.2282907251155839 23.945286133101039 4.7933302211895343 132.61279871562147 11.13229850161588 7.2134361481294027 0.98755981854628849 7.217442935332655 8.4410420987842372 112.51652122655651 9.6449097897857445 9.4126259894110245 394.31792413874064\nleaf_count=3640 4422 1975 1397 307 1703 171 53 50 77 462 1008 161 82 18558\ninternal_value=0.00086646 0.00126876 0.0219668 0.0336215 0.000469727 0.00924149 -0.0185095 0.0127529 -0.0130021 -0.0396743 -0.00139924 0.00684377 -0.0187213 -0.00305288\ninternal_weight=1003.98 994.148 35.9669 28.7386 958.181 167.605 18.3457 149.259 16.646 9.4286 790.577 131.574 19.0575 659.003\ninternal_count=34066 30426 3679 1704 26747 2516 224 2292 589 512 24231 1251 243 22980\nis_linear=0\nshrinkage=0.05\n\n\nTree=168\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 12 12 2 11 1 1 6 6 9 9 13 13\nsplit_gain=4.5119 10.3036 9.01197 6.96791 6.74995 8.73635 6.70981 9.04124 8.80696 9.42833 6.3802 10.8962 5.52203 5.28163\nthreshold=2.5000000000000004 0.11918615595049482 0.11520419728812863 0.12947591132444949 -0.5498725608388263 0.088508261124055423 6.8997397422790536 6.6454498767852792 12.500000000000002 7.5000000000000009 0.64309270329754664 0.58595927517517843 1.5007010094932978 -0.44077704520667127\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -3 -4 5 -2 7 -6 9 -8 11 -7 -11 -9\nright_child=4 2 3 -5 6 10 8 13 -10 12 -12 -13 -14 -15\nleaf_value=-0.02720538254813186 0.03630484714795125 0.03465882734957558 -0.037970428164840031 -0.0014703417894787649 0.0043864148513530123 -0.012471083118418581 0.015265082509038045 0.016008713459538082 0.0598598347291385 -0.062593280204414944 -0.063194887930074634 0.051520559918537302 0.027894814928650438 -0.042929317072307807\nleaf_weight=40.802718145496328 9.6273019232030475 15.837632776936514 12.514212979280272 406.71392373521667 264.45381950370211 100.97847404160711 113.56056323867233 4.3623084688733824 9.4055506731383485 5.4324308121867935 4.6245268238708368 6.1358561194501808 1.6307946206070481 15.473321515106361\nleaf_count=848 564 160 309 5509 15159 9441 1254 104 83 239 105 71 10 384\ninternal_value=4.48295e-05 -0.00349427 -0.00121323 -0.00264766 0.00318939 -0.00714052 0.00622523 0.00190349 0.0155874 0.0116715 -0.0111725 -0.0083628 -0.0408325 -0.0298213\ninternal_weight=1011.55 475.868 435.066 419.228 535.685 121.366 414.319 284.289 130.029 120.624 111.739 107.114 7.06323 19.8356\ninternal_count=34240 6826 5978 5818 27414 10181 17233 15647 1586 1503 9617 9512 249 488\nis_linear=0\nshrinkage=0.05\n\n\nTree=169\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 11 13 7 3 2 13 2 2 9 7 12 9\nsplit_gain=9.02127 7.37992 6.76634 6.71936 6.28974 5.80487 5.80076 5.74819 6.36476 9.46923 8.57551 6.37629 6.35563 10.0576\nthreshold=2.5000000000000004 0.12043646866653869 0.12462106435257471 -0.67432252158699446 5.5000000000000009 2.6316286664499118 -1.2034596960634947 0.82124236120517702 0.61051104804217138 0.76098550396778364 0.48753231203628372 24.500000000000004 0.084535385028355392 0.11499787120625572\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 -1 6 -4 8 12 11 -11 -10 -2 -14\nright_child=7 2 5 -5 -6 -7 -8 -9 9 10 -12 -13 13 -15\nleaf_value=0.024993294513488279 0.029989887358499341 0.036990573359413748 0.047206518294485425 -0.039222711055368079 -0.032688982706482857 0.045451894380667573 -0.0061782769456970154 -0.0020750119324956153 0.05289853635166121 0.014190570988934779 -0.04703881601907757 -0.02558362450027445 -0.024405980233633856 0.0057490217935445113\nleaf_weight=9.9164729284239019 18.491748941061815 8.9114264139207062 4.1642678820207939 28.652565746262553 7.3337767380362484 4.6725302832783191 403.45877487905091 249.11463469222872 22.229104815247414 69.96827342947654 5.2469244541134676 2.3154516662834785 33.321674769156743 141.27704147216537\nleaf_count=193 603 108 31 537 173 100 5689 4117 343 2770 104 490 9518 9328\ninternal_value=-0.000881313 -0.00596709 -0.00386212 -0.0247882 2.27909e-05 -0.00483248 -0.00551871 0.00350494 0.00824247 0.0184319 0.00935814 0.0447831 0.00291027 -0.000112189\ninternal_weight=1009.07 467.11 421.207 45.9028 17.2502 412.296 407.623 541.965 292.85 99.7598 75.2152 24.5446 193.09 174.599\ninternal_count=34104 6831 5928 903 366 5820 5720 27273 23156 3707 2874 833 19449 18846\nis_linear=0\nshrinkage=0.05\n\n\nTree=170\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 13 13 6 9 8 14 9 2 0 1 2 1 1\nsplit_gain=5.8646 7.6053 4.62955 4.99683 11.6949 7.19246 7.6013 6.79211 6.57844 5.68433 11.4361 11.2727 8.87939 8.87077\nthreshold=0.75381749868392955 0.043496472104380256 2.9496234745417209 2.5000000000000004 0.078907978039233523 22.500000000000004 0.39728985235740838 0.081388599864904806 1.875273743832488 0.76680442430599671 7.1975305080413827 1.2928514382138581 4.1540565490722665 6.6137495040893564\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 5 -2 -7 -6 9 11 13 -5 -13 -11\nright_child=2 -3 -4 8 7 6 -8 -9 -10 10 -12 12 -14 -15\nleaf_value=0.0084865611543161468 -0.062188995880807833 -0.062197980405045816 0.03569872853721124 0.0030425067942006397 0.044067964945343402 0.0063005757927399432 -0.044467287084658497 -0.0015153557291127861 0.036414767483127379 0.011032031397842501 0.038586553065287521 -0.0013301361974577593 -0.056871644813083003 -0.043597729308910954\nleaf_weight=6.5873906436436283 10.611395103187531 7.3711605466542096 8.5423331487108936 415.54923021454852 7.3350005341926581 17.357474767602984 11.553483514922844 417.571468193084 14.235634136362931 31.110398105229251 30.641795667586848 14.382872302347097 13.499944778799543 8.6698585231788439\nleaf_count=2811 171 1234 252 20968 81 311 279 5628 143 1062 236 346 264 222\ninternal_value=0.000574202 -0.0305028 0.00103754 0.00070359 -0.00307595 -0.0287411 -0.0147958 -0.000626847 0.00402704 0.0030608 0.016152 0.000950104 -0.0292595 -0.00165131\ninternal_weight=1015.02 13.9586 1001.06 992.519 464.429 39.5224 28.911 424.906 528.09 513.854 70.4221 443.432 27.8828 39.7803\ninternal_count=34008 4045 29963 29711 6470 761 590 5709 23241 23098 1520 21578 610 1284\nis_linear=0\nshrinkage=0.05\n\n\nTree=171\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 0 4 1 2 0 11 12 11 5 2 6 0 3\nsplit_gain=6.60346 5.46753 4.48019 7.12791 6.75218 6.33781 6.86133 6.31162 13.0549 11.3502 7.26384 7.06205 9.39208 7.66415\nthreshold=0.34755730289246772 0.93488923097199472 2.5000000000000004 6.2170217037200937 1.4900624917218295 0.4993943273353797 0.47815812583848105 0.19338065372620974 0.23670318574735247 17.500000000000004 1.1771025309410685 3.5000000000000004 0.13376347902723115 0.41106258932710005\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 5 7 6 -1 8 10 11 -4 12 -9 -14\nright_child=-2 -3 4 -5 -6 -7 -8 9 -10 -11 -12 -13 13 -15\nleaf_value=-0.029793372121525913 0.0043014068563258234 0.022721816991636473 0.00098244822826342696 0.0019228954959746765 0.033019279882179342 -0.0057849430865931249 0.033001617037329074 0.036756821104115704 0.042434161846527652 0.039447809109417552 -0.043537013494302487 -0.032219223425228406 0.027948409103317681 -0.031736533316303349\nleaf_weight=73.118754483948578 459.6162586993305 17.770964725641534 166.29943607693713 82.259814877063036 13.176350140565772 74.298921625500952 3.8171836804831392 9.9018709028023277 17.744456846121466 8.3162929709469591 8.6944062407492293 44.341360703132523 6.2264649558346745 21.411043840576895\nleaf_count=1706 5045 278 19319 1152 246 1742 30 85 244 412 445 2793 99 448\ninternal_value=-0.000112191 -0.00381924 -0.00475097 -0.00991397 -0.000660098 -0.0163226 -0.0262943 -0.00233932 0.00277541 -0.0131794 -0.00146427 -0.018936 -0.00281692 -0.0177837\ninternal_weight=1006.99 547.377 529.606 233.495 296.112 151.235 76.9359 282.935 192.738 90.197 174.994 81.8807 37.5394 27.6375\ninternal_count=34044 28999 28721 4630 24091 3478 1736 23845 20008 3837 19764 3425 632 547\nis_linear=0\nshrinkage=0.05\n\n\nTree=172\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 5 5 2 11 11 7 14 12 2 13 13 12 11\nsplit_gain=6.12646 8.27316 8.04266 6.9819 7.29196 8.00457 9.49473 7.50608 7.56251 7.72641 7.40654 7.16803 6.89015 7.10541\nthreshold=2.5000000000000004 9.5000000000000018 14.500000000000002 1.3438334896653068 0.12043646866653869 0.16821326847638382 17.500000000000004 0.42253279867808763 0.14324172638303317 1.0929501479680097 -0.49248288101553256 0.83711243686168813 0.33976823339759277 0.30198526726146607\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 10 7 -7 -6 9 -9 -1 -12 13 -8\nright_child=-2 2 -4 -5 5 6 12 8 -10 -11 11 -13 -14 -15\nleaf_value=-0.0010343535982444256 0.003431608832341294 -0.065322216440310096 0.0018721975009093495 0.0057784990084583723 0.045457394354522222 -0.0047578666674077614 -0.037048723520654102 -0.044859782317720441 0.025689394520605192 0.024707079351211969 -0.076845986688956647 -0.010359409588467048 -0.067770060321879391 0.02684740017593398\nleaf_weight=12.511096425652797 531.02791393290499 9.6874173191026802 7.5410968337673685 149.50522200370324 12.686116898898034 206.71600868998212 11.528063588135415 10.877780152652127 14.981662200851131 5.2310767155140629 9.8506696753756859 7.0367576873395583 9.6270784884691221 5.7956661952193818\nleaf_count=282 27206 155 92 1972 220 2682 192 326 283 51 296 84 108 63\ninternal_value=-0.000252749 -0.0043826 -0.0374213 -0.00305619 -0.00736558 -0.00479945 -0.0084809 0.0146016 0.000987092 -0.0221454 -0.0306285 -0.0512699 -0.0358451 -0.0153737\ninternal_weight=1004.6 473.576 17.2285 456.347 306.842 277.443 233.667 43.7766 31.0905 16.1089 29.3985 16.8874 26.9508 17.3237\ninternal_count=34012 6806 247 6559 4587 3925 3045 880 660 377 662 380 363 255\nis_linear=0\nshrinkage=0.05\n\n\nTree=173\nnum_leaves=15\nnum_cat=0\nsplit_feature=5 10 0 13 12 5 10 0 3 3 0 12 3 9\nsplit_gain=6.19985 8.3593 9.16103 7.26899 5.35681 7.24732 5.00179 4.89183 5.68325 10.9391 8.05566 6.54061 6.97666 9.09326\nthreshold=22.500000000000004 0.31929669030732866 0.97043570271994106 -0.31824396407007477 0.33266628102458634 3.5000000000000004 0.23117408906882594 0.6062644305068664 0.30771841529455551 0.6281986210363516 0.5669415828282881 0.09978491514898781 -0.18361356884518543 0.19453332722939159\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -3 -2 7 -6 -5 8 10 -10 11 -1 13 -13\nright_child=3 2 -4 6 5 -7 -8 -9 9 -11 -12 12 -14 -15\nleaf_value=0.02065813124944238 -0.041882068789259652 0.0012993129821989025 0.048368073445549845 0.054851834474251272 -0.0042744185705015014 -0.065118001359818198 -0.033593003685328686 0.0024674321206080629 -0.042850252611777641 -0.009436312447409223 -0.058960219829255145 -0.029204983342809521 0.011991655660638579 0.010071283250141325\nleaf_weight=31.4995902740975 2.1429164307755864 519.00476023457304 9.5496077714487893 11.134917014349407 10.856476752553137 8.1447856608720013 1.161913845397067 126.95052687087082 28.475961644044219 151.18197547420277 4.814897808355453 40.126490914082751 36.523214619788632 22.327309462991252\nleaf_count=1226 1015 6670 65 802 205 137 105 4618 2205 4804 700 7098 3661 780\ninternal_value=-0.00156229 -0.00205087 0.00223945 0.0298817 -0.00696515 -0.0323064 0.0445969 -0.00580502 -0.00913676 -0.0149403 -0.00137693 0.00118031 -0.00516316 -0.0152262\ninternal_weight=1003.9 989.456 528.554 14.4397 460.901 19.0013 12.2968 441.9 314.949 179.658 135.292 130.477 98.977 62.4538\ninternal_count=34091 32169 6735 1922 25434 342 907 25092 20474 7009 13465 12765 11539 7878\nis_linear=0\nshrinkage=0.05\n\n\nTree=174\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 10 6 14 2 0 1 10 10 1 2 1 13 10\nsplit_gain=4.79369 4.43718 12.7096 8.08111 7.18806 6.09127 9.47727 3.93811 5.28876 7.23881 3.89566 7.59269 5.75702 3.75338\nthreshold=24.500000000000004 0.12712842712842715 4.5000000000000009 0.4795549589540728 -1.2425192343971152 0.012031525820100544 2.9345310926437382 0.12018518518518519 0.10135754907356449 2.9345310926437382 1.3673476462045733 3.8484723567962651 -0.54420942749954193 0.14354395604395606\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 5 -1 6 -4 8 9 -7 -6 -12 -13 -3\nright_child=-2 13 3 -5 10 7 -8 -9 -10 -11 11 12 -14 -15\nleaf_value=0.040626123281173002 -0.05821624184329708 0.014310567239881156 0.036891002622705275 -0.015507184913803262 -0.025764688404388194 -0.031120593284385118 -0.052291271173673042 -0.032582634093127609 0.0635324854981974 0.041132712788083697 -0.037924882317026919 0.038913217175320161 -0.0087151323550019952 0.00012358385669903951\nleaf_weight=4.2859072772553182 2.6175933882841482 48.600755086427853 5.0169256022163591 17.476578092247109 77.525465647708188 4.3509547932662889 4.9294344912050292 1.2246079264477874 12.488864467410165 9.3022785164939688 11.394399730663279 9.2571158863138407 15.906371722317999 776.37800396304374\nleaf_count=71 618 3269 126 4776 2168 677 92 263 620 267 248 76 260 20667\ninternal_value=-0.000762949 -0.000554164 -0.0078065 0.0119579 -0.0169784 0.0248909 -0.00804717 0.0367238 0.0410512 0.0175549 -0.0194767 -0.0058113 0.00962395 0.000975674\ninternal_weight=1000.76 998.138 173.159 54.7896 118.369 37.3131 9.94636 27.3667 26.1421 13.6532 114.083 36.5579 25.1635 824.979\ninternal_count=34198 33580 9644 6821 2823 2045 218 1827 1564 944 2752 584 336 23936\nis_linear=0\nshrinkage=0.05\n\n\nTree=175\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 5 13 13 8 3 0 9 0 5 2 8 5 0\nsplit_gain=4.03588 4.11232 6.16816 5.24798 11.2719 9.64454 5.60723 10.2445 10.9427 8.55032 8.05437 6.59701 6.30293 5.92933\nthreshold=0.34755730289246772 22.500000000000004 -0.31824396407007477 -1.0000000000000195 25.500000000000004 1.3544876714586842 0.080737732012600816 0.24949058931584997 0.020884627018111463 6.5000000000000009 0.73126556468747761 44.500000000000007 7.5000000000000009 0.2350987293572229\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 4 -1 -6 7 8 9 -5 11 -8 -10 -12\nright_child=-2 2 -4 6 5 -7 10 -9 12 -11 13 -13 -14 -15\nleaf_value=0.022609772855886917 0.0025088293771707224 -0.040396050069039798 0.041350396115318544 -0.015392365781603355 -0.041939449780735688 0.030494709340430262 -0.021479497781375603 -0.030345535442922456 0.043687395015708827 0.030435745340653516 -0.031931405122146685 0.016410596432313539 -0.021177722437304584 -0.00082306888156092424\nleaf_weight=44.577562285658132 456.13607722707093 2.1000141741496909 9.4424400056705036 35.818995655528852 14.779689708389926 5.651301554622477 133.88822189865004 16.516731963900384 26.193802252520985 13.07310591256646 15.566351055218545 11.834911432117222 3.7310680576738369 204.04225274235432\nleaf_count=2721 5009 1043 879 618 1078 77 14931 288 941 670 324 233 871 4327\ninternal_value=-0.000946945 -0.00388036 0.0244426 -0.00454749 0.00865852 -0.0214158 -0.00642582 0.00433776 0.0118951 -0.00277411 -0.00923877 -0.0183114 0.0351766 -0.00316284\ninternal_weight=993.353 537.216 11.5425 525.674 65.0086 20.431 460.665 95.3337 78.817 48.8921 365.332 145.723 29.9249 219.609\ninternal_count=34010 29001 1922 27079 3876 1155 23203 3388 3100 1288 19815 15164 1812 4651\nis_linear=0\nshrinkage=0.05\n\n\nTree=176\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 11 11 3 11 11 11 6 0 10 2 9 11 2\nsplit_gain=6.58525 7.40545 6.41065 16.6189 8.52793 8.27712 7.6726 7.52863 7.85166 8.16451 6.66937 6.46577 6.40111 5.94489\nthreshold=2.5000000000000004 0.11918615595049482 0.15253578947861293 0.9164627150260779 0.1463062475137259 0.13928496946513533 0.19200570707404305 13.500000000000002 0.012031525820100544 0.069605943152454788 -0.74960001319577663 0.12648515653440548 0.82415404743218235 -1.2601658302236523\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 5 7 -4 8 9 -2 13 -6 -12 -8\nright_child=2 -3 6 -5 11 -7 10 -9 -10 -11 12 -13 -14 -15\nleaf_value=-0.024525872724703631 0.021965009474648134 -0.0028365154249530371 -0.017857446985270971 -0.042680903466797593 0.0027421767513761599 -0.03119280064883382 0.0058320981229614422 -0.036860628277115469 0.031206174130229289 -0.037132651379108587 0.0067365817075259729 0.075055538294398497 -0.037674549996870539 -0.027762966628347364\nleaf_weight=42.463461841252865 12.618675635370893 421.02021817133937 50.057751541067773 11.029955120968223 4.4374660065659546 8.4489034065409196 22.418871248308278 4.9240894094327805 60.043718081111365 9.2482416838793124 307.26187668618877 9.8153329301621834 7.3495024461299172 29.930239713728952\nleaf_count=859 95 5923 6341 1139 921 1613 670 3649 4817 240 5922 354 131 1337\ninternal_value=-0.000510179 -0.00487218 0.00325262 0.013356 0.0193338 0.0136101 0.000306619 0.0182748 0.0219939 -0.00356017 0.00282621 0.0541973 0.00558303 -0.013536\ninternal_weight=1001.07 463.484 537.585 120.566 109.536 95.2836 417.018 86.8347 81.9106 21.8669 366.96 14.2528 314.611 52.3491\ninternal_count=34011 6782 27229 12828 11689 10414 14401 8801 5152 335 8060 1275 6053 2007\nis_linear=0\nshrinkage=0.05\n\n\nTree=177\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 6 10 2 12 3 3 14 11 0 2 2 14 10\nsplit_gain=7.16633 5.48276 7.98395 5.62669 10.2811 7.69563 15.01 8.72054 7.67353 7.31509 8.08349 8.0641 8.8646 7.30273\nthreshold=0.029776858282843214 2.5000000000000004 0.061381475667189961 -0.6064945943510146 0.19488605834408215 -1.2214177550289611 -0.9999468844480115 0.52810343530048642 0.088508261124055423 0.080737732012600816 -0.3306222700042753 -0.39932064797367844 0.89678331622138285 0.17089216944801031\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 -3 4 5 -4 -7 -8 -5 10 -10 12 13 -11\nright_child=1 2 3 8 -6 6 7 -9 9 11 -12 -13 -14 -15\nleaf_value=-0.0427360971333249 -0.003435508493484687 0.035252114919112183 -0.048699770784664739 -0.047932879308990095 -0.022420439467795185 0.069763842848992882 -0.025417819815407756 0.020208814227504757 -0.016492433791151982 -0.021649837447721067 0.033908158952552496 0.0016445125147789052 -0.047662345239589725 0.051659820087503527\nleaf_weight=8.6429348016972636 455.82587277153652 18.734499665592011 4.4229704521003432 5.609266436786128 54.276203604011243 8.7655485079758346 17.282495847450264 23.516057791273852 9.2311798605442146 3.4878607190585234 39.884326375467936 320.96465648232152 2.4130910057028805 19.597830758668351\nleaf_count=130 6699 496 912 495 2011 421 2510 3228 720 1352 713 13176 265 755\ninternal_value=0.000158255 0.000578016 0.00404164 0.0028271 -0.00724299 0.00814599 0.0140816 0.000735415 0.00555584 0.00643325 0.0242954 0.00383844 0.0303502 0.0401449\ninternal_weight=992.655 984.012 528.186 509.451 108.263 53.9871 49.5641 40.7986 401.188 395.579 49.1155 346.463 25.4988 23.0857\ninternal_count=33883 33753 27054 26558 9082 7071 6159 5738 17476 16981 1433 15548 2372 2107\nis_linear=0\nshrinkage=0.05\n\n\nTree=178\nnum_leaves=15\nnum_cat=0\nsplit_feature=5 13 10 1 1 4 3 1 1 13 3 12 10 9\nsplit_gain=4.5053 5.89471 4.20152 7.29286 7.09865 9.47786 7.75102 7.08105 8.56105 6.71236 6.07517 5.88702 8.849 6.43841\nthreshold=22.500000000000004 -0.31824396407007477 0.27326839826839833 5.0610318183898935 5.503817319869996 1.5000000000000002 -1.0511661868229158 4.3886544704437265 4.2955996990203866 0.4271357933468744 0.064246569147159552 0.12570481843286005 0.10733486943164364 0.057128223454471175\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 9 7 5 -5 -6 8 -4 -1 -7 13 -13 -11\nright_child=1 -3 3 4 6 10 -8 -9 -10 11 -12 12 -14 -15\nleaf_value=0.0021700745842657783 -0.039418973361749973 0.039663322677121901 0.0081960752927624052 0.014780636239402784 -0.03594292386158434 -0.00045599721823765611 0.0027049271929866798 0.028476819242353419 -0.062114136197134601 -0.031524884378631483 -0.06321837957627828 -0.040699037732200273 -0.0092657381263181997 0.020589123227749501\nleaf_weight=211.53017828526117 2.0136765228162412 12.539482952641261 39.551762215909548 9.6395448610710464 12.364314235630443 5.4500600976753075 444.29481128594489 47.841671980306273 3.8470194954425088 6.7117730658210375 12.301576634956289 25.710262244494515 145.55778239254505 25.150703968884045\nleaf_count=18407 1045 897 914 138 263 185 5488 608 83 81 243 664 4764 400\ninternal_value=0.000188357 0.02689 -0.00023092 0.00253293 7.74176e-05 -0.0241831 0.00158226 0.0154179 0.000704904 -0.00406229 -0.0450012 -0.0105312 -0.0141933 0.00898615\ninternal_weight=1004.5 14.5532 989.951 575.291 484.05 27.3912 456.659 91.2405 43.3988 414.661 17.7516 203.131 171.268 31.8625\ninternal_count=34180 1942 32238 7922 6317 566 5751 1605 997 24316 428 5909 5428 481\nis_linear=0\nshrinkage=0.05\n\n\nTree=179\nnum_leaves=15\nnum_cat=0\nsplit_feature=5 13 10 9 12 12 3 1 12 10 5 2 13 12\nsplit_gain=3.88758 5.61303 4.54206 4.09416 4.5104 5.29769 6.02995 5.03399 4.74743 4.52441 4.00598 7.16474 5.88738 7.42098\nthreshold=22.500000000000004 -0.31824396407007477 0.23117408906882594 0.34755730289246772 0.26410036853271063 0.31905847858317887 0.51298751142788357 5.1241953372955331 0.26016240578361016 0.10403138528138529 5.5000000000000009 -1.4244562613774308 -0.92626928906613226 0.2020546430968527\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 -3 4 8 7 -7 -6 10 -9 12 -12 13 -1\nright_child=1 2 -4 -5 5 6 -8 9 -10 -11 11 -13 -14 -15\nleaf_value=0.014403641303012988 -0.039535796258257212 0.046446267515198994 -0.034906493069329129 0.0035653949396720779 0.0067675053695527556 -0.028011693611118466 0.010122008103823769 0.018370401478034964 0.049175331358431183 -0.056362325731535559 -0.057174337728665328 0.0061732333565056958 -0.0085033456092567638 -0.051519190318833424\nleaf_weight=52.786065644897462 2.0407578570993801 11.919877990943855 1.2379742820048711 459.49796659420826 5.3442378691979711 15.793569799279792 26.182606184389442 1.8417511584702904 3.6398400457401285 14.751929483230922 3.5922009652422267 164.96355037110243 242.61703347093498 3.6602058892603955\nleaf_count=1575 1040 801 110 4987 221 573 444 21 82 658 823 15454 7242 103\ninternal_value=0.000503383 0.024801 0.0368672 0.000107579 -0.0028624 -0.0152194 -0.00454377 -0.0345575 -0.00115783 -0.047495 -0.00165371 0.00449395 -0.00512203 0.00930653\ninternal_weight=1009.87 15.1986 13.1579 994.671 535.173 63.9141 41.9762 21.9379 471.259 16.5937 467.619 168.556 299.063 56.4463\ninternal_count=34134 1951 911 32183 27196 1917 1017 900 25279 679 25197 16277 8920 1678\nis_linear=0\nshrinkage=0.05\n\n\nTree=180\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 6 4 1 9 10 5 1 3 10 13 3 10 9\nsplit_gain=4.8874 4.17352 8.18027 10.6347 9.16211 6.85817 7.44399 6.01003 5.47528 5.33152 9.78393 9.38095 7.41113 6.4942\nthreshold=0.75381749868392955 11.500000000000002 14.500000000000002 6.7416048049926767 0.29122929824397153 0.40114942528735636 11.500000000000002 3.4489090442657475 -1.7894353859448957 0.11872797449236451 1.0278617765834659 1.0446204841859017 0.12104562104562107 0.075623222642289656\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 9 5 -4 -5 7 -7 8 -3 10 13 -12 -11 -2\nright_child=1 2 3 4 -6 6 -8 -9 -10 12 11 -13 -14 -15\nleaf_value=-0.030975331030590438 -0.016959258849621763 0.041519306130623872 -0.029633069504193206 0.055935624285241292 -0.020918177321004303 0.032603698643908184 -0.045784323645771563 0.048283834645794985 -0.043713985773584763 0.043348553933444722 -0.062899872971744542 -0.0015345141679011687 0.00087500747724573611 0.0070703141573187078\nleaf_weight=11.276436282075336 46.859396473217203 2.3673095071717407 19.40245724138731 8.5542771024047379 5.8248601192608467 4.9259438037406635 5.2374610465485603 25.250430464107922 3.3108605596862635 9.4083436492728634 13.828086805908244 10.825994844140949 780.52410900727773 67.245491418088932\nleaf_count=4062 1385 131 5565 123 153 39 168 1062 1619 218 226 167 16611 2613\ninternal_value=0.000382267 0.000765097 0.0120403 -0.00612182 0.025467 0.026762 -0.00827438 0.0381738 -0.00728291 -0.000155911 -0.00915844 -0.0370636 0.00143505 -0.0028595\ninternal_weight=1014.84 1003.57 74.8736 33.7816 14.3791 41.092 10.1634 30.9286 5.67817 928.691 138.759 24.6541 789.932 114.105\ninternal_count=34142 30080 8860 5841 276 3019 207 2812 1750 21220 4391 393 16829 3998\nis_linear=0\nshrinkage=0.05\n\n\nTree=181\nnum_leaves=15\nnum_cat=0\nsplit_feature=1 9 6 2 14 5 1 3 10 10 2 6 1 3\nsplit_gain=3.15502 5.46728 3.1586 8.33728 9.59197 7.1819 5.68505 8.25732 6.32227 7.94612 4.28558 5.19479 8.46209 6.82854\nthreshold=0.6179969608783723 0.070281643234268623 11.500000000000002 -0.5498725608388263 0.90244349365186294 11.500000000000002 3.4489090442657475 -1.7894353859448957 0.40114942528735636 0.64899964899964913 -1.4103809358165493 8.5000000000000018 6.5555982589721689 0.41106258932710005\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 10 5 6 -4 7 -5 -8 -10 -2 -12 13 -13\nright_child=2 -3 3 4 -6 -7 8 -9 9 -11 11 12 -14 -15\nleaf_value=0.012156678969891161 0.023731140508364924 -0.056074572277780489 0.0066150446087051319 0.047320860867660082 -0.048366256684467895 -0.047316880832019896 0.051036692694804278 -0.047785608111278979 -0.037300041572050607 0.037863961669035114 0.00077770839198724366 -0.019270843145329165 -0.061363304086402864 0.029784841543852705\nleaf_weight=4.8958985221506719 18.341610011273588 5.3736893631712519 18.052507932627126 2.9066009682937857 3.4344104898518699 8.2257226633064402 25.392961200876016 4.5401430219899348 5.0145096874330175 7.5154657047241926 862.43870135764155 24.205322688285378 7.6453267333636168 8.8971061740176065\nleaf_count=844 571 2862 2413 143 418 2608 1126 2273 110 46 17414 2193 363 715\ninternal_value=0.000956038 -0.0253538 0.00125229 0.0110439 0.0231055 -0.0113827 0.0294421 -0.00945635 0.0371271 0.00724967 0.000443403 -5.5239e-05 -0.0172812 -0.00559913\ninternal_weight=1006.88 10.2696 996.61 75.0823 48.8041 26.2782 45.3697 7.44674 37.9229 12.53 921.528 903.186 40.7478 33.1024\ninternal_count=34099 3706 30393 9137 4116 5021 3698 2416 1282 156 21256 20685 3271 2908\nis_linear=0\nshrinkage=0.05\n\n\nTree=182\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 10 11 13 10 11 1 8 2 11 8 11 2 12\nsplit_gain=3.87986 4.41874 10.2917 9.84592 10.5854 7.76925 6.74551 6.52684 6.33158 6.10409 5.96046 5.73472 7.12823 6.63577\nthreshold=0.95849462365591409 0.22696969696969699 0.15089642070402184 1.1020406375351024 0.21635553078852052 0.1457572911529163 6.0601730346679696 18.500000000000004 1.2413453555885363 0.28299061324924563 24.500000000000004 0.19668356768404752 0.87099168877543065 0.17056733839577878\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 6 7 -3 -1 -5 11 -9 -8 12 -4 -13\nright_child=-2 2 8 4 -6 -7 10 9 -10 -11 -12 13 -14 -15\nleaf_value=-0.004025663739999746 -0.036226972245668412 -0.0029435780745772047 -0.046023185965697967 -0.041002148418417525 -0.071906144606440278 0.073686080311249569 0.020012757260063328 0.0013880296124546482 0.012203781256266227 -0.041328013719997274 -0.0082092273588699159 0.029327921226562206 0.019585413076233578 -0.00032313662922089502\nleaf_weight=203.45469968600082 6.079893529531545 5.1100880759258862 12.860751693817294 14.843564463400979 7.3076616963662664 8.0063337395040417 66.437109946040437 55.218637424892222 131.41592309958651 8.8439505620626715 25.249349869322032 18.721727714575536 5.2361176874255753 436.01082495334049\nleaf_count=20038 160 154 837 252 104 57 1564 1358 1442 137 589 401 76 7012\ninternal_value=0.000657998 0.000918555 0.00353035 -0.00330776 -0.0180681 0.0457379 0.00105034 -0.0122446 0.00254009 -0.00505356 0.012236 -0.000165874 -0.0270086 0.000959297\ninternal_weight=1004.8 998.717 617.362 381.355 86.2138 13.1164 295.141 78.9062 604.245 64.0626 91.6865 472.829 18.0969 454.733\ninternal_count=34181 34021 9979 24042 1851 211 22191 1747 9768 1495 2153 8326 913 7413\nis_linear=0\nshrinkage=0.05\n\n\nTree=183\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 13 8 4 9 12 8 1 1 0 0 5 6 4\nsplit_gain=5.28759 5.8441 6.4887 5.48823 5.26301 10.5359 6.16685 7.59036 7.92815 7.3526 5.73999 5.42571 5.3283 5.25593\nthreshold=2.5000000000000004 0.77517283292263039 50.500000000000007 22.500000000000004 0.69450985341528104 0.53142246749396205 24.500000000000004 6.2170217037200937 5.0610318183898935 0.51877627139250015 0.36621320965455945 18.500000000000004 3.5000000000000004 7.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 -2 5 6 11 8 9 -8 -10 -3 -9 -14\nright_child=1 4 -4 -5 -6 -7 7 12 10 -11 -12 -13 13 -15\nleaf_value=-0.003609614544191779 0.011453745088865165 -0.011529004384644874 -0.018850105689094725 -0.045169957567292529 0.020417714413241559 -0.03708464522730566 0.024154910472943068 -0.0091713659875967066 -0.051895771153398156 -0.028613260674497595 0.0080510472648087795 0.026629674774158004 0.047836455742800647 0.0039341742829028395\nleaf_weight=458.72543326077721 264.4719787564718 106.48400840279282 19.239113198360428 3.38532707068316 22.864802327007055 21.078248146921396 23.778040353055989 8.5395383466966432 10.546677733538674 8.068071622255955 5.7372371840174301 9.2555058390498726 21.342594516463581 10.086162023479117\nleaf_count=6763 20684 2254 512 1651 136 201 723 53 101 281 78 349 83 114\ninternal_value=0.00032477 0.00369945 0.00854247 0.0105728 -0.0019238 -0.00427535 -0.000718157 0.00919184 -0.00419169 0.0103222 -0.0315312 -0.00827554 0.0250173 0.0343031\ninternal_weight=993.603 534.877 287.096 267.857 247.781 224.916 203.838 88.0983 48.13 31.8461 16.2839 115.74 39.9683 31.4288\ninternal_count=33983 27220 22847 22335 4373 4237 4036 1433 1183 1004 179 2603 250 197\nis_linear=0\nshrinkage=0.05\n\n\nTree=184\nnum_leaves=15\nnum_cat=0\nsplit_feature=0 13 9 9 6 1 9 6 3 13 3 13 7 10\nsplit_gain=5.0383 4.68707 7.22171 13.001 6.50946 8.65197 7.52444 6.36321 6.34206 6.05419 6.49258 5.4762 4.77125 4.55861\nthreshold=0.70626634622016105 1.5007010094932978 0.1664082439200707 0.3135379665119214 4.5000000000000009 6.3985919952392587 0.72224482408762225 1.5000000000000002 1.5039199563579408 0.87952391891555892 1.1361560482789164 1.8602164531472061 4.5000000000000009 0.8034250585480095\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=13 2 -2 -4 6 -6 7 9 -3 -5 -11 -10 -9 -1\nright_child=1 8 3 4 5 -7 -8 12 11 10 -12 -13 -14 -15\nleaf_value=-0.0027802917154472005 0.022515351043466608 0.034468821727311971 -0.029352287547207413 0.013766401844581923 0.013125894876778675 -0.055787390298860533 -0.033421759974783888 -0.0030096008949959509 -0.047118612670626059 -0.053403568184391063 0.017354089153951306 0.012106119050666699 0.031501151896959942 0.012964904572872447\nleaf_weight=723.89449949408481 36.485954837917234 29.39435133949155 32.780794257756497 20.4218271644786 8.3676926357438806 8.2480638932902348 7.9340210934169582 12.437251994386314 4.0924352742731598 8.0992316337942594 4.4235294818063258 15.728732985677196 47.15570082096383 48.097427317232359\nleaf_count=28940 1591 290 1050 224 173 209 39 127 69 123 52 301 323 588\ninternal_value=0.000170414 0.00655514 0.00288501 -0.00201163 0.00582665 -0.0223076 0.0106534 0.0147493 0.0201179 -0.00285493 -0.0289742 -0.00179762 0.0243678 -0.00178382\ninternal_weight=1007.56 235.57 186.354 149.868 117.087 16.6158 100.472 92.5375 49.2155 32.9446 12.5228 19.8212 59.593 771.992\ninternal_count=34099 4571 3911 2320 1270 382 888 849 660 399 175 370 450 29528\nis_linear=0\nshrinkage=0.05\n\n\nTree=185\nnum_leaves=15\nnum_cat=0\nsplit_feature=11 11 4 2 7 1 0 4 0 2 12 13 1 13\nsplit_gain=5.2814 7.82936 7.27568 8.3844 8.17259 12.0518 6.38152 7.54575 6.13925 5.61953 5.12385 5.26755 4.83459 4.62956\nthreshold=0.54136429796339314 0.52985700832157623 6.5000000000000009 1.3438334896653068 11.500000000000002 6.3710157871246347 0.77366324840924527 4.5000000000000009 0.87645216386918368 0.42896023551267498 0.53142246749396205 -0.028659912574114984 8.3477363586425799 0.27731332837924033\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 -2 -6 12 10 -9 -7 11 -8 13 -1\nright_child=2 -3 -4 -5 5 9 7 8 -10 -11 -12 -13 -14 -15\nleaf_value=0.0019461580813300398 0.0031961709610303524 -0.043737274560259012 0.025517058093731445 0.023627624726790988 0.017824876212433179 -0.0093461495620019274 -0.0062947164532622989 0.0077097746983121581 -0.04749759907863265 -0.069601270351814265 -0.035558701886660125 0.021697168229874594 0.030593071369526312 -0.0068479093566785236\nleaf_weight=250.10194249596316 119.98194397462066 9.6902918985579145 43.636379857663997 37.960403983277502 9.356989186722787 6.1177380764856961 20.251627614343306 9.8863320095042599 8.7833608856308256 11.457589325495062 4.0338525664992622 87.823092997336062 9.7059797146357578 370.31175297449045\nleaf_count=19722 1165 120 502 328 55 46 458 615 355 104 36 1178 81 9319\ninternal_value=0.000736785 -0.00124467 0.007395 0.00300334 -0.00244937 -0.0268139 -0.000646822 0.00936182 -0.0193574 -0.0502595 0.0142734 0.0164423 -0.00273561 -0.00330541\ninternal_weight=999.099 770.588 228.511 184.875 146.914 26.9323 760.898 130.778 18.6697 17.5753 112.109 108.075 630.12 620.414\ninternal_count=34084 31884 2200 1698 1370 205 31764 2642 970 150 1672 1636 29122 29041\nis_linear=0\nshrinkage=0.05\n\n\nTree=186\nnum_leaves=15\nnum_cat=0\nsplit_feature=9 4 13 13 3 13 2 10 14 9 6 10 10 10\nsplit_gain=4.72718 4.77628 8.32681 13.0047 6.57536 6.02416 8.70598 7.85594 6.70268 8.84074 11.1374 9.27055 8.6427 5.89596\nthreshold=0.10791723503898144 2.5000000000000004 -0.97342025210221095 -0.67432252158699446 -0.64598341937415527 0.23860957374911687 -1.0825981551352715 0.73598008385744251 0.50753029406901284 0.37935766637759955 16.500000000000004 0.51136950904392775 0.68841813923781148 0.073606811145510848\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 13 4 -4 -3 6 -5 -7 -8 10 -10 -11 -13 -2\nright_child=1 2 3 5 -6 7 8 -9 9 11 -12 12 -14 -15\nleaf_value=-0.0075643353766283372 -0.044633802573862562 -0.010487183655856866 -0.053699582910337489 -0.041990181558305292 0.039748080112762652 0.0044391336387107167 0.045370228968222688 -0.017316733384979539 -0.022626896487042494 0.049917766204752446 0.049022536442376405 -0.042903174836780897 0.033409503939743956 -0.0016007357314683753\nleaf_weight=160.9690840790945 7.1452467064373186 8.0339956129864714 8.8433657357854809 5.121209985094537 26.838370607496472 302.45404751699061 19.150076732323214 47.072668771725148 24.350749472130705 20.050347379408777 5.9171603355998732 6.178365699015556 6.6886077979579568 355.14678216648463\nleaf_count=13037 126 886 1196 723 737 6369 604 499 3413 255 1026 130 56 5037\ninternal_value=0.000260034 0.00176129 0.00502422 0.00315197 0.028205 0.00442247 0.0160797 0.00146811 0.0201522 0.0119213 -0.00750009 0.0294742 -0.00368517 -0.00256997\ninternal_weight=1003.96 842.991 480.699 445.827 34.8724 436.983 87.4565 349.527 82.3353 63.1852 30.2679 32.9173 12.867 362.292\ninternal_count=34094 21057 15894 14271 1623 13075 6207 6868 5484 4880 4439 441 186 5163\nis_linear=0\nshrinkage=0.05\n\n\nTree=187\nnum_leaves=15\nnum_cat=0\nsplit_feature=4 1 0 14 1 3 0 1 2 2 0 5 2 1\nsplit_gain=4.79062 6.74255 9.04955 9.30362 9.8014 8.6629 6.37512 5.91242 5.76857 5.74193 5.64508 5.40448 6.50628 6.42231\nthreshold=1.5000000000000002 8.0035862922668475 0.11122955540524941 0.85443103819729782 5.503817319869996 1.0922634794594657 0.93488923097199472 1.8575797677040102 1.1771025309410685 1.9357224602941756 0.74834841027005417 5.5000000000000009 -1.4103809358165493 2.81056821346283\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 5 -5 7 9 -4 -8 11 -9 12 -2 -13\nright_child=6 -3 3 4 -6 -7 8 10 -10 -11 -12 13 -14 -15\nleaf_value=-0.04411480874302709 0.02375157703348179 0.015858196225687536 0.025754125864656488 0.019423220798284419 -0.039184783560683528 0.017441839137020753 0.0092873621258348447 -0.019587265336624422 0.057236013524863676 0.026517712787922639 0.007188114458457075 -0.01412118319391034 -0.0048446065824352185 0.0087721272049633061\nleaf_weight=16.61711376067251 19.859632079496805 28.294658733066171 10.405230744261642 8.3450690108293184 33.32254091091454 44.653197396663018 21.986464694840837 71.442143960623071 8.0770192000927619 19.552422102366108 26.429429504671134 34.722203197878116 434.58132916406612 217.04711647208205\nleaf_count=243 667 283 154 100 318 803 249 1457 75 249 475 9720 9423 9827\ninternal_value=-0.000538222 -0.00668496 -0.00976537 -0.00662166 -0.0272668 -0.000856039 0.00141584 -0.00849262 0.0235974 0.000465805 -0.0123572 -0.000291955 -0.00354551 0.00557147\ninternal_weight=995.336 239.509 211.215 194.598 41.6676 152.93 755.826 108.277 30.0635 725.763 97.8716 706.21 454.441 251.769\ninternal_count=34043 3833 3550 3307 418 2889 30210 2086 324 29886 1932 29637 10090 19547\nis_linear=0\nshrinkage=0.05\n\n\nTree=188\nnum_leaves=15\nnum_cat=0\nsplit_feature=2 4 10 6 8 1 3 10 8 4 11 1 8 12\nsplit_gain=4.15179 8.05178 4.64613 4.60636 11.8379 10.9853 7.01669 5.47487 4.9322 10.6231 9.105 5.96501 6.17346 4.77996\nthreshold=-1.4103809358165493 4.5000000000000009 0.45857843137254906 11.500000000000002 12.500000000000002 5.7881557941436776 -0.71381031133021522 0.10403138528138529 13.500000000000002 8.5000000000000018 0.22964059561341896 7.1009039878845224 10.500000000000002 0.30604502668814709\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -3 8 -5 7 -7 -6 9 10 -2 -12 -13 -14\nright_child=3 2 -4 4 5 6 -8 -9 -10 -11 11 12 13 -15\nleaf_value=0.050140518489014643 -0.026105109961588126 -0.019860559621243841 0.027048268081561862 0.06562494057700928 0.020517259511932312 -0.044168644441473066 0.032082793374515847 -0.029265932058454864 0.00011695120585476136 -0.070388556161597518 0.03089225848108796 0.024588154488272462 -0.048823343534836999 0.0074499663803790474\nleaf_weight=11.218872930039653 32.784639949240955 12.292808037309443 7.7574677714728741 8.0746205244358844 6.4619509802132624 2.4928523348062326 28.091065399363288 22.187984405997668 818.04542975071217 6.3156225025013546 16.71462884277571 8.4728979934006947 8.4902005586773139 5.8752991035580626\nleaf_count=130 600 1457 90 124 1819 175 726 7474 20738 214 193 63 105 69\ninternal_value=0.000503401 0.0181453 -0.00128874 -8.7052e-05 0.0124296 0.00420908 0.024666 -0.0177241 -0.00103939 -0.0129165 -0.00700743 0.00907561 -0.00751735 -0.0268213\ninternal_weight=995.276 31.2691 20.0503 964.007 67.3085 59.2339 30.5839 28.6499 896.699 78.6533 72.3377 39.553 22.8384 14.3655\ninternal_count=33977 1677 1547 32300 10318 10194 901 9293 21982 1244 1030 430 237 174\nis_linear=0\nshrinkage=0.05\n\n\nTree=189\nnum_leaves=15\nnum_cat=0\nsplit_feature=6 5 9 9 7 9 0 1 11 2 9 1 1 2\nsplit_gain=4.48322 8.43116 6.9765 7.83225 5.44599 4.93017 4.83949 5.34756 4.60244 11.2182 9.43022 7.16445 9.92684 7.53378\nthreshold=2.5000000000000004 9.5000000000000018 0.078907978039233523 0.081853989203032898 9.5000000000000018 0.084789595248866576 0.21875313378574363 5.423553705215455 0.56629291394080161 -0.43634239975746453 0.48753231203628372 6.279320716857911 6.5555982589721689 -0.12823173267677504\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -4 6 -5 7 -1 10 -10 11 -2 13 -13\nright_child=8 -3 3 5 -6 -7 -8 -9 9 -11 -12 12 -14 -15\nleaf_value=-0.048193346616511971 -0.0018595951820998928 -0.039187695634317427 0.048366820364629637 -0.050311724887744318 -0.047547338802678411 -0.00039393286991336752 0.023075567106708585 0.0113519684206314 -0.022992762592832294 0.019668742896307202 -0.020934530729133951 0.054662177582305516 0.0042817648285221072 0.011285351234391198\nleaf_weight=8.0856565127542144 230.14594072122691 14.693331938877234 7.1924195376923299 4.0093107067878 12.933405645639139 395.67133721360733 7.6491486200538921 5.9028379949741057 17.600498290266842 91.877618639497086 42.238806345267221 19.469503603628254 111.04194535007264 20.882498120699893\nleaf_count=161 22483 235 97 92 209 5631 308 65 370 644 588 381 2462 336\ninternal_value=0.000464695 -0.00317072 -0.00188604 -2.95378e-05 -0.0231205 -0.00101899 -0.00706449 -0.0239856 0.00357447 0.0126641 0.00120198 0.00370094 0.0121107 0.0330301\ninternal_weight=989.394 456.137 441.444 406.873 34.571 399.681 21.6376 13.9885 533.257 109.478 423.779 381.54 151.394 40.352\ninternal_count=34062 6798 6563 5820 743 5723 534 226 27264 1014 26250 25662 3179 717\nis_linear=0\nshrinkage=0.05\n\n\nTree=190\nnum_leaves=15\nnum_cat=0\nsplit_feature=10 11 0 4 11 10 10 3 2 12 4 12 12 12\nsplit_gain=4.0468 6.90883 8.00309 5.58204 5.11147 6.72257 4.92326 6.62133 7.80511 5.38145 4.80443 7.56718 7.40625 5.74399\nthreshold=0.12712842712842715 0.16162050241984516 0.39686519946586502 17.500000000000004 0.15784628527155695 0.27608892921960076 0.12952069716775602 -0.47870520148800483 2.1520662742152665 0.17800936357259559 9.5000000000000018 0.15015596373093926 0.086958361481737009 0.15980846598300777\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 3 -3 5 6 -2 8 -8 -9 13 12 -12 -1\nright_child=4 2 -4 -5 -6 -7 7 9 -10 -11 11 -13 -14 -15\nleaf_value=-0.0023127393756315106 0.046959792680521147 -0.059971886048457414 -0.0049276415661450822 0.02314837674398711 4.1454001343806237e-05 0.047099398452196628 -0.039599531371930716 0.0049498290469254629 0.037383433732338364 0.045274087187619633 0.035861269730712374 0.05754872854402613 -0.03601404301787689 -0.046050420929846014\nleaf_weight=105.16041318201133 6.1696256630348207 14.894716332040845 22.154920459573987 1.7802115514859895 725.63827952528845 11.090412833145818 13.725868734690268 56.266880694316569 3.38394594809506 8.7712470359656418 5.5595673664574878 7.3441849524315321 6.9132330783977523 7.0956514686113215\nleaf_count=4421 390 393 716 93 16777 99 2654 4159 20 111 349 391 3297 216\ninternal_value=5.21482e-05 -0.00692753 -0.0251816 -0.0502904 0.00150458 0.012078 0.00720216 0.00368734 -0.0231505 0.0109913 -0.0014119 0.0206775 -0.00369286 -0.00545957\ninternal_weight=995.949 170.903 38.8298 16.6749 825.046 99.408 88.3176 82.1479 17.1098 65.0381 132.073 19.817 12.4728 112.256\ninternal_count=34086 9876 1202 486 24210 7433 7334 6944 2674 4270 8674 4037 3646 4637\nis_linear=0\nshrinkage=0.05\n\n\nend of trees\n\nfeature_importances:\nz_sup=302\nsup_max=288\nsup_top1=271\nz_frag=237\nneural=226\nz_neural=217\nfrag=193\nsup_wsum=192\nsup_mean5=169\nbest_nb_sim=133\nbase_rank=132\nrank_frag=128\nrank_neural=76\nheavy=66\nn_short=44\n\nparameters:\n[boosting: gbdt]\n[objective: lambdarank]\n[metric: ndcg]\n[tree_learner: serial]\n[device_type: cpu]\n[data_sample_strategy: bagging]\n[data: ]\n[valid: ]\n[num_iterations: 3000]\n[learning_rate: 0.05]\n[num_leaves: 15]\n[num_threads: 0]\n[seed: 0]\n[deterministic: 0]\n[force_col_wise: 0]\n[force_row_wise: 0]\n[histogram_pool_size: -1]\n[max_depth: -1]\n[min_data_in_leaf: 50]\n[min_sum_hessian_in_leaf: 0.001]\n[bagging_fraction: 0.8]\n[pos_bagging_fraction: 1]\n[neg_bagging_fraction: 1]\n[bagging_freq: 1]\n[bagging_seed: 7719]\n[bagging_by_query: 0]\n[feature_fraction: 0.8]\n[feature_fraction_bynode: 1]\n[feature_fraction_seed: 2437]\n[extra_trees: 0]\n[extra_seed: 11797]\n[early_stopping_round: 0]\n[early_stopping_min_delta: 0]\n[first_metric_only: 0]\n[max_delta_step: 0]\n[lambda_l1: 0]\n[lambda_l2: 1]\n[linear_lambda: 0]\n[min_gain_to_split: 0]\n[drop_rate: 0.1]\n[max_drop: 50]\n[skip_drop: 0.5]\n[xgboost_dart_mode: 0]\n[uniform_drop: 0]\n[drop_seed: 21238]\n[top_rate: 0.2]\n[other_rate: 0.1]\n[min_data_per_group: 100]\n[max_cat_threshold: 32]\n[cat_l2: 10]\n[cat_smooth: 10]\n[max_cat_to_onehot: 4]\n[top_k: 20]\n[monotone_constraints: ]\n[monotone_constraints_method: basic]\n[monotone_penalty: 0]\n[feature_contri: ]\n[forcedsplits_filename: ]\n[refit_decay_rate: 0.9]\n[cegb_tradeoff: 1]\n[cegb_penalty_split: 0]\n[cegb_penalty_feature_lazy: ]\n[cegb_penalty_feature_coupled: ]\n[path_smooth: 0]\n[interaction_constraints: ]\n[verbosity: -1]\n[saved_feature_importance_type: 0]\n[use_quantized_grad: 0]\n[num_grad_quant_bins: 4]\n[quant_train_renew_leaf: 0]\n[stochastic_rounding: 1]\n[linear_tree: 0]\n[max_bin: 255]\n[max_bin_by_feature: ]\n[min_data_in_bin: 3]\n[bin_construct_sample_cnt: 200000]\n[data_random_seed: 38]\n[is_enable_sparse: 1]\n[enable_bundle: 1]\n[use_missing: 1]\n[zero_as_missing: 0]\n[feature_pre_filter: 1]\n[pre_partition: 0]\n[two_round: 0]\n[header: 0]\n[label_column: ]\n[weight_column: ]\n[group_column: ]\n[ignore_column: ]\n[categorical_feature: ]\n[forcedbins_filename: ]\n[precise_float_parser: 0]\n[parser_config_file: ]\n[objective_seed: 8855]\n[num_class: 1]\n[is_unbalance: 0]\n[scale_pos_weight: 1]\n[sigmoid: 1]\n[boost_from_average: 1]\n[reg_sqrt: 0]\n[alpha: 0.9]\n[fair_c: 1]\n[poisson_max_delta_step: 0.7]\n[tweedie_variance_power: 1.5]\n[lambdarank_truncation_level: 30]\n[lambdarank_norm: 1]\n[label_gain: ]\n[lambdarank_position_bias_regularization: 0]\n[eval_at: 1,3,10]\n[multi_error_top_k: 1]\n[auc_mu_weights: ]\n[num_machines: 1]\n[local_listen_port: 12400]\n[time_out: 120]\n[machine_list_filename: ]\n[machines: ]\n[gpu_platform_id: -1]\n[gpu_device_id: -1]\n[gpu_device_id_list: ]\n[gpu_use_dp: 0]\n[num_gpu: 1]\n\nend of parameters\n\npandas_categorical:[]\n', 'rerank/meta.json': '{\n  "numeric": [\n    "frag",\n    "neural",\n    "z_frag",\n    "z_neural",\n    "rank_frag",\n    "rank_neural",\n    "base_rank",\n    "n_short",\n    "heavy",\n    "sup_max",\n    "sup_top1",\n    "sup_mean5",\n    "sup_wsum",\n    "z_sup",\n    "best_nb_sim"\n  ],\n  "params": {\n    "objective": "lambdarank",\n    "metric": "ndcg",\n    "eval_at": [\n      1,\n      3,\n      10\n    ],\n    "learning_rate": 0.05,\n    "num_leaves": 15,\n    "min_data_in_leaf": 50,\n    "feature_fraction": 0.8,\n    "bagging_fraction": 0.8,\n    "bagging_freq": 1,\n    "lambda_l2": 1.0,\n    "verbose": -1,\n    "seed": 0\n  },\n  "best_iteration": 191\n}'}
C3_SHA256 = {'library.py': '6666c6028d1d5d41ae825c157009c4ecb0c3d14f847ef5c152ab7cfbfb2a060b', 'transforms.py': 'e8b786bc027ff53c1ab42bc616d0f586910a7456e7ef075f1ed2602fdbbe3a6f', 'neural.py': '30f7d9805c3e310b64000be2f59bc8edd9fee50ff5d2c736b3d0a6605aa4ff9e', 'candidates.py': '994588e8187e8609d773834546460c34818fefb9e3976be2437c5c997ef159fb', 'stage.py': '76927a688516e1b0dc7ae0abb34ae97d888776502d9a322a4b1dbc02dca70c33', 'ranker/ranker.txt': '77b57061932ec1d118d01f2a1f68fb7d3bd1769be19948506486a9d98471e80e', 'ranker/meta.json': 'fd51e2d6fca5d2dd8c2917991449c4ef66e06dae8d8f21904356a6cf6fa6a66f', 'features.py': 'b2b223c34ad1b9eff9850ca31ab8d12b064d231f5ff66bf1d79b7d56845ba564', 'rerank/ranker.txt': 'aa45815f393ec99a3d015f29527cde939d30dbbe3a2cfcfdc5825c4160f439ca', 'rerank/meta.json': '78e304438ad860dadbc11138947f12a2e2ccc51799f85ced6cf0b585a0ce88ce'}
C3_DIR = CODE / 'class3'
for _name, _text in C3_FILES.items():
    (C3_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
    assert __import__('hashlib').sha256(_text.encode()).hexdigest() == C3_SHA256[_name], _name
    (C3_DIR / _name).write_text(_text)
for _p in (str(C3_DIR), str(ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

CLASS3_REPORT = dict(status='disabled')
if globals().get('CLASS3', False):
    _sub = ROOT / 'submission.csv'
    _backup = ROOT / 'submission_before_class3.csv'
    shutil.copyfile(_sub, _backup)
    try:
        if (time.time() - NB_T0) / 3600 > TIME_GUARD_HOURS:
            CLASS3_REPORT = dict(status='skipped_time_guard')
        else:
            _bundle = sorted(glob.glob(str(INPUT / '**/casmi_spectrum_model_v1/config.json'), recursive=True))
            assert _bundle, 'casmi_spectrum_model_v1 bundle not attached'
            import stage as _c3
            CLASS3_REPORT = _c3.run(ROOT, COMP, Path(_bundle[0]).parent, globals().get('LIB_MAX') or {},
                                    CLASS3_CFG, ranker_dir=C3_DIR / 'ranker')
    except Exception as exc:
        traceback.print_exc()
        shutil.copyfile(_backup, _sub)
        CLASS3_REPORT = dict(status='restored_after_error', exception=repr(exc)[:600])
(ROOT / 'class3_report.json').write_text(json.dumps(CLASS3_REPORT, indent=2, default=str))
_final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert _final.molecule_id.tolist() == SAMPLE_IDS
print('CLASS3', json.dumps({k: v for k, v in CLASS3_REPORT.items() if k != 'cfg'}, default=str))


## 4. What this run did

The table counts, layer by layer, how many answer lists changed against the layer before. In the commit run the
pipeline ranks only the first 12 visible molecules (the rest are placeholders), and the visible molecules are copies
of training spectra, so every leader has a library match of 1.0 and the fragment check's rank-1 rule cannot fire;
only its swaps can. The neural fusion can still move a leader (the V1 row). The last row is the merge: the 0.75 order with the fragment check's leader moves from the V2 order; with no leader move in the commit run it equals the 0.75 order. The hidden rerun that produced the
score runs every molecule and is not shown to anyone.

In [ ]:
# What each layer did in this run, read from the files the stages wrote.
if SHOW:
    rep = {}
    for name in ('spectral_parallax_report.json', 'provenance_tournament_report.json', 'merge_report.json'):
        p = ROOT / name
        if p.is_file():
            try:
                rep[name] = json.loads(p.read_text())
            except Exception as exc:  # noqa: BLE001 - a report is only shown, never used
                rep[name] = {'unreadable': repr(exc)}
    base = pd.read_csv(BASELINE, dtype=str, keep_default_na=False) if BASELINE.is_file() else None
    final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    layers = [('0.465 pipeline', BASELINE), ('V1, neural 0.25', VARIANTS_DIR / 'submission_V1.csv'),
              ('V2, ranks 2-25 at 0.50', VARIANTS_DIR / 'submission_V2.csv'),
              (f'{VARIANT}, ranks 2-25 at {CUSTOM["w_mid"]:.2f}', VARIANTS_DIR / f'submission_{VARIANT}.csv')]
    if globals().get('FRAG_ON_V2'):
        layers += [('submitted: CUSTOM + the fragment check\'s leader moves on the V2 order', ROOT / 'submission.csv')]
    elif PROVENANCE:
        layers += [('after the fragment check', ROOT / 'submission_physical0472_control.csv'),
                   ('after the provenance reranker (submitted)', ROOT / 'submission.csv')]
    else:
        layers += [('after the fragment check (submitted)' if PHYSICAL else 'submitted', ROOT / 'submission.csv')]
    prev, out = None, []
    for label, path in layers:
        if not path.is_file():
            continue
        f = pd.read_csv(path, dtype=str, keep_default_na=False).set_index('molecule_id').smiles
        if prev is not None:
            common = f.index.intersection(prev.index)
            moved = int((f[common] != prev[common]).sum())
            top1 = int(sum(a.split(';')[0] != b.split(';')[0] for a, b in zip(f[common], prev[common])))
            out.append({'layer': label, 'lists re-ordered vs previous': moved, 'rank 1 changed': top1})
        else:
            out.append({'layer': label, 'lists re-ordered vs previous': '-', 'rank 1 changed': '-'})
        prev = f
    table_html(pd.DataFrame(out).set_index('layer', drop=False), numeric=('lists re-ordered vs previous', 'rank 1 changed'))
    print('Commit run: the counts cover the 12 visible smoke molecules; the hidden scoring rerun is not shown to anyone.')
    for k, v in rep.items():
        print(k, json.dumps(v)[:600])

## 5. Every submission of this pipeline

Each row is a kernel on our account (busyaprime/33-casmi26-px-e1, e2, e3, e4, e5, e6, m1, m2, m4, m6, m5, m7, n2) running this pipeline on the hidden
test, submitted between 2026-10-27 and 2026-10-29 (our four earlier submissions, 2026-09-21, ran a different pipeline).
Besides the switches, e1-e6 carry one patch this version does not: their fragment check reads the 0.75 file
(variants/submission_CUSTOM.csv) instead of the V2 file, so it runs on the 0.75 order. This notebook is a new
version of m1, so the score on this page is m1's. e3-e5 point the neural bundle pin at Ahmed Berat Özer's long-trained `final_model.pt`; e6 adds his NX isomer reranker; m1 and every m, t and n run after it keep the fragment check on the V2 order, and each changes one setting of its leader rule (or adds NX) against m1.

| run | ranks 2-25 weight | fragment check | provenance reranker | neural model | other | public LB | submission |
|---|---:|---|---|---|---|---:|---|
| e1 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 |  | 0.472 | 57713785 |
| e2 | 0.75 | on, 0.75 order | on | Saner Turhaner v1 |  | 0.472 | 57713786 |
| e3 | 0.75 | on, 0.75 order | off | long (Ahmed Berat Özer) |  | 0.453 | 57714284 |
| e4 | 0.75 | off | off | long (Ahmed Berat Özer) |  | 0.461 | 57715407 |
| e5 | 0.75 | on, 0.75 order | on | long (Ahmed Berat Özer) |  | 0.453 | 57715244 |
| e6 | 0.75 | on, 0.75 order | off | Saner Turhaner v1 | NX isomer reranker, weight 2 | 0.414 | 57720847 |
| m1 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only | **0.477** | 57722897 |
| m2 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; library gate 0.9 | 0.477 | 57731403 |
| m4 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; from 2 spectra | 0.477 | 57742127 |
| m6 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; old-config margin 0.015 | 0.477 | 57752007 |
| m5 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; vote step 0.015 | 0.473 | 57752070 |
| m7 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; new-config margin 0.08 | 0.473 | 57752139 |
| n2 | 0.75 | on, V2 order | off | Saner Turhaner v1 | leader moves only; NX isomer reranker on the V2 order | 0.469 | 57752191 |

How to read a difference: the leaderboard page says the public board uses about 31% of the test data, and
participants estimate the public split at about 125-130 molecules ([Roman Rozen](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures)), so one molecule
moved from rank 2 to rank 1 is worth about 0.004 and the board prints three decimals. This notebook's 0.477 is +0.005 against 0.472, the score Spectral Parallax shows and the score a fork running alex chilton's 0.75 weights shows (his own page shows 0.467): the size of about one net leader move from rank 2 to rank 1. The hidden lists are not shown, so how many leaders actually moved is not known.

A full scoring run takes hours. The only public timings are for the earlier E007 pipeline (0.440) that our base grew
from: about 2 h 38 min and, for a later version, 2 h 44 min, each a full run over the 400 visible molecules
([Kino Lutor](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/749251)).

## 6. Run it yourself

Join the competition, then Copy & Edit this notebook (the competition data and the datasets in the credits stay
attached). Keep the accelerator on GPU T4 and internet off. Run All in the editor is a smoke test: it ranks 12
visible molecules and gives no score. For a score, Save Version (our commit runs took 21-30 minutes on a T4), then
submit that version; the scoring rerun on the hidden test takes hours.

| switch | this notebook | what it does |
|---|---|---|
| `VARIANT`, `CUSTOM` | `CUSTOM`, w_top 0.25, w_mid 0.75 | the leader-safe weights; `V2` is the 0.25 / 0.50 recipe |
| `PHYSICAL` | `True` | Roman Tamrazov's fragment check |
| `PROVENANCE` | `False` | his provenance-aware reranker |
| `FRAG_ON_V2` | `True` | keep the fragment check's leader moves from the V2 order on top of the 0.75 order; False leaves the check's own V2 output |
| `COMMIT_MODE` | `'smoke'` | the commit run ranks 12 visible molecules; the scoring rerun ranks all |

Our string patches to his code, each asserted by our builder to apply exactly once:

1. Configuration cell: `VARIANT = 'CUSTOM'`, the CUSTOM weights, and the `PHYSICAL`, `PROVENANCE` and `FRAG_ON_V2`
   switches.
2. Fragment-check source (hidden): unchanged. It reads the V2 file, as upstream does.
3. Run cell (`BUNDLE = materialise_sources()`): it accepts the configured variant instead of only V2, and in a fast
   commit it writes the sample file and stops.
4. Fragment-check cell and provenance cell: an `if not PHYSICAL` and an `if not PROVENANCE` gate.
5. Our merge cell, after the provenance cell: per molecule, if the fragment check moved the V2 leader, the answer is
   [its new leader, the V2 leader] followed by the 0.75 order; otherwise the 0.75 order unchanged. It applies only
   when V2 and the 0.75 order share the leader and the fragment check's output holds the same candidates as the 0.75
   order; a list where the check also put an outside candidate at rank 25 keeps the 0.75 order without its leader
   move. Any error restores the 0.75 order.

## 7. What we concluded

1. The fragment check's leader moves from the V2 order on top of the 0.75 order (m1): 0.477, against 0.472 for the 0.75 order alone (its forks) and 0.472 for both changes stacked the plain way (e1 0.472).
2. The provenance reranker on top of the plain stack: e1 0.472 -> e2 0.472 (+0.000, no change at three decimals).
3. The fragment check run on the long-trained model's order: e4 0.461 -> e3 0.453 (-0.008, a loss at three decimals). This fits a check that depends on the order it is handed, but the pair switches all three parts of the check and does not single out the rank-1 rule.
4. Ahmed Berat Özer's long-trained neural model instead of Saner Turhaner's: e1 0.472 -> e3 0.453 (-0.019, a loss at three decimals). Its offline diagnostic (MRR 0.484 to 0.718 on sanpier's random holdout) did not carry to the hidden test.
5. Ahmed Berat Özer's NX isomer reranker at weight 2 on the plain stack: e1 0.472 -> e6 0.414 (-0.058, a loss at three decimals).
6. Opening the leader rule up to library match 0.90: m1 0.477 -> m2 0.477 (+0.000, no change at three decimals).
7. Opening the leader rule to molecules with two spectra: m1 0.477 -> m4 0.477 (+0.000, no change at three decimals).
8. The leader rule's old-config margin .028 -> .015: m1 0.477 -> m6 0.477 (+0.000, no change at three decimals).
9. The leader rule's vote step .025 -> .015: m1 0.477 -> m5 0.473 (-0.004, a loss at three decimals).
10. The leader rule's new-config margin .095 -> .08: m1 0.477 -> m7 0.473 (-0.004, a loss at three decimals).
11. Ahmed Berat Özer's NX isomer reranker on the V2 order, its leader moves merged: m1 0.477 -> n2 0.469 (-0.008, a loss at three decimals).
12. Rank 1 is where this lineage gains and loses. The public runs that let the neural model move the leader more freely scored lower: rank-1 weight 0.50 shows 0.448 and 1.00 shows 0.359 on the 0.472 base, and a uniform weight of 0.35 at offset 3 shows 0.447 on two accounts.

## 8. Credits

The answer lists come from other people's code, run with the patches listed in section 6:

- **Roman Tamrazov**, [Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax), version 22: the notebook this one runs, including the
  fragment check and the provenance reranker. His notebook thanks Kozykorpesh's
  [LB [0.468] balanced acquisition evidence](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) first, for the reproducible V2 consensus pipeline, the
  neural spectrum-structure ranking, the candidate-evidence exports and the evidence-guided candidate recovery, and
  credits Heliosli's [training-structure protection](https://www.kaggle.com/code/heliosli/happy-to-make-the-dataset-public-if-it-helps) for the provenance idea.
- **Roman Rozen**, [From Peaks to Structures](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures): the leader-safe consensus (`consensus.py`) and the
  outside-candidate collection (`outsiders.py`).
- **alex chilton**, [Leader-Safe, Ranks 2-25 at 0.75](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-472-leader-safe-ranks-2-25-at-0-75): the 0.75 weight on ranks 2-25.
- **Saner Turhaner**, [PubChem + Neural025](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466): the spectrum-candidate model, the reciprocal-rank
  fusion and the six helper modules.
- **Dmitrii Gluzdov**, [The Data Behind Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures): the data notebook that assembles
  and documents the shared 0.465 pipeline, and the datasets that carry it.
- **Huseyin Emre Aksoy** ([profile](https://www.kaggle.com/huseyinemreaksoy)): `frag_rescore.py`, the untrained
  MetFrag-lite fragment scorer that both the fragment check and the pipeline's fragment-gap score run (Apache-2.0,
  reached through Ahmed Berat Özer's v7c notebook).
- **Ahmed Berat Özer**, [the long-schedule spectrum model](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-spectrum-model-long): Saner Turhaner's model trained 40 x 2,000 steps instead of 16 x 500, same code; runs e3-e5 use it.

The embedded 0.465 pipeline keeps its own full credits and licence table (its section 6), unchanged. Its main
components: canqiang's E006 (the complete inference base); Ahmed Berat Özer's main engine, rankers, fingerprint
weights, candidate pool and the Kaggle packages of ICEBERG and GLACIER (the ms-pred models of the Coley group);
prvsiyan, megayak and seyitkaangunes (second engine, analogue propagation, engine fusion); flexonafft (the N1
challenger); hengck23's DreaMS resource; and the further authors and public spectral libraries listed there.

| dataset | author | used for | licence |
|---|---|---|---|
| [dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) | Dmitrii Gluzdov (a collection of other authors' public files, each folder with its own notices) | models, candidate sets and engine code of the pipeline | CC BY-NC-SA 4.0; each folder keeps its source licence (`ATTRIBUTION.md`) |
| [dmitriigluzdov/casmi26-pubchem-popularity-prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Dmitrii Gluzdov | PubChem popularity prior | Other: PubChem courtesy of the U.S. NLM; LOTUS CC0; NPAtlas CC BY-NC 4.0 |
| [dmitriigluzdov/casmi26-natural-product-panel-benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Dmitrii Gluzdov | natural-product panel | Other: competition data terms |
| [dmitriigluzdov/casmi26-fold-safe-fpnet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | Dmitrii Gluzdov | fold-safe FPNet checkpoints | Other: checkpoint under competition data terms; scripts Apache 2.0 |
| [dmitriigluzdov/casmi26-natural-product-knowledge-table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) | Dmitrii Gluzdov | natural-product knowledge table | Other: LOTUS CC BY 4.0, NPAtlas CC BY-NC 4.0, PubChem counts |
| [sanpier/casmi-spectrum-model-v1](https://www.kaggle.com/datasets/sanpier/casmi-spectrum-model-v1) | Saner Turhaner | the spectrum-candidate model | trained on competition data (CC BY-NC 4.0) |
| [sanpier/catalogue-experiment-saner](https://www.kaggle.com/datasets/sanpier/catalogue-experiment-saner) | Saner Turhaner | attached as in Spectral Parallax; this run does not read it | as on its page |

The competition data is CC BY-NC 4.0, and the models above are trained on it: use them non-commercially, with
attribution. We wrote the field record, the figure and the audit cells.